# Two-stage vector sum

A GPU reduction can't be one loop: thousands of programs each reduce a block, and the partial results then have to be combined. Write the classic **two-stage** version.

Write `vector_sum(x)` that returns the sum of all elements of `x`.

## Inputs
- `x`: contiguous `float32` CUDA tensor, any shape, any number of elements (including 0).

## Output
- A 0-d `float32` CUDA tensor holding the sum. An empty input sums to `0.0`.

## Requirements
1. **Stage 1:** a kernel with `triton.cdiv(n, BLOCK)` programs. Program `pid` sums its block (masked lanes contribute 0) with `tl.sum` and writes one value to `partials[pid]`.
2. **Stage 2:** a kernel launched with **one** program that loops over `partials` in chunks and writes the final value.
3. No `torch.sum` on the host and no atomics, so the result is the same on every run.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a PyTorch reference that sums in float64; your float32 tree sum is graded with a small tolerance.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def vector_sum(x):
    """x: float32 CUDA tensor. Return a 0-d float32 tensor with the sum, using two Triton stages."""
    raise NotImplementedError


@triton.jit
def _partial_sum_kernel(x_ptr, partial_ptr, n, BLOCK: tl.constexpr):
    pass


@triton.jit
def _final_sum_kernel(partial_ptr, out_ptr, n_partials, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0, 4.0], "dtype": "float32"}], "e": {"__nd__": 10.0, "dtype": "float32"}}, {"a": [{"__nd__": [[0.5, -0.5], [10.0, -2.5]], "dtype": "float32"}], "e": {"__nd__": 7.5, "dtype": "float32"}}, {"a": [{"__nd__": [42.0], "dtype": "float32"}], "e": {"__nd__": 42.0, "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}], "e": {"__nd__": 0.0, "dtype": "float32"}}, {"a": [{"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float32"}], "e": {"__nd__": 5000.0, "dtype": "float32"}}, {"a": [{"__nd__": [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, -1000.0, 0.25], "dtype": "float32"}], "e": {"__nd__": 0.25, "dtype": "float32"}}, {"a": [{"__nd__": [-0.8019314408302307, -1.3243589401245117, -0.24836161732673645, 0.4204452335834503, 1.1360465288162231, 0.10970640182495117, -0.5526472926139832, -0.7847803831100464, 0.7487457990646362, 1.6347830295562744, 0.2727687656879425, -1.2333287000656128, -0.9582651853561401, 1.600019097328186, 0.20288243889808655, -1.7321348190307617, -0.08369619399309158, -1.1632260084152222, -0.6292880773544312, -0.4880058169364929, -0.7133134007453918, 0.5533784627914429, -0.06308597326278687, -0.5894312858581543, 0.40963783860206604, 0.8298553228378296, -1.6430233716964722, -0.2567301392555237, -0.9807473421096802, -0.17315521836280823, -1.2894186973571777, 0.020690394565463066, -0.03788574039936066, -0.304337739944458, -1.0479265451431274, -0.396190345287323, -1.0913288593292236, -1.3552087545394897, 0.2247857302427292, -1.1093499660491943, 1.1702960729599, 0.7165876626968384, -1.9978166818618774, 0.2721288800239563, -1.1017166376113892, 0.03305722028017044, 0.04363199323415756, -1.9884297847747803, -0.23342251777648926, -0.25579002499580383, 0.9620005488395691, -1.1814467906951904, 0.738041877746582, -1.0989727973937988, -0.3312909007072449, -0.8404731750488281, 1.4487313032150269, 0.5682131052017212, 2.431732416152954, 0.6419163942337036, 0.8449927568435669, 0.840682864189148, -0.6066115498542786, -0.07002844661474228, 1.3503888845443726, -0.3965507745742798, 0.18879953026771545, -0.021223461255431175, 0.6092165112495422, -0.36490875482559204, -0.1523618847131729, 0.24238142371177673, 0.10302314162254333, -0.8649727702140808, 0.8957830667495728, -1.2984812259674072, -1.201115608215332, -1.2824918031692505, 0.966972291469574, -0.36060836911201477, -0.9710363745689392, -1.136021375656128, 0.4211311340332031, -1.0548406839370728, -1.2720781564712524, 0.6139930486679077, -1.1967077255249023, -0.32243815064430237, -0.006761540193110704, -0.44533535838127136, -0.05409396067261696, 1.338773488998413, -0.5168941617012024, -1.2593071460723877, -1.8367457389831543, -0.20476610958576202, -0.35225701332092285, 0.2650909125804901, -0.4642446041107178, -0.4786384701728821, -0.7213157415390015, -0.5197597742080688, 0.1602267026901245, -0.3803527355194092, 0.10044185072183609, 1.9011871814727783, 0.47911813855171204, -1.576089859008789, 1.7335295677185059, 0.3478103578090668, -0.9414132833480835, 0.9070489406585693, 0.017655611038208008, -0.6152185201644897, -0.6335089206695557, -0.993431806564331, 0.04811898246407509, 1.0688166618347168, -0.3250521123409271, 0.4208241105079651, 1.8756459951400757, -1.2145906686782837, 0.257522314786911, -0.3064369261264801, -1.0592560768127441, -1.0258471965789795, -0.015285143628716469, 0.43386340141296387, -0.5315898060798645, 0.054390545934438705, 0.7222086787223816, 0.29656124114990234, 0.8686701655387878, 0.46139562129974365, 0.488005667924881, 1.8266340494155884, 0.623171865940094, 0.12947113811969757, 0.7099438309669495, -0.919123649597168, -0.33770477771759033, 0.7932208776473999, 0.6307867169380188, 1.5483616590499878, 0.010473028756678104, -1.4623278379440308, 1.9472469091415405, 1.0928927659988403, -1.0587373971939087, 1.3758236169815063, 0.032402798533439636, -1.8139287233352661, -0.4203539192676544, -0.5089038610458374, 1.5908482074737549, -0.7920687198638916, -0.2536173462867737, -0.2767608165740967, -0.37897276878356934, -0.9140740633010864, 0.2190869152545929, 1.0768046379089355, 0.6239776611328125, -0.9274991154670715, -1.1498157978057861, 0.11897355318069458, -0.7065801620483398, -0.6301656365394592, -1.6796945333480835, 1.9504915475845337, 0.9166191816329956, -0.9739073514938354, 0.908210277557373, 1.342487096786499, -2.3895533084869385, -0.5489447712898254, -0.38797321915626526, 0.6482544541358948, -0.12146542221307755, -0.23043133318424225, -0.05837138742208481, 1.8533257246017456, 2.159980535507202, -0.5248264074325562, -0.9262440204620361, 2.6925530433654785, -0.9797446131706238, -0.5734020471572876, 0.036581508815288544, 0.48324623703956604, 1.0290355682373047, 0.39097723364830017, -0.8723830580711365, 0.5063823461532593, 0.24991758167743683, 1.8769190311431885, -0.014999798499047756, -1.336972951889038, -1.0450198650360107, 1.450153112411499, -0.5401312112808228, -2.104465961456299, -0.5806996822357178, 1.5099831216502935e-05, 1.1888307332992554, -1.0144681930541992, 0.6666833162307739, 0.7952991127967834, -0.699388325214386, -0.18758970499038696, 1.7694501876831055, 1.720484733581543, 0.8555219769477844, 0.33194637298583984, 1.1383095979690552, -0.1406772881746292, -0.09509306401014328, -0.8603601455688477, 0.0055633848533034325, -0.08201662451028824, 2.7736315727233887, -0.19284316897392273, 1.2708179950714111, 1.3204972743988037, -0.18647870421409607, 1.169397234916687, -2.1768136024475098, 0.09492718428373337, 0.8572614192962646, -2.3978652954101562, -1.159171223640442, 1.0560736656188965, -0.2570107579231262, -1.098198652267456, -0.373595654964447, -0.5425882935523987, 0.7244473099708557, 0.45388588309288025, -0.2802283465862274, -0.6653426885604858, -0.05387566238641739, 1.3251559734344482, 0.3490261137485504, 0.6389455795288086, -0.15068159997463226, -1.291816234588623, -0.710843563079834, 0.6512365937232971, 0.10321610420942307, -0.9038398861885071, -1.6178704500198364, 0.11068189889192581, -0.3395310044288635, -0.350530207157135, -0.32944995164871216, 0.28390511870384216, 0.29444772005081177, 0.9621285796165466, 0.9284215569496155, 1.3320931196212769, 0.806363046169281, -0.33560803532600403, -0.05426894500851631, 0.49421510100364685, -0.3102922737598419, 0.5494387745857239, 1.1536095142364502, -0.8012459874153137, -1.7703840732574463, 1.193655014038086, 0.017759909853339195, 0.7086698412895203, -1.3323053121566772, 1.7329925298690796, 0.738619863986969, -0.1260741800069809, -0.3519880473613739, 0.11389587819576263, -0.1667313426733017, -0.2786901295185089, -1.1195999383926392, 1.2628093957901, -0.03957768529653549, 0.07463563978672028, 0.09291598945856094, -2.358217239379883, 0.5271414518356323, -0.32915493845939636, -0.2892284691333771, 0.0697878748178482, -0.9724755883216858, 0.8433808088302612, -0.6958646178245544, -0.9036580324172974, -0.8621895909309387, 0.1587526947259903, 0.7457794547080994, -0.5950532555580139, -1.5101866722106934, 1.062490701675415, -1.065083622932434, 1.1948304176330566, -0.05429587513208389, -0.6983498930931091, -0.6400399208068848, 0.461866170167923, 0.8893718123435974, -0.1981702297925949, 0.9469344019889832, 2.0869781970977783, -0.20830833911895752, -1.1494466066360474, -1.0544003248214722, -1.3903353214263916, -0.639665424823761, -0.7237792611122131, -1.3251032829284668, 0.530184268951416, -2.6113390922546387, -1.3207989931106567, 2.015251874923706, 1.0378024578094482, 0.5227916836738586, 0.45725905895233154, 1.770937204360962, 0.5356296300888062, 0.07757708430290222, -1.230160117149353, -0.07366681843996048, -0.7338839769363403, -0.7061752676963806, -0.7142389416694641, -0.044646862894296646, -0.32971081137657166, 0.8711104989051819, -1.326181411743164, -0.1694701910018921, 1.8594975471496582, 0.03889733552932739, 1.1986151933670044, 1.1368634700775146, 1.3166477680206299, 0.7381191849708557, -1.3804460763931274, -0.267387717962265, -0.38033556938171387, -1.5488553047180176, -0.46388161182403564, -1.250079870223999, -1.6655514240264893, 0.21089625358581543, -0.043591272085905075, -0.6878148913383484, -0.9505863785743713, -0.5412734150886536, -0.3940903842449188, -0.40811029076576233, -0.052707746624946594, -0.43554067611694336, 0.38697198033332825, 0.5228708982467651, 0.3578696846961975, -0.6631035208702087, -0.6165061593055725, 0.4348641037940979, -0.197042778134346, -0.04315171018242836, -2.491995334625244, -2.700103998184204, 0.33714818954467773, 0.32955867052078247, 1.3934319019317627, 0.7211630940437317, 0.221738800406456, 0.01636245660483837, -1.4267404079437256, 0.8831759095191956, -0.6035860180854797, 0.5024844408035278, 0.20004165172576904, 0.27209702134132385, 0.8926752209663391, -0.48986610770225525, -0.38723519444465637, -0.855861485004425, -0.3771520256996155, 1.665940761566162, -0.36331865191459656, 1.1856162548065186, 0.8260114192962646, 0.7292817831039429, 0.8030151128768921, -1.0142765045166016, 0.3734476864337921, 1.5884120464324951, -0.6957959532737732, -0.04710708558559418, -0.6891591548919678, 0.3170115053653717, -1.0785300731658936, 1.6096798181533813, -0.37683477997779846, -1.5426615476608276, 1.7907909154891968, 0.9705228805541992, 0.5185114741325378, 0.5325875282287598, -0.3577980101108551, -0.22329559922218323, 0.7240192294120789, 0.9176249504089355, 0.932612419128418, 0.21255098283290863, -0.46638810634613037, -0.192087322473526, 1.1095997095108032, 1.4399181604385376, -0.34131237864494324, -0.47730401158332825, 1.2475496530532837, 0.7236490249633789, 0.6972581744194031, -1.3123337030410767, -0.3737574517726898, -0.5149955153465271, 0.207904651761055, -1.6849420070648193, 1.2717528343200684, 0.30013179779052734, 0.4348370134830475, -1.1841403245925903, -0.02642037160694599, 1.3346502780914307, -1.5696955919265747, 0.30576804280281067, 0.6988049149513245, -0.6919471025466919, 0.8147412538528442, -0.33006903529167175, 1.229766607284546, 0.5275117754936218, 0.22436589002609253, 2.008915901184082, 0.8318964838981628, 1.3086357116699219, -1.1895196437835693, -0.6466070413589478, 0.8365690112113953, 0.9056118726730347, -0.055740028619766235, 0.779726505279541, -1.0583537817001343, 0.2138352245092392, -1.147283911705017, 0.3405705392360687, -0.4861719012260437, -0.6409975290298462, 0.056299030780792236, 1.817460060119629, -1.0117617845535278, -0.6017506122589111, 1.0676641464233398, -0.42356666922569275, 1.7692642211914062, 0.22252970933914185, 0.09063191711902618, 0.3568841516971588, 1.7411795854568481, 0.655361533164978, 0.5167121291160583, 0.2518577575683594, -1.562418818473816, -0.8852720260620117, 1.4833440780639648, 0.8218783140182495, 0.5060648322105408, 0.6370627284049988, -0.5435930490493774, 0.33796426653862, -0.9007946252822876, 0.3913623094558716, -0.7141351103782654, -0.21381983160972595, 0.4534148573875427, -0.32678771018981934, 0.647581934928894, 0.4560382664203644, 0.35945814847946167, 1.324411392211914, -1.4738847017288208, -1.5962822437286377, 0.6162418127059937, 0.9407941102981567, 2.0116403102874756, 0.7562568187713623, 0.8302149772644043, -0.8830800652503967, 0.7526217699050903, 0.6738389134407043, -1.0558651685714722, -0.8963174819946289, -0.11103120446205139, 0.2882537543773651, -2.9447855949401855, 0.07030551135540009, 0.23085449635982513, -0.17683635652065277, 0.2668249011039734, 0.4559062421321869, -1.0116380453109741, 0.41618016362190247, -0.2513071894645691, -0.945878803730011, 0.20914079248905182, 1.4876441955566406, -1.4601155519485474, -1.6531099081039429, -1.6531661748886108, 1.9773117303848267, -0.2515837550163269, -0.0594867542386055, -0.7865757346153259, -0.9404478073120117, -2.026860237121582, 0.1704062968492508, 0.2595598101615906, -1.9765340089797974, -1.0431492328643799, -0.7441137433052063, -0.02600245364010334, 0.24412916600704193, 1.1257929801940918, 1.364456057548523, 1.0684545040130615, 1.3415285348892212, -1.1664166450500488, -0.5544917583465576, 1.1436097621917725, 0.28866109251976013, 0.018721718341112137, -1.9789881706237793, -1.8592429161071777, 0.7308894395828247, 0.34157639741897583, -0.2960576117038727, -0.41297173500061035, -1.726102590560913, 0.3712151348590851, 1.0344208478927612, 1.642171025276184, 0.04913794994354248, -0.20419667661190033, -1.0959970951080322, 1.2232437133789062, 1.7068555355072021, -0.6131700277328491, 0.29075875878334045, 0.09067010879516602, -0.57382732629776, -2.4974496364593506, -0.36599910259246826, 0.607444167137146, -1.443302035331726, -0.838881254196167, -3.2835605144500732, 1.2845276594161987, 0.5002763867378235, 0.04547853022813797, -0.2704733908176422, -0.637283205986023, -0.07880625128746033, -0.31031492352485657, -0.24835671484470367, -0.033811114728450775, 0.12565097212791443, 1.5489784479141235, -1.449432373046875, 0.26196861267089844, -0.6245608329772949, -1.1394811868667603, -2.368170738220215, 0.6068921685218811, -0.9903792142868042, -0.7596859931945801, 0.45987868309020996, -1.093472957611084, 0.016045130789279938, -0.08488040417432785, 0.7918434143066406, 1.0006409883499146, -0.8768518567085266, 0.907177746295929, -0.1067456603050232, 0.1127975657582283, -0.2225136160850525, 0.7655396461486816, -1.2222003936767578, 0.2500842809677124, 1.1657798290252686, -0.2936699092388153, 2.0177724361419678, -0.745905876159668, 0.2036961317062378, -0.49260851740837097, -0.6287503242492676, -1.1257710456848145, -0.5500664710998535, -0.06223707273602486, -0.8130971789360046, 0.8723957538604736, 1.510326623916626, -1.5388962030410767, 0.8248433470726013, 0.17806008458137512, 0.2333207130432129, 2.4504992961883545, 1.1943246126174927, 0.3243098556995392, 0.17311415076255798, -2.165717124938965, 0.7723909020423889, -0.7486869692802429, 0.17573262751102448, 0.09763062745332718, 0.9603515267372131, 2.0305027961730957, 0.9836022853851318, 1.2729486227035522, -0.23454320430755615, 0.4557735025882721, 0.769248902797699, -1.43962824344635, 2.0415430068969727, -0.7148682475090027, -1.277543306350708, 1.6340441703796387, 2.118438720703125, -0.7031057476997375, -0.6133099794387817, 3.111912965774536, 0.1126171126961708, -1.984058141708374, -0.6122687458992004, 2.0021188259124756, 0.6591240763664246, 0.7520990967750549, 1.2245573997497559, 0.6757786870002747, -0.3332044780254364, -1.5943117141723633, 0.44637084007263184, -1.1160348653793335, -0.05623851716518402, 1.758971095085144, 0.5406503081321716, 0.05949153006076813, 0.06568005681037903, 0.730792760848999, 0.9212337732315063, -1.3168138265609741, 0.1854793280363083, 1.7156471014022827, 0.7543686628341675, -0.7651258707046509, -0.6796277165412903, -2.061922550201416, -0.7933788299560547, 1.6069653034210205, -0.0695239007472992, -0.855010986328125, -0.5761204957962036, -0.11074551194906235, -0.10654617100954056, -0.35397371649742126, -0.7690944671630859, 0.6365700960159302, 1.5385345220565796, -0.5679569840431213, -0.5412718653678894, 1.189353108406067, -1.4637436866760254, -0.5913947224617004, -0.3780970871448517, 0.16778159141540527, -0.27493882179260254, -0.8866033554077148, 0.31050634384155273, -1.1877104043960571, -1.742513656616211, -2.0282576084136963, 0.7486985325813293, -1.7368555068969727, 0.4470572769641876, 0.6707091331481934, 0.8068993091583252, -0.17468272149562836, -0.5482001304626465, 0.0856713354587555, -0.5937349200248718, -1.286202311515808, 1.2554398775100708, -0.8944374322891235, 1.7958890199661255, 0.5147645473480225, -0.45455676317214966, 0.43764764070510864, 1.230149507522583, -1.3784692287445068, -0.926772952079773, -1.0492675304412842, 0.16503140330314636, 0.4042508602142334, 0.028968343511223793, -0.7282098531723022, -0.5284513235092163, 1.187043309211731, 1.1169426441192627, 0.13135528564453125, -1.310608983039856, 0.6281268000602722, 0.2876591682434082, 0.25273048877716064, 1.7300316095352173, 0.8205397129058838, -0.974485456943512, -0.9647605419158936, 1.3803138732910156, 0.20571652054786682, 0.8870580196380615, -0.10639959573745728, -0.9494908452033997, 0.32420140504837036, 1.6189154386520386, 0.39913585782051086, -1.065564751625061, -0.24585874378681183, -0.42841270565986633, 0.2473440170288086, 1.0980541706085205, 2.0542898178100586, 0.3144912123680115, -0.3364866077899933, 0.1715981662273407, 0.11086616665124893, -0.32979634404182434, 0.1186041608452797, 0.09529157727956772, -1.0805187225341797, 0.6438245177268982, -0.2667025625705719, -0.8877440690994263, 0.5859832167625427, -0.39159297943115234, 1.8382551670074463, 0.5709066987037659, 0.1686367392539978, -0.665776252746582, -0.31839120388031006, 0.9355282783508301, -0.6024860739707947, 0.8436412811279297, -2.0179641246795654, 0.28045257925987244, 0.8953021764755249, 0.41629400849342346, 0.07274018973112106, 1.086340069770813, -0.7792912125587463, -0.9079190492630005, 0.1802140772342682, 1.2776236534118652, 0.5065013766288757, -0.11938237398862839, 0.37507161498069763, 0.5171626806259155, -1.1704579591751099, -2.7371013164520264, -0.950610339641571, -1.4196820259094238, -0.5050193071365356, -1.1275153160095215, -0.7912658452987671, 0.5087776780128479, 0.17351014912128448, 0.05337090045213699, 1.7401034832000732, -0.3785295784473419, -0.2968817353248596, 1.3511508703231812, -1.025349497795105, -1.5425773859024048, 1.397329568862915, 0.42847737669944763, 0.6939471364021301, 0.7324030995368958, 1.000512719154358, -0.37342125177383423, 0.024766210466623306, -1.0105915069580078, 0.24289265275001526, 0.3374541997909546, -0.3463873863220215, 0.7765704989433289, 3.064624071121216, -1.0395714044570923, 0.9449557662010193, 1.060904860496521, 1.186570167541504, -0.6814988851547241, 1.3214032649993896, 0.7814787030220032, 0.004755706060677767, -0.21026098728179932, 0.3801894783973694, 1.071524977684021, 1.024174451828003, 2.4328603744506836, -0.2064686417579651, 0.7699827551841736, -0.2774447500705719, 0.9362072348594666, 0.28631535172462463, -0.44236698746681213, -1.170732021331787, -0.6131279468536377, 0.3971872627735138, 1.6269680261611938, 0.9308051466941833, -1.0413693189620972, -0.33361268043518066, 0.1840648353099823, 0.01132616400718689, 0.19017614424228668, 0.8913050889968872, 0.31386786699295044, -0.601158618927002, -1.8446869850158691, -0.2656921446323395, -1.6127432584762573, -0.10446330159902573, 0.363433837890625, -1.2998757362365723, -0.11584888398647308, -0.3271665871143341, 1.906078815460205, 0.44876807928085327, -1.9007844924926758, 0.42046672105789185, 0.3862980902194977, -0.2349327951669693, -0.4137638509273529, 0.5544675588607788, 0.6138501167297363, 0.38203006982803345, -1.0521268844604492, -0.3142268657684326, -1.6254591941833496, -1.566889762878418, -0.6347933411598206, -1.1325814723968506, -0.4956102967262268, 2.3927721977233887, 0.4025426506996155, -0.9809014201164246, -2.2614598274230957, -0.9428343772888184, -1.5433629751205444, -2.3115298748016357, 0.6050446629524231, -0.6025458574295044, 0.3164512813091278, 2.966238260269165, 0.14657582342624664, -0.0074188197031617165, 0.22416597604751587, -0.391497403383255, -0.9520021677017212, -0.3374938666820526, 1.0182347297668457, -0.3933661878108978, -1.492048978805542, -1.368557095527649, -1.1497435569763184, -0.1072094589471817, 0.3657081127166748, 0.36833882331848145, 1.0810695886611938, -0.30665305256843567, 0.7394214272499084, 0.23684793710708618, 0.6244451403617859, -1.9157264232635498, 1.1791285276412964, -1.6059455871582031, -1.1812087297439575, 0.728539764881134, -0.43165335059165955, 1.728952169418335, 0.12705281376838684, -1.0572758913040161, 0.128393292427063, -0.42599353194236755, 1.5300532579421997, 1.6091804504394531, 0.7201277017593384, -0.42249882221221924, 0.06931523978710175, 1.0352593660354614, -0.8136550784111023, -0.8067298531532288, 0.8386394381523132, 0.7259921431541443, -0.2420131117105484, -0.2782663106918335, -1.9611250162124634, 1.173648715019226, -0.1261032074689865, -1.4362707138061523, -1.1350212097167969, -0.9003010988235474, 1.6969619989395142, 0.6850574612617493, -1.341442346572876, 1.8512842655181885, 1.3351792097091675, -1.0612409114837646, -2.66099214553833, 0.9614986181259155, -0.6027781367301941, -0.13628479838371277, -0.524908185005188, 1.2592880725860596, 1.4789767265319824, 0.810117781162262, -0.15063685178756714, 0.26184359192848206, 0.6211815476417542, -1.5746363401412964, 1.2433815002441406, -0.31869974732398987, 0.26307061314582825, 0.14000190794467926, 0.9083163738250732, 1.0714818239212036, 2.2131998538970947, -0.34747856855392456, -0.8965520858764648, 0.14481224119663239, -0.6418266892433167, 0.8751027584075928, -0.7909221649169922, 0.15036310255527496, 0.9588245153427124, 1.034327507019043, 0.22643126547336578, 1.4341362714767456, 1.3108830451965332, -1.6749932765960693, 0.3853166103363037, 0.20764310657978058, -0.2698063850402832, -0.607272744178772, 0.07507789134979248, 0.41873955726623535, 0.3086544871330261, 1.3998866081237793, 1.0281139612197876, -0.5532593727111816, -0.7224798202514648, 0.8842297196388245, -0.6599516868591309, 0.20378567278385162, 0.5136725306510925, 1.0678472518920898, 0.5548705458641052, 1.483630657196045, -0.41991153359413147, 0.7100406289100647, -0.6179177165031433, 0.22685877978801727, 0.16443012654781342, -0.8879100680351257, -2.5632784366607666, 0.9970925450325012, 0.4042292833328247, 0.18224069476127625, -0.6591445803642273, -0.29145991802215576, 0.4969172775745392, 0.06155714392662048, 1.200844168663025, 0.1582016795873642, -0.7803479433059692, 0.6506232023239136, -0.593580424785614, 0.2554992139339447, -1.7722288370132446, -0.31255680322647095, 0.5930131077766418, -0.6167822480201721, 0.005562063306570053, 0.6214967966079712, 0.09812398999929428, 1.8157238960266113, -0.22971957921981812, -0.15062332153320312, -0.13217906653881073, -1.9484788179397583, -1.505789041519165, 0.5606258511543274, -0.30178529024124146, 1.41615629196167, -0.05088788643479347, -0.1429254710674286, 1.0557904243469238, 0.3035111725330353, 1.6309014558792114, -1.0298982858657837, 0.46728771924972534, 0.7073956727981567, 0.5844882130622864, 1.1814675331115723, 1.0607037544250488, 0.0700731948018074, -0.5216994285583496, 1.0194790363311768, 0.0929674580693245, 1.0061924457550049, 1.4043821096420288, 0.25849786400794983, -0.8391388058662415, -0.188700869679451, 0.8577681183815002, -0.16371336579322815, 1.2553577423095703, 0.12203249335289001, 0.21839630603790283, -0.5367452502250671, -0.008135730400681496, 1.1862748861312866, -0.25078389048576355, -0.24314925074577332, -0.6909047365188599, 0.8662309646606445, -1.438070297241211, -0.5731570720672607, 0.9874779582023621, 1.4973156452178955, -0.1496429741382599, 1.033983588218689, -1.914989709854126, 0.8238314390182495, 0.35727834701538086, -1.1592878103256226, -1.716760277748108, -0.44351816177368164, 0.32016366720199585, 0.4285535216331482, 0.17032885551452637, 0.8300293684005737, 0.7591947317123413, -1.325154423713684, -0.3256736993789673, -0.4546818733215332, 1.082566738128662, -0.4368080496788025, 1.0607962608337402, 0.14688044786453247, 2.4004673957824707, 0.005745717324316502, 0.059517305344343185, -0.056776802986860275, 0.20182116329669952, 2.482877731323242, -0.4495837688446045, -0.8182248473167419, -0.14498426020145416, 0.7236251831054688, -0.9369282126426697, 0.8460574746131897, 0.8127056360244751, -0.1168634369969368, -0.2049449384212494, -0.7416505217552185, -0.22885537147521973, -0.7548636794090271, 1.2864465713500977, 1.6693135499954224, -0.24161919951438904, 0.00040231773164123297, -1.8577793836593628, -0.48929232358932495, 0.3772006034851074, 0.3102934658527374, 0.15976007282733917, 0.880998969078064, 0.1265142410993576, -0.35277992486953735, 1.6342995166778564, -1.8823978900909424, -1.1231029033660889, -0.7060480117797852, -0.6988998055458069, -0.7454692721366882, -0.19863399863243103, -0.6876280903816223, -0.03457644209265709, 0.6096417307853699, 0.046772900968790054, 0.14978359639644623, -0.9177576303482056, -1.1184498071670532, -0.2779383361339569, 0.936154842376709, 0.3869842290878296, 1.095211386680603, 1.0011459589004517, 1.130170226097107, -0.793280839920044, 1.2445451021194458, 0.765385091304779, 2.0772807598114014, 0.597710132598877, 0.3354604244232178, -0.1174328625202179, 0.1754874438047409, -1.6325438022613525, -0.23683799803256989, -0.026065850630402565, 0.9965258836746216, -0.6405432224273682, 0.14371009171009064, 1.8275507688522339, -0.5140311121940613, 0.8498762845993042, -0.5180673003196716, -0.08717534691095352, 0.46998757123947144, 0.8892123699188232, -0.4767798185348511, -1.0020980834960938, 1.5188612937927246, 0.19799013435840607, 0.8002609014511108, 0.9185786843299866, 0.7643093466758728, -0.24615730345249176, -0.12190839648246765, -1.5274707078933716, 0.1913345605134964, -0.4519769847393036, -0.009605282917618752, 1.86375093460083, 0.6937499642372131, 0.06547141075134277, -1.3327445983886719, 0.7337110638618469, 1.5181533098220825, -0.8004674315452576, 0.43257465958595276, -0.49807146191596985, -0.1366279423236847, -0.3479275703430176, -1.5764728784561157, -0.49307113885879517, -1.5852549076080322, 1.3782265186309814, 1.0388894081115723, 0.05506785586476326, 0.2867772877216339, 0.0687292069196701, 0.20761720836162567, 0.20490533113479614, -0.22359073162078857, -0.15187609195709229, -1.3104325532913208, -0.9730347990989685, -0.5838696360588074, 0.21642760932445526, -0.3533463776111603, 1.885884165763855, -0.173744335770607, -0.337854266166687, 0.4379376769065857, 1.0245460271835327, 1.833386778831482, 0.9708805084228516, -0.5633888840675354, -0.15201693773269653, 1.0467700958251953, 0.26900050044059753, 0.8014563322067261, -0.05585052818059921, 1.5882296562194824, 1.437668800354004, 1.1231521368026733, 0.1743306666612625, -1.5570294857025146, -1.299076795578003, 0.3423963189125061, 0.6638202667236328, -2.8262147903442383, -0.4391079843044281, 1.1521999835968018, 0.9599878787994385, -0.4598434567451477, 0.6500676274299622, 0.057154640555381775, -0.185583233833313, 1.1234819889068604, 1.1430305242538452, -0.4834875762462616, -0.8938437700271606, -1.2477530241012573, -1.35883367061615, 1.3349007368087769, 1.8648263216018677, 0.24048985540866852, 1.062922477722168, -0.6329414248466492, -0.03174551576375961, 0.9313143491744995, 0.34497955441474915, 0.0025368696078658104, -1.5862555503845215, -1.3807862997055054, -0.36637210845947266, -0.5022204518318176, -0.03396618366241455, -0.584236741065979, -0.17405374348163605, 0.0976613238453865, 0.5410217642784119, 0.10228528082370758, -0.49703362584114075, -0.23362646996974945, 0.9192822575569153, -0.16407977044582367, -0.9060814380645752, -0.9460440874099731, 0.31115853786468506, 0.5796923041343689, 1.2263473272323608, -0.27965250611305237, 1.3735370635986328, -0.3059794306755066, -0.1946568638086319, -0.5217711329460144, 0.5411524176597595, 1.05014967918396, 0.004995408002287149, -0.45856037735939026, -1.4729292392730713, 0.1721293330192566, -0.3063606917858124, -0.1690288484096527, -0.5875093340873718, -0.825576901435852, -0.3926343321800232, 0.45955032110214233, 0.9546487927436829, 0.01128454227000475, 0.3360567092895508, -0.17004065215587616, -0.0998295247554779, -1.3070999383926392, 0.8779817819595337, 0.599260687828064, 1.4344311952590942, -0.5745570063591003, 0.1446526050567627, -0.8759451508522034, 0.2539585530757904, -0.09897676855325699, 0.6767484545707703, -0.12040253728628159, 0.6449835300445557, -0.4546484053134918, 0.5981537103652954, -0.2716369926929474, -0.4019356966018677, 0.1071111410856247, 0.3789922595024109, -1.3443832397460938, -1.4471772909164429, 0.20568685233592987, -0.03210219740867615, -1.9469980001449585, -1.6273365020751953, -0.7208101153373718, 0.33340221643447876, 0.22141604125499725, 0.999830424785614, 1.159284234046936, 2.3200325965881348, -1.3723598718643188, -0.013785125687718391, 1.0572185516357422, -1.1914669275283813, -1.340640902519226, 0.21681663393974304, 1.0876883268356323, 1.2406551837921143, -1.6978176832199097, -1.151814579963684, -0.06735824048519135, -0.48677846789360046, 0.3394567668437958, 0.15675535798072815, 1.7359110116958618, -0.033442456275224686, 0.2049727439880371, 0.10196290910243988, 1.1016439199447632, -1.0495768785476685, 0.4038074314594269, -0.7196693420410156, -1.3453353643417358, 1.1762868165969849, 0.5426834225654602, 1.3904682397842407, 0.2827655076980591, 0.6671103239059448, 0.16937315464019775, 0.5917483568191528, -0.3609093725681305, 1.701602578163147, -0.8201969861984253, 1.7057604789733887, -0.1485859602689743, 0.8921769261360168, 0.18080472946166992, 0.4822855293750763, 1.4990304708480835, 0.18873842060565948, -0.8698955178260803, 0.2848009467124939, 0.684927225112915, -2.343674421310425, 1.005519986152649, 1.1186068058013916, -1.675768494606018, -0.09786126762628555, 0.8562725186347961, -1.0792711973190308, -0.3587975800037384, -0.27950209379196167, 0.6004593372344971, 1.143613576889038, 0.02294977754354477, 0.1153683140873909, 0.9619436860084534, 0.5853611826896667, -0.05394911393523216, 1.3585437536239624, -1.3352926969528198, 0.2020653635263443, -0.3604046702384949, -0.44249820709228516, 0.24635453522205353, 0.08954466134309769, 2.466902732849121, 0.6843624114990234, 0.3969602584838867, 1.674640417098999, 1.0493732690811157, -1.7812647819519043, 0.17044976353645325, 0.781863272190094, -0.5458417534828186, 0.33988142013549805, -0.4015931189060211, 0.4161466658115387, -1.5575698614120483, -1.874250888824463, -0.4973315894603729, 0.26492151618003845, 1.3531397581100464, -0.232381671667099, -0.6981062293052673, 0.9420226812362671, 0.07280247658491135, 1.8154182434082031, -0.0507391132414341, -1.8845176696777344, -0.7511929869651794, -0.9051104784011841, -1.3399128913879395, -0.7324886918067932, -0.6487211585044861, -0.25051265954971313, -0.5116637349128723, 0.4315355718135834, 0.4358985722064972, 0.20217981934547424, 0.22221021354198456, -1.8342421054840088, -1.6215094327926636, -0.8437504768371582, 0.27673444151878357, -1.4215724468231201, -0.3913985788822174, 0.2697241008281708, 1.9867075681686401, 0.28176867961883545, -0.5059260725975037, 1.1024010181427002, 0.22433927655220032, -0.27569183707237244, 1.1198322772979736, -0.8906885981559753, 0.2054339051246643, 0.09374996274709702, 0.9072456955909729, -1.1242132186889648, -1.2728910446166992, -0.37835124135017395, -0.2663916349411011, 0.04562491178512573, -0.3687709867954254, 0.42910754680633545, 0.6845958828926086, 0.22120583057403564, -1.0464434623718262, 1.1411778926849365, -1.5240790843963623, -0.20895320177078247, 0.8774041533470154, -0.6181960701942444, 1.2746925354003906, 1.8907825946807861, 1.5551873445510864, 0.42311325669288635, 0.9650139212608337, -0.5183897614479065, 0.3883185386657715, 0.4431073069572449, 0.0834994912147522, 0.14439734816551208, -0.21514803171157837, -1.957853078842163, 0.38814401626586914, 0.3022112250328064, 2.0293054580688477, 0.6977156400680542, 0.058108072727918625, 0.8594762086868286, 0.6098951697349548, -0.025558190420269966, -0.3873504102230072, -1.4679991006851196, 0.11340352892875671, 1.6074281930923462, -1.8636541366577148, -0.9536433219909668, 0.7432786226272583, 0.18359945714473724, 0.4093780219554901, -1.857773780822754, -0.5945903658866882, 0.4945397675037384, 0.5778988003730774, 0.28581249713897705, -0.8860844969749451, 0.5290326476097107, -1.1968166828155518, -0.9440393447875977, -1.1677287817001343, 1.425971269607544, -0.41940295696258545, 0.046004146337509155, -2.1984241008758545, 1.0813047885894775, 0.45218610763549805, -1.6580231189727783, -1.609822154045105, 1.6429228782653809, -0.9511963725090027, 1.8240394592285156, -0.4779016375541687, -2.4858362674713135, 1.09334135055542, 0.4894861876964569, 0.6681703925132751, -0.5189090371131897, -0.22533050179481506, 0.9421473741531372, -0.8329755663871765, -1.975229024887085, 0.8923466801643372, -0.6322474479675293, 1.2579083442687988, -0.49562975764274597, -0.44075384736061096, -0.26612457633018494, -2.3663153648376465, 1.1586400270462036, -0.6432483196258545, -0.21072284877300262, 1.4466402530670166, 0.09996433556079865, 0.8790737986564636, 0.3202430307865143, 0.7567090392112732, 0.2622949481010437, 1.1387934684753418, -0.22413885593414307, 0.4465140104293823, -0.7214215993881226, 1.1713738441467285, -2.1937522888183594, 0.17275454103946686, 0.3479059636592865, 0.32904183864593506, -0.01979093626141548, 1.3260551691055298, 2.101428985595703, -0.6073958277702332, -1.9200876951217651, -1.0991789102554321, -0.8243827819824219, -0.148102268576622, -0.06763698905706406, 1.0654654502868652, -0.43930330872535706, -0.04962938278913498, 0.09713341295719147, -0.8906799554824829, -0.3303353190422058, 2.742591381072998, -0.10513272136449814, -0.975318193435669, -2.1901962757110596, 1.0241239070892334, -0.335629940032959, -0.6474533081054688, 2.4308433532714844, 1.4182775020599365, -1.7253302335739136, -0.3614228069782257, 0.9838337302207947, 0.8207517862319946, -1.127099871635437, 1.762094259262085, 1.835807204246521, -1.122765302658081, -1.7803300619125366, -1.405688762664795, 0.6507818102836609, 1.0621857643127441, 0.16741704940795898, -0.6129648089408875, -0.1111510768532753, 0.5452548861503601, -1.2326768636703491, -1.7168357372283936, -0.105614073574543, 1.2133082151412964, 0.16984093189239502, 0.17263193428516388, -0.6962419748306274, 0.8977471590042114, -1.7302011251449585, -0.0884302407503128, -0.6173286437988281, -0.17867448925971985, -0.9521907567977905, 0.7226251363754272, -0.10454924404621124, -0.6244663000106812, 0.48045769333839417, -0.08159482479095459, -0.5501055717468262, -0.6215181350708008, 0.7462332844734192, -0.9719409346580505, -0.25569018721580505, -0.3024829626083374, 0.2788255214691162, -0.2284586876630783, 2.1212375164031982, -1.7823542356491089, 0.5563126802444458, -0.06899502873420715, 1.0719588994979858, -1.661516785621643, 1.1792610883712769, -1.4204144477844238, 0.6526589393615723, 0.4729805290699005, -0.007830378599464893, -0.7760050892829895, -0.8554891347885132, 0.05861421674489975, -0.7905116081237793, -1.9903665781021118, -1.847069501876831, 0.7606856226921082, 0.785500705242157, 0.47872257232666016, 0.012259075418114662, 0.3462595045566559, 0.46212875843048096, 0.21987619996070862, -0.8225317597389221, -0.7628558874130249, -1.1806305646896362, 0.12368922680616379, -1.0183985233306885, -0.18521061539649963, 1.2895960807800293, -0.4007664918899536, 0.9477011561393738, 1.5366467237472534, -1.8462074995040894, -1.4550038576126099, 0.4535783529281616, -0.727559506893158, 0.5174764394760132, 0.6908771395683289, -0.5290382504463196, 0.8123423457145691, 1.7308964729309082, 0.9444819688796997, -0.8852678537368774, -0.15462060272693634, -0.5446020364761353, 1.5613423585891724, -0.19860419631004333, -0.6944221258163452, 0.04367179423570633, 0.5124685764312744, -0.37633129954338074, 0.10705791413784027, 0.0024917887058109045, 0.7415816783905029, 0.9778999090194702, 0.27968519926071167, -0.4603579342365265, 0.4139566421508789, -0.8825797438621521, 1.0847476720809937, 0.36036601662635803, 0.2574433386325836, -2.4102981090545654, 2.384186029434204, 1.4711592197418213, 1.288474440574646, 0.147059366106987, 0.6938448548316956, 2.8812804222106934, -0.6559854745864868, 0.7759380340576172, 0.6342946887016296, 0.17647245526313782, 0.3914646506309509, -0.9390909075737, 0.6915391087532043, -1.263176679611206, 1.8821666240692139, 0.8521352410316467, -0.7967486381530762, -2.0363402366638184, 0.16343924403190613, 0.5680439472198486, -0.9327842593193054, 1.741498589515686, 0.3390537202358246, -0.39467480778694153, -1.3252567052841187, -0.04735438898205757, -0.12439973652362823, 0.7260642051696777, 1.125786304473877, 0.10802280902862549, 0.5389137864112854, 1.5030630826950073, 0.6967734694480896, -0.17905555665493011, 0.6983010768890381, -0.44238340854644775, 0.5312957167625427, 0.13035549223423004, -1.0187915563583374, 0.6830295920372009, 0.16322802007198334, 0.8106619119644165, -1.24861478805542, 1.18617582321167, 1.2288119792938232, -0.2741584777832031, -0.022449886426329613, 0.12253140658140182, 0.052425842732191086, 0.9073825478553772, -0.6430671811103821, -1.0808987617492676, -0.1760164350271225, 1.962971568107605, -0.3276088237762451, -0.9405286908149719, -0.8623668551445007, 0.22092878818511963, 1.228193759918213, -0.6862431168556213, 0.22234377264976501, 0.2604827582836151, 0.5109481811523438, 0.18895459175109863, 1.0941689014434814, 0.7297549843788147, -0.5733395218849182, 2.4672462940216064, -1.3218557834625244, -1.2772600650787354, 1.0755003690719604, 0.22601525485515594, -0.20893940329551697, 0.23949386179447174, 0.724484384059906, -0.38553953170776367, -0.5983303189277649, -1.0125545263290405, -0.8265764117240906, -0.7659786343574524, 1.1788495779037476, 0.3116062581539154, 0.5756569504737854, -0.5320655107498169, 1.0790150165557861, -1.5308595895767212, 0.5036996603012085, 0.7234464287757874, -0.5512312650680542, -0.9989850521087646, 0.5466575026512146, -0.684355616569519, -0.8702754378318787, 0.0020186363253742456, -1.0957164764404297, -0.11396929621696472, 2.049099922180176, -0.41492578387260437, 0.9298009872436523, -1.0312168598175049, 0.10533149540424347, -0.2494734525680542, -0.14563637971878052, 0.7782558798789978, 2.1698338985443115, -0.1063050627708435, 1.3247079849243164, 0.3704155683517456, 0.7564952969551086, -0.44869694113731384, -0.6872261762619019, -0.1815694272518158, -0.7536616921424866, 0.009123672731220722, -1.0375372171401978, 0.023145683109760284, 0.19463421404361725, 1.8339016437530518, 1.7419975996017456, 0.5283176302909851, -0.4836968779563904, 0.6267964839935303, 0.8358477354049683, -0.17668025195598602, -0.8293713331222534, -1.771276593208313, -0.01900007575750351, 1.9416769742965698, -0.1866835653781891, 0.4056864082813263, 0.3856768012046814, 0.5051284432411194, 0.7945407032966614, -1.9836710691452026, -0.12875913083553314, 0.3988375961780548, -0.06705044955015182, 1.2387086153030396, -0.3480604887008667, 0.9469493627548218, 1.7920414209365845, -0.2254084050655365, 0.057759691029787064, -0.2623107433319092, 0.7974582314491272, -2.1079933643341064, -1.4305156469345093, -0.2600786089897156, 0.9074793457984924, 0.6277872323989868, -0.7818407416343689, 1.6447612047195435, 0.1508508175611496, -0.585057258605957, 0.5723842978477478, 1.3474761247634888, -0.024584978818893433, -1.475991129875183, -0.5078163146972656, 0.13803061842918396, 0.9861943125724792, -0.958077609539032, 0.30879470705986023, -1.1394537687301636, 0.12330487370491028, -1.2868380546569824, -0.41652217507362366, -1.4775577783584595, -0.9329387545585632, -1.544848918914795, -1.9235820770263672, 1.2739397287368774, 1.228738784790039, 0.5300590991973877, -0.30698293447494507, 1.5223150253295898, -0.229792520403862, 0.4800225496292114, 1.1913615465164185, 0.279014527797699, 1.5368289947509766, -0.08661879599094391, -0.09483730792999268, 0.19175629317760468, -1.0959360599517822, -1.0146629810333252, 0.8395438194274902, -1.6665738821029663, 1.1950830221176147, 0.33906376361846924, 1.2519441843032837, 0.8313003182411194, 0.668026864528656, 1.8131299018859863, -0.0066193523816764355, -0.03390422835946083, 1.1996455192565918, -0.5904881954193115, 1.2064158916473389, 0.7611380219459534, -0.43198567628860474, -1.2055048942565918, -1.955995798110962, 1.9460760354995728, -1.4253811836242676, -1.141318440437317, 0.9324416518211365, 0.5389015078544617, 0.2745162844657898, 0.14609888195991516, -0.3657054603099823, -0.8646954298019409, -0.6852208971977234, 0.6529102921485901, 1.1334609985351562, 1.2387707233428955, 0.014396779239177704, -2.2000515460968018, 0.5596252679824829, -0.0993688553571701, -2.6409964561462402, 1.4376649856567383, 0.2722074091434479, -0.0033549123909324408, 0.24939411878585815, 0.1734665185213089, 0.2923654615879059, 0.17090201377868652, 0.07970554381608963, 0.630560576915741, 0.4072439968585968, 2.48055362701416, 0.20415771007537842, -0.44723913073539734, -0.5324724316596985, 0.45582911372184753, 0.19513754546642303, 0.322083443403244, -0.12491130083799362, -0.10494505614042282, 0.833539605140686, 1.4451780319213867, 0.42491820454597473, -0.02818743884563446, -1.084568738937378, -1.0856214761734009, 0.12048393487930298, -0.17535515129566193, -0.029498199000954628, -1.0472146272659302, -0.06253772974014282, -1.8932443857192993, -1.1398168802261353, -0.7421925663948059, 0.6393010020256042, -0.5662417411804199, -0.9038388729095459, 0.3102126121520996, -0.06312230974435806, 0.957832932472229, 0.3540431261062622, 0.24079760909080505, -0.16631868481636047, 0.6837397813796997, -0.7159861922264099, -1.5155874490737915, 0.9188432097434998, -0.4650467038154602, 0.05570192635059357, 0.6328984498977661, 1.02590012550354, -2.90317702293396, -1.0468684434890747, 1.2337498664855957, -1.7055178880691528, 0.47626402974128723, -0.3627007007598877, -0.8921003341674805, -0.1311357170343399, -1.6965107917785645, 0.40589582920074463, 1.1765064001083374, -1.4324692487716675, -0.6737122535705566, 1.201277494430542, 0.3682346045970917, -0.3279290795326233, 1.4351848363876343, -1.6871209144592285, -0.2809986174106598, 0.08910386264324188, -0.3242110311985016, 0.7949813604354858, -1.6841487884521484, 0.0419415645301342, -0.7951081395149231, 1.1987022161483765, 1.0446423292160034, -0.7540777921676636, -1.110982060432434, -0.05243905261158943, -0.86948162317276, -0.5787657499313354, 0.42370519042015076, -0.4685525596141815, 1.8138983249664307, 0.059520427137613297, -0.5323241949081421, -1.1301554441452026, -0.5311712622642517, 1.1020461320877075, 0.5418751239776611, 1.4253020286560059, 1.3063513040542603, 0.5486820340156555, -0.7083648443222046, -0.16049906611442566, 0.0593281127512455, -1.6690553426742554, 0.3216547667980194, -1.0518375635147095, -0.09585031121969223, -0.5272992253303528, 0.5736538171768188, 2.6331241130828857, -1.2069087028503418, -0.027186580002307892, 2.0459628105163574, -0.938170850276947, -0.2334383726119995, -0.11366590857505798, 0.30666616559028625, -1.6295677423477173, 0.3029867112636566, 0.60346519947052, 2.554124355316162, -0.08022479712963104, -0.5807622075080872, 0.13223762810230255, -2.6241302490234375, 0.33901074528694153, 1.565286636352539, -0.03310404345393181, -0.03400510922074318, -0.24262818694114685, 0.14173579216003418, -1.2457361221313477, 1.556157112121582, -0.45594120025634766, -0.4500073790550232, 1.6717227697372437, 0.03138326480984688, -0.41074836254119873, 1.6706821918487549, 0.19934625923633575, 0.9619471430778503, -0.5526202321052551, -1.7544792890548706, 0.19771327078342438, 1.4599751234054565, -0.43941211700439453, 0.3906843364238739, 0.7519165873527527, 0.29140812158584595, 0.10033553838729858, 0.8053041100502014, 0.2440894991159439, -0.5051175951957703, 1.0927962064743042, 0.8403662443161011, 0.41254448890686035, 0.7595425248146057, -2.6139895915985107, 0.305851548910141, 0.07747472077608109, -1.9600838422775269, 2.1217827796936035, 0.3910723030567169, -0.4022234380245209, -0.33886682987213135, 0.3079967498779297, 0.024515388533473015, -1.5770111083984375, -0.3604246973991394, 0.49143627285957336, 0.6596852540969849, 0.913526177406311, -1.0220931768417358, -0.5900863409042358, 0.4200451970100403, -0.5641858577728271, 0.26408806443214417, -0.7852447628974915, 0.6409056186676025, 0.11563713848590851, 0.6402919292449951, 0.09382452815771103, -0.7299577593803406, -1.164474606513977, 0.10823257267475128, -0.5488094091415405, -1.4282312393188477, -0.483622670173645, 0.8639504313468933, 1.3144090175628662, 0.8382378220558167, 1.4408756494522095, -0.29636409878730774, -0.07282564043998718, -0.8781378865242004, -0.6975635290145874, 1.6998779773712158, 0.5192322731018066, 0.7358409762382507, 2.7045629024505615, 2.023057222366333, -0.7820925116539001, 1.0720834732055664, -0.6707949638366699, 1.0838983058929443, -1.800173044204712, -0.8540792465209961, -0.8314402103424072, 2.106879711151123, -1.6960121393203735, 1.2090988159179688, 1.1745225191116333, 0.21123374998569489, 0.9305042028427124, 2.6903953552246094, 0.3472551703453064, -0.1859285682439804, 1.4567855596542358, 0.8817878365516663, -1.0671945810317993, 0.28148236870765686, 0.37875786423683167, -1.0624080896377563, -0.8689408898353577, -0.5779282450675964, -1.1256519556045532, 0.6099210977554321, 0.3916228711605072, -2.1584393978118896, -0.34197917580604553, 0.24037520587444305, -1.2846587896347046, -0.2018256038427353, 0.3723554313182831, -1.6439969539642334, -0.23576058447360992, -2.324458599090576, 0.8270865082740784, -0.6788032054901123, -1.4082980155944824, -1.813498616218567, -1.6186257600784302, 1.3463342189788818, -0.8091784119606018, -0.5387489199638367, -0.36283519864082336, -1.4947246313095093, 0.8040604591369629, -2.514575481414795, 0.9299538731575012, -0.7476893067359924, 0.062265776097774506, 0.31638234853744507, 0.864777147769928, -1.245194435119629, 0.6316952705383301, -0.042534977197647095, -0.1497233510017395, -0.279050350189209, 1.1402994394302368, 0.4842745363712311, 1.2972815036773682, 0.9357013702392578, -2.0091545581817627, -0.10172495245933533, 1.0270110368728638, -0.9157797694206238, -1.709349513053894, -2.3339133262634277, 2.0197603702545166, -0.9423668384552002, 0.16915524005889893, -1.0996613502502441, 0.4261535108089447, 1.680699348449707, -1.4839746952056885, -0.07135990262031555, -0.7865275144577026, -1.9219657182693481, -0.9180206656455994, 0.022351490333676338, 1.2539845705032349, 1.299833059310913, -0.02115975320339203, -1.6937263011932373, 0.6718358397483826, -0.008554019033908844, -0.01956520415842533, -1.1728698015213013, -0.162626251578331, 2.0049469470977783, 1.5216397047042847, 1.301401972770691, 0.6831285357475281, -0.40635454654693604, 0.0019021439366042614, 0.1545584350824356, 0.3991532623767853, -0.3562451899051666, -0.8836852312088013, 1.1446409225463867, -0.38013988733291626, 0.8623542189598083, -0.17686472833156586, 0.695436954498291, 0.5290508270263672, 0.760155200958252, 0.7070104479789734, -0.9656563997268677, -1.3841845989227295, -0.04374552145600319, 1.6943448781967163, -0.8474162817001343, -0.9452128410339355, 2.3303966522216797, -1.5767143964767456, -0.5816718935966492, -0.41499564051628113, -0.6903476715087891, -0.6569643616676331, 1.1038422584533691, 0.2124847173690796, 0.18789586424827576, -1.6894078254699707, 0.8133772015571594, 0.1701420545578003, 0.15670719742774963, 0.8441293239593506, 0.3222537636756897, -0.026573948562145233, 0.4295389950275421, -0.7183917760848999, -0.9197143316268921, -1.3769526481628418, 0.9888535737991333, -0.6619219779968262, 0.18392722308635712, 1.1356672048568726, 1.847802996635437, -0.3610532581806183, 0.1748092919588089, 0.10296069085597992, -1.32536780834198, -0.4818105101585388, 2.8531901836395264, -0.057821985334157944, 1.8103140592575073, 0.784964382648468, 1.798912763595581, 1.359023928642273, 0.23863103985786438, -1.2540137767791748, 1.4669009447097778, 0.7286076545715332, 0.5011237859725952, 0.8914228677749634, -0.801724374294281, 0.1388131082057953, 0.002337648067623377, -1.5085853338241577, -1.217527985572815, -0.12283164262771606, -2.2543201446533203, 0.753483235836029, 0.7880116105079651, -0.0745398998260498, 0.2870529890060425, -1.8507272005081177, 0.17429620027542114, 0.7635998129844666, 0.5131475329399109, 0.9346216320991516, -0.06840269267559052, 2.0264604091644287, 0.43739089369773865, 0.8038463592529297, 1.0973892211914062, -0.7842851281166077, -1.14949631690979, 0.790923535823822, -1.279953956604004, -0.6160572171211243, 0.9996698498725891, -1.1180498600006104, -1.004956603050232, -1.269560694694519, -1.0677045583724976, 1.9004151821136475, 0.5517888069152832, 0.9722903966903687, -1.9816550016403198, -2.457585096359253, -1.179125189781189, 1.0562944412231445, 1.1865676641464233, -2.0245754718780518, -1.791841983795166, 0.6510434746742249, 0.2402716428041458, 0.4296501576900482, 0.6572490334510803, -1.046489953994751, 0.10760582983493805, -0.30710411071777344, 1.2317569255828857, -0.33581119775772095, -1.947200894355774, 1.2642539739608765, 1.8175526857376099, 0.9674302935600281, -0.3589092791080475, -0.9893718361854553, 0.9997820854187012, -0.5646253824234009, 0.5054417848587036, 0.7701002955436707, -0.16397438943386078, -2.0658774375915527, 0.4813714921474457, -1.049729824066162, -1.6199027299880981, -0.07810775190591812, 0.10766378045082092, -0.12551642954349518, 0.6125551462173462, -1.9776705503463745, 0.5594844818115234, 1.6242954730987549, -0.11603474617004395, -0.5438166856765747, -0.20839431881904602, 1.215266227722168, -0.22255584597587585, -0.8388031721115112, -1.7559230327606201, 1.4067758321762085, 1.4250935316085815, -0.019607912749052048, 2.085129737854004, -0.4282550513744354, 0.7128222584724426, -1.3589022159576416, -1.0492678880691528, -0.03090570494532585, 0.4628725051879883, 0.6063375473022461, -0.9331610202789307, -0.5591902136802673, -0.13281725347042084, -0.30639317631721497, -0.2832806706428528, 0.8107817769050598, 1.165527105331421, -1.8971138000488281, 1.323367714881897, 0.1668599396944046, 0.3179636001586914, 0.3290240466594696, 0.9359439611434937, 0.6962593793869019, -1.5688550472259521, 0.8707560896873474, -1.1553701162338257, 0.3822036683559418, 0.21011683344841003, -1.5739116668701172, 0.00763573357835412, -0.2656339704990387, -0.688567578792572, 2.055619239807129, 0.1570737510919571, -0.38075757026672363, 1.765931248664856, -0.035051796585321426, -0.14131540060043335, -2.0396265983581543, 0.6059733033180237, 1.0125623941421509, 1.936902403831482, -1.0317901372909546, 1.0567675828933716, 1.4211300611495972, 0.7312470078468323, 0.0004376191063784063, -1.6787775754928589, 0.1879522204399109, -0.5667224526405334, -1.7716076374053955, 0.6081501841545105, -0.12629400193691254, -0.24238882958889008, 1.999988079071045, -0.5465682744979858, -2.113917350769043, -2.260662317276001, 0.5262123346328735, -0.2282325029373169, -0.7765992283821106, -2.1633615493774414, -0.8526902794837952, -1.1533225774765015, -0.12701547145843506, -0.8885976076126099, -0.9055554866790771, -0.06567306071519852, -0.8213518857955933, 0.30681097507476807, -3.4890921115875244, -1.283579707145691, 1.2037767171859741, 0.1792767196893692, -1.1309115886688232, -0.219748392701149, -0.6168520450592041, -0.4062117338180542, 1.4187238216400146, -0.4551678001880646, 0.17878110706806183, -0.8977140784263611, -0.788196325302124, -1.6043168306350708, -0.7006797194480896, -0.4537413716316223, -0.3494471311569214, 0.01974780112504959, 0.061055779457092285, 0.6308714747428894, -0.2514100968837738, 1.2957452535629272, 0.5811610817909241, 1.3666226863861084, -0.08464314788579941, 1.1991623640060425, 0.06524880975484848, -1.7045265436172485, -0.05154397338628769, -0.20740807056427002, -1.5042614936828613, -1.028358817100525, -0.6435770988464355, 2.279287099838257, 0.28586164116859436, 1.8088774681091309, 0.9187138080596924, 1.070163369178772, 0.797220766544342, -1.5280883312225342, 1.2445708513259888, -1.035203218460083, 0.42310360074043274, -0.12211237847805023, 1.6191723346710205, 1.192905068397522, 1.8358203172683716, 0.4801208972930908, -0.00454776082187891, 0.7659798264503479, 2.4078879356384277, 0.5153617858886719, 0.03587067499756813, -1.2473448514938354, -0.760235071182251, -0.09207724034786224, 0.6738526821136475, 0.620242714881897, -0.1053115651011467, 0.12852054834365845, 1.5394366979599, 1.8534750938415527, 1.2020151615142822, 1.5357857942581177, 1.2501819133758545, 1.8346267938613892, -1.266554355621338, -0.5137951374053955, 0.23736003041267395, 0.39106374979019165, 1.3472763299942017, -1.8483145236968994, 0.15010730922222137, 0.269502192735672, -0.6765498518943787, -0.13676701486110687, 1.0100003480911255, 0.47231218218803406, -0.12119347602128983, 0.2133304625749588, -0.11790558695793152, 0.9489354491233826, -0.09370438754558563, 0.9949321746826172, -0.5499192476272583, -0.023564215749502182, -1.11411452293396, -0.19425441324710846, 0.8555719256401062, -1.0062843561172485, -0.33976125717163086, 0.11938902735710144, -0.3426598906517029, 0.8570409417152405, -1.2319366931915283, 0.8793560266494751, 1.344630241394043, 1.6850330829620361, -0.046261727809906006, -0.7885118722915649, -1.566705584526062, 0.7056165933609009, 0.8463314771652222, 0.31323671340942383, 0.3673610985279083, 0.6974493861198425, -0.341010183095932, -1.45907461643219, -1.4159059524536133, -0.7706493139266968, -0.9017643332481384, -1.5783023834228516, -0.41286250948905945, -1.0220776796340942, 0.6832398772239685, -0.05107981339097023, 0.8219492435455322, -1.311927318572998, -1.0842654705047607, -1.5840054750442505, 0.3100616931915283, 3.1597750186920166, 0.24542008340358734, -0.4130656123161316, 0.23803766071796417, -0.007701708935201168, 0.9950780272483826, 0.01981261931359768, -0.8630015254020691, -1.8907221555709839, 0.38540878891944885, 1.0339497327804565, 0.17416469752788544, -0.8695354461669922, -0.41264787316322327, -0.30134317278862, 0.609575629234314, -0.0373137928545475, 1.2036628723144531, -0.1780203878879547, -0.6263366341590881, 1.8862498998641968, -0.983154296875, -0.7684646248817444, 2.1231918334960938, 0.3077166676521301, 1.8227672576904297, -2.4914472103118896, 1.2892688512802124, 0.024232281371951103, 0.1291380077600479, 0.5211510062217712, -2.7540884017944336, -0.880375325679779, 0.4748156666755676, 3.280484437942505, -0.734463095664978, 2.4022715091705322, 0.6666901707649231, 1.0606590509414673, -0.9445372223854065, 0.6408244371414185, -0.8412412405014038, -1.19634211063385, 1.3122743368148804, -0.6845633387565613, -0.4900459945201874, 0.7229471802711487, -0.7951647043228149, -1.2512199878692627, 0.25068992376327515, 0.29657167196273804, 1.5548866987228394, -1.1132643222808838, 0.7431970238685608, 0.741662859916687, -0.4214533269405365, -0.14950981736183167, 0.04107408598065376, -1.0215051174163818, -1.315905213356018, 0.7301647663116455, -1.0596551895141602, -0.9686294794082642, 0.8134691715240479, -0.534144401550293, -0.3964405953884125, -1.4020283222198486, -1.2373522520065308, -1.3910726308822632, -1.3860958814620972, 1.258571743965149, -0.9760587811470032, -2.3431038856506348, -1.1263753175735474, 0.6967557072639465, -1.1906239986419678, -0.0696740373969078, -0.9768961071968079, -1.8019131422042847, -0.5871549248695374, -1.0703202486038208, 0.13781099021434784, 0.6738025546073914, -0.10938538610935211, 2.619720220565796, -1.110555648803711, 1.3088946342468262, 0.38069814443588257, 1.367080569267273, 0.49181103706359863, -2.3132083415985107, -0.3493176996707916, 0.9929517507553101, -0.02758386917412281, -1.3802120685577393, -0.6042711734771729, 0.2782595455646515, 1.1835925579071045, -0.662032961845398, 0.04236043244600296, -1.0001064538955688, -1.2735131978988647, -0.5881301164627075, 1.2959563732147217, 0.6613765954971313, -0.3469172716140747, -0.09621936827898026, 0.46980953216552734, -0.19815851747989655, -0.5142219662666321, -0.38081440329551697, 0.5668272376060486, -0.4785928726196289, 0.7067695260047913, -1.961521863937378, -0.854141354560852, -0.5196807384490967, 0.5035271048545837, -1.021882176399231, -0.6350343227386475, 0.22382725775241852, -0.20150126516819, -0.7761430144309998, -0.4345782399177551, -1.0312949419021606, 0.7105249762535095, 0.3398021459579468, 0.7663951516151428, 2.173912763595581, 1.1747143268585205, 0.4217837452888489, -0.16563311219215393, 2.218447208404541, 0.19466865062713623, -0.25797784328460693, 1.2692523002624512, 0.4117867648601532, -0.26381590962409973, 0.44421863555908203, 1.9802014827728271, 0.166892871260643, 0.41853103041648865, -0.8326524496078491, -0.41872110962867737, 0.9571453928947449, -0.021325645968317986, 0.19744333624839783, 1.183922529220581, -0.2429741770029068, 1.6476994752883911, -0.555851399898529, -0.6657913327217102, -0.7138451337814331, -1.4098362922668457, 0.6448240876197815, 0.11049598455429077, 1.4621315002441406, -0.6484695076942444, 0.8371844291687012, 1.4197953939437866, -0.14677178859710693, -0.09642885625362396, -2.4290921688079834, 2.00850510597229, -0.06405671685934067, 0.7521620988845825, -0.46113643050193787, 0.4048464596271515, -1.095330834388733, 0.807016909122467, 0.15720871090888977, -0.31606048345565796, 0.48843416571617126, -1.7440528869628906, 0.21442176401615143, -0.3899080455303192, 1.4976296424865723, 0.8168826103210449, -0.14408200979232788, -0.25363150238990784, 3.027841091156006, -0.9625516533851624, 1.9403542280197144, 0.9593336582183838, 0.8167067766189575, -1.8807340860366821, 0.861061692237854, -0.4763568937778473, -0.8841438889503479, 0.6219140291213989, -0.6908262968063354, 0.5945615768432617, -0.9252094626426697, 0.6085829734802246, 0.4558362066745758, -0.8358479142189026, -0.9486767053604126, -0.6734093427658081, -0.5301512479782104, -0.6025046110153198, 1.852903127670288, 0.4144555926322937, -0.3521604537963867, -0.3363032341003418, -0.8189142346382141, 0.2642752230167389, -0.07895781099796295, -0.6958801746368408, -0.7114008069038391, -0.4240822196006775, -1.077291488647461, 0.09817306697368622, -0.930112361907959, 0.5227359533309937, -0.3476731777191162, 0.5173784494400024, -0.5966096520423889, -1.88602876663208, -1.3306903839111328, -0.30026423931121826, 0.3775738775730133, 0.40221843123435974, 0.5155473351478577, 0.6871378421783447, 0.36985957622528076, 0.4444774091243744, -0.21929647028446198, 0.5644184350967407, -0.10225122421979904, -0.06550837308168411, 0.1972229778766632, 2.3073863983154297, -1.4591786861419678, -0.46659308671951294, -0.25178998708724976, 0.4394550919532776, 0.030403653159737587, 1.6543138027191162, -1.5394114255905151, 1.8360337018966675, -1.2312308549880981, -0.6932327747344971, 0.5829170346260071, 0.10989007353782654, 0.0028024918865412474, 0.1010860726237297, 0.41663408279418945, 0.990801215171814, 0.4378989636898041, -1.2280985116958618, -0.7072792053222656, 1.4000340700149536, 1.9535826444625854, 0.9641135931015015, 1.0990877151489258, 0.3367008864879608, 2.1004538536071777, -0.5456416606903076, -0.30883529782295227, -0.7497166395187378, 1.592954397201538, -0.49618202447891235, 0.5355872511863708, -1.3538793325424194, -1.4378212690353394, 1.0183064937591553, 0.7379837036132812, 0.878753125667572, 0.23666289448738098, -0.2587861716747284, -0.3739846348762512, -1.0723695755004883, -0.34636470675468445, 1.3794361352920532, -0.12160094082355499, -1.2731781005859375, -0.20390133559703827, -0.9462082386016846, -1.1088943481445312, -0.07637085020542145, -0.7296434640884399, -1.2250570058822632, 1.5490106344223022, 1.7468217611312866, 0.17093303799629211, 1.5040730237960815, -0.4287012815475464, -0.0072801001369953156, 1.3300824165344238, 0.3224194347858429, 0.8401212096214294, -2.7249093055725098, 0.3085741102695465, -0.4050181806087494, -0.09118630737066269, -1.5000832080841064, -0.7580959796905518, 0.6101497411727905, 0.5960333347320557, 0.7825926542282104, 0.85992032289505, 1.1054673194885254, 0.3939518630504608, -0.22872893512248993, -0.5660899877548218, -0.41188210248947144, -0.9606953263282776, -0.9880048036575317, -0.08396982401609421, -2.273559093475342, -1.2232555150985718, -0.7990161180496216, 0.5299466848373413, 1.0802807807922363, 0.33431708812713623, -0.03870897367596626, -0.48555073142051697, 1.118587613105774, 0.7146350145339966, -0.6497265696525574, -0.017248759046196938, 0.4223219156265259, 0.9239057898521423, -0.3174450993537903, 1.5150575637817383, 1.1879867315292358, -0.40283674001693726, -0.17259639501571655, -0.8989632725715637, -0.8745676279067993, 0.34387657046318054, 1.4575897455215454, 1.4731487035751343, 0.2334730476140976, -0.32460805773735046, -0.594416618347168, 0.48383259773254395, -0.4040856659412384, 0.772034227848053, -0.05897550657391548, 0.355744332075119, -0.45271843671798706, 2.0842161178588867, 1.2032252550125122, 0.15742062032222748, -0.8073006272315979, 0.633104145526886, 0.824955940246582, -1.0576523542404175, 0.14946265518665314, 2.1114773750305176, 0.2559845745563507, -0.3155668377876282, 0.6699381470680237, 1.8983629941940308, -0.35976630449295044, -0.3484162986278534, -0.1523588001728058, 0.16521680355072021, 1.2786155939102173, 0.42656999826431274, -2.1184041500091553, 0.5315694212913513, 0.4979761242866516, 0.7367697358131409, 0.37691470980644226, -0.020146511495113373, 0.6139426231384277, -0.5744167566299438, -1.5768693685531616, -0.2356051802635193, -0.9987313151359558, 1.7751117944717407, -0.02259250357747078, 1.6441134214401245, -1.2590882778167725, -1.1267911195755005, 0.5007064342498779, -1.6315810680389404, -0.5709758400917053, -1.868010401725769, 0.4814133644104004, -1.660117745399475, -0.08186803758144379, -1.711890697479248, 0.5296099781990051, 1.1102181673049927, 0.16424964368343353, -0.9433190822601318, 0.46220409870147705, 1.0122227668762207, 0.7989479303359985, 1.2946593761444092, 1.9302805662155151, 0.2597943842411041, -0.22692182660102844, -0.6023266315460205, 0.018776608631014824, -0.20093780755996704, -0.499337375164032, 0.36964184045791626, 2.00644850730896, -0.7636719346046448, -0.45962753891944885, -0.331974059343338, -1.0034511089324951, 1.0679951906204224, -0.13810276985168457, 0.6084507703781128, -1.5930707454681396, 0.5707660913467407, -0.11970788985490799, -1.4413307905197144, 0.8660524487495422, 0.9487757086753845, 0.2321598380804062, 0.47581371665000916, -0.9398449659347534, 1.0351611375808716, -0.22718358039855957, 0.48386818170547485, 0.6052852272987366, 0.4902229905128479, -0.7192102670669556, 0.28369516134262085, 0.15395978093147278, 1.1887308359146118, -0.7401849627494812, 0.9670329689979553, 0.4880593419075012, 0.0675697848200798, 1.8977575302124023, 0.8222788572311401, -0.5326683521270752, 0.3140498995780945, -0.7374113202095032, -1.153651237487793, -0.48984450101852417, -0.4775482416152954, 1.7176231145858765, 0.810215175151825, -2.4088127613067627, -0.14535285532474518, -0.6888182759284973, -0.13287459313869476, 0.1854962557554245, 0.04844951257109642, 0.4300859868526459, -0.7742994427680969, 1.4190995693206787, -0.12620404362678528, 1.237034559249878, -0.09160531312227249, 0.5015392899513245, -0.9523482322692871, 1.1138787269592285, 0.29490765929222107, -0.39645102620124817, 0.38532671332359314, 0.4753826856613159, -1.5354543924331665, -0.7465856075286865, -0.9922381639480591, 0.8310493230819702, 0.18762171268463135, -0.6644507050514221, -0.7077614665031433, -0.039350010454654694, -2.2990148067474365, -0.16877107322216034, 1.5640983581542969, -1.4035494327545166, -1.1968687772750854, -1.1314923763275146, 0.4406205415725708, -0.7534507513046265, 1.870538592338562, -0.20240260660648346, -0.49794867634773254, -1.1012840270996094, 0.6805511116981506, 1.071608543395996, 0.17080950736999512, 1.3044053316116333, 1.242796540260315, 0.3613174557685852, 0.06728771328926086, -0.5427314639091492, 0.22998665273189545, -0.33789461851119995, 1.489331841468811, 0.767981767654419, -0.15290775895118713, -0.3733164966106415, 0.43045011162757874, 1.0660746097564697, 0.30078452825546265, 1.0407698154449463, -0.42374515533447266, 0.2492169290781021, 0.678574800491333, 0.2271493375301361, 0.4186370074748993, 1.0504661798477173, 0.03492581099271774, 1.037218451499939, -0.4703746438026428, 0.5396657586097717, -0.3772940933704376, -0.32690781354904175, 0.655674934387207, 1.3336074352264404, -0.058609090745449066, 0.16296041011810303, 1.394927978515625, -0.7243450284004211, 0.9893326759338379, -1.0811567306518555, -1.8002487421035767, 0.4312465190887451, -0.20289508998394012, -0.42567312717437744, -2.3121469020843506, 0.4457888603210449, -0.38042548298835754, -1.0975983142852783, -0.03929280489683151, -1.119173288345337, -0.7006880640983582, -0.743288516998291, 1.036751627922058, -0.0502912774682045, 1.3461846113204956, -0.7511456608772278, -1.1358426809310913, 1.8109080791473389, -1.326167106628418, 1.2654954195022583, -1.0491005182266235, 0.7671260833740234, -0.6859959959983826, 1.5593405961990356, 2.025953769683838, 1.0971591472625732, 0.4107394814491272, -0.30582094192504883, 0.8485366106033325, 2.1631383895874023, -0.38581642508506775, -1.0673532485961914, 1.855400562286377, -0.15129543840885162, 0.8064190745353699, 0.2669338285923004, 0.34848177433013916, -0.1976155787706375, -0.056623492389917374, -1.589194893836975, -0.4844615161418915, -0.7823276519775391, 1.2475695610046387, -0.22885139286518097, 0.19063137471675873, -1.5898244380950928, 0.15996578335762024, 1.609471321105957, 0.2534945607185364, 0.9643060564994812, -0.22611163556575775, -0.5263950824737549, 1.046466588973999, -0.3636203408241272, -0.9595946073532104, -0.015127908438444138, -0.29786303639411926, 1.1378110647201538, 0.8100088834762573, -1.3530939817428589, 0.596145510673523, -1.2496598958969116, 0.2651059925556183, 0.6556275486946106, -0.4312736392021179, -0.4396377205848694, 0.8494310975074768, -0.21529564261436462, -1.0392838716506958, 1.7186962366104126, 0.9644219279289246, 0.5621758699417114, 0.4356314241886139, 1.1098506450653076, 0.17943821847438812, 0.8839022517204285, 0.42547059059143066, -0.5741647481918335, 1.1636534929275513, 0.2796519100666046, -0.6976014971733093, -0.3311852514743805, -2.2853822708129883, -1.5536330938339233, -1.348146915435791, 0.7956990599632263, 0.762464165687561, 1.2402788400650024, 0.5089648365974426, 1.8096579313278198, -0.37223732471466064, -0.6800718307495117, -0.8567442297935486, 0.26151347160339355, -0.5864918231964111, -0.8570237755775452, -0.4819689989089966, 0.20421355962753296, -1.241112232208252, 0.03054048679769039, 0.018350427970290184, 1.7065799236297607, 0.18260003626346588, -1.2554631233215332, 0.28380337357521057, 0.2198420614004135, -1.2326818704605103, -3.0200252532958984, 0.7750603556632996, -0.16169552505016327, -0.6993016600608826, -0.15083551406860352, 1.3017535209655762, 1.7662708759307861, -0.4445127844810486, 1.545525074005127, -0.029086990281939507, 0.02635383978486061, -0.07475262880325317, -0.04334151744842529, -0.12689724564552307, -0.6685713529586792, 0.03407662361860275, -0.9958080649375916, 0.577201247215271, -0.6321569681167603, 0.5038849115371704, -0.10716787725687027, -1.7749274969100952, 1.1907894611358643, 0.30621784925460815, -0.05482300743460655, -0.32525354623794556, 1.8049708604812622, -0.23402263224124908, 0.9681760668754578, 0.21120670437812805, -1.3210361003875732, 0.728491485118866, -1.5151286125183105, 0.4537009000778198, 0.41374462842941284, 0.2375398874282837, -0.7435579299926758, 1.397802472114563, 1.7701517343521118, -0.9155606031417847, -0.38045865297317505, 1.6291191577911377, -0.4323292672634125, 0.961184561252594, -0.9784068465232849, -0.633400559425354, 0.26013097167015076, 0.45214128494262695, -1.4457045793533325, -1.0499062538146973, 0.5112688541412354, 1.2757476568222046, 0.5147407054901123, 0.35967937111854553, 0.43580010533332825, -1.2190502882003784, 0.3962579369544983, 0.8991712927818298, 0.5954474210739136, -1.166929006576538, -0.958499014377594, 1.1200820207595825, -1.384940266609192, -1.0681498050689697, 1.583078384399414, -0.9018523693084717, -0.7317481637001038, 0.6386840343475342, 0.7848655581474304, -0.16740551590919495, -1.4323614835739136, -0.5178028345108032, 0.16304154694080353, -0.9197327494621277, -0.9756007194519043, 1.2500271797180176, 0.5410477519035339, 0.7814403176307678, 0.8961166143417358, -1.0051637887954712, 0.4836586117744446, 1.1929501295089722, 0.679314911365509, -1.1566542387008667, -1.5547057390213013, -0.803153395652771, 0.5467050075531006, -0.18544551730155945, 0.278155654668808, -1.2629302740097046, 0.8317814469337463, -0.7319740653038025, 1.1288713216781616, -0.02692096121609211, -2.983304738998413, -0.20099715888500214, -0.8784583210945129, -0.8480167388916016, -0.16243065893650055, 1.0687962770462036, 0.843956470489502, 1.3950250148773193, -0.006775322370231152, -1.8736416101455688, -0.5627657771110535, -0.9102773070335388, -0.6916677355766296, 2.660832643508911, 0.7955988645553589, -0.9716129899024963, 1.1903032064437866, -0.2986043393611908, -1.3516161441802979, 1.168671727180481, -0.479875773191452, 0.12517546117305756, 2.297947645187378, -1.0188980102539062, 1.0360150337219238, -0.31614920496940613, -1.1174938678741455, -0.25544971227645874, 0.17025402188301086, 0.18396884202957153, -1.5423492193222046, 0.8352594971656799, -0.8572018146514893, 0.3042236566543579, 0.20090848207473755, -0.5590395331382751, 0.5946524143218994, -0.7217796444892883, -0.949777364730835, -1.4248555898666382, -1.867972493171692, -0.7026321887969971, -2.198394775390625, -0.3678354322910309, -0.5476751327514648, 1.6355128288269043, -1.1710472106933594, 1.1364130973815918, 0.3579765260219574, 0.28504258394241333, -0.6431961059570312, -0.8386884927749634, -0.33704936504364014, -0.0642470046877861, 0.5602267384529114, -0.5295238494873047, 1.5474653244018555, 1.6693686246871948, -2.483076333999634, 0.12001407146453857, -0.13005711138248444, -0.9967870116233826, -2.704066038131714, -1.7473961114883423, -0.2014160305261612, 0.6785237789154053, 0.4372639060020447, 0.5011740922927856, -0.23343618214130402, 1.0114980936050415, 0.4695163667201996, 1.1200000047683716, -0.14076656103134155, -0.665596067905426, -1.5401561260223389, 1.5397344827651978, -0.5880365967750549, -1.0492141246795654, -2.1425108909606934, 0.5628193020820618, -1.3827499151229858, 0.0653817355632782, 0.2795105278491974, 1.4245837926864624, -0.7208463549613953, -1.949139952659607, 0.8661922216415405, 0.2561570405960083, -0.6391749978065491, -0.03710775822401047, 1.8843717575073242, 0.05465118587017059, -0.6617418527603149, -0.08113501965999603, 0.5789934396743774, 0.36482372879981995, 0.33901694416999817, 0.32801559567451477, -1.1705477237701416, 0.6045888066291809, 1.042079210281372, 0.6626666784286499, -0.06385974586009979, -0.07897458970546722, 0.19271844625473022, 0.06323280930519104, -1.649280309677124, -0.12423093616962433, 1.2666898965835571, -0.3213159441947937, -0.5224445462226868, 0.6217910051345825, 0.09953209012746811, -0.6863774657249451, 0.18987660109996796, 1.7299407720565796, 1.8567564487457275, 0.17497456073760986, -0.14183993637561798, 1.2208938598632812, 0.4903475344181061, 0.574055552482605, -0.03750361502170563, -1.276310682296753, -1.4215664863586426, -0.05281725898385048, -1.065730094909668, 0.8345025181770325, 0.9644421339035034, 1.5105040073394775, 0.14101266860961914, 1.9865413904190063, 1.3150159120559692, 0.42888861894607544, 0.80147784948349, -0.5164341330528259, -3.118182897567749, 1.0983068943023682, 0.5442060828208923, 1.736991286277771, 0.5411150455474854, -0.12752681970596313, -0.4405460059642792, -0.5650120377540588, 0.21583804488182068, -1.7724534273147583, -1.0814930200576782, 0.8762532472610474, -2.0076615810394287, -1.5287182331085205, -0.5604720711708069, -0.37819704413414, 0.09841744601726532, 0.8811007142066956, 0.31697705388069153, 0.623593270778656, -0.559839129447937, 0.051346153020858765, -0.6860194206237793, -0.08713014423847198, 0.6069824695587158, 0.5255100131034851, -1.87220299243927, -1.1198173761367798, 2.0182058811187744, -2.0275206565856934, 0.08992525190114975, -1.149557113647461, -0.192964568734169, -0.5637987852096558, 0.09119017422199249, -1.088201880455017, 1.069815993309021, 2.21403169631958, -2.4384334087371826, -0.8260675072669983, -1.5027443170547485, 0.14151571691036224, 1.4273366928100586, 1.4152957201004028, 1.3436318635940552, 0.24831610918045044, -0.536372184753418, 1.1526192426681519, -0.592350423336029, -0.7679307460784912, 0.08941148966550827, -1.220232367515564, -0.16935715079307556, -0.920637845993042, 0.12135239690542221, 0.992047905921936, -0.5344865918159485, 0.4972158968448639, 0.9880134463310242, -0.7551485896110535, -0.049712300300598145, -0.3396786153316498, 1.8394063711166382, -0.4816047251224518, 0.14651602506637573, -1.3879245519638062, -0.9070892930030823, 0.6201841831207275, -0.26400578022003174, 0.7687034606933594, 0.09005006402730942, 2.6467742919921875, -0.21276643872261047, 0.9800443649291992, -0.6944661140441895, -0.4127323031425476, -1.5103859901428223, 2.6688411235809326, -2.077820062637329, 0.14751334488391876, -0.41501426696777344, 0.8284316062927246, -0.5147935152053833, 1.9319367408752441, 1.176098108291626, 0.5752525925636292, 0.5455314517021179, 1.1120167970657349, 0.4912285804748535, 1.659765362739563, 0.9524226188659668, 0.030035853385925293, 0.5216901302337646, 0.2936675250530243, 1.282731056213379, -0.16396719217300415, 0.20250225067138672, -1.1694313287734985, 0.7778228521347046, 0.34858885407447815, -0.5647812485694885, 1.8266050815582275, 0.13034836947917938, -1.0519931316375732, 1.2555075883865356, -0.0755084976553917, -3.0023438930511475, 0.45173877477645874, 0.08945809304714203, 0.18226958811283112, -0.10465089976787567, -0.7131412625312805, 0.22052015364170074, -1.5977740287780762, -0.08372026681900024, 0.9634135961532593, 1.0978418588638306, -0.40297722816467285, 2.037614583969116, 0.40261754393577576, -0.3468180298805237, 0.25993403792381287, -0.5926457047462463, 0.9643853306770325, -0.8455010056495667, -0.6122591495513916, -1.0585112571716309, -0.7160094380378723, -0.18803143501281738, 1.9595186710357666, 0.33877381682395935, 1.2755241394042969, 0.1449190229177475, 1.0415313243865967, -0.9555771350860596, -1.035093069076538, 1.9607577323913574, 0.4695088565349579, -1.4889787435531616, 0.6967544555664062, -0.10883867740631104, -0.18226486444473267, 0.9165145754814148, -1.1422555446624756, 1.3184292316436768, 0.3490830063819885, -0.39783433079719543, -1.265401840209961, 1.844338297843933, -0.28609588742256165, -1.302280068397522, -0.07707741856575012, 0.5476939082145691, 0.7013795375823975, 1.5576955080032349, 0.636326253414154, 0.01624738797545433, 2.320136785507202, 2.0990424156188965, -1.0546531677246094, 1.0578608512878418, -0.5170668959617615, 1.583986759185791, -0.8697493076324463, 0.38141393661499023, 1.6703126430511475, 0.4210970401763916, 1.1836241483688354, -0.6241598725318909, -1.164780855178833, -1.099281668663025, 0.6985853314399719, -0.5676637291908264, 1.5615769624710083, -0.2562851309776306, 1.438626766204834, 0.43415161967277527, -1.008139967918396, 0.1386002153158188, 0.5796401500701904, 0.5829049944877625, -0.014744335785508156, 1.2071722745895386, 1.0491958856582642, 1.0417866706848145, -0.6953059434890747, -2.0423152446746826, 0.6714187860488892, -0.7508044838905334, -0.4588521122932434, -0.16163498163223267, 0.4102363586425781, -1.677219033241272, -0.04861164093017578, 1.4559720754623413, -1.3821014165878296, -0.21119609475135803, 1.5442267656326294, 0.9272860884666443, 0.7106500864028931, 0.24335119128227234, -1.3469922542572021, 0.20281638205051422, -0.8722490668296814, 0.21567144989967346, -0.6804844737052917, -1.0944935083389282, 0.08393443375825882, 1.1605384349822998, -0.4103006422519684, 0.27684521675109863, 0.813885509967804, 0.6065249443054199, -1.1890093088150024, -0.4749634265899658, 0.4454396367073059, 0.11049844324588776, 1.8578436374664307, 0.679482102394104, -0.8042291402816772, 0.7697632908821106, 1.2138804197311401, 0.7421479821205139, 0.5988601446151733, -1.3343217372894287, 0.026255056262016296, 0.682756245136261, -0.310396671295166, -0.2561720609664917, 0.5350619554519653, -0.5647339820861816, -0.8186501264572144, -0.07931062579154968, -0.016269981861114502, 0.9288008809089661, -0.0957370474934578, -1.0884650945663452, 0.5529963374137878, -0.7272505760192871, 0.8126926422119141, -0.508187472820282, 0.6492453813552856, 0.3831597566604614, -0.1258544623851776, -1.2019609212875366, -0.3282935619354248, 1.4864698648452759, 0.2596432864665985, 1.2317891120910645, 0.1379862129688263, 0.07829545438289642, -0.9018182754516602, 0.4542944133281708, -0.7724809646606445, -0.7655104994773865, 1.111363410949707, -0.9317512512207031, 2.370243549346924, 0.06538983434438705, -0.49641308188438416, 1.4049818515777588, -0.9127324819564819, 0.1981116086244583, 0.10118179023265839, 1.0183123350143433, 1.798588514328003, 0.4112475514411926, -1.225598692893982, -1.261889934539795, -0.42825639247894287, 0.8730645775794983, 0.1969636082649231, 0.3364540934562683, -0.2508205473423004, 1.4187977313995361, -1.0688376426696777, 1.990577220916748, 1.26265287399292, 0.7657533884048462, 0.039348162710666656, -0.23543761670589447, 0.05939410254359245, -0.48150646686553955, -0.36307594180107117, -0.7521176934242249, 0.8223510980606079, 0.15944361686706543, 0.501099705696106, -0.6747158169746399, -1.1779736280441284, -0.8672982454299927, 0.2819685637950897, -1.1254199743270874, 0.6416962146759033, -0.8955376148223877, -0.4036557972431183, 0.2831224799156189, -1.1471447944641113, -2.0295753479003906, 1.191953420639038, -0.1358472853899002, -0.6067324280738831, 1.6278566122055054, 2.2002360820770264, 0.05231134220957756, 0.341178834438324, 2.657649278640747, -0.5864152908325195, 1.229604959487915, 1.0042905807495117, -0.5409359931945801, -0.9458861947059631, -0.7368704676628113, 0.17119915783405304, 0.37926244735717773, 2.640115737915039, -0.9913631081581116, 0.2300710380077362, -0.7514102458953857, 0.9043951630592346, 0.46724733710289, -1.2744417190551758, -0.4049414098262787, 0.10765296220779419, 0.772471010684967, 0.5813251733779907, 0.7263111472129822, 0.07743563503026962, 0.6570204496383667, 0.13751058280467987, 1.0316225290298462, 0.6429873108863831, -0.8199806809425354, 0.015138978138566017, -0.7021170258522034, -0.05452507361769676, 0.632208526134491, 0.16194570064544678, -0.04425327479839325, -0.5379634499549866, 1.5462924242019653, 0.07825995236635208, -0.8941624760627747, -0.4053542912006378, 1.0155236721038818, 2.0947463512420654, 1.3427225351333618, -0.2598535716533661, 0.9273083806037903, -1.5896728038787842, 0.9028159379959106, -0.1547987461090088, 0.337981641292572, -1.156127691268921, 0.4501473307609558, -1.1998841762542725, -1.5599243640899658, 1.5622144937515259, -1.1675564050674438, 0.9930763840675354, -0.4901351034641266, -0.039744727313518524, 0.06694691628217697, 1.1796844005584717, 0.5012404322624207, -0.8121668100357056, -0.44964227080345154, 1.3879607915878296, 1.4151084423065186, -0.4072864353656769, 0.9538639187812805, -0.005431880243122578, 1.0714361667633057, -0.9404116868972778, 0.8740315437316895, -0.4989400804042816, -1.2197439670562744, 0.6629244685173035, 0.6253814697265625, -1.4380460977554321, -2.2581748962402344, -0.06952077895402908, 0.4943099915981293, -0.48744890093803406, 0.13446544110774994, -0.3825136721134186, -0.5112877488136292, 0.4277516305446625, 1.2649694681167603, 0.49866095185279846, -0.28250616788864136, -0.6538451910018921, 0.23434822261333466, -2.171501398086548, 0.1375926434993744, -0.08335935324430466, 0.006405659485608339, -0.24392864108085632, -0.17330209910869598, -1.4279048442840576, 0.6775554418563843, -0.29543396830558777, 0.038633182644844055, -0.4739074409008026, -1.4527223110198975, 0.6427724957466125, -0.19842591881752014, -0.23208148777484894, 0.40083184838294983, 1.7148016691207886, -0.01067575067281723, 0.5707886815071106, 1.8718814849853516, -0.27517208456993103, 1.247967004776001, 0.07521532475948334, 1.0412615537643433, 0.15740154683589935, 0.9957903027534485, -0.701547384262085, -0.4429342448711395, -0.573255717754364, -0.25896376371383667, 1.141019344329834, 0.47779518365859985, 0.7084728479385376, -0.16866256296634674, 0.1391945481300354, 0.46999624371528625, 0.408950537443161, -0.5055137276649475, -0.33724498748779297, -1.0063786506652832, 0.8035193681716919, 1.1075878143310547, 1.9465858936309814, -3.022599220275879, 0.8855026960372925, 0.6209810376167297, 2.6916885375976562, -0.40931326150894165, -0.7286598682403564, 0.261883020401001, -1.3990834951400757, -1.3640228509902954, 0.6135121583938599, -1.6320226192474365, -1.0834375619888306, 1.2185560464859009, 0.616369366645813, -1.2684286832809448, 1.4197355508804321, 0.09837165474891663, -0.3274315595626831, 0.16301009058952332, -0.07891504466533661, 1.179919719696045, 0.5907275676727295, -0.8738268613815308, -0.5202476382255554, 0.7822255492210388, -0.6043688654899597, -0.9650158882141113, -2.3792288303375244, 0.8469629287719727, 0.008675879798829556, -0.20740407705307007, -0.8806806802749634, -0.5542364120483398, -0.0361345037817955, -1.1340765953063965, 0.09976045042276382, 1.7121368646621704, -0.5213478803634644, 0.0158638097345829, 1.5823760032653809, 0.10097140073776245, 1.5719095468521118, 1.0225367546081543, 0.9305958151817322, -0.16413745284080505, -0.4539240300655365, 1.333112120628357, 0.14839796721935272, -0.10384184122085571, -2.1861491203308105, -0.13131318986415863, 2.5089852809906006, 0.6596968770027161, 1.8052024841308594, 0.26568523049354553, -1.3633335828781128, 0.18430054187774658, 1.817602515220642, 0.2254113405942917, -0.2435978353023529, -0.1254524439573288, -1.1273407936096191, -0.30678290128707886, 0.18408356606960297, 1.007245421409607, 0.8182203769683838, -1.0563197135925293, -1.8846975564956665, -0.30578288435935974, -0.7182192206382751, 0.8816930651664734, -0.9771134853363037, 0.6500548720359802, -0.3910331726074219, 1.180495023727417, 0.09764722734689713, -1.9364246129989624, -2.326242208480835, 0.33993229269981384, -0.9981257319450378, 0.9973939657211304, -0.020869823172688484, 1.8069126605987549, -0.01322134118527174, -0.7462223768234253, -2.008134126663208, 0.5002263188362122, -0.21392115950584412, -0.31688588857650757, 2.70031476020813, 0.7976053953170776, -0.9404081106185913, 0.6705132126808167, -1.549302577972412, -0.6488956809043884, 0.8580150604248047, -1.3166965246200562, -0.002242449903860688, 0.06336915493011475, 0.17389526963233948, -0.9803891777992249, -1.4209308624267578, -0.015031237155199051, 0.6358557939529419, -1.0020148754119873, 0.8084143996238708, 1.1043095588684082, 0.32442060112953186, 1.0261847972869873, 0.9863293766975403, 1.216808557510376, 0.27726149559020996, 0.30499300360679626, 0.3611016869544983, 1.2193549871444702, -0.49719586968421936, 1.5071804523468018, -0.9183217883110046, 1.9850945472717285, 0.6710376143455505, 2.741076946258545, 0.2953648865222931, 0.7170491814613342, -1.5355044603347778, -0.24160020053386688, 0.23035173118114471, 0.4512726664543152, 0.7002689838409424, 0.18588511645793915, -0.020422454923391342, 0.986422598361969, -1.4825314283370972, -0.4149706959724426, 0.7296224236488342, -0.04177474230527878, 1.2389990091323853, -0.6432457566261292, -0.3435799777507782, 0.6666651368141174, 1.8195815086364746, -1.2647656202316284, -1.6835395097732544, -1.1772680282592773, -1.2185548543930054, 0.29450681805610657, 0.9712257385253906, 0.01290073711425066, 0.2720201015472412, 0.6628457307815552, 0.41368475556373596, 0.32981154322624207, 0.270978718996048, 1.1760356426239014, -1.0972440242767334, -0.4595486521720886, 0.3020695149898529, 2.744908332824707, -0.8269578814506531, 0.8197419047355652, -0.7017862200737, -0.12611179053783417, -0.7437007427215576, 1.9193602800369263, 0.42006969451904297, -0.459930419921875, 1.0233185291290283, -0.6631523370742798, 0.31333446502685547, -1.2996395826339722, -0.26468712091445923, 0.38684210181236267, 0.384397953748703, -0.8066387176513672, -0.7274262309074402, -1.7075073719024658, -2.084909677505493, -0.19645905494689941, 1.1618103981018066, 0.5110016465187073, 0.6180379390716553, 0.04809848964214325, 0.012796456925570965, -0.0036893223877996206, 1.7159843444824219, -0.29239246249198914, 1.2473583221435547, -1.3711719512939453, -1.0330928564071655, -0.030937839299440384, -1.4534504413604736, 1.5877515077590942, 0.28032374382019043, 0.46985962986946106, -0.40859249234199524, 0.5461751222610474, -0.5864737033843994, 1.4033607244491577, 1.3743858337402344, -0.5971161723136902, -0.7790312767028809, -1.4193953275680542, -1.368478536605835, 1.3592506647109985, 0.8901622295379639, 1.6870664358139038, 0.6417639851570129, 1.0144795179367065, 0.8079535961151123, -0.34841135144233704, 0.34505653381347656, 1.0906347036361694, 0.29581889510154724, -0.808639407157898, -0.8305936455726624, 3.1173787117004395, -0.7110452055931091, -0.5067533254623413, 0.22049134969711304, 0.07538223266601562, 1.1806684732437134, -0.2449997514486313, 0.8252784609794617, 0.2478552609682083, 0.25034207105636597, -0.01222191657871008, -0.7047547698020935, -0.348072350025177, -0.9887223243713379, -0.27914971113204956, 0.2795493006706238, 0.38812455534935, -0.9812436103820801, -0.5982367992401123, -0.18218640983104706, 1.467629313468933, 0.3585329055786133, 0.5158430337905884, 1.3789113759994507, 0.6407270431518555, -0.3610503077507019, 1.8214436769485474, 0.02097543701529503, 1.3352537155151367, 0.5559730529785156, 0.7857832312583923, 1.4477472305297852, -0.2973117232322693, -1.2787977457046509, 0.4580390751361847, 0.08680824935436249, -0.12419930845499039, 0.14738775789737701, -1.4336656332015991, 0.11309107393026352, -0.1667010337114334, 0.4109669029712677, 0.01962805911898613, -0.252989798784256, 0.7525773644447327, 1.1807031631469727, 0.009176774881780148, -0.7648316025733948, -0.5947780013084412, 1.282982349395752, -1.5955275297164917, 0.09632227569818497, 0.3107394874095917, 0.49746307730674744, -1.9795383214950562, 0.22296489775180817, -2.180830240249634, -0.5805161595344543, -1.0273971557617188, -2.083286762237549, 1.620255947113037, -0.480985552072525, -1.7515766620635986, -1.2753428220748901, -0.31034770607948303, 0.5307537913322449, -0.14811673760414124, -1.3603079319000244, 1.409977912902832, 0.2054949551820755, 1.5031423568725586, 1.2679322957992554, 0.21525603532791138, 1.2690597772598267, -0.1754191517829895, 0.9741461277008057, -1.6886873245239258, -1.8811297416687012, -1.1591875553131104, 0.45454639196395874, 0.07814662903547287, -0.7875313758850098, -0.6560848355293274, -0.6241730451583862, 0.0701427310705185, 1.0197868347167969, 0.9252178072929382, -0.246258944272995, -0.0731838271021843, 0.5370272994041443, -1.1194283962249756, -0.037257060408592224, 0.5378785133361816, 0.4163036048412323, -0.48437991738319397, -0.8481769561767578, -0.9332056641578674, 1.3815529346466064, 0.004185743164271116, -1.2492139339447021, -2.0324342250823975, -0.0012230048887431622, 1.484907627105713, -0.2852284014225006, -0.8980448246002197, 0.7887653112411499, -0.333074152469635, -0.15907470881938934, 0.2706158459186554, -0.3367537260055542, 0.6610546708106995, 0.803743839263916, -0.10491452366113663, -0.5684873461723328, 0.2959232032299042, -2.5477612018585205, -0.43735185265541077, -0.574709951877594, 0.7080899477005005, -1.0851657390594482, 1.0531055927276611, -0.1559278517961502, 0.3917694091796875, 1.2776176929473877, 0.13758280873298645, -0.6245824694633484, 0.2689867615699768, 0.2914106845855713, 1.9139188528060913, -0.0689375251531601, 0.9147412180900574, 1.3252241611480713, 1.1022758483886719, -0.22185564041137695, -1.1573714017868042, 1.0608969926834106, 1.7141567468643188, 0.6813824772834778, 1.2061262130737305, 0.19358403980731964, 0.890209436416626, 1.3827964067459106, 0.07748039066791534, 1.2969330549240112, 0.04575445130467415, -0.830608069896698, 0.469716876745224, -0.07496832311153412, 0.5358691215515137, 0.6505110859870911, -0.08989182859659195, 0.4967024028301239, -0.3564531207084656, 0.8994153141975403, -0.8239049911499023, -1.280012845993042, 1.1010522842407227, -1.8915107250213623, -0.19339871406555176, -0.1414763629436493, 0.8939318060874939, -0.1832791119813919, -0.1838146150112152, -1.7774192094802856, 0.9990590214729309, -1.022274374961853, -0.005682750139385462, -0.777435302734375, 1.1304495334625244, -0.9250909686088562, -1.3490660190582275, 1.298081874847412, -0.6556217670440674, 0.22845375537872314, 0.4188864231109619, -1.5086030960083008, 1.0030704736709595, -1.0337074995040894, 0.061122260987758636, 0.4351845383644104, -0.08686766773462296, -0.8916633725166321, 0.929705023765564, -0.8886168599128723, -1.322123646736145, 1.171439528465271, 1.230987787246704, -1.878366470336914, -0.22098511457443237, -0.012426040135324001, -2.8408007621765137, -0.5758945941925049, 1.137115240097046, -1.0194538831710815, 1.3009381294250488, 1.973767876625061, -0.1630055457353592, -0.743348240852356, 0.05804387107491493, -1.4529953002929688, 1.2814292907714844, 0.4957069754600525, -1.427046298980713, -0.8022620677947998, -0.3241294324398041, 1.0888526439666748, 0.836169958114624, -0.6527667045593262, 0.17185263335704803, -0.44802284240722656, 0.4788268506526947, -0.27778080105781555, 2.1177985668182373, 0.46831974387168884, -0.26938575506210327, -1.0667262077331543, -1.4131758213043213, 0.7295089960098267, 0.8277548551559448, 1.5849480628967285, -1.0354958772659302, 0.7566903829574585, -1.2410223484039307, 0.6890734434127808, 0.573789119720459, 0.23780126869678497, -0.3094487488269806, -0.486259788274765, -0.37309759855270386, -1.9478912353515625, 0.67911696434021, -1.1434104442596436, 0.15981535613536835, -0.05379559099674225, 0.725807249546051, 0.07691055536270142, 0.02950834482908249, 0.30519717931747437, 1.0108662843704224, -0.3782815635204315, -0.8681812882423401, 0.0511961355805397, -0.08230289816856384, 0.7649745941162109, 0.4328940808773041, 0.19973082840442657, 1.1731125116348267, 0.34558340907096863, 0.675385057926178, -0.3522491157054901, 1.5677682161331177, 0.7143169641494751, -0.8778802156448364, -0.8327956199645996, 1.8438302278518677, 0.2975374460220337, 1.8098818063735962, 2.439117670059204, 0.2193886637687683, 0.38346973061561584, 0.8977211713790894, -0.8163151144981384, -1.7417490482330322, 1.9581966400146484, -0.7247838973999023, -0.04142562672495842, 1.5978004932403564, 0.17470817267894745, 1.0384231805801392, 0.4251341223716736, -1.183004379272461, -0.6662361025810242, 0.19472649693489075, -0.8675304055213928, 1.0816898345947266, 0.4670279026031494, -1.485260248184204, 0.5791328549385071, -0.4230351448059082, -0.22250095009803772, 0.8114640712738037, -1.786007285118103, -0.8571941256523132, 0.35326680541038513, -1.4082790613174438, 1.3620105981826782, -3.2235703468322754, 1.1367980241775513, 0.795490026473999, -0.6096446514129639, 0.11581864207983017, -0.70713210105896, 1.930295705795288, 0.01676533743739128, 0.5297750234603882, 0.3240806460380554, -1.3678534030914307, -1.9216625690460205, 0.19571027159690857, 1.455841302871704, 0.6762587428092957, -0.1477973461151123, 0.4661216139793396, -0.3326398432254791, 1.3233586549758911, 0.6402652859687805, 0.3462657034397125, 2.1564290523529053, -0.5545221567153931, -0.9038958549499512, -0.21247628331184387, -0.5039451718330383, -0.5528506636619568, 0.6496098041534424, 0.562119722366333, -1.7146941423416138, -0.6664971113204956, -0.5114247798919678, -1.850893259048462, 0.6288632154464722, 0.12625189125537872, 0.480479896068573, -0.6603986024856567, -0.6047845482826233, 0.23425008356571198, 0.062325816601514816, -0.47108694911003113, 0.4740225076675415, -0.7133165001869202, 0.9183318614959717, 0.39484554529190063, -0.6064939498901367, -1.3082178831100464, 0.27151912450790405, 1.0833890438079834, 0.01165634673088789, 1.4111902713775635, 0.7336990833282471, 1.2809360027313232, -1.5086115598678589, -0.6716793775558472, 0.09154467284679413, 0.22495202720165253, -0.1563691645860672, -0.4259496331214905, -2.1936912536621094, 1.1164088249206543, 0.751190721988678, -0.49839040637016296, 0.022532103583216667, -0.6546835899353027, 0.9935736656188965, -0.25065097212791443, 1.1508915424346924, -0.5228586196899414, -0.9375754594802856, -1.6657934188842773, 0.416054904460907, -0.648723304271698, 0.5414428114891052, 1.0652812719345093, 0.307271271944046, 1.4959256649017334, -0.4146920144557953, -0.8145109415054321, 0.27764415740966797, -0.16281543672084808, 1.5854729413986206, -1.7585898637771606, -1.1065566539764404, 1.2899729013442993, -0.803180992603302, -0.24482229351997375, -0.5156071186065674, -0.06362397968769073, 2.0720388889312744, 1.0703520774841309, -0.007531690411269665, 1.9056308269500732, -0.9686286449432373, 0.176248237490654, 1.9178740978240967, -1.2776687145233154, -0.9462248682975769, -0.955971896648407, -0.37081918120384216, 0.7791385054588318, 1.5112498998641968, 1.1446962356567383, 1.322918176651001, 0.3622460663318634, -0.541554868221283, -0.9122899174690247, 0.002183526288717985, -0.913764476776123, -1.2283377647399902, 0.542465329170227, -0.8754155039787292, -0.3201415538787842, 2.069988489151001, 0.48145902156829834, 0.5056977868080139, -0.2505156695842743, -0.9197723865509033, 0.5628900527954102, -0.4241677224636078, -0.18308478593826294, -0.6374444961547852, 0.3763900101184845, 0.22274883091449738, -2.2196829319000244, -0.9814381003379822, -0.2099340707063675, 1.8388869762420654, -0.9947884678840637, 0.3375232219696045, 1.0879923105239868, 1.0771023035049438, -0.5437743067741394, 0.8907426595687866, 0.23739676177501678, 0.05376880243420601, -0.5204508304595947, -1.803023099899292, -0.5493133664131165, -1.5938389301300049, 0.8351545333862305, 0.38948968052864075, 1.7228944301605225, -1.534836769104004, -1.6342179775238037, -0.2599266767501831, 0.13811154663562775, 0.8086139559745789, -1.226036548614502, -1.8970955610275269, -1.085283875465393, 3.300605297088623, -1.1031277179718018, -0.7220099568367004, 0.7452561259269714, 0.49810877442359924, -0.5398253798484802, -3.1994268894195557, 0.5183766484260559, -0.6529751420021057, 1.2164244651794434, -0.03439180552959442, 0.47351735830307007, -2.005758285522461, 0.007817280478775501, -0.15269735455513, -0.5296470522880554, 0.3557344377040863, 0.5215749740600586, -0.49737805128097534, 0.5819966793060303, 1.4654425382614136, 0.4993889629840851, -0.1841312050819397, 0.27192771434783936, -0.22531485557556152, 0.25572875142097473, 0.7854230403900146, -0.35654959082603455, 0.6618443727493286, 1.660173773765564, -0.22056207060813904, -0.7647355198860168, -0.4060629606246948, 0.6630030870437622, 0.3919092118740082, -0.5968066453933716, -2.6187360286712646, 0.4890161454677582, 0.43832284212112427, 0.41469696164131165, -0.7040902972221375, -2.2411065101623535, -0.38688182830810547, -0.772457480430603, 0.5731558799743652, 1.5451154708862305, 0.8699582815170288, -0.28885456919670105, 0.8150743246078491, -0.5147525668144226, 0.774875819683075, 0.10919222980737686, -0.4634404480457306, 0.4234868586063385, 0.19661743938922882, 1.438188910484314, -0.019295869395136833, -0.08849328756332397, 1.4673798084259033, 0.43405938148498535, 0.30220746994018555, 2.1054508686065674, -2.3599815368652344, -1.3833439350128174, -0.4919191598892212, 1.4510550498962402, 1.6868363618850708, 0.48839855194091797, -0.43337148427963257, -1.1848162412643433, 0.7935181856155396, 0.6275580525398254, -0.32427501678466797, 1.303299903869629, 1.9426275491714478, -0.6267013549804688, 0.5900862812995911, -1.0156712532043457, 0.5118036270141602, -1.0017659664154053, 0.3267207443714142, 2.1813621520996094, -0.3241310119628906, 1.7359906435012817, 1.6164398193359375, 1.4702837467193604, -0.14000746607780457, -1.18061363697052, 0.2884925603866577, 0.87519770860672, -0.33240559697151184, -0.6647538542747498, -1.0363606214523315, 3.231355905532837, -1.8582448959350586, 0.9991834759712219, -0.3084733784198761, 1.367713451385498, 1.4582114219665527, -0.8536527156829834, 0.1040748804807663, -0.15155692398548126, 0.736409068107605, 0.04138306900858879, 0.9767021536827087, 0.668829083442688, -0.4192309081554413, 1.2396997213363647, 0.6507455110549927, 0.6967126131057739, -0.25227710604667664, 0.8942233920097351, -0.23650552332401276, 1.4801119565963745, 0.5928309559822083, -2.1389002799987793, 0.7977235317230225, 0.7035781741142273, 0.2202211618423462, 0.987251877784729, -1.325478434562683, 0.5145424008369446, -0.5983438491821289, 0.378528356552124, -0.5617437362670898, -1.1301753520965576, -0.1791311353445053, -1.5344429016113281, 0.18851712346076965, 0.8742617964744568, 1.4413152933120728, 0.5508870482444763, -0.32710781693458557, -0.0483885183930397, 0.02827717736363411, 1.3887914419174194, -0.8073453903198242, -0.38403618335723877, -1.9343458414077759, 1.3846020698547363, -1.0681082010269165, -1.1452676057815552, -1.821010708808899, 0.3776572048664093, -0.15825296938419342, -0.2503286898136139, -1.2847799062728882, -0.44067302346229553, 1.2569974660873413, -1.4418058395385742, 0.20821894705295563, 1.4275099039077759, -1.0566538572311401, 0.7179573774337769, 0.7354019284248352, -0.7170832753181458, -0.019570140168070793, 0.728198230266571, 0.9398220181465149, -0.5282438397407532, -1.4046063423156738, 0.2257322371006012, 0.22266478836536407, 0.7346832752227783, 1.4495301246643066, 1.185120940208435, 0.9539763927459717, -0.22919951379299164, 1.3462656736373901, 1.2469595670700073, -0.2644408643245697, -0.39962276816368103, -1.2755473852157593, -0.3912849724292755, 0.10604032129049301, -0.13747794926166534, 0.692025899887085, 0.14151883125305176, -0.7692358493804932, 1.3322162628173828, -1.5929110050201416, 1.1265217065811157, 0.16440390050411224, 0.37914565205574036, -0.561843752861023, 0.23158057034015656, 1.716468334197998, -0.5230008959770203, 0.4197006821632385, 0.23364906013011932, -2.9572997093200684, -0.47725361585617065, 0.40757590532302856, -0.5801024436950684, -0.09796924144029617, -1.1183990240097046, 1.7162610292434692, 0.39009061455726624, -0.10636631399393082, -0.6637086868286133, 1.0307412147521973, 0.22040659189224243, 0.8949521780014038, 0.6844693422317505, -1.272091269493103, 0.5709240436553955, -2.308544158935547, -2.6406824588775635, -0.23886987566947937, 1.748420000076294, 0.5827145576477051, 0.5868804454803467, -0.831283450126648, -0.13493883609771729, 0.5361659526824951, -2.089913845062256, -0.7653740048408508, -0.4763548672199249, 0.9014745354652405, 1.229602575302124, -0.5839805006980896, 1.5176352262496948, 0.16222669184207916, -0.06325722485780716, 1.3406933546066284, -0.7634558081626892, -0.6397113800048828, -1.2340511083602905, -0.7788592576980591, -2.221921920776367, 2.488284111022949, 0.7014798521995544, 1.9129794836044312, -1.5365393161773682, -1.831802248954773, 1.4670580625534058, -0.2779097259044647, -0.34215447306632996, -0.48074328899383545, -1.3485223054885864, -0.734042227268219, 2.119088888168335, -1.0937628746032715, 1.46792733669281, 0.38787829875946045, 0.8205649852752686, 1.162490725517273, -0.9827112555503845, 0.8757071495056152, 0.4536117911338806, 2.746666431427002, 0.4512075185775757, 1.0482177734375, 0.2391623854637146, 0.40618932247161865, 0.31035521626472473, 0.009234639815986156, 1.0952003002166748, 0.35260236263275146, 0.5628849267959595, 0.937782883644104, -0.6196615695953369, -0.24927493929862976, 0.357412725687027, 0.5408771634101868, 1.8080708980560303, -2.0749099254608154, -0.749607503414154, -0.9511321187019348, 0.0850684717297554, -0.23895052075386047, -1.709924340248108, -0.9410361647605896, -0.9363610148429871, -1.7693899869918823, -1.0548179149627686, -0.6413883566856384, 1.0338715314865112, 0.08378145843744278, -0.5365338921546936, -0.7737843990325928, -0.24408479034900665, 1.1843680143356323, 1.2808606624603271, -0.8704200983047485, -0.37676674127578735, 0.1332108974456787, 0.2899605929851532, -1.0702589750289917, 0.5765387415885925, 0.8096770644187927, 0.8899059295654297, 1.8787288665771484, 0.9865936040878296, -0.1559108942747116, -1.419926643371582, -0.06226211413741112, -0.6617076992988586, 1.104094386100769, -0.08852441608905792, -1.2490501403808594, -0.5025092363357544, 1.4913378953933716, -0.6656333208084106, 0.1882135272026062, 1.0533355474472046, 0.34556710720062256, -1.3727366924285889, -1.4131646156311035, -0.8346819877624512, 0.44925224781036377, 0.2663756012916565, -0.7568382024765015, 0.011371023021638393, 0.23882469534873962, -1.6815943717956543, -1.2659319639205933, 0.06686481833457947, 1.1025456190109253, -0.19889360666275024, 0.18772520124912262, 1.4884920120239258, -0.8106886744499207, -0.02987673692405224, -0.2881856858730316, 1.8541910648345947, -0.5197080373764038, -0.8500446081161499, 0.984063982963562, 2.941011905670166, 0.9557345509529114, 1.3183172941207886, 1.338739037513733, 0.5710428357124329, -0.13785943388938904, 1.8556818962097168, 0.5640795230865479, -1.0602965354919434, -0.42747873067855835, 0.07436669617891312, -0.7656161785125732, 0.01915420964360237, -0.35148945450782776, -0.3571849465370178, -1.4221549034118652, 1.5407733917236328, -1.0759063959121704, -1.267290472984314, 0.6938391923904419, -0.163191556930542, -0.09963271021842957, 1.8423821926116943, -0.7013712525367737, -1.269592523574829, -0.8409980535507202, -0.6261940002441406, 0.2970099449157715, 0.5634037852287292, 1.1823716163635254, -0.40109992027282715, 0.517025887966156, -0.9443305134773254, -0.4681321680545807, 0.1562928706407547, -1.1318333148956299, 1.7320783138275146, -0.4181094765663147, -0.896774411201477, -0.9102271199226379, 0.24188244342803955, 1.717306137084961, -1.3495492935180664, 0.15143030881881714, -0.8831983208656311, 0.834039032459259, -0.5362573862075806, -2.2259268760681152, -1.1682049036026, -0.3086549937725067, -0.33190372586250305, 1.5183475017547607, 0.41749194264411926, -1.051293969154358, 0.7881765365600586, -0.14137689769268036, 2.0652332305908203, -0.28867900371551514, -0.6490789651870728, 0.29939067363739014, 1.7889533042907715, -0.39325469732284546, 1.5432764291763306, 0.07910846173763275, 0.2537718117237091, 1.3100544214248657, 1.119065523147583, 0.6159331202507019, 1.9124252796173096, -0.4499320685863495, -2.454457998275757, 0.7417799830436707, 0.231427863240242, -0.19995218515396118, 1.225999355316162, -2.2055985927581787, -0.30375638604164124, 1.310563325881958, 0.3109147846698761, 1.2712477445602417, 0.2703321874141693, -1.2198766469955444, -1.2181570529937744, 0.1788962334394455, -1.3295496702194214, 1.6056102514266968, -0.8961048722267151, -1.82364022731781, -0.2922292947769165, 0.504667341709137, -0.11795013397932053, -0.22850355505943298, -0.49848753213882446, -1.829942226409912, 0.11415518075227737, 0.767385721206665, -1.0313938856124878, 0.4766293168067932, -1.7563821077346802, 0.16852223873138428, -1.0866023302078247, 0.8074703216552734, -1.4986063241958618, -1.2568796873092651, -0.37685462832450867, -0.08961267024278641, -0.35765644907951355, 0.6653513312339783, -0.05698070302605629, -0.35984256863594055, 1.6284034252166748, 0.9158811569213867, 0.20308683812618256, 0.327973872423172, -1.8808190822601318, -1.306904673576355, 2.120358943939209, -1.5506370067596436, 0.9441961646080017, 0.39116573333740234, -0.437399685382843, 2.5407304763793945, -0.2983216941356659, 1.4734892845153809, -0.6977433562278748, 0.2806962728500366, 0.3104807436466217, 0.19660913944244385, 1.1373932361602783, 0.269223153591156, -1.3233532905578613, 0.17706887423992157, -1.7846089601516724, 1.3841500282287598, 0.964046835899353, 0.8791561126708984, -0.24478214979171753, 0.4822591245174408, -2.262791395187378, -0.6630811095237732, 0.11351140588521957, -2.3652944564819336, 0.13565871119499207, -0.4509366750717163, 0.26246944069862366, -0.633801281452179, 1.3326425552368164, -0.10827410966157913, 1.1771965026855469, 0.8529519438743591, -1.2928738594055176, -2.1866183280944824, -0.38439106941223145, 0.27305397391319275, 0.31291475892066956, -0.519102156162262, 2.1992383003234863, -0.1357472687959671, 1.495552659034729, -1.75140380859375, 0.4746882915496826, -0.16530963778495789, 1.0645830631256104, 2.5297794342041016, -0.6903075575828552, -0.5940001010894775, -0.12490882724523544, -0.31560778617858887, -0.024828974157571793, 0.040798645466566086, -0.587444007396698, 0.007899089716374874, -0.23677779734134674, 1.3452060222625732, -0.6073068380355835, 0.8240179419517517, 0.5986742973327637, 0.16685166954994202, 1.1304454803466797, -1.9383450746536255, 0.4103485345840454, -0.7540855407714844, -2.3641984462738037, -0.39639726281166077, 0.3113541603088379, -2.0502066612243652, 0.26978227496147156, -1.0393439531326294, -0.8783811330795288, 1.5180894136428833, 1.1839470863342285, 0.6623528599739075, 0.39367109537124634, -0.9402511119842529, 1.8601704835891724, 1.109928011894226, 0.7128311395645142, 1.45442533493042, 0.21132813394069672, 0.05869413912296295, 0.7973194122314453, 0.8306521773338318, 1.6861491203308105, -0.38253119587898254, 1.8551812171936035, 0.10898423939943314, 1.0866895914077759, 0.37622225284576416, -1.8155654668807983, -0.13791286945343018, -0.14933717250823975, -0.3754732310771942, 2.5752830505371094, -0.3785961866378784, 0.5970938205718994, 0.7128015756607056, -0.4201318323612213, -1.430572748184204, -0.9412431716918945, 1.5892817974090576, -0.7443103790283203, 0.3464356064796448, 0.3096601366996765, -1.1070961952209473, 0.832427442073822, 0.5256113409996033, -1.3402981758117676, -0.9707240462303162, -0.4504861533641815, 1.0859100818634033, -0.7349144816398621, -2.0300445556640625, 1.7827898263931274, -1.0944197177886963, -0.6299670934677124, -0.6534979343414307, -2.067902088165283, 1.3160443305969238, 0.9990230202674866, 1.0820132493972778, 0.2403668910264969, 3.2732276916503906, -1.4326262474060059, 0.08576612174510956, -0.32789289951324463, -2.4104762077331543, -0.8524674773216248, -0.26041722297668457, -0.1670616716146469, -0.3967569172382355, -0.07948307693004608, 0.16059094667434692, -0.2841622829437256, 1.280636191368103, -0.08225618302822113, 1.3942691087722778, 1.501094102859497, 0.18753927946090698, -0.8707332611083984, 0.4392290711402893, 1.3750724792480469, -2.310857057571411, -0.30985191464424133, 2.475829601287842, -0.2092994749546051, -0.7502995729446411, -0.7794491052627563, -0.622065007686615, -0.5144359469413757, -1.3651959896087646, 0.13037057220935822, 0.013852369040250778, -1.431694507598877, 1.274221658706665, 1.11076021194458, -0.5641658902168274, -0.28695979714393616, 0.4127286672592163, 0.015705671161413193, 0.5900428891181946, 1.1439707279205322, 1.636017918586731, 1.0924015045166016, -0.1641114503145218, -0.32468101382255554, -1.5454367399215698, 0.15364602208137512, 1.716671347618103, 0.6499969363212585, -0.6306609511375427, 0.37544941902160645, -0.39376747608184814, 0.10394161194562912, -0.6198724508285522, 1.7189702987670898, -2.2070703506469727, -1.0571423768997192, 0.652655839920044, -2.2411932945251465, -0.4433800280094147, -1.314754605293274, 0.3448668420314789, 0.009701893664896488, -0.09811045974493027, -0.00970652885735035, -0.23702266812324524, 1.98464035987854, -0.35759878158569336, -1.0788922309875488, 0.061641838401556015, -1.590362310409546, -1.123458981513977, 1.0264973640441895, 0.17820684611797333, -0.11718665063381195, 1.1401007175445557, -0.9621736407279968, -1.9316285848617554, 1.063501000404358, -0.27034300565719604, 2.513941764831543, -1.6334583759307861, 0.0718780979514122, -0.5751168131828308, 0.2411363124847412, -0.08716582506895065, -0.36485517024993896, 1.592492938041687, 1.1292383670806885, 0.40056121349334717, 0.6978684663772583, 0.36867234110832214, -1.8088046312332153, -0.8574612736701965, 0.36830082535743713, 0.21920114755630493, 1.808619499206543, -0.7493268251419067, -0.130567267537117, 0.06377211958169937, 0.2709585130214691, 1.3073303699493408, -0.8643958568572998, 1.0922484397888184, 0.43319085240364075, -1.488219976425171, -1.3377563953399658, -0.08540556579828262, 1.8580971956253052, 0.40956032276153564, 0.6514831781387329, 0.3587650954723358, -1.221747636795044, -0.6507357358932495, -1.6597981452941895, 0.8998413681983948, -0.42286476492881775, 1.3308764696121216, -0.07909727096557617, -1.501686453819275, -0.4217556416988373, 1.0586249828338623, -0.3201506435871124, 0.08488515764474869, -1.5912894010543823, -0.38501161336898804, 0.7834437489509583, -0.09614534676074982, 0.23204046487808228, 0.29744574427604675, -0.8104091286659241, 0.07524097710847855, -0.07016508281230927, -0.7050802707672119, 1.842074990272522, 1.6934869289398193, 1.407400369644165, -0.04928077384829521, -1.5695983171463013, 0.8214635252952576, -0.670816957950592, 0.8401554226875305, -0.374175101518631, -0.6954014897346497, -1.9354413747787476, 0.34189870953559875, 0.6357929706573486, 0.6959586143493652, 0.9965394139289856, -1.6866213083267212, 0.19383804500102997, 0.9462235569953918, 1.1370705366134644, -1.6307276487350464, 0.32061833143234253, 1.1263102293014526, 0.7430835962295532, -0.8926247358322144, 1.0083364248275757, 0.331466943025589, 1.539768099784851, -0.6651175022125244, 0.20702241361141205, 0.06969541311264038, 0.7518766522407532, -1.419697880744934, -0.04124514013528824, -1.3127243518829346, -0.1893422156572342, -0.562842607498169, -0.029891226440668106, 0.8500314950942993, 0.40397512912750244, 0.5295793414115906, 0.4711354374885559, -0.10630005598068237, -0.6290395259857178, 0.359324187040329, -1.782119870185852, 0.2458002120256424, -0.5170716047286987, -0.7352774143218994, 0.08773908764123917, 0.10938441008329391, -1.110041618347168, -0.013981220312416553, 0.5594531297683716, -0.7203802466392517, 0.7293131947517395, -1.4648712873458862, -0.8783168792724609, 0.7245556712150574, -0.2732691168785095, -0.5666722655296326, 0.508558988571167, 0.7557640075683594, -0.053451597690582275, 0.37059924006462097, 0.06824751198291779, -0.5704014897346497, 0.26177820563316345, -0.2726665139198303, -2.108954906463623, -0.5292848944664001, 0.5081583261489868, 0.03297705203294754, -0.038254667073488235, -0.6426332592964172, -1.5597003698349, -1.0235408544540405, -1.3948842287063599, -1.5805702209472656, 0.449943870306015, -0.5095123052597046, -1.2747927904129028, 0.4535273611545563, 0.7786860466003418, -1.5304663181304932, 0.6446101665496826, -1.300795555114746, 2.562025308609009, 0.31455302238464355, -2.0598866939544678, 1.1535439491271973, -1.0212448835372925, 0.4341508448123932, 1.725103497505188, 0.9578354358673096, -1.6462304592132568, 0.11101148277521133, -0.6522006392478943, 0.2298366129398346, -0.34214988350868225, -0.14784203469753265, 1.5492324829101562, -0.644245982170105, 1.2221477031707764, -0.9852613806724548, -0.23082248866558075, -1.5296506881713867, 1.1576136350631714, 0.07138579338788986, -1.3866899013519287, 0.10306810587644577, -0.41614899039268494, -0.47738492488861084, -0.5634847283363342, 0.6296229958534241, -0.6759562492370605, 1.0127488374710083, 0.6117656230926514, 0.9512954354286194, 1.10902738571167, 0.4446694552898407, -0.7094317674636841, 0.8943852782249451, -1.4269027709960938, -0.6764688491821289, 1.1463491916656494, -1.8082952499389648, -0.8859174847602844, -1.3372513055801392, 0.3736979067325592, -1.02860689163208, 0.19449077546596527, 0.043949492275714874, 1.979201078414917, 2.0958123207092285, -0.8327865600585938, -0.6717298030853271, -0.05900539085268974, -0.3766714334487915, -0.3415672481060028, -0.15892665088176727, 0.9098086953163147, 1.372442603111267, -1.4148451089859009, 0.5944443941116333, 0.14774231612682343, -0.9890608787536621, -0.05647042393684387, -0.5188241600990295, 0.10929341614246368, 0.10399053990840912, -1.7016546726226807, -0.5163238644599915, 0.23603498935699463, -0.6285950541496277, -3.1091034412384033, 1.3137089014053345, -0.1568969339132309, 0.6279906034469604, -0.6560253500938416, -0.8922040462493896, 0.9882558584213257, 0.10492151975631714, -0.486226350069046, 0.3620564341545105, -0.3193877935409546, 0.2488717883825302, 0.8613300323486328, 0.04190696403384209, -0.4379061758518219, 0.6884281635284424, 1.204564094543457, 0.25200197100639343, 0.03607378154993057, 0.04703017324209213, -0.6341204643249512, 0.6808624267578125, -0.39703401923179626, -0.3545634150505066, 0.4031873345375061, 1.4778506755828857, 1.1717076301574707, -1.062908411026001, 1.439521312713623, 0.8025926947593689, 0.1514747589826584, 0.9817519187927246, 0.2966782748699188, 1.922666072845459, -0.7173696756362915, 1.4971003532409668, -0.41782450675964355, -0.2691618502140045, 0.8883659839630127, -1.9553759098052979, 0.9094483256340027, 1.319114327430725, -1.680073618888855, -0.633281946182251, 1.060203194618225, -0.9616311192512512, -1.3203486204147339, 1.8076236248016357, -0.3892544209957123, 1.6459420919418335, 1.0479435920715332, 0.391729474067688, -0.7843952178955078, 1.1159218549728394, 0.5206010341644287, -0.5596349239349365, -0.7297086119651794, -0.33412590622901917, -1.1359909772872925, -0.4860326647758484, 0.4842219650745392, -0.8894041180610657, -0.35507723689079285, -0.30706682801246643, -0.13724498450756073, -0.6799649000167847, 0.4193349778652191, 1.1986234188079834, -1.2843053340911865, -0.5419999361038208, -0.1517971009016037, 1.1768007278442383, -0.13478809595108032, -0.522971510887146, 0.8836696743965149, -0.1663135439157486, -1.0703206062316895, 0.25000908970832825, 0.10676278918981552, -0.6321485638618469, -1.6834800243377686, -0.5440505146980286, 0.3968663215637207, 0.3381110429763794, -1.5034687519073486, -1.1678626537322998, 0.9026063680648804, -1.8426244258880615, 0.8732308149337769, 0.6527636647224426, 2.015495538711548, -1.6676045656204224, 0.8358821272850037, -0.6080018281936646, 0.027867209166288376, -1.0483587980270386, 0.9192641973495483, 0.5542119145393372, -1.1629598140716553, -1.4636321067810059, 0.3239675760269165, -0.47687599062919617, 0.19949114322662354, 0.18514060974121094, 0.35669150948524475, 0.0713285431265831, -0.5130950212478638, 1.0061553716659546, 0.7355831265449524, -0.035393279045820236, 0.38582512736320496, -0.008950408548116684, -0.10535839200019836, -0.4893485903739929, 1.3763456344604492, -0.5269622206687927, 1.3603503704071045, 0.6695204973220825, 0.5828642249107361, 0.7016631364822388, 0.44255688786506653, -1.2608070373535156, -1.492821455001831, -0.7636751532554626, -0.5978853702545166, -0.04890973120927811, -0.17302407324314117, 0.3164033889770508, -1.7196252346038818, 0.09780445694923401, -0.7836015820503235, -0.7513781189918518, -0.7647578120231628, -0.08814771473407745, -1.0017905235290527, 0.6183966994285583, 0.6447989344596863, -0.7979119420051575, -0.7958347797393799, -0.11332419514656067, 1.109632134437561, 0.5300443768501282, -0.7856763601303101, -0.2061791568994522, -1.3263742923736572, -0.5754131078720093, -1.1959589719772339, 0.4208774268627167, -0.36765044927597046, -0.17918722331523895, -1.7079375982284546, -0.5871677398681641, -0.522705614566803, 0.9729191660881042, -1.8086309432983398, -1.0716532468795776, -1.0886861085891724, 0.29471346735954285, -0.7221387624740601, 1.069154143333435, 1.445449948310852, -2.0357542037963867, 1.177430272102356, 0.6065773367881775, 0.7102747559547424, 0.44544118642807007, -0.3412761390209198, -0.5641161799430847, -0.6523860692977905, -0.9445432424545288, 0.22162765264511108, -0.13771748542785645, 0.12488869577646255, 0.09333427250385284, -1.726348638534546, -0.1268308460712433, -0.8091046214103699, -0.9525368213653564, -0.1212654709815979, 0.7907422780990601, -0.2882816791534424, 0.22042728960514069, 1.5082272291183472, 0.6470072865486145, 0.5331809520721436, -0.7266383171081543, 0.8471091985702515, -0.34290778636932373, 1.5095937252044678, 1.9556955099105835, -0.6744992733001709, 0.3035876750946045, 0.2052280753850937, 1.5523861646652222, -1.2408734560012817, 1.2250313758850098, 1.1440564393997192, 0.16911372542381287, 0.6299594640731812, -0.015245816670358181, 0.40197306871414185, -0.0847088173031807, -1.29597008228302, -0.47954854369163513, -0.4434421956539154, 0.6289181709289551, -0.3469647467136383, 0.5434751510620117, 1.1056432723999023, -0.30568599700927734, 0.5861474871635437, -1.2443137168884277, 0.20009282231330872, 1.0718218088150024, 1.014975905418396, 3.5673112869262695, -0.4409671127796173, 0.18436837196350098, 0.13930298388004303, -0.5929490923881531, 0.028835270553827286, -1.1736884117126465, -0.2304847538471222, 3.0203986167907715, -0.6822523474693298, 0.9831365346908569, 1.3868460655212402, 1.5015201568603516, 0.7088115811347961, 0.3626127243041992, 0.9326211810112, 1.060015082359314, -1.435847282409668, 1.5338211059570312, 0.1744007170200348, 1.0180249214172363, 1.0685348510742188, 0.4230239689350128, -1.1126775741577148, -1.6690762042999268, 0.9320728778839111, -0.6101217269897461, -1.4374793767929077, -0.4998711347579956, -0.002834590617567301, 1.563658356666565, -1.0722678899765015, 0.46185317635536194, 0.15708063542842865, -0.4446231722831726, -1.014357566833496, 1.4231094121932983, -0.2042001336812973, 0.2998610734939575, -0.2489088475704193, 1.818052053451538, 0.31657183170318604, -0.4377683401107788, 0.9046235084533691, 0.15963315963745117, -0.9867634177207947, -1.4866127967834473, -0.6034660935401917, -0.550509512424469, -1.5842849016189575, -2.1086671352386475, 0.08413279801607132, 0.003395203733816743, 0.5348941683769226, -1.2621228694915771, -0.09992027282714844, -0.0009984277421608567, -0.4402512311935425, -0.9596166610717773, -0.07679780572652817, -0.6163137555122375, -0.7154316902160645, -0.4595522880554199, -1.0566318035125732, -1.1085840463638306, -0.2633673846721649, -0.10762066394090652, 0.6040247082710266, -0.3608371615409851, -0.9232916831970215, -1.2750123739242554, 0.7855157852172852, -0.6493817567825317, -1.1817185878753662, 1.3302538394927979, 0.920102596282959, -0.6697805523872375, -0.19037850201129913, -0.9396185874938965, -0.22762344777584076, 1.2986669540405273, -0.2039039582014084, 0.9434242248535156, 1.2223377227783203, -1.909683346748352, -0.7994086742401123, 1.2624633312225342, -0.6573380827903748, -0.6673133969306946, -0.7998536825180054, 0.8196184635162354, -0.17181019484996796, 2.3786094188690186, 0.2351534515619278, -2.117511034011841, -1.295159935951233, 0.09008654206991196, 1.8315507173538208, 0.9193454384803772, -0.7152693271636963, -0.6145361065864563, -1.6496227979660034, 0.9659966230392456, -1.6817933320999146, 0.7210935354232788, 0.8792908787727356, 0.3623609244823456, 1.1138193607330322, 2.1475753784179688, -0.8094921708106995, 1.7383432388305664, -2.136404275894165, -0.4001534879207611, -0.21634535491466522, -0.1001284196972847, 1.1321702003479004, -0.23220698535442352, -0.9391445517539978, -0.9851633310317993, -0.4003034830093384, 0.5093330144882202, -1.1112287044525146, -0.7496314644813538, -0.6930408477783203, 0.4449767470359802, 0.27159377932548523, -1.166816234588623, -1.252226710319519, -0.7757284045219421, 0.265610933303833, 1.239011526107788, 0.08439398556947708, -0.733564019203186, 2.068077564239502, 0.7658389210700989, -2.429597854614258, 0.6259574294090271, 1.11574125289917, 1.0360976457595825, -0.5827099680900574, -1.6387712955474854, 0.041770175099372864, 0.6230417490005493, 0.5107347369194031, 0.30974385142326355, -0.7572271823883057, 0.06357128173112869, -0.0908389613032341, -0.551916241645813, 1.7818981409072876, -0.34286391735076904, -0.3243288993835449, 1.8203895092010498, -0.4510249197483063, -0.4770224988460541, -0.1829509437084198, -1.3457223176956177, 0.5456275343894958, -0.4376218914985657, 0.14485102891921997, 1.0282936096191406, -0.6867288947105408, -0.5724254250526428, -1.6912389993667603, 0.6811192035675049, -0.5750874280929565, 0.7118764519691467, -0.19357949495315552, -0.6427901387214661, -0.9549718499183655, 1.0946266651153564, 0.5498635768890381, 0.9548963308334351, -0.15797169506549835, -1.4800033569335938, -0.21443970501422882, 0.5532279014587402, 0.6352826952934265, 0.8350695371627808, 1.027267575263977, 0.9047111868858337, 1.6210627555847168, -1.7838870286941528, 0.6620467305183411, 0.6275163888931274, 0.8135183453559875, 0.11769786477088928, -0.7375169992446899, 0.2566777765750885, 0.7859528660774231, 0.11987647414207458, 1.0928168296813965, 0.9399109482765198, -2.0839767456054688, 2.3498482704162598, -1.0708492994308472, 2.4623608589172363, 0.7926816940307617, -0.5858049988746643, 0.3179114758968353, -0.06710135191679001, 0.6199281215667725, -1.3996895551681519, -0.3038055896759033, -2.373401641845703, -0.0018137304577976465, -0.5057306289672852, -1.1171354055404663, -1.0261698961257935, -0.18624024093151093, -0.6032435297966003, -0.17400169372558594, -0.5202615857124329, -0.581221342086792, -0.6659215092658997, 1.0639845132827759, -0.3995739817619324, -0.0237455815076828, 0.7303004860877991, 1.3135446310043335, -1.4243388175964355, -0.3537812829017639, -0.965740442276001, 0.17820216715335846, 0.16999439895153046, -1.679962158203125, -1.0156183242797852, -0.8059147000312805, 0.9065230488777161, -2.142859935760498, 0.5250865817070007, 0.02191033400595188, -0.6009505391120911, 0.15073111653327942, -0.7178676128387451, 1.9166771173477173, 0.08365476876497269, -0.4903925359249115, 1.1148492097854614, 0.07285487651824951, -0.8576450347900391, -0.2296055257320404, 0.405775785446167, 0.08062738925218582, 0.2561505436897278, 0.26827287673950195, 0.7135244607925415, -0.05450987443327904, 1.8493956327438354, 0.5918864011764526, 1.2361071109771729, 0.04546704888343811, -0.6481087803840637, -0.027397265657782555, 0.19679543375968933, -0.5326281189918518, -2.902712345123291, -0.551166832447052, 1.15830397605896, 1.5810022354125977, 1.5065174102783203, -0.3532458543777466, -0.3678855299949646, 1.7277625799179077, 0.7343124747276306, -1.1649571657180786, -1.0895551443099976, -0.902155876159668, -0.5989304184913635, -2.403777837753296, -0.7736782431602478, -0.62723308801651, 0.65675288438797, -0.31350794434547424, -1.7329713106155396, 0.3420681655406952, 0.9381769299507141, 1.7533916234970093, 0.9790100455284119, 0.887135922908783, 1.7484426498413086, -1.2241500616073608, 1.8061933517456055, 1.8824130296707153, -0.7167029976844788, 1.161973476409912, -1.3483363389968872, 2.9494147300720215, 0.8988279700279236, 0.031326454132795334, -0.8932703733444214, 0.15789411962032318, 0.7961908578872681, 0.07977962493896484, 1.1627459526062012, 0.3850899338722229, -0.48946869373321533, 1.1001945734024048, 3.189073085784912, -2.0696680545806885, 0.11438465863466263, 1.0663965940475464, -0.5741991400718689, -1.0296965837478638, -0.7000496983528137, 0.19054685533046722, -1.0445725917816162, -1.7761727571487427, -1.4074822664260864, 0.2319781631231308, -0.17965900897979736, 0.9651010036468506, 0.1207309141755104, -1.1784987449645996, -0.7218870520591736, -0.8613566756248474, 0.8150973916053772, -0.10899592936038971, -0.2095891684293747, 0.2920805811882019, 0.5511004328727722, -0.7352827191352844, -1.4503347873687744, -0.11593370884656906, -0.47743460536003113, 0.3524051904678345, -1.3727442026138306, 1.393254280090332, -0.11905152350664139, 1.2917627096176147, 0.5210446715354919, -0.11773208528757095, -0.934425413608551, -0.16670095920562744, -0.8871771097183228, 1.6970772743225098, -0.8113023042678833, 0.04613278806209564, -0.7830929160118103, -0.6320576667785645, 0.16719764471054077, 2.114938974380493, -1.1465184688568115, 0.08030100166797638, 1.0949546098709106, 0.01656132936477661, -0.47267940640449524, 2.8966751098632812, 0.37298583984375, -0.5921692252159119, -1.6615408658981323, -0.9433864951133728, -0.5785303711891174, 0.9783948659896851, 0.035062529146671295, -0.8032646179199219, 1.7297933101654053, 1.1550370454788208, -0.41155683994293213, 0.9132372736930847, 0.04969555512070656, -0.5175139307975769, -1.0658416748046875, -0.20912989974021912, 0.11002882570028305, 0.3161357641220093, 0.17875659465789795, 1.0579993724822998, 1.5154297351837158, -0.02087102085351944, 0.6771761775016785, 1.274680733680725, -0.24154174327850342, -0.7013731598854065, -2.0616202354431152, -0.24207167327404022, -2.1513290405273438, -1.0055196285247803, 0.41423743963241577, 0.8350659608840942, -1.3102165460586548, -2.1053590774536133, -1.380631446838379, 0.3243478834629059, 2.095951795578003, -0.11143551021814346, -2.0513670444488525, -1.1758959293365479, 1.1898902654647827, 0.4166967272758484, 0.638815701007843, -0.14055192470550537, -0.08746081590652466, -2.0720248222351074, 0.912452220916748, 1.6349561214447021, -0.6527641415596008, -0.7882694602012634, -0.09787412732839584, -0.378527969121933, -1.284877061843872, -1.162109375, 0.23162078857421875, 1.1910109519958496, -0.6086152791976929, -1.5126032829284668, 0.4439091384410858, 1.1310937404632568, -0.7105218768119812, -1.1273638010025024, -0.5661898255348206, -0.2694997191429138, 1.5110726356506348, -1.0620620250701904, 0.3518851101398468, 0.45665988326072693, -0.44298484921455383, 1.7807590961456299, 0.6445503234863281, -0.6059103012084961, -0.28282439708709717, 0.050780102610588074, 1.6694740056991577, -0.315757691860199, 0.7461427450180054, 0.44474464654922485, -1.026462435722351, 0.3434520959854126, -1.0671406984329224, -0.4233205318450928, -0.14164400100708008, 0.6045663952827454, -0.11205337941646576, -0.42172592878341675, -0.5652124285697937, -0.45104923844337463, 0.17213821411132812, -1.3549357652664185, -0.7203165292739868, -0.020635245367884636, -2.486882448196411, 0.6467735171318054, 0.8566967248916626, -0.725279688835144, -0.07952059060335159, -0.09864263981580734, -0.4496613144874573, 0.13944891095161438, -1.1889351606369019, 0.7590968012809753, -0.6861369609832764, 1.0561244487762451, 2.6870663166046143, 0.26473096013069153, 0.48823845386505127, 0.2995580732822418, -0.11562822014093399, 0.6959853768348694, 1.0416502952575684, -0.5440841913223267, 1.3410097360610962, -0.24565909802913666, 1.4944301843643188, -0.19581827521324158, -2.1069986820220947, -0.41552937030792236, -0.9402577877044678, -0.20668281614780426, 0.8323878049850464, -0.6934300661087036, -0.6725265979766846, 0.15019501745700836, -1.4208316802978516, -0.47103556990623474, 1.3006439208984375, 1.4739469289779663, -0.5705240964889526, -0.3819601237773895, 0.7485178709030151, 1.2367621660232544, -0.3563151955604553, -0.8675207495689392, -0.7140162587165833, 1.2010722160339355, 0.5514052510261536, -0.7877522706985474, 1.657217025756836, -0.28888264298439026, -0.7833958864212036, 1.0739519596099854, 1.0733839273452759, -0.658380925655365, -0.5101917386054993, 0.6535712480545044, -0.2902759313583374, 1.2732518911361694, 0.8794219493865967, -0.4909162223339081, -2.785942792892456, 0.10056933015584946, -0.01538501214236021, 2.225454330444336, -1.002367377281189, -0.10403374582529068, 1.6493291854858398, -0.1668483316898346, 0.9911093711853027, 0.2756485641002655, -0.01437193900346756, 0.6910043954849243, -0.3619421422481537, -0.8401368260383606, 0.5256587862968445, -0.28461506962776184, 0.4774847626686096, 1.3869210481643677, 0.8919420838356018, -1.107468605041504, -0.8812538981437683, -1.3305654525756836, -0.5001844763755798, -0.6995905637741089, 0.5013925433158875, 1.2758833169937134, 0.46279457211494446, -2.0223660469055176, -0.23175691068172455, 0.07605810463428497, 1.3366448879241943, 0.4811779260635376, 0.8144302368164062, 0.2928929626941681, -1.4625734090805054, 0.5484941005706787, 1.0686147212982178, -1.0800366401672363, 1.7852076292037964, 2.3322341442108154, -0.8176872730255127, -0.5964502096176147, -1.0260601043701172, 0.3039773106575012, -0.8102144598960876, 1.7806837558746338, -0.5206454396247864, 0.006084630265831947, 0.7532508969306946, -0.3666476309299469, 1.661636471748352, 0.004683622159063816, 0.24243110418319702, 0.7920036911964417, -1.2625850439071655, 0.21334950625896454, -0.5892475843429565, 0.2808832824230194, -1.071326732635498, 1.4275027513504028, 0.6683521270751953, -0.32268622517585754, -0.08973655849695206, -1.6086349487304688, 0.48136091232299805, -1.1775418519973755, 0.873238742351532, -0.6808737516403198, 1.6913580894470215, -0.265873521566391, -0.5241625308990479, 1.1472299098968506, -0.12409181892871857, 2.026481866836548, 2.0486998558044434, -0.5572580099105835, 1.2261829376220703, 0.6659159064292908, 0.21805201470851898, -1.8022713661193848, -0.05654536932706833, -0.4776655435562134, 1.1926982402801514, 1.2784969806671143, -1.0840094089508057, -1.352274775505066, 0.24642325937747955, -0.38440269231796265, 0.9618211388587952, -0.50386643409729, 1.3858751058578491, 1.7485994100570679, -1.2116349935531616, -0.9311841726303101, 0.16196748614311218, -0.2306334525346756, 1.954773187637329, -0.0922708585858345, -0.18976104259490967, -1.5163140296936035, 0.5690574645996094, -0.9476891756057739, -0.6061000227928162, -1.9566380977630615, -0.2070157378911972, 0.15937048196792603, 0.1928853690624237, -1.5225390195846558, 0.1829889863729477, 0.3792528510093689, 0.7313146591186523, -1.4871175289154053, 0.23890957236289978, 0.2612496018409729, -0.0760100781917572, 0.34789663553237915, -1.0448111295700073, -0.8278177976608276, 2.286282539367676, 1.1180987358093262, 0.8371188640594482, 0.4227409362792969, -0.3331139087677002, -0.721198558807373, -1.3401923179626465, -0.16838869452476501, -1.6816905736923218, -0.00589874479919672, -0.8009464144706726, -0.38837796449661255, -0.5258849263191223, 1.8006924390792847, 1.943821668624878, -1.1254684925079346, 2.2226641178131104, 0.9667279720306396, 0.632936954498291, 1.1696771383285522, -2.4694859981536865, 0.3038582503795624, 1.8215856552124023, -0.397116482257843, 1.3514695167541504, -0.5575533509254456, -0.9451057314872742, 0.5276089906692505, -0.281904399394989, -0.5971977114677429, -0.1940467655658722, 0.19336630403995514, 0.8282409906387329, 0.0034860463347285986, 0.19435609877109528, -0.08308684080839157, -0.6756108403205872, -0.7915363311767578, -0.34897059202194214, -0.6120470762252808, -0.4492850601673126, 0.6536174416542053, 0.13391342759132385, 0.11103737354278564, -0.8472356200218201, -0.40043285489082336, -0.3318954408168793, -0.644652247428894, 0.752005934715271, 1.8611152172088623, 1.065205693244934, -0.9258942604064941, -0.3567332923412323, 0.6750712990760803, -0.25203773379325867, 0.124990314245224, -1.2066665887832642, -1.2169902324676514, 0.5921534895896912, -0.6818658113479614, 0.6822362542152405, 0.9620978832244873, -0.7281033992767334, -0.8037620782852173, 1.6192330121994019, -0.106594517827034, 0.44247353076934814, 0.42737066745758057, -0.11280351132154465, 0.12017311900854111, 0.7475338578224182, -0.13255898654460907, -1.7780025005340576, -0.37565070390701294, -0.5137608051300049, -0.42468252778053284, 1.0375975370407104, -1.616410255432129, 0.6744170188903809, -1.2442947626113892, -0.5670765042304993, 0.5068130493164062, -0.8604033589363098, 1.297218680381775, 0.9931066036224365, -0.3269566297531128, -0.34139567613601685, -0.4370616376399994, -2.5845959186553955, -0.7899416089057922, -0.7233834862709045, 0.10579467564821243, 0.6232095956802368, -3.34254789352417, -0.8290130496025085, 0.667599618434906, 0.6080718636512756, 1.4520139694213867, -0.5872042179107666, -1.276552677154541, -0.04261567443609238, 0.5261742472648621, -1.069409966468811, -0.9480657577514648, -1.2489149570465088, 0.5761538743972778, -1.2343143224716187, 0.07732219994068146, -0.0430561788380146, -0.28202009201049805, 0.3582209050655365, -0.025060798972845078, -1.3215785026550293, -0.9385651350021362, 0.32126176357269287, 1.7163664102554321, 0.7268438935279846, 0.0422331802546978, 0.22044673562049866, -0.8781000375747681, 1.420742392539978, 1.1155004501342773, -2.2221121788024902, 0.6948548555374146, -1.7874395847320557, -0.04900744557380676, 1.2307541370391846, -0.8020151853561401, 0.07729672640562057, -0.22445333003997803, -3.2846643924713135, 2.083717107772827, 0.5839570760726929, -2.6031744480133057, 1.4761407375335693, -0.6968451738357544, -0.5695924162864685, -0.5524478554725647, 0.9880194067955017, 0.49623537063598633, -0.9830886125564575, -0.09211330115795135, -1.1616291999816895, 1.1547589302062988, 0.27897292375564575, 0.9197271466255188, -1.3264442682266235, -0.010691859759390354, 1.8354469537734985, -0.5726884603500366, -0.11663331091403961, -0.5527477860450745, -0.7236055731773376, 0.11061165481805801, -0.21891574561595917, -0.8760361075401306, -0.8337336778640747, 0.5058481097221375, 1.4370992183685303, 0.36911630630493164, 0.48022300004959106, -0.6535972356796265, -0.34790775179862976, -0.8374881148338318, 0.02206178940832615, 0.013182888738811016, 1.2302244901657104, 1.3357030153274536, -0.4501102864742279, 0.7637640833854675, -0.15581750869750977, -1.1720982789993286, -0.5930302739143372, 0.3229505121707916, 0.2740674912929535, 0.6931359171867371, -0.7698983550071716, -0.22606179118156433, -1.8208484649658203, 0.9152842164039612, 3.4374983310699463, 0.3508489429950714, -2.1545486450195312, 0.02427651360630989, -0.0293339304625988, 0.8218543529510498, 0.609031617641449, 1.5338897705078125, 1.2194589376449585, 0.4631563723087311, -0.6331737041473389, -0.6024254560470581, 0.19330397248268127, -0.42363524436950684, 1.1290391683578491, 0.7487088441848755, 0.8166717290878296, -2.481024742126465, -0.9183474183082581, 1.1916863918304443, 0.1680484414100647, 0.15365883708000183, 0.8641029000282288, -0.5934807658195496, -0.6296699047088623, -0.36915087699890137, -1.213889479637146, -0.24535802006721497, -2.0327625274658203, 1.4960826635360718, 1.768652081489563, 0.729274332523346, 0.20396143198013306, -1.9716213941574097, 2.104060649871826, 0.6950942277908325, -0.7469016909599304, -0.8515223860740662, 0.215719535946846, -0.7890348434448242, 0.45944616198539734, 0.9960584044456482, -1.2871137857437134, -1.545632243156433, -0.6743455529212952, 0.9968156814575195, 0.2716684341430664, 0.1493447870016098, -0.19335272908210754, 1.206682562828064, -0.9945492744445801, 1.3325273990631104, -0.7734037041664124, -0.421720951795578, 0.4113522469997406, 1.3482378721237183, 0.20328286290168762, -1.3184165954589844, 0.08138656616210938, -1.6370784044265747, 0.8903610110282898, 0.13461121916770935, -0.08006524294614792, 0.6031178832054138, -1.1619292497634888, -1.9120562076568604, -0.35646969079971313, -0.42201247811317444, -1.9919483661651611, 0.14886035025119781, 0.973504900932312, 1.5702497959136963, 0.6665960550308228, 0.011287190951406956, -2.704575538635254, -2.480792284011841, 0.9032522439956665, 1.4930813312530518, -0.7116197347640991, 0.11985185742378235, 0.605167806148529, 0.3636355400085449, 0.8009308576583862, -0.7459663152694702, -1.7445483207702637, -0.2033787965774536, -0.025012679398059845, -2.7039244174957275, -0.12161006033420563, 2.755110740661621, 0.8847886919975281, -0.24220021069049835, 0.6795443296432495, 0.656981348991394, 0.13499975204467773, -0.26785463094711304, -1.3704311847686768, -1.2016208171844482, 0.6364750862121582, -0.6345766186714172, -0.38839712738990784, 0.7103890180587769, 0.6898196935653687, -0.5182366967201233, -1.6068466901779175, -0.5054689645767212, -0.9463163018226624, 0.6288233399391174, -1.7542431354522705, 0.09206689894199371, 0.4256588816642761, -1.945345401763916, 0.29422688484191895, 0.24584831297397614, 1.6477679014205933, 0.542008638381958, 0.8491106629371643, 0.19968603551387787, 1.5556334257125854, -0.5951604247093201, 1.8135415315628052, 1.1261776685714722, -1.6873834133148193, -0.6011731028556824, 1.4067797660827637, 1.140290379524231, -0.2254898101091385, -0.3412075340747833, 0.7605263590812683, -1.5522981882095337, 0.026583373546600342, 1.407947063446045, -1.4376176595687866, 1.7180759906768799, 1.8576717376708984, -0.6231099963188171, -1.1760344505310059, -0.025834767147898674, 0.47607776522636414, 0.8948854804039001, -0.343863308429718, -1.1520988941192627, 1.2355684041976929, -0.7279402017593384, -1.1200921535491943, -0.8507999777793884, 0.6682566404342651, -0.5586385726928711, -0.5070435404777527, 0.981611967086792, -1.0465402603149414, -2.5902819633483887, 0.33759018778800964, 0.4032415449619293, 0.4132152497768402, -0.650031328201294, -1.4146579504013062, -0.47281256318092346, -1.1700899600982666, 0.9550498723983765, 0.31329578161239624, 1.9712265729904175, 0.7899218201637268, -0.22259646654129028, 1.4386425018310547, -0.8279028534889221, 1.196254849433899, 1.1264411211013794, -0.26473698019981384, -0.7888810634613037, 0.020853696390986443, -0.45334210991859436, 0.0996149480342865, 0.5201098322868347, 0.3421051800251007, 0.13095925748348236, 0.9654352068901062, 0.3531239330768585, 1.7259016036987305, -1.620312213897705, 0.2662351727485657, -0.5921801328659058, 1.2262197732925415, 0.12452042102813721, 1.2975975275039673, 1.6699188947677612, -0.1101493090391159, 1.1577295064926147, 0.9623712301254272, -0.6914200782775879, -0.564709484577179, 0.8470078706741333, -1.9299921989440918, -0.4031526744365692, -0.5591127276420593, -0.1526847779750824, 0.016906458884477615, 2.030733108520508, -0.40436941385269165, -1.2529168128967285, 0.05146360769867897, -0.9524176120758057, 0.9264519810676575, 0.8744187951087952, -1.4202691316604614, -1.3746707439422607, 0.30312424898147583, -0.48644018173217773, -0.05058625712990761, 1.5220937728881836, -0.10497400909662247, 0.4083549380302429, -1.2929778099060059, -1.444392204284668, -1.0376129150390625, -0.6254894137382507, 0.5712074041366577, 0.9332399368286133, 1.7052420377731323, -0.895646333694458, 0.8229856491088867, -0.781237781047821, -1.6984175443649292, 1.5907405614852905, 0.5636669397354126, -0.7735167145729065, -0.22781804203987122, 0.8484704494476318, 1.9563467502593994, 1.0391325950622559, 0.8286913633346558, -0.7347940802574158, -1.615257978439331, -0.006473924964666367, -1.3265957832336426, -0.4333399534225464, 1.3843841552734375, 0.282093346118927, -1.0594866275787354, -1.7616956233978271, -1.0979303121566772, 0.6807953119277954, -2.4529407024383545, -1.1823585033416748, 0.5488222241401672, -0.6138034462928772, -0.05295569449663162, -1.1356760263442993, 0.9453004598617554, 2.344595432281494, 0.5353612899780273, 1.532835602760315, -0.3714263439178467, 0.41627037525177, 0.04970235377550125, -1.7143573760986328, -0.09267795830965042, -0.037677861750125885, -0.4816981852054596, -0.3135668635368347, -1.823951244354248, 0.6384023427963257, -0.7880344986915588, 0.23204068839550018, -0.5069634914398193, 0.9938857555389404, 0.08191949129104614, 0.7019092440605164, -1.1911687850952148, 0.6750460863113403, 0.47283580899238586, -0.1456248015165329, -0.4782790243625641, 0.3552609980106354, 0.5156674981117249, -0.030132507905364037, 0.13364839553833008, -0.405962735414505, -0.68131023645401, -1.350484013557434, 1.042153000831604, 0.4190343916416168, 0.16358882188796997, -0.3169989585876465, -0.19954873621463776, -0.6331744194030762, 0.8837574124336243, 0.24794241786003113, 0.09455978125333786, -0.21314017474651337, -0.8213616609573364, 1.378279447555542, -0.14551204442977905, 0.9328216314315796, -0.62176114320755, 0.1986643373966217, 1.1359578371047974, -1.3116897344589233, -0.8056994080543518, -0.9799292087554932, 0.3439631760120392, -1.7288790941238403, -0.4941812753677368, -1.477062702178955, -1.7083483934402466, 2.2749948501586914, -0.32272735238075256, 1.2523545026779175, 1.1838099956512451, 1.569666862487793, -0.139162078499794, -1.1326251029968262, 2.260035991668701, -1.6344363689422607, -1.8799909353256226, 0.6670931577682495, 0.9103142023086548, 0.8894047737121582, -0.29963478446006775, -1.3832966089248657, 0.2518208920955658, 1.0611765384674072, -0.08162710070610046, 1.4998586177825928, -0.07158855348825455, 0.9284087419509888, -0.7146397829055786, 0.4521181285381317, 1.512770414352417, 1.3887156248092651, 0.7150481939315796, -0.05897825211286545, -0.6227583289146423, -0.001390890683978796, 1.102766990661621, 0.8784970045089722, -1.0547735691070557, 1.3136777877807617, 1.5840703248977661, -0.6692838072776794, 0.515053391456604, -0.9876570701599121, 0.17001758515834808, -0.062388282269239426, 0.35507193207740784, -0.021324768662452698, -0.45788511633872986, 0.4718484878540039, -2.468313455581665, -1.1020554304122925, 0.013128926046192646, -0.09577500075101852, 0.5765439867973328, -0.9563531875610352, -1.4600740671157837, -0.03059164434671402, -0.5708193182945251, -2.019652843475342, 0.9232614636421204, 0.40748193860054016, -0.13205529749393463, 2.5841548442840576, -0.9323763251304626, 0.14126801490783691, 0.6972410678863525, -0.019493380561470985, -0.5074684619903564, 0.05821641534566879, 0.3699660897254944, 0.8616219162940979, -0.5769234299659729, -0.4470294117927551, -0.851676881313324, -0.5730496644973755, -0.5430657267570496, -0.6752426028251648, 1.8789315223693848, -0.601634681224823, -0.5553030371665955, 2.284723997116089, 0.839337170124054, -2.7908899784088135, -0.06614029407501221, -0.29702994227409363, -1.5700162649154663, -1.3409830331802368, 0.6956264972686768, 0.3860680162906647, 0.051935888826847076, -0.8930606245994568, 1.2549015283584595, 0.2207430899143219, 0.005342231132090092, 0.059168629348278046, -1.670541524887085, 0.3807331621646881, 1.4561846256256104, 0.34612277150154114, -0.507881224155426, 1.1356773376464844, -1.3353190422058105, -0.6403961777687073, 0.27470630407333374, 0.5331928133964539, 0.3766433894634247, -1.6821831464767456, 0.5091484189033508, 0.34131935238838196, -0.7811291813850403, 1.6300379037857056, -0.2739735543727875, -0.14292462170124054, 0.5675291419029236, 0.3372799754142761, 0.3434775173664093, -0.6091005802154541, 0.2305477261543274, -0.0070664179511368275, -1.1855071783065796, -1.5954086780548096, 2.2633488178253174, -1.1375826597213745, -0.8352458477020264, 0.6895403861999512, 0.21924564242362976, -1.7912381887435913, 0.6076593995094299, 0.744586169719696, 0.005988870281726122, -0.4493226110935211, -0.035404354333877563, -0.9513779282569885, -1.239656925201416, 0.38977909088134766, 1.8019905090332031, 1.1065620183944702, 0.42001834511756897, 0.7838947176933289, 0.5054168105125427, 0.9687461256980896, 0.9649083018302917, -1.19186270236969, -0.7187884449958801, 1.5398460626602173, -0.85878586769104, -1.8298577070236206, 1.2785967588424683, 0.09199512004852295, 0.5409001708030701, 1.2822494506835938, -0.9324820637702942, 0.20751549303531647, 0.3195757567882538, -0.24133028090000153, -0.09103785455226898, -0.28334543108940125, 0.5140523314476013, -0.7393580675125122, -0.10627766698598862, -1.1299480199813843, 0.12714357674121857, 0.9476136565208435, -0.7605385184288025, -0.2523583471775055, -1.060828447341919, 1.8155885934829712, -0.019391732290387154, 0.4182718098163605, 2.131227731704712, -0.25396236777305603, -0.09531241655349731, -0.7413858771324158, -1.2621725797653198, 1.5981435775756836, 0.2761390805244446, 0.08378727734088898, 0.4385521709918976, -0.6730919480323792, 0.2765045166015625, 0.5380395650863647, 1.0719013214111328, -0.10434254258871078, -0.4210846722126007, 0.11686112731695175, 0.7484068274497986, 1.0358035564422607, -0.3712756931781769, -0.6193749308586121, -1.4635848999023438, -0.5482596755027771, 0.20108459889888763, 0.5004163384437561, -1.1224216222763062, -1.762628436088562, -0.9395259022712708, 1.5066139698028564, 0.6433097124099731, -0.22144275903701782, 2.010270833969116, 0.21964344382286072, -0.27697181701660156, 0.249564066529274, 0.07494346052408218, -0.16062822937965393, 0.9187678098678589, -0.04942769184708595, 0.10794281214475632, 0.4221767485141754, 1.0131428241729736, -1.4308778047561646, -0.8599252104759216, -0.3672284185886383, -0.6743965148925781, -0.18707983195781708, -0.9847589135169983, -0.9671480059623718, -1.2636590003967285, 0.18556803464889526, 1.438539981842041, -0.9962128400802612, 1.4054213762283325, 1.2028369903564453, -1.9328242540359497, 0.23250187933444977, 0.3878405690193176, -2.1689293384552, 0.3181452453136444, -0.2554861605167389, 0.13337360322475433, 1.4374910593032837, 0.14067673683166504, 0.3930247724056244, -1.2400636672973633, 0.7485615611076355, 0.7086167931556702, 0.8094612956047058, 3.209430694580078, -0.08833437412977219, 0.8457263708114624, -0.15435971319675446, 0.6159102320671082, 2.2969675064086914, -0.28665605187416077, -1.0740448236465454, 1.201499342918396, -0.5396712422370911, -0.21388202905654907, 0.34531134366989136, -0.25507116317749023, 0.16606774926185608, 0.7740017771720886, 1.3758983612060547, -1.010311245918274, 2.244762659072876, 0.21196608245372772, -0.6677922606468201, -0.8562031388282776, 0.5832969546318054, 1.8201786279678345, -0.42257246375083923, -1.089469313621521, 1.4492202997207642, -1.0714870691299438, 0.5839068293571472, 0.7351388931274414, -0.016495466232299805, -1.3410420417785645, 0.756488561630249, 0.5641587972640991, -0.0029218322597444057, 1.004654049873352, -0.4106641113758087, -1.526393175125122, -2.2575302124023438, 0.23859737813472748, -0.5562986135482788, 0.6953704357147217, 0.3404626250267029, 0.15129968523979187, -0.6203209161758423, 0.063806913793087, 0.9556921720504761, -1.0382581949234009, 1.3428839445114136, -0.1775844693183899, 0.6758065819740295, -0.6917272806167603, 0.8570429682731628, -0.1043778732419014, -0.001517641358077526, 0.33716660737991333, 1.7465507984161377, 0.7136356234550476, 0.2239709049463272, -1.1539636850357056, 0.012101172469556332, -1.238045573234558, 0.3685716986656189, -0.36055371165275574, 0.031235797330737114, -0.8264997005462646, -0.01167932990938425, 0.2668876051902771, 1.6840014457702637, 0.5267770290374756, 0.7078707814216614, -0.9992595314979553, 0.8918015360832214, -0.5841625928878784, -0.9892972111701965, -0.3919677436351776, 0.23650150001049042, -0.44198405742645264, -0.5960873365402222, 0.5667828321456909, -0.871913492679596, -0.380439817905426, -0.587908923625946, -0.013651865534484386, 0.13961847126483917, -0.836245596408844, 0.9898120164871216, -2.8554301261901855, 0.33680614829063416, 1.2173455953598022, 1.5702567100524902, 0.5069186687469482, 0.5804793238639832, -0.4200607240200043, -0.40663957595825195, -1.0223637819290161, -0.568574070930481, -1.6494179964065552, -2.1037511825561523, 1.0410175323486328, 0.31300222873687744, 1.2333980798721313, -0.3030479848384857, -1.350504755973816, 0.2949436604976654, 1.1531990766525269, -2.011070966720581, -0.38047170639038086, 0.054904405027627945, 2.128750801086426, 0.22440873086452484, -1.2694746255874634, -1.209602952003479, -1.146852731704712, -0.9923461675643921, 0.5868125557899475, -0.23126612603664398, 0.056178655475378036, -1.5770328044891357, 2.105036735534668, -0.7164774537086487, -1.5560753345489502, -0.512919545173645, 0.5452418327331543, -0.4897894263267517, -0.2233441174030304, 2.0557546615600586, -1.0299097299575806, 0.6280494332313538, -2.668330669403076, 0.06683304160833359, -0.6971495747566223, 0.7145875096321106, -0.05761535465717316, -0.8951294422149658, -0.12770245969295502, -0.07688138633966446, -0.40133944153785706, -0.914107620716095, -1.3307371139526367, -0.9239300489425659, -0.5489113330841064, -1.0649186372756958, 0.4963129162788391, -1.561405897140503, -2.066798686981201, -1.492009162902832, -1.1996415853500366, -0.5139029622077942, 0.4937843084335327, -0.260924756526947, 0.7878021597862244, -0.11639469861984253, 0.07733634859323502, 1.099268913269043, 0.002393025206401944, 0.35443392395973206, -0.5562495589256287, -1.1504085063934326, -0.5187245011329651, 1.5052831172943115, 0.12850889563560486, -2.5081727504730225, -1.362428903579712, 0.3671036660671234, -0.7674247622489929, 0.6514360308647156, 0.23178640007972717, -0.4764327108860016, 1.6697980165481567, -0.34524449706077576, 2.0695712566375732, 0.3641725480556488, 0.5600192546844482, 0.25023820996284485, -1.0634558200836182, -0.35904693603515625, 0.3554201126098633, -1.0170890092849731, 0.9945668578147888, 0.08421322703361511, 0.09665525704622269, -0.09078460186719894, -2.5115504264831543, 0.3320245146751404, 0.8143190145492554, 2.077313184738159, 0.7856117486953735, -0.047735102474689484, 0.21725349128246307, -0.43049174547195435, -0.15159983932971954, -1.730840802192688, 0.09637484699487686, 1.4444059133529663, 0.2682484984397888, -1.032415509223938, -0.5011813044548035, 1.0629050731658936, 1.2705570459365845, -1.402072548866272, -0.9204955101013184, -0.16330304741859436, 0.6171742677688599, 0.14191630482673645, 0.2636215090751648, 0.9756128191947937, 0.6684443354606628, -1.3206429481506348, 0.49622589349746704, -0.43643462657928467, -0.3996821343898773, 0.07055272161960602, -0.14079627394676208, -0.14497819542884827, 0.9118967652320862, 2.021278142929077, -0.31508854031562805, 0.42562732100486755, -0.8233931660652161, -1.307288646697998, -1.3422399759292603, -1.473580002784729, -0.11141392588615417, -0.24878768622875214, 1.2382745742797852, 0.31413745880126953, 0.36372390389442444, -0.4757508933544159, -0.5638067722320557, 0.8231837153434753, 0.2501618564128876, 0.6020488739013672, 0.7098177075386047, -1.0660945177078247, -0.2369580715894699, 1.6201928853988647, 0.7105090022087097, 1.9236706495285034, -0.04942002147436142, 0.6037179827690125, 1.575691819190979, 3.098357915878296, 0.6821735501289368, -0.6302111744880676, 0.0034660242963582277, -0.6857143640518188, -0.4619493782520294, 0.08819479495286942, -0.5416510105133057, 0.41368597745895386, -1.6099778413772583, 0.16098198294639587, -0.8082388639450073, -0.3404647707939148, 1.4887067079544067, -0.5158957242965698, 0.13361112773418427, 0.6512170433998108, 1.0306516885757446, 0.9626544117927551, -0.37627479434013367, -0.67633455991745, 0.9881640076637268, 1.333283543586731, -1.118929386138916, 0.9854008555412292, -0.0168533343821764, 0.5753874778747559, -0.7428656220436096, 0.9817422032356262, 1.0569665431976318, -2.0024991035461426, 1.508378028869629, 0.28081706166267395, -1.0761548280715942, 0.9623749256134033, -1.0347895622253418, 0.788589358329773, 1.4554024934768677, -0.48890239000320435, -0.7082083821296692, -1.5783082246780396, -0.6206771731376648, 1.7796722650527954, -1.094179630279541, 1.4440689086914062, 0.8876802921295166, -0.37380409240722656, 0.6375541687011719, 0.103664830327034, 0.22190959751605988, 0.7557260990142822, 0.5829514265060425, -0.5211575627326965, 0.19264675676822662, 0.24666668474674225, -0.17053650319576263, -0.6662943363189697, -0.09407810866832733, -0.08009974658489227, 0.11214112490415573, -1.2102422714233398, -1.9722485542297363, 0.7821441292762756, 0.17914681136608124, -0.975715696811676, 0.8059186339378357, -0.3310781419277191, -0.8140372633934021, 1.292372226715088, 1.0878539085388184, -0.3116684854030609, 0.9851370453834534, 0.6020064353942871, -1.4095549583435059, -1.5808022022247314, 0.16800585389137268, -0.5817394256591797, -1.0385444164276123, -0.27645811438560486, 0.2438696026802063, 0.7480424046516418, -0.2147715836763382, -0.8249471187591553, 0.894482433795929, -1.2201793193817139, -1.230031967163086, -0.5986416339874268, -0.6802225708961487, 0.30633261799812317, -0.5755220055580139, 0.8812527060508728, -0.7378180623054504, -0.6719323396682739, 1.7047152519226074, -0.3520689010620117, -0.602418065071106, 0.29352790117263794, 1.1967328786849976, 1.2692457437515259, -0.8181480765342712, -1.7465485334396362, -1.1991254091262817, -0.46909138560295105, 1.0246649980545044, 1.289578914642334, -0.7801135182380676, -1.0157361030578613, 1.0710642337799072, -0.13918302953243256, -0.13066576421260834, 1.4851638078689575, 0.693018913269043, -1.3669573068618774, 0.3423025906085968, -0.006052742246538401, 0.9004168510437012, -1.4527206420898438, -0.22787746787071228, -0.8512318730354309, 0.04989797621965408, -1.026625394821167, 0.7068508267402649, -1.082442045211792, 1.0134449005126953, 1.0995088815689087, -0.14998850226402283, 1.3312879800796509, 1.338014841079712, -0.16784027218818665, -0.9498894810676575, -0.6024373173713684, -0.6238706707954407, 1.0353435277938843, 0.7977229356765747, 0.2786141633987427, 0.7325548529624939, 0.8908532857894897, -0.7932363152503967, -0.47650113701820374, -1.3007442951202393, 0.7603474259376526, 0.5230316519737244, 0.8778253197669983, 0.2476375848054886, 0.2967016398906708, -0.6265799403190613, -0.7391803860664368, 1.441675066947937, -2.0677647590637207, 1.2472718954086304, -1.22791588306427, -0.6884373426437378, 1.0220812559127808, -0.7787095904350281, -1.835768222808838, 0.6508118510246277, 0.8719595074653625, 0.4449690580368042, -1.3926411867141724, 0.27443167567253113, -0.06139953434467316, -0.31569039821624756, 0.2801777720451355, -1.024943470954895, 1.3174673318862915, -0.6301632523536682, 0.1548791378736496, 0.4129609167575836, -0.07001426070928574, 0.7945465445518494, 0.041081104427576065, 0.5517203211784363, -1.1782623529434204, 0.46982088685035706, 0.004740480333566666, -1.2917652130126953, -0.023715466260910034, -0.6925429701805115, -0.561538577079773, -1.506384015083313, -1.0710582733154297, -0.8297767043113708, -0.1290997415781021, 0.027186434715986252, -0.29170915484428406, -0.9366260170936584, 0.013616473414003849, 1.1745659112930298, 0.6639390587806702, 0.2909291386604309, 0.3353727459907532, -0.19486062228679657, 0.09311743825674057, 0.36810216307640076, 0.31239694356918335, 0.6737570762634277, 1.076369285583496, 1.1380754709243774, 0.524297833442688, -0.24860087037086487, -1.0942487716674805, -1.6498135328292847, 1.022222876548767, 0.6211788058280945, -0.07957213371992111, -1.0141019821166992, 0.29273414611816406, 1.387865424156189, 1.4318499565124512, 2.3910272121429443, 0.9516160488128662, 0.9512190222740173, 1.3018566370010376, -1.5704575777053833, 0.8587192893028259, -0.6908383369445801, 0.42054301500320435, -1.5439469814300537, -1.0254284143447876, -0.9492470026016235, -0.5431014895439148, 0.7233948111534119, -1.0671905279159546, 0.7102158665657043, 1.0388729572296143, -1.1404279470443726, 0.9865483045578003, 1.7776020765304565, -0.6226539015769958, 0.6478496789932251, 1.480932354927063, -0.5868710875511169, -0.6231276392936707, -0.1307341307401657, -0.13755415380001068, -0.9965152144432068, -0.3977276086807251, -0.9202225804328918, 0.557403564453125, 2.3761556148529053, 1.0811265707015991, -1.3868377208709717, 1.228020191192627, -0.935524046421051, 0.5965280532836914, -0.279348760843277, 1.4463459253311157, 0.06359265744686127, 0.12031606584787369, 0.015882888808846474, -0.11934179067611694, 0.3579094707965851, 1.8525935411453247, 1.6328771114349365, 0.78110271692276, 0.12531240284442902, -0.05019533634185791, 1.4214770793914795, -0.8233321309089661, -0.9639562368392944, -0.643222987651825, 0.3277381360530853, 0.34441307187080383, 0.4474184811115265, -0.21767795085906982, -0.7044922113418579, -2.061532497406006, 0.3853450119495392, -0.7282955050468445, 1.0543184280395508, -1.117812991142273, 0.12191388010978699, 0.5444864630699158, 0.357079416513443, 0.770431399345398, 0.9263312816619873, 0.3274388909339905, 0.9055979251861572, -0.01342124305665493, -0.0698675736784935, 2.6798934936523438, -0.050890929996967316, -0.25570249557495117, -1.833501935005188, -0.9767597913742065, 0.038281477987766266, -0.7735971212387085, 0.021758724004030228, -1.290818214416504, 1.223110318183899, 0.15264274179935455, -0.4423760771751404, -0.4224531948566437, -0.24355284869670868, 0.7376647591590881, -0.6746431589126587, 0.3806312680244446, -0.32882457971572876, -1.300124168395996, -0.4380318820476532, -0.0031836088746786118, 0.4442856013774872, 1.5505515336990356, -1.0369185209274292, -0.6713353991508484, 0.47545769810676575, -0.10785908252000809, -1.5473101139068604, 0.8536614179611206, -1.1364012956619263, -1.4680951833724976, -1.0881061553955078, -1.9614566564559937, -0.8376564383506775, 0.3315235376358032, 1.0519400835037231, -0.45519036054611206, -0.7931910753250122, 0.02703843265771866, -0.559280276298523, 2.75072979927063, 1.668297529220581, 0.5324532389640808, -0.2325395792722702, 0.5093146562576294, 0.8449515104293823, -0.15920484066009521, 0.8561148643493652, 2.8641560077667236, -2.0386078357696533, 0.33423271775245667, 0.9835999011993408, -0.7014506459236145, -1.048711895942688, 0.4508914053440094, -0.09566190093755722, -0.35191449522972107, -1.124457597732544, 1.5770654678344727, -0.2669842541217804, 0.8729398250579834, 0.05067698657512665, -1.2834104299545288, 0.01779438555240631, -1.8440262079238892, 0.5364727973937988, -0.0008468187297694385, -1.6429386138916016, 0.515324592590332, 1.4200340509414673, -0.8402597308158875, -1.991278052330017, -1.884737253189087, -0.6473025679588318, 1.5753830671310425, -1.2744665145874023, 1.8963240385055542, -1.2016324996948242, 0.11579283326864243, -1.6344393491744995, 0.7476978898048401, -0.29614976048469543, 1.233794927597046, 0.103936105966568, 0.4748380780220032, 0.7171542048454285, 0.459953248500824, 0.3818747103214264, -0.23145581781864166, -0.15102888643741608, -0.6347934603691101, 1.3002485036849976, -0.4673907160758972, -2.1454646587371826, 0.40371066331863403, 0.9520857930183411, -0.6246804594993591, -0.08968441188335419, -0.448149710893631, 0.6055550575256348, 1.4635823965072632, 0.1575136035680771, -0.07249808311462402, 1.0814580917358398, 0.5565301775932312, -0.4168778657913208, -0.022524289786815643, 1.6593611240386963, 0.24149754643440247, 0.4873451888561249, -0.951802670955658, 1.4929081201553345, -0.47988244891166687, 0.12101221829652786, 0.26706990599632263, -0.8128165006637573, -1.0834583044052124, 1.2434717416763306, 0.0760277584195137, 1.0035152435302734, 2.4543447494506836, -0.6869756579399109, 0.09037258476018906, 0.17637589573860168, -1.612082839012146, -0.7915107607841492, 2.0376086235046387, 0.07485342025756836, 1.1731663942337036, 2.0016653537750244, -0.3014466464519501, 1.6856789588928223, 1.1270383596420288, -0.3084764778614044, 0.5642104148864746, 1.0287328958511353, -0.17441754043102264, 1.0070695877075195, 0.8592528104782104, -2.676353931427002, 0.3203377425670624, -0.6908507347106934, 0.7965518236160278, -0.03546055778861046, 0.9542853236198425, -0.3359668254852295, -1.1718419790267944, 1.7823712825775146, -0.3133690059185028, 1.029672622680664, -0.09886310249567032, -0.9996832609176636, -1.0366504192352295, 1.3897426128387451, 0.4461825489997864, 0.2595154047012329, -1.7561167478561401, 0.1688583642244339, -0.35086673498153687, 0.3951159119606018, -0.020496077835559845, -0.2854534983634949, 0.056494735181331635, -0.4297613203525543, 1.209861397743225, -0.5795184969902039, -0.4363517761230469, 1.286057710647583, 0.7864657044410706, 0.1788426637649536, 0.2059105932712555, 0.7374028563499451, -0.015345445834100246, -1.3074350357055664, 1.2314364910125732, -0.16548936069011688, -1.9827580451965332, -1.0510029792785645, -0.7575930953025818, -0.15954189002513885, 1.5111451148986816, -0.21218881011009216, -0.2237548679113388, -1.3096826076507568, 1.127375841140747, 0.043639831244945526, -1.0850536823272705, 0.7040722966194153, 1.5840582847595215, -0.3931346535682678, -0.17784075438976288, 1.0151631832122803, 2.2618961334228516, -1.3404477834701538, 1.450069785118103, -0.5893085598945618, -2.298924207687378, 0.8174931406974792, -1.7727570533752441, -0.8952125906944275, 2.0874528884887695, -0.8671852946281433, 1.1222206354141235, -1.2108792066574097, -0.3632550537586212, 0.430969774723053, -2.281891345977783, -0.6079671382904053, 1.8516777753829956, -0.7515299320220947, -0.26244089007377625, 0.018632451072335243, -0.6072034239768982, 2.066802978515625, -1.7029403448104858, 1.4047539234161377, 0.87850022315979, -0.017252996563911438, 1.2333168983459473, -1.5238844156265259, 0.11439212411642075, 1.006446361541748, 0.4349026083946228, -0.9236762523651123, 0.32567667961120605, 0.6640288233757019, -1.2346168756484985, -0.04779166728258133, -1.2451326847076416, -0.9613367915153503, -1.3764008283615112, 0.2765912711620331, -0.5604491233825684, -1.1097817420959473, 0.5066201090812683, -1.0108176469802856, -0.5151920318603516, -1.1983963251113892, -0.9168599843978882, 1.4421888589859009, -0.18622276186943054, -1.1528664827346802, 0.6586501598358154, -0.4964248239994049, -0.09020132571458817, 0.1589718610048294, 1.3513884544372559, 1.0484634637832642, -1.4813687801361084, 1.046642541885376, 1.538170576095581, -1.0175244808197021, -0.5012051463127136, 0.2403106391429901, 1.1787309646606445, -1.2767549753189087, 1.6043494939804077, 1.2013884782791138, 0.317741721868515, 1.9035661220550537, -1.0923784971237183, 1.996508002281189, 0.08641110360622406, 1.3108819723129272, -1.3447552919387817, 1.3745760917663574, 0.3623647093772888, -1.312438726425171, -0.22787296772003174, 2.150524616241455, -0.7434682250022888, 0.14053210616111755, 0.9357025623321533, 1.518667221069336, 0.08026202023029327, 0.3971741497516632, 0.22638583183288574, 0.8528579473495483, -1.4706530570983887, -0.8476631045341492, -0.9210489988327026, 1.3920220136642456, -1.2840622663497925, -0.8988012075424194, 1.0164191722869873, -1.9585981369018555, -1.0465610027313232, -0.6108725070953369, 1.359671950340271, 1.4255973100662231, 2.3126883506774902, -2.993704319000244, -1.3874874114990234, 0.6143301129341125, -1.9221243858337402, 0.8063139915466309, -1.5399023294448853, 1.0185858011245728, -0.5303502082824707, -1.2559502124786377, -0.32695263624191284, 2.153719902038574, 0.8433046936988831, 2.090036392211914, -1.0708286762237549, -0.07000871002674103, 0.7943392395973206, 1.1748913526535034, 0.06659334897994995, 0.21436768770217896, -0.42988070845603943, -0.07080940157175064, -0.5812516212463379, 1.0262020826339722, 0.40308472514152527, 0.006723609287291765, 0.49892091751098633, 0.7401888966560364, -0.460604190826416, -0.371256023645401, 0.22593453526496887, -1.7546731233596802, -0.5850371718406677, -0.6892315149307251, 0.764668345451355, -1.2901188135147095, 0.020473003387451172, 0.3233400285243988, -1.1375141143798828, 1.9081567525863647, -2.2590575218200684, -0.9953539371490479, 2.2312698364257812, 2.819115400314331, 0.2919973134994507, 0.9786405563354492, -0.5304405093193054, 0.08263450115919113, -0.2482275515794754, 1.043709397315979, 0.5629456043243408, -0.6697129607200623, 0.7435029745101929, -1.7635571956634521, 1.9518953561782837, -1.3774383068084717, -0.9983581304550171, 0.7627129554748535, 0.11856646090745926, 0.8499701619148254, -0.23036816716194153, 2.1337826251983643, -2.285409450531006, 1.0890907049179077, 2.695960760116577, 1.114061951637268, 0.317135751247406, 0.7814971804618835, -0.04022393748164177, 0.9380069375038147, -0.1841518133878708, 0.5552043914794922, 0.03728075325489044, 0.07296070456504822, -0.779215931892395, -0.41164374351501465, -1.555912733078003, -0.4568081796169281, -0.12271203845739365, 0.3523021638393402, -0.1099400594830513, -0.49601149559020996, -0.17680180072784424, -0.46651965379714966, 0.14701934158802032, 1.0340675115585327, -0.47204703092575073, 0.3694378137588501, -0.2135612964630127, -0.09220227599143982, 1.1975001096725464, -1.8690019845962524, -0.2060755342245102, -1.0548170804977417, -1.0123869180679321, -0.7715350985527039, -0.1284681260585785, 0.6082091927528381, -0.9689562916755676, -1.2474989891052246, 0.35806700587272644, 1.465334177017212, 1.117425560951233, 1.1232800483703613, -0.07473570108413696, 0.8215402364730835, 0.8016750812530518, -1.8521602153778076, 0.481738418340683, -1.1002578735351562, 1.6999613046646118, 0.05665784701704979, -0.500060498714447, 0.5419171452522278, 0.2536711096763611, 0.9344347715377808, 0.827714741230011, -1.2280470132827759, -0.6827393174171448, -0.15774306654930115, 0.605900764465332, 1.5551056861877441, -0.45965126156806946, 0.4063553214073181, 1.031361699104309, 0.7200781106948853, -0.6824256777763367, 0.04375452920794487, 0.7539891004562378, -0.05832012742757797, -0.02319895476102829, -0.46999701857566833, -1.0856297016143799, -0.41241219639778137, -0.9388788938522339, -0.3876267671585083, -0.2471262365579605, 1.6873306035995483, 1.015700101852417, -0.13052795827388763, 0.5827577710151672, -2.350677013397217, 0.15519529581069946, 1.38551664352417, -1.7359105348587036, -1.158949375152588, -0.661357045173645, -0.5808553695678711, 0.4778587818145752, 0.8121609687805176, -1.0576281547546387, -1.0206950902938843, 0.7189271450042725, 1.030380368232727, 1.090140461921692, 0.40127941966056824, -0.4614313542842865, -0.13636818528175354, -1.11601984500885, -1.1178617477416992, -1.2615872621536255, 0.024051157757639885, 0.20130743086338043, -0.34913405776023865, 0.08163019269704819, 0.47446250915527344, 0.3263437747955322, -0.4781292676925659, -0.1672191619873047, 0.4021010398864746, 0.49983710050582886, 1.0581822395324707, 3.098681926727295, 0.7763373255729675, -0.6603796482086182, 0.8049245476722717, -0.1770569533109665, -0.7928357720375061, -1.2734228372573853, -0.35779470205307007, -0.75249844789505, 1.8444817066192627, -0.1810499131679535, 2.0381786823272705, -0.7165346145629883, -0.0654601901769638, -0.37204036116600037, -0.6677572727203369, 0.4558664858341217, 0.7374860644340515, -0.213368758559227, -0.03692160174250603, 0.048815689980983734, 0.1962134838104248, -0.5470411777496338, -0.8113362789154053, 0.6096752285957336, -1.1371744871139526, 0.8665317893028259, -0.3198443353176117, 0.1866666078567505, -0.34500497579574585, 1.375518798828125, 0.18081432580947876, 1.9067174196243286, -0.4787919819355011, -0.4695799946784973, 1.2053254842758179, 2.0853264331817627, -0.9223760962486267, -0.8396336436271667, 0.8346738815307617, -0.11057782173156738, 1.0043903589248657, 1.0492349863052368, -0.16617166996002197, -0.30670925974845886, 0.5078884959220886, 1.1778969764709473, 0.7835136651992798, -0.9055512547492981, 0.26973721385002136, 0.7856980562210083, 0.9078106880187988, 0.027340643107891083, 0.051227591931819916, -1.5183472633361816, -0.6300894618034363, 1.83566415309906, -0.1455450803041458, -0.14522069692611694, 0.3694973289966583, -0.4246174991130829, -2.447916030883789, -2.07246732711792, -0.13274817168712616, 0.5626733303070068, -0.27668559551239014, 0.7453561425209045, -0.6462580561637878, 1.0655146837234497, 0.5842932462692261, -1.1563912630081177, -0.04400604963302612, 0.6392338275909424, 1.7789403200149536, -1.3228927850723267, -0.5049335360527039, 0.19721537828445435, -3.112128973007202, -0.3441047966480255, -0.06272953748703003, 0.09515140950679779, 1.5885111093521118, 1.2461236715316772, 0.5186995267868042, 1.0833635330200195, -0.43495434522628784, 0.3169543743133545, 0.549990177154541, -1.0901373624801636, 1.150537133216858, -0.1669478565454483, -1.9095181226730347, 1.1403839588165283, -0.8215040564537048, -1.4262313842773438, 0.15052339434623718, -0.03435678780078888, 1.2676067352294922, -0.9185501933097839, 1.5013359785079956, 1.0269508361816406, 0.4038267135620117, 0.5027269721031189, 0.9591103196144104, -0.7972054481506348, 0.7311956882476807, 0.5648959875106812, -1.566750407218933, -1.4464983940124512, 0.048598311841487885, 0.17966262996196747, -0.901223361492157, 1.25188410282135, 0.2904493510723114, 0.9778818488121033, -0.07078235596418381, -0.25239846110343933, 0.9736960530281067, 0.08135128021240234, 1.4764312505722046, 0.2859351933002472, 0.38838160037994385, -0.5348414778709412, 1.068085789680481, 1.8057951927185059, -0.23737139999866486, 3.4231441020965576, 0.15032753348350525, -0.01199411042034626, 0.6430261731147766, -0.3185162842273712, 0.4022747278213501, 1.0370306968688965, 0.44768333435058594, 0.11370252072811127, -0.6689943671226501, 0.6243605017662048, -0.14957737922668457, -0.8231247067451477, 1.47602379322052, 1.9025695323944092, 0.25762122869491577, 1.0016531944274902, 1.61436128616333, 0.8322001695632935, 0.37267300486564636, 0.2315828502178192, 1.3931628465652466, -0.9982144236564636, -0.8310883641242981, 0.6114102602005005, -0.6684849262237549, -1.2028636932373047, 0.13625185191631317, -0.03335382416844368, -0.7195682525634766, 0.7182945013046265, -0.1416747123003006, -0.6427590847015381, -0.3733343780040741, -1.369177222251892, 0.13933251798152924, -0.1835576593875885, 0.7436398863792419, -0.3926229476928711, 1.1520506143569946, 0.48766523599624634, 0.41144976019859314, -0.5018351674079895, 0.21435262262821198, -0.7475334405899048, -0.3346719443798065, 1.0008296966552734, 0.7018142342567444, -0.4643677771091461, 1.9028269052505493, 1.112383246421814, 0.7738712430000305, 0.7617173194885254, 1.2022106647491455, -0.0912233218550682, 0.5479734539985657, 0.7806429862976074, -0.9923449754714966, -1.4749407768249512, 0.3072822093963623, -0.6114837527275085, 0.2798720896244049, -1.269793152809143, 0.6153466701507568, 1.5459872484207153, 0.12027372419834137, 1.4340656995773315, -0.07841476052999496, -0.42092207074165344, -0.49965983629226685, -2.070422410964966, -0.005637360271066427, 0.14242100715637207, 0.36555200815200806, -0.175174742937088, -0.099821537733078, -0.30373501777648926, -1.86897611618042, -0.841509997844696, 0.30759015679359436, -1.2255609035491943, 0.803866446018219, 1.8362350463867188, 1.8545925617218018, -0.14204616844654083, -0.4241408407688141, 1.2836079597473145, -0.5961849689483643, -0.6248025894165039, 1.4060499668121338, -1.238849401473999, -0.8854982852935791, 0.8212612867355347, -0.25308817625045776, -0.2350536435842514, 0.8225547671318054, 0.9437452554702759, -0.32607635855674744, 0.19290724396705627, 0.8223180770874023, 1.9302417039871216, 0.09638053178787231, -0.5904714465141296, -0.018125373870134354, 0.6289617419242859, 0.5269557237625122, -1.601881742477417, -0.22195270657539368, -0.8140097856521606, 1.1399801969528198, 0.39076387882232666, -0.37341493368148804, 2.134692668914795, -1.2475998401641846, -0.3189769983291626, 0.7740548849105835, 0.6381967067718506, -0.5189910531044006, 0.4593605101108551, 0.773316502571106, 0.9453734755516052, -0.8775589466094971, -0.9187382459640503, -0.5094241499900818, -0.33576950430870056, 0.3639453947544098, -1.616464614868164, 2.778794288635254, 1.3525397777557373, 0.4750608205795288, 0.048741765320301056, 1.6309183835983276, 0.0569160170853138, 0.4232271909713745, -1.101260781288147, 0.20932158827781677, -1.3282233476638794, -0.3914427161216736, 0.742676317691803, -0.9810468554496765, -0.19538816809654236, -1.615723729133606, -0.8057589530944824, -0.5024837851524353, 0.3315887451171875, 1.1073817014694214, 1.2199519872665405, -0.44235193729400635, -0.9519596695899963, -0.37480100989341736, -0.6210317611694336, -0.3663500249385834, -1.279200553894043, -0.2128491997718811, 0.09171521663665771, 0.0256052166223526, -0.6312025189399719, -0.37164098024368286, 0.3151082992553711, 0.2093677967786789, 0.2778266370296478, 0.1395806223154068, -0.2803475260734558, 0.751820981502533, -0.970319390296936, 1.948349952697754, -0.7532632946968079, 0.35396185517311096, 1.616392970085144, 0.7906880974769592, 0.6041668057441711, -0.4568193852901459, 1.370819091796875, -0.3044435977935791, 0.34571707248687744, -0.398257315158844, -0.17073224484920502, -0.73673415184021, -1.3000491857528687, 1.1029006242752075, -0.9088062047958374, -0.015145082026720047, -0.2773882746696472, -0.5041791200637817, -0.6182844638824463, 0.1172352060675621, 0.05156558007001877, -2.3826141357421875, -0.9869762063026428, -0.23864729702472687, 0.7493827939033508, -0.12188761681318283, 1.1214648485183716, -0.13906678557395935, 0.023066259920597076, -0.273506760597229, 0.3027508854866028, -2.6428332328796387, 0.14131906628608704, 0.2536396086215973, -0.37358400225639343, -0.6528441309928894, 0.7266069054603577, -1.1340376138687134, 0.3053279221057892, 0.5146104693412781, -1.7715649604797363, -1.6964929103851318, -0.39637765288352966, 0.7607346773147583, -0.3900768458843231, -0.7105863094329834, 0.3825356364250183, 0.18612121045589447, -0.6396470069885254, 0.17906148731708527, 1.1366349458694458, -0.2811450660228729, 1.3919709920883179, 0.4761494994163513, 1.6217516660690308, 0.5991334915161133, 0.6980730295181274, -0.5535948872566223, 2.360935926437378, -1.5122928619384766, 0.722973108291626, 0.061902716755867004, -1.3476834297180176, -0.809354305267334, -0.5664619207382202, 0.6531075835227966, -0.15695396065711975, -0.2110522985458374, -3.272679090499878, -1.2971993684768677, -0.7718679904937744, -0.42996111512184143, -1.6068991422653198, 1.8468666076660156, -0.3271321654319763, 0.8998811841011047, -1.6724525690078735, -0.9032995700836182, 0.5209499597549438, -0.328082412481308, -0.5634551644325256, 1.4227224588394165, 0.013691144064068794, -0.8087970614433289, -1.1125609874725342, 0.3819234073162079, 0.5581685304641724, 1.349461317062378, 0.25133469700813293, 0.15148267149925232, 0.475730299949646, 2.309997320175171, 0.27040523290634155, 1.3635239601135254, -0.4248751401901245, -0.7073823809623718, -0.9627370238304138, 0.03841901570558548, -0.42465049028396606, 0.8620533347129822, 0.7250973582267761, 0.07366704195737839, -0.21472136676311493, -1.1742273569107056, -0.008106537163257599, 0.511496901512146, 1.8993550539016724, 0.7472545504570007, -1.030758023262024, 1.6758275032043457, 0.7487645149230957, -0.018341418355703354, -1.189374566078186, 0.2977895140647888, -0.9254221320152283, 0.03847305849194527, -1.5111382007598877, -0.5790376663208008, 0.16073814034461975, 0.6924355030059814, 0.9755429029464722, 1.1507842540740967, -0.23907433450222015, 0.27874448895454407, 1.1223704814910889, -0.7548146843910217, 0.15293703973293304, 0.12750020623207092, -0.1261242926120758, 2.0517654418945312, -0.9991444945335388, 0.28531068563461304, -0.27705076336860657, -0.5008499622344971, 0.5414227247238159, -1.293707013130188, -0.7479076981544495, 0.20398208498954773, 0.7368305921554565, 0.008434930816292763, -0.21143648028373718, -2.3218743801116943, -1.1152150630950928, -0.8593639731407166, -1.9247325658798218, 0.990803062915802, -0.6473472118377686, 0.07181411981582642, -0.6519342064857483, 0.3319482207298279, 2.087502956390381, -1.4470939636230469, 0.04311120882630348, 0.5056166648864746, -1.1301127672195435, -1.351857304573059, 0.886946439743042, 1.3307636976242065, 1.3306653499603271, -1.8695605993270874, 2.5331244468688965, -0.600114643573761, -0.12313149124383926, 2.5101735591888428, -0.7815437316894531, 2.5205304622650146, 0.13628363609313965, -1.5777077674865723, 2.0506064891815186, -0.4333832263946533, -1.1588351726531982, 0.08944623172283173, 1.341809868812561, 0.5365514159202576, 0.7201502919197083, -1.1969757080078125, -0.5206983089447021, -0.3670833706855774, 1.5721182823181152, 0.26364707946777344, -0.2856065034866333, -1.5998477935791016, -1.497666597366333, 0.09559670090675354, 1.049782395362854, 0.7606340050697327, 0.5206311345100403, -0.8651037812232971, 0.5896475315093994, 1.1693049669265747, -1.7779414653778076, 2.140326976776123, 0.20744504034519196, -0.8591791987419128, 1.755409836769104, -1.8986343145370483, -2.9495203495025635, -0.8988288640975952, 0.3390250504016876, -0.1168571263551712, 0.5329219698905945, -0.7187793850898743, 0.9328434467315674, 1.1550379991531372, 0.2762061059474945, -0.25969749689102173, -0.2008938491344452, -1.0344390869140625, 2.5171873569488525, -0.26769015192985535, 0.9134128093719482, -1.4844456911087036, -0.35610395669937134, -0.10965461283922195, -0.18221253156661987, 0.20705267786979675, -0.34546685218811035, -0.04063970223069191, -0.022343313321471214, -0.6170628666877747, 1.4321919679641724, 0.7252956628799438, -0.49257904291152954, 0.6185025572776794, 0.8271253705024719, -0.17988547682762146, -0.8419985771179199, -0.7743129730224609, 1.9901957511901855, -1.0843427181243896, -0.34124523401260376, -0.036197420209646225, -0.7212685942649841, 0.10790996253490448, 0.04163016006350517, 0.49497678875923157, -0.4084993302822113, 0.9521082639694214, 0.27177244424819946, 1.7383310794830322, 1.1864389181137085, 0.09492229670286179, -1.3250354528427124, -0.16960306465625763, -1.095820665359497, 1.5498982667922974, -0.12045329064130783, -0.4742981493473053, -1.1015018224716187, -0.8328718543052673, 1.1351121664047241, -0.47953930497169495, 1.3738147020339966, -0.08282195031642914, -0.08579310029745102, -2.101907730102539, -0.02063450589776039, -1.0872405767440796, 0.7004830241203308, 0.18480068445205688, -1.0386170148849487, 1.1733566522598267, -0.8927441239356995, 0.43378135561943054, -0.2886607050895691, -0.2534105181694031, -1.1143426895141602, 1.4015978574752808, 0.48302140831947327, -1.4354702234268188, -0.9698762893676758, 0.5584094524383545, -0.3340882956981659, 0.11554157733917236, 0.34803882241249084, -0.1381947249174118, -1.010888695716858, 1.6456536054611206, 1.043554425239563, 0.641879677772522, -0.47027260065078735, 0.3403353989124298, -1.637318730354309, 0.31059178709983826, -1.0324575901031494, 0.80331951379776, -0.9692904353141785, -1.2482426166534424, -0.9580772519111633, -0.349570631980896, -2.395490884780884, 1.02120840549469, 0.1742871254682541, 2.193549394607544, -0.40297234058380127, -1.4282649755477905, 0.9330501556396484, -0.4895865321159363, -0.1309286504983902, -0.047895535826683044, 0.4803883135318756, 0.00036197411827743053, 0.4494345784187317, -0.8527477979660034, 0.09871921688318253, -0.5783620476722717, 0.4067985713481903, 0.35303765535354614, 1.1843852996826172, -1.3189589977264404, 0.4158176779747009, -0.4868813455104828, 0.35888752341270447, -1.0400264263153076, 0.721124529838562, 0.07362053543329239, -0.009954816661775112, 1.8179506063461304, 0.8250675201416016, -0.11009766906499863, 0.9191710948944092, -0.49459218978881836, 0.2183225005865097, 0.46514394879341125, 2.0810329914093018, 2.092360019683838, 0.42165613174438477, 0.5897051692008972, -1.5766115188598633, -0.3534921705722809, -1.1650294065475464, 1.7173633575439453, 0.046504098922014236, -0.9440734386444092, 0.3013751804828644, 1.4529871940612793, 0.17433778941631317, -0.25926175713539124, 1.1157087087631226, 0.3603856861591339, -0.19759204983711243, -1.4541035890579224, -0.15644890069961548, -0.14616918563842773, 0.8535190224647522, -0.18719595670700073, -0.1969655454158783, 0.3844602108001709, 0.8258597254753113, 2.0108067989349365, -1.2293431758880615, -0.7758119106292725, 1.8345099687576294, -0.07525230944156647, -0.8250781893730164, 0.35499870777130127, 1.5823025703430176, 0.33051419258117676, 0.15427318215370178, 0.3558332026004791, -0.4041898548603058, -1.0460964441299438, 2.1138546466827393, -0.5906659960746765, 0.03172648698091507, 1.42988920211792, -1.124498724937439, -0.7889022827148438, 0.960867166519165, -0.054666101932525635, 0.32112205028533936, 0.2336588054895401, 2.23572039604187, -0.6754054427146912, -0.8338848352432251, 0.7780001163482666, 1.135817050933838, -0.28879469633102417, -0.3203764855861664, -2.0567471981048584, -0.9597899317741394, -0.13978539407253265, 0.006344988010823727, -0.1446458250284195, 0.17440220713615417, 0.3829522728919983, 1.7985862493515015, -0.3145253360271454, 1.0374982357025146, -1.0583378076553345, -0.15215668082237244, -0.4811697006225586, -0.6461989283561707, -0.12981659173965454, 0.7955266833305359, -0.4756702780723572, -1.8531233072280884, 1.0557363033294678, 1.974324107170105, 0.5996620059013367, 0.02336476556956768, -0.1830005645751953, 0.5410553812980652, 2.1879780292510986, 0.6724725365638733, -0.6934752464294434, 0.7672923803329468, -0.8942654132843018, 1.2651159763336182, -0.03594416379928589, 0.7445619702339172, -0.20458011329174042, -2.227382183074951, 0.3187079429626465, -0.5541889071464539, 0.14796197414398193, 1.306410789489746, -1.322993516921997, -0.7481926679611206, -1.166988730430603, -0.05214514210820198, 0.3211866021156311, 0.02934529259800911, 0.433289110660553, 0.884604275226593, 1.2009330987930298, -2.2927305698394775, -1.4063972234725952, 0.01705646701157093, -0.8951613306999207, 0.7812317609786987, 1.2832293510437012, -0.9088362455368042, -0.020795177668333054, -0.19019284844398499, 1.0426193475723267, -0.5563148260116577, -0.15895888209342957, 0.7616363167762756, -0.09478789567947388, -0.7054675221443176, 1.5714842081069946, -0.9216277599334717, -0.7346015572547913, 2.0279526710510254, -0.027256395667791367, 0.9563698172569275, -0.994199812412262, -2.1476123332977295, 0.9094680547714233, -0.11861547827720642, -1.1559230089187622, 0.21687103807926178, -0.17896172404289246, -0.04178827255964279, 0.8629907369613647, -0.7816668748855591, 0.8083857893943787, -0.43306148052215576, 0.7003906965255737, 1.1607213020324707, 1.6878571510314941, 1.198935866355896, -1.2071900367736816, -1.2366849184036255, 1.1824573278427124, -0.6931714415550232, -0.5483407974243164, -0.6146999001502991, 2.519216537475586, -1.6103076934814453, 0.004777722992002964, 0.0860762670636177, 0.5288235545158386, -0.38092637062072754, -0.2510833144187927, 0.25671687722206116, 0.22421661019325256, -0.2630936801433563, 0.13736124336719513, 0.47631657123565674, -1.1629101037979126, -0.9732142686843872, 1.4627606868743896, 0.18768750131130219, 1.3252322673797607, -0.754182755947113, 0.8083850145339966, -1.015851378440857, 0.8240447044372559, 0.1722269505262375, 1.1671926975250244, -0.9007735848426819, 0.9096366763114929, -0.5721641182899475, -0.8452796339988708, 0.571258544921875, -1.134080410003662, 0.4919615089893341, -0.12199939787387848, 0.506162703037262, 0.14834406971931458, 0.10151249915361404, -0.2596858739852905, -0.5617241263389587, -0.9058637022972107, 0.6702986359596252, -0.255083292722702, 1.753920555114746, 1.0937068462371826, -0.4542454183101654, 0.49546751379966736, 1.2256767749786377, -0.7961490750312805, -1.893797755241394, 0.9757081270217896, -0.7272136807441711, -1.6787720918655396, 0.1761254072189331, -0.3869483172893524, 0.6995066404342651, 1.4556081295013428, 1.7510865926742554, 0.6957346200942993, 0.7486576437950134, -1.0942045450210571, 1.345004916191101, 0.7510249018669128, -0.7552568316459656, -0.2929602861404419, -1.036940097808838, -0.10376632213592529, -0.42488035559654236, 0.06119374930858612, -1.8599605560302734, 1.1867321729660034, -0.051807042211294174, -0.5871240496635437, -0.13920146226882935, -1.007132649421692, 0.5388684272766113, -0.13234686851501465, -0.751606285572052, 0.16687604784965515, -0.6438098549842834, -0.08488720655441284, -0.5632094144821167, 1.474257469177246, 0.5490497350692749, 0.3353990614414215, 0.5300071239471436, 0.8431238532066345, 1.7612402439117432, 0.07438197731971741, 0.6598042845726013, 0.5806466341018677, 0.8949946165084839, 0.634346067905426, -0.6518761515617371, 0.26683729887008667, -0.6775082945823669, -0.6037220358848572, -0.24994339048862457, -0.8069313168525696, 1.3725370168685913, -0.42962607741355896, 0.9745369553565979, 0.4316500127315521, -0.4374271035194397, 0.9488667249679565, -3.0322108268737793, -0.9825255870819092, 1.8125609159469604, 0.11017833650112152, -0.4304686486721039, 0.7969231009483337, -1.2141494750976562, -1.7004917860031128, -0.6877467632293701, -1.4547467231750488, -0.5658293962478638, -0.3835189640522003, -0.22086840867996216, -0.42439648509025574, -1.6292603015899658, -0.2520989775657654, -1.982629418373108, -0.6938722133636475, 1.7100268602371216, 0.4324895739555359, -1.1039224863052368, 0.19299177825450897, -0.08561372011899948, -0.5560290813446045, -1.3751596212387085, 0.15151169896125793, 0.2434602975845337, 0.785773754119873, 0.4561866819858551, 1.9754674434661865, 0.07401866465806961, -3.5269598960876465, -1.221888780593872, 0.2700630724430084, -1.1179602146148682, 1.2079943418502808, -1.7790645360946655, -0.6838284134864807, -0.6564601063728333, -0.7786406874656677, 1.3596372604370117, -0.6778258085250854, 0.4664056897163391, -0.7531455159187317, 1.309615135192871, 0.8819910287857056, 2.0653650760650635, -1.0113542079925537, -0.07778719067573547, 0.551493763923645, -0.2250053584575653, -0.8699713945388794, -1.7411236763000488, 0.2454504668712616, 0.2149377167224884, -0.06298650056123734, -0.017357345670461655, -0.3924051821231842, -0.4594787061214447, -2.8271641731262207, 0.601815938949585, -0.058316320180892944, 0.22621706128120422, -1.1280720233917236, -0.906105101108551, -0.5881222486495972, -0.07925226539373398, -0.5883989930152893, 0.5542637705802917, 0.29943564534187317, 0.37188076972961426, 0.4066994786262512, -0.03139590844511986, -0.7362465262413025, -0.8253023624420166, -1.644286036491394, -2.0498039722442627, -0.7428134083747864, 1.0202833414077759, -0.4502522945404053, -0.3559405207633972, -0.034260135143995285, -0.4308425486087799, -1.4185696840286255, -1.1634151935577393, 1.5580384731292725, -0.35859158635139465, 1.142473816871643, 0.8004852533340454, -0.5174044370651245, 1.5506336688995361, 0.5110357999801636, -1.9554469585418701, -0.6696092486381531, 0.4199124574661255, -1.7386000156402588, 0.10796462744474411, 0.8928626179695129, -0.28401118516921997, 0.2130529135465622, -0.16590455174446106, 0.4673893451690674, 0.5103428363800049, -0.3328480124473572, -2.3695619106292725, -0.8476943969726562, -0.4949849247932434, 1.1422886848449707, -0.04283105581998825, -2.1300172805786133, 0.37622493505477905, -0.846796452999115, -0.3750384449958801, 0.6296527981758118, 0.7113891243934631, 2.7826921939849854, 1.1983537673950195, 0.0036916618701070547, 0.8514811992645264, 0.31044965982437134, -1.1405121088027954, -0.23484428226947784, 1.0877807140350342, 0.9097931981086731, 0.5477544665336609, 1.5108529329299927, 0.5317018032073975, 1.0679398775100708, -0.46985703706741333, -0.8187240958213806, 0.3754863739013672, 0.9476084113121033, 0.5764954686164856, 0.620622456073761, 1.62830650806427, 0.1036040261387825, 0.793591320514679, -0.2167225331068039, -0.046653635799884796, 0.8390509486198425, 2.0027880668640137, -0.26363739371299744, -1.0306788682937622, 1.1329033374786377, 1.548959732055664, 0.37427088618278503, -0.24942079186439514, 0.20713143050670624, -0.14288420975208282, -1.4485104084014893, 0.24054470658302307, 1.0481668710708618, -0.526020348072052, 1.4934685230255127, -1.918393611907959, 1.0062718391418457, -1.5745880603790283, -0.47217655181884766, -0.9591631293296814, -0.9813115000724792, 2.2057242393493652, 0.00361064774915576, 1.6808359622955322, 2.0980870723724365, 0.5476152300834656, -0.3094250559806824, -0.3183049261569977, -0.9133190512657166, 0.3735787868499756, 1.2430529594421387, -0.7793294191360474, -0.8237756490707397, -0.04408667981624603, 1.6917836666107178, -0.22032348811626434, 1.6930510997772217, -1.2110638618469238, 0.5003395676612854, 1.6070125102996826, -0.8057965040206909, -0.5293818116188049, 1.0145018100738525, -0.1714615374803543, 0.5417417883872986, 0.6597731709480286, 1.2126590013504028, 1.8490939140319824, -0.8935555219650269, 0.11363638192415237, 1.1567258834838867, 0.8070096373558044, 0.7738924622535706, -0.6375279426574707, 0.6961275339126587, -0.8286320567131042, 1.5739965438842773, -0.26958101987838745, -2.7348530292510986, -2.843966484069824, 0.7632014751434326, 1.8495872020721436, 0.45514237880706787, 0.6343014240264893, 0.2185051292181015, -0.00021515499975066632, -0.31086137890815735, -1.653571367263794, -0.035153016448020935, 0.5835023522377014, -0.42455336451530457, 0.3681681156158447, 0.8050702810287476, -1.6696693897247314, 0.23026694357395172, -0.254885196685791, 1.1802427768707275, 0.6063275337219238, 0.45702821016311646, -1.803636908531189, 0.844356119632721, -0.843498945236206, -0.8400782942771912, -0.36785247921943665, 0.5212110280990601, -0.4955922067165375, 2.060671329498291, 1.801753044128418, -0.5531667470932007, 0.16140824556350708, -0.8455464243888855, -1.369185447692871, -0.992654025554657, 0.8641502261161804, -0.1252424269914627, -0.7145529985427856, -0.6069270372390747, -1.2103816270828247, -0.19077341258525848, 0.47987228631973267, 1.6234309673309326, 0.49736616015434265, -0.3566955029964447, -0.7719172239303589, 2.7637827396392822, -0.3646998405456543, -0.0202327910810709, -1.7627086639404297, -0.30997371673583984, -0.4344753324985504, 0.1900818645954132, 1.2685935497283936, 0.7055661082267761, -0.8991524577140808, -1.5819804668426514, -0.17899231612682343, 0.492775559425354, -0.7202668190002441, -0.42802736163139343, 0.3514484763145447, -1.0021454095840454, 0.6346113681793213, -0.7759697437286377, 1.8846770524978638, -0.02434002235531807, -0.5541463494300842, -0.29208728671073914, -1.088049292564392, 0.24675355851650238, -1.308998703956604, 0.8512901663780212, -0.12723548710346222, -1.3989498615264893, 0.598767101764679, 0.3616858422756195, -1.3116480112075806, 0.44790077209472656, -0.22203952074050903, 0.289366751909256, -1.4789578914642334, 1.1919533014297485, -0.6255374550819397, 0.333749920129776, 1.4728518724441528, -0.10408619791269302, 0.6943622827529907, 1.2830051183700562, -0.9817785024642944, -0.6732786893844604, 0.2888522446155548, 0.9233606457710266, -0.5647050738334656, 1.4293580055236816, 0.9955511689186096, -0.03587799519300461, -0.5648581385612488, 1.8201402425765991, -0.6586771607398987, 0.7385415434837341, -2.3101487159729004, 0.11645790189504623, 0.16511230170726776, -0.20040272176265717, 0.9843433499336243, 0.6201378107070923, -1.2558435201644897, 1.185249924659729, -0.013208632357418537, 1.572298526763916, -0.2571812868118286, -0.1687449812889099, 0.5325621366500854, -0.8824224472045898, 0.4450269043445587, -0.18777428567409515, -1.0126686096191406, 0.9155533909797668, 0.6662111282348633, -1.1765234470367432, 0.8700821399688721, 1.299789309501648, 0.6978880167007446, -0.8059459328651428, -0.33108144998550415, -0.45614880323410034, 0.7944746613502502, -1.5574874877929688, 0.3269787132740021, 0.4845273196697235, -2.010056734085083, 1.239193081855774, 0.4800659418106079, -1.7669016122817993, -0.4182935655117035, 0.8485519289970398, -0.9344991445541382, -1.9082621335983276, -0.9052777886390686, -0.5795618295669556, 0.27629029750823975, 0.6038072109222412, -0.8654646277427673, -2.082045316696167, -0.579657793045044, -0.7515884041786194, -0.05290907993912697, 1.2314743995666504, -1.1729902029037476, 1.6400971412658691, 0.3849848806858063, 1.0456244945526123, -2.3417346477508545, -0.15211234986782074, 0.6733006238937378, -0.531130313873291, -0.12791390717029572, -0.4466612637042999, -0.3967263698577881, -0.52825528383255, 0.9102756381034851, -0.6954976320266724, -0.5215584635734558, 1.2963110208511353, -0.05588262528181076, -0.4482376277446747, 2.5020642280578613, -2.3961124420166016, 0.6187834143638611, 0.1229967400431633, 2.455721616744995, 0.3803538978099823, -0.8127440214157104, -0.18434345722198486, -0.9744108319282532, 1.763518214225769, 0.15738245844841003, 0.228117898106575, -0.23588529229164124, 0.646166980266571, -0.9622747898101807, 0.11936915665864944, -0.4201800525188446, 0.38179436326026917, -0.15483535826206207, 0.32987263798713684, 1.292147159576416, 0.06334470957517624, 0.6779922842979431, -0.38461264967918396, -0.649545431137085, -0.18720197677612305, -0.38024649024009705, 0.16952306032180786, -0.2974901497364044, -0.3701842427253723, -1.0593035221099854, 0.7046212553977966, -0.6509758234024048, -0.41367393732070923, 1.1095768213272095, 1.998611569404602, 1.4478235244750977, 1.0159547328948975, 1.5395406484603882, -0.5828961730003357, 1.296472430229187, 0.7756607532501221, 0.07048378139734268, 0.22201792895793915, 0.5348454713821411, -1.1599208116531372, 0.8871437311172485, 0.45420581102371216, 0.10602313280105591, -1.2276360988616943, 0.651050865650177, -0.7307875156402588, 1.3873615264892578, -0.30329668521881104, 0.18324504792690277, -2.2757821083068848, 0.677388072013855, -0.10720115900039673, 1.0162253379821777, 1.3063050508499146, -0.9446171522140503, -1.3065649271011353, -0.640247106552124, 0.774061381816864, 0.046341340988874435, 0.4913913309574127, -0.45208755135536194, -0.46349871158599854, -0.8157921433448792, 1.3432273864746094, 1.1058207750320435, 0.6596536636352539, 2.212134599685669, 1.654234766960144, -0.6803878545761108, 1.1553601026535034, 0.7008917331695557, 0.9626690149307251, -0.8523895144462585, 0.6998773813247681, 0.10575052350759506, 1.5863280296325684, 0.3113170564174652, 0.4531663656234741, 1.2708722352981567, -0.43231329321861267, 1.4781829118728638, 1.9055699110031128, 0.395920991897583, -0.11045685410499573, -0.1144433245062828, -0.22452689707279205, 0.5398847460746765, -0.33241358399391174, 2.003830671310425, 0.46424537897109985, 0.8324267864227295, -0.014392365701496601, 0.48725706338882446, 0.6234115362167358, -0.11800512671470642, -0.28849557042121887, -1.438502311706543, -0.994087278842926, 0.12897397577762604, 1.9276483058929443, 0.12535423040390015, -0.41686397790908813, 0.7098380923271179, 0.6318171620368958, 0.23555588722229004, 0.25616979598999023, -1.7417831420898438, 1.5843356847763062, -1.265227198600769, -1.2636888027191162, -1.0653390884399414, -1.0149686336517334, -0.6428592801094055, -0.32534343004226685, 1.137855887413025, -1.1525840759277344, -0.13503585755825043, -0.22047515213489532, -0.20154190063476562, 0.09199944138526917, 1.9562021493911743, -0.2756762206554413, 0.22250770032405853, 0.25261273980140686, 1.4145053625106812, 1.28972327709198, 0.2497473657131195, 0.5443851947784424, 0.41735029220581055, -0.35749781131744385, -0.0014471383765339851, 1.0118188858032227, -1.9551002979278564, -0.44758856296539307, 1.158389925956726, 0.1346183866262436, 0.1719084084033966, 0.2388676404953003, 1.4240175485610962, 0.9205069541931152, -0.8550165891647339, 1.190624475479126, -0.32849037647247314, -0.5276917219161987, -0.24827787280082703, -1.29964280128479, 0.15287061035633087, -0.26303979754447937, -1.6222939491271973, 0.42036768794059753, 1.450477123260498, -1.0999783277511597, -1.1207869052886963, -2.178100824356079, 1.2485156059265137, -0.011144801042973995, -0.8152625560760498, 1.0820852518081665, 0.27824175357818604, 2.645566940307617, -0.1866065412759781, 0.7544766068458557, -0.1388787180185318, 1.1766908168792725, -0.8280033469200134, 1.372958779335022, -0.20308496057987213, 0.1693515032529831, 0.23036104440689087, 0.0753839984536171, -0.3379809260368347, -0.5395922660827637, -0.27986228466033936, 0.4392601251602173, -0.03681855648756027, 0.8722963929176331, 0.041257426142692566, 1.1011477708816528, -0.4502023160457611, 1.1437747478485107, -2.206181764602661, -1.405090570449829, 0.1012272760272026, 0.8671480417251587, 1.4724653959274292, -0.13858802616596222, 1.2291747331619263, -0.02470695972442627, -1.1390926837921143, 1.1387284994125366, 0.4866555631160736, -0.18451784551143646, 0.4240858852863312, 0.38679206371307373, -0.0065453690476715565, -0.027642622590065002, -0.6204506158828735, 0.583794116973877, -1.0846387147903442, -0.6703898906707764, 1.4941751956939697, -0.21019414067268372, 1.0962533950805664, -1.2651764154434204, -1.8944867849349976, 0.5561314821243286, -0.15348899364471436, -0.20372578501701355, -1.200441598892212, 0.5052489042282104, -1.8935153484344482, 0.09751754999160767, 1.4589464664459229, -0.32943102717399597, 0.6081149578094482, -2.1343834400177, -0.9765879511833191, -0.3552951216697693, 0.6440419554710388, -0.6363621950149536, 0.23533189296722412, 0.10079850256443024, -0.6904414296150208, 0.3831084370613098, -1.2361682653427124, 0.477484792470932, 1.2753100395202637, -0.4408130347728729, -0.6975442171096802, -0.41048210859298706, 0.07097875326871872, 0.12036346644163132, 0.027857020497322083, -0.926244854927063, 0.25810620188713074, 1.1367337703704834, -0.3001821041107178, 1.5943522453308105, 0.9836565852165222, 1.00142502784729, 0.4911838471889496, -0.14149878919124603, 1.2003779411315918, -0.15667827427387238, -0.45112746953964233, 1.3256815671920776, 0.6878237128257751, -1.8332717418670654, 1.061937928199768, 0.30619025230407715, -0.23204803466796875, 0.06973563134670258, 0.11592838913202286, -0.8893889784812927, 0.978636622428894, 1.803151249885559, 1.3864381313323975, 0.5151457190513611, -1.1640093326568604, -0.5563866496086121, -1.677516222000122, -0.21263344585895538, 0.035552285611629486, 0.24527597427368164, -1.3859037160873413, -0.320792555809021, -1.5833600759506226, -1.3927323818206787, -0.12313906103372574, -0.6137268543243408, 1.303622841835022, 1.3447312116622925, 0.6052956581115723, 1.1216405630111694, 0.1237095296382904, -2.192207098007202, -0.8572380542755127, 0.11746706068515778, 1.9416508674621582, -0.6600398421287537, -1.1740590333938599, -2.6152873039245605, -0.30711305141448975, -0.18655288219451904, -0.6122815608978271, 0.2057551145553589, 1.3651012182235718, -0.7227754592895508, -1.4828625917434692, 1.191133975982666, -1.1081688404083252, 0.1881529539823532, 1.6552884578704834, 0.047489456832408905, 1.2720025777816772, -0.2851639986038208, 0.7043618559837341, 0.6555595397949219, 0.9178925156593323, -1.0130956172943115, 0.7635275721549988, -0.910542368888855, -0.36555561423301697, -0.8403845429420471, -1.1576813459396362, -1.1111235618591309, -0.327563613653183, 1.694427728652954, -0.9723818302154541, 1.3977676630020142, -0.8245160579681396, 0.9484590888023376, -1.8738168478012085, 0.5769997835159302, 0.16996149718761444, 1.3055540323257446, -2.1455466747283936, -0.21046370267868042, -1.7801061868667603, 0.6510570049285889, -1.907694697380066, -1.4416487216949463, 0.3865306079387665, 0.47305363416671753, 0.10977110266685486, 0.3809099495410919, -0.6778548359870911, 1.4243677854537964, -0.027751101180911064, -1.8437016010284424, -0.6413598656654358, -0.6379948258399963, 1.1039814949035645, 0.834296464920044, 0.5493330955505371, -1.2533884048461914, 0.7202978730201721, -0.8232497572898865, -0.2826038599014282, 0.3206382393836975, -1.6114169359207153, 0.39319244027137756, -0.3789574205875397, -1.3189369440078735, 1.1936808824539185, 0.24072760343551636, 0.3160307705402374, -1.5271096229553223, 0.7587989568710327, -0.22651559114456177, -1.4444955587387085, 0.6129714250564575, 0.24387216567993164, 0.2719672918319702, 0.6581708192825317, -0.8046095371246338, -0.05397602170705795, 0.25221237540245056, 0.7283226847648621, -1.3032097816467285, -1.4085551500320435, 1.257759690284729, -0.3903457522392273, -2.0043420791625977, 0.6879453659057617, -0.8676369786262512, -0.4051544666290283, 0.08926921337842941, -0.8459975123405457, -1.8993531465530396, -0.9174767136573792, -0.10938658565282822, -1.3445030450820923, -1.2883301973342896, 1.5708588361740112, 0.28874149918556213, -0.22959227859973907, -0.7086868286132812, 0.39481720328330994, 0.26903894543647766, -1.6405611038208008, -1.0465071201324463, -0.02242831327021122, 0.5132469534873962, 1.0133219957351685, 1.1499049663543701, -0.257793128490448, -0.343418687582016, 0.46145230531692505, -1.4985921382904053, -1.092057466506958, 0.9258062839508057, -1.5086678266525269, 0.26615577936172485, 0.2800973057746887, -1.6301958560943604, 0.04852263256907463, 0.5525507926940918, 1.4505118131637573, 1.5981765985488892, 0.0864635780453682, -0.22426117956638336, -0.006411214359104633, -0.9308244585990906, -0.8712772727012634, -0.9833381772041321, 2.0533626079559326, -0.0013330749934539199, 0.019905518740415573, 1.6542667150497437, -0.8342505097389221, 0.6940916776657104, 0.9622942805290222, 0.16763071715831757, -0.008901608176529408, 0.9018924236297607, 1.7586411237716675, -0.34705209732055664, 2.8943638801574707, 1.6327780485153198, -1.6984455585479736, -0.025865139439702034, -0.5131883025169373, 1.322936773300171, 2.2946534156799316, 0.7409218549728394, -0.1232469230890274, -0.6332864761352539, 1.141118049621582, -0.8617064952850342, 2.272268056869507, 1.1405768394470215, 2.0610954761505127, 0.2042611986398697, 1.9521714448928833, -0.26406678557395935, 0.39606937766075134, 0.7642431855201721, -0.32181671261787415, -0.2428097277879715, 1.7416540384292603, -1.0879478454589844, 2.13153338432312, -0.3908078968524933, -0.6114645004272461, -1.4880280494689941, -1.4631530046463013, -1.0565649271011353, 0.043497323989868164, 0.47842511534690857, 0.567094087600708, 0.10118638724088669, 0.017955457791686058, 1.528372049331665, 0.13258656859397888, 0.15945599973201752, -0.11496848613023758, -0.6673130393028259, -0.286099374294281, -0.39111945033073425, -0.13604110479354858, 3.5178210735321045, 1.3668889999389648, -1.01339590549469, -1.4137153625488281, -0.5022480487823486, 0.07745496183633804, 0.38862141966819763, 0.1477193534374237, -1.6568145751953125, -1.7813773155212402, -0.5138444304466248, -0.7832794785499573, -2.285306930541992, -1.442802906036377, 0.1341405212879181, -1.2943392992019653, -0.5389161109924316, 0.33881667256355286, 1.2698975801467896, -0.9180983304977417, 0.0445450097322464, -0.09201318770647049, -0.758849024772644, -0.7403687238693237, 1.3833351135253906, -0.6452460289001465, -0.03079674020409584, 0.13655142486095428, 1.6838027238845825, 0.08240833133459091, 1.7578758001327515, 1.1219369173049927, -0.6138486266136169, 0.7913243174552917, -0.8650104403495789, -0.11599523574113846, -0.39066535234451294, 1.4669184684753418, -0.8504008650779724, 0.9801990389823914, 0.22092624008655548, 0.8349664211273193, -2.3756494522094727, -0.19557863473892212, -0.17326252162456512, 2.5137205123901367, 1.9150186777114868, -0.08509179949760437, 0.07335860282182693, -1.9273062944412231, 0.2520577013492584, -0.9186553359031677, 1.2402271032333374, -1.2311217784881592, 0.20422431826591492, -0.062390413135290146, -0.3569563925266266, -0.4003745913505554, -0.3122195601463318, -1.5240541696548462, 0.5995477437973022, 1.2585855722427368, -1.0156769752502441, -0.7585468888282776, -1.253064513206482, 0.24470211565494537, 0.11376633495092392, -1.3101149797439575, -1.72486412525177, -0.14190854132175446, 1.0808608531951904, 1.3752336502075195, -0.3895503282546997, -0.38962069153785706, 1.8238861560821533, 1.419551134109497, -0.8965878486633301, -0.989684522151947, -1.7230662107467651, 0.49143263697624207, -0.330195814371109, -0.9048953056335449, -0.3653501868247986, -1.3306965827941895, 1.1798471212387085, -0.08967771381139755, 0.7065779566764832, -1.6369787454605103, -0.8827138543128967, 0.6572413444519043, 0.5688979625701904, -0.11750917881727219, -1.0150200128555298, -0.7312965393066406, 0.5528442859649658, 0.004776351619511843, -0.337612122297287, 0.6417142152786255, -1.5046929121017456, -0.7990649938583374, 0.02153613604605198, 1.5040756464004517, -0.6577517986297607, 1.2229238748550415, -0.44088634848594666, 0.854735255241394, 0.7745774388313293, 1.3506625890731812, 0.6358617544174194, 0.5600656867027283, -0.1689896285533905, 1.217393398284912, 1.1514263153076172, -0.3629404902458191, -1.741645097732544, 1.0914721488952637, 0.39496731758117676, -0.4955832362174988, -0.7303245663642883, 0.16983316838741302, -0.5634880065917969, 0.5954596400260925, -0.25581124424934387, 0.3867185115814209, -2.7940945625305176, -0.7531731128692627, -1.579795241355896, -0.8507612347602844, 0.39931389689445496, -0.06512415409088135, 0.4533911943435669, -0.6453508734703064, -1.4382553100585938, 0.3185591697692871, -0.719753623008728, 0.5961838364601135, 0.7761504650115967, 0.5233759880065918, -2.044419527053833, 0.31264108419418335, -1.1439011096954346, 0.897842526435852, -0.37191152572631836, 0.23723691701889038, 0.8707143664360046, -0.008719327859580517, 1.4424506425857544, -1.2584372758865356, -0.8724331855773926, 2.4495232105255127, 0.818223237991333, 1.7007948160171509, 0.46954309940338135, -0.524945855140686, 1.4943585395812988, 1.3859496116638184, 0.3656843304634094, -0.5679981708526611, 0.6360520124435425, 1.3802399635314941, -1.0064877271652222, -0.37931764125823975, -0.49375221133232117, 0.6938718557357788, 1.6112068891525269, -0.6629020571708679, 0.5224304795265198, 1.7981646060943604, -0.6019996404647827, 1.2324695587158203, -0.8681746125221252, -1.059667706489563, -0.2025073915719986, 1.0290621519088745, 0.04572214186191559, -0.8583725690841675, -1.4162273406982422, 0.6977763772010803, 1.9925379753112793, 1.6692208051681519, 0.04203606769442558, -0.7261157035827637, 0.44619086384773254, -1.0943267345428467, -0.8867244124412537, -0.7558334469795227, 0.5592957139015198, 1.9739632606506348, -0.7574946284294128, -0.9221000671386719, 1.2186360359191895, -0.8044843673706055, 1.353046178817749, -0.531865656375885, -1.1196763515472412, 0.115109384059906, -0.7663249373435974, -1.4434605836868286, -1.1690500974655151, -0.4480040967464447, 0.6453943848609924, 1.3806445598602295, -0.6948428750038147, -0.14362700283527374, 0.410291463136673, 1.4218358993530273, 0.051429346203804016, -1.1981643438339233, 1.1185364723205566, -0.9789149761199951, 0.19189682602882385, 0.5995882749557495, 1.2332159280776978, -0.42542779445648193, -0.6731253862380981, -0.5597110986709595, -0.85722416639328, 2.3743410110473633, 0.09817852824926376, 0.33089372515678406, -0.9017242789268494, 0.279610812664032, 0.5077676773071289, 0.10001464933156967, 1.4862741231918335, -0.17130224406719208, 1.3127951622009277, -0.31140777468681335, -0.8582549095153809, 0.33686861395835876, -1.8804869651794434, -1.327035903930664, -0.09305224567651749, -0.860805332660675, -0.22864720225334167, 0.733532190322876, 0.9935156106948853, 0.7794314026832581, 0.7016611099243164, -0.901523768901825, -0.35460761189460754, -0.8750571012496948, -1.1727598905563354, -1.1453853845596313, 1.8551961183547974, -0.8303028345108032, 0.16991718113422394, 0.7002075910568237, 0.38279834389686584, -0.7044036984443665, 0.17961134016513824, -1.5431348085403442, -1.2323243618011475, -0.21130584180355072, 0.7673062086105347, 1.0276182889938354, 0.7370584011077881, 0.24671146273612976, 0.47319960594177246, 1.4297953844070435, -0.9097868204116821, -0.4838130474090576, -1.0822988748550415, -0.8057454824447632, -0.66303551197052, 0.36074841022491455, -0.903481125831604, 0.5527318716049194, -0.7162327766418457, 1.208599328994751, 1.0143612623214722, -0.6364022493362427, -0.5184544324874878, -0.1063094213604927, 0.4910375475883484, -0.20879819989204407, 0.7772449254989624, -1.1250687837600708, -1.0731416940689087, 0.849738597869873, 0.6985753774642944, 0.20334933698177338, -0.6620802283287048, -0.5730706453323364, 0.03538592904806137, -0.8739436864852905, -0.15587061643600464, -0.048891469836235046, 2.1790990829467773, -0.3007460832595825, -0.6727169156074524, -0.5264837741851807, 1.2350234985351562, -1.5482386350631714, 0.7112827897071838, 0.45029687881469727, 0.4980895519256592, 1.2091999053955078, -0.8207481503486633, 0.7706156969070435, -0.9289768934249878, 0.8383402228355408, 0.1318904459476471, 1.18648362159729, -0.38775256276130676, -1.2470431327819824, 0.16811025142669678, -0.20575930178165436, -0.3801710605621338, 0.7921314239501953, 0.5090847015380859, -0.42780253291130066, 0.5077677369117737, 0.666936457157135, 1.4955263137817383, 0.560651957988739, -0.2883415222167969, -1.4537112712860107, 1.1537424325942993, -1.1560792922973633, -0.9624719023704529, -0.08769858628511429, -0.2942228615283966, -1.4847935438156128, -2.2408082485198975, 0.472007155418396, 0.04020201787352562, 0.23302724957466125, 2.6821818351745605, -0.2150619626045227, 0.4156235456466675, 1.0855635404586792, 1.2391307353973389, 0.3678038716316223, -1.3621903657913208, 0.2967628538608551, 0.2205335795879364, 0.431503564119339, -1.6893913745880127, 1.7459454536437988, -0.3218272030353546, -0.2725972831249237, 2.2841601371765137, 2.3284242153167725, -0.1138525977730751, 0.6938652396202087, -1.5313377380371094, -0.14434903860092163, -0.3100528419017792, -2.0346570014953613, 1.3752696514129639, -1.9336069822311401, 2.669997215270996, -1.4494987726211548, -2.5147669315338135, 0.0635421872138977, 1.1948732137680054, -1.9763400554656982, 0.10582486540079117, 0.050295159220695496, -0.8476631045341492, 0.03863675147294998, -0.13360798358917236, -0.17139476537704468, -0.38250118494033813, -0.7066915035247803, 0.34523507952690125, 0.6779600977897644, -1.1312785148620605, -1.2321171760559082, 1.9534449577331543, -0.4115036725997925, -0.8824042677879333, -1.768646001815796, 1.1059203147888184, 0.42086073756217957, -0.8888649344444275, 1.4526221752166748, 1.1055846214294434, -1.6256426572799683, -0.6045635938644409, 0.29445263743400574, 0.50991290807724, 0.921759843826294, -2.2590601444244385, -1.1493407487869263, 0.39625540375709534, 0.4590328633785248, 0.6908928155899048, -0.5617008209228516, 1.0803275108337402, -0.12230786681175232, 0.42945167422294617, 0.9230348467826843, -0.14984656870365143, 0.6615203022956848, -1.638252854347229, 0.7400050163269043, -0.12130674719810486, 0.11711329221725464, -0.5882457494735718, -3.097338914871216, 0.813178539276123, 0.6506825685501099, 1.6390595436096191, -1.7466803789138794, -1.781825065612793, 0.14213083684444427, -0.6058157682418823, -0.3609592616558075, 1.4323318004608154, 0.7512513399124146, 0.5924474596977234, 1.1745926141738892, -1.3091648817062378, -0.3628794550895691, 0.6658983826637268, -0.7406931519508362, 1.1506378650665283, -0.7294014692306519, 1.3443851470947266, -2.0875155925750732, 0.8008920550346375, -0.24247108399868011, -0.44119325280189514, 1.0293747186660767, 1.0320963859558105, -0.5896912813186646, -1.072398066520691, 1.1201918125152588, 0.3943588137626648, 2.145477771759033, 0.683037281036377, 0.5574367046356201, 0.12032121419906616, -1.047497272491455, -0.17631778120994568, 1.8107913732528687, 0.17675244808197021, -1.0245765447616577, -0.03140632435679436, 0.6935263276100159, -0.41132694482803345, -0.7475412487983704, -0.7960383892059326, 0.49656105041503906, 0.3263842761516571, 1.3564541339874268, 1.1271135807037354, 0.9369884729385376, -1.697670817375183, -0.005952952895313501, 0.46136611700057983, 0.22253109514713287, 0.6086367964744568, -0.5710323452949524, -0.26840120553970337, 0.3908405005931854, 1.226071834564209, 1.1291624307632446, 0.9716792702674866, -0.4441337287425995, 1.8576126098632812, 1.1773123741149902, 0.5504891276359558, 0.9419156312942505, -0.3630301356315613, -1.0084975957870483, -2.1579198837280273, 0.9621029496192932, 0.5304996371269226, -0.20262204110622406, 0.2995777726173401, -1.350258231163025, 1.5229328870773315, 1.6111146211624146, -0.6478345394134521, -0.5952853560447693, -1.180929183959961, 0.11376428604125977, -1.9019720554351807, -0.8296894431114197, -0.41305381059646606, 0.7210129499435425, -0.33188024163246155, 1.6360758543014526, 1.4564529657363892, 0.6549007296562195, -0.18007223308086395, 0.8085372447967529, 0.8764578700065613, 2.0512213706970215, -0.9600472450256348, 0.7018045783042908, 0.033134881407022476, 0.6989873647689819, -0.24011921882629395, -0.46202778816223145, 1.747852087020874, -1.0188277959823608, 0.5159691572189331, -0.5803763270378113, 1.3906632661819458, -0.1391303390264511, 0.24579183757305145, -0.7408368587493896, -0.7420985102653503, -2.100783348083496, 1.4571717977523804, 0.8626437187194824, -1.273447871208191, 0.5031960606575012, -0.8659777641296387, -0.1915934532880783, 1.0965425968170166, 1.2512080669403076, 1.443078875541687, 0.8904556632041931, -0.5764138102531433, -0.8496009707450867, 0.890821099281311, -0.1666364073753357, 0.0016675482038408518, -2.0962870121002197, -2.0646004676818848, -0.6799240112304688, 1.8145722150802612, -1.369937777519226, 0.6237272620201111, 1.9421323537826538, -0.41675177216529846, -0.6658526659011841, -0.14205294847488403, 1.4440124034881592, -0.12104585766792297, -0.4839646518230438, -0.3991847634315491, 0.30141133069992065, 2.3684751987457275, -0.5446182489395142, -0.6700097918510437, 1.4215961694717407, -0.559655487537384, -0.5201317071914673, 0.0106126070022583, -0.10589422285556793, -0.795382022857666, -0.4811779260635376, -0.8093088865280151, 1.533994436264038, -0.47339069843292236, 0.6294263005256653, -0.7920819520950317, -0.6913578510284424, -0.6392260193824768, 0.6939657926559448, 0.01445859670639038, 1.1621384620666504, 2.1208949089050293, 1.0330026149749756, -0.24970300495624542, -0.024208681657910347, -0.20064771175384521, -1.583141565322876, 0.19112852215766907, -2.236741304397583, 0.7816398739814758, 0.6169225573539734, -0.02442581206560135, 0.5657406449317932, -1.1765552759170532, -0.7629585266113281, 0.03470088168978691, -0.21661284565925598, -1.6693459749221802, 1.0703611373901367, -0.5966671109199524, -0.5375010371208191, -0.5978214740753174, -0.23653090000152588, -1.2951624393463135, 0.4981583058834076, 0.6651197075843811, 0.3790356516838074, 0.025634106248617172, 0.32540127635002136, 0.4395900368690491, 0.535282552242279, 0.9224082827568054, 1.040570616722107, 1.7611503601074219, 0.34864741563796997, -2.202754497528076, -1.6882834434509277, 0.5341277122497559, -0.8970386385917664, 0.6910337805747986, 0.7941861748695374, 0.7986721992492676, -1.4317141771316528, -0.2731330990791321, 0.6983967423439026, -0.34710457921028137, 0.020176760852336884, -0.5874713659286499, 0.45921415090560913, 0.8414655327796936, -0.670353889465332, 2.159644842147827, -0.04972940683364868, 0.1521439552307129, -0.10102448612451553, 1.3312054872512817, 0.5918578505516052, -0.20944498479366302, -1.2884554862976074, 0.04213104769587517, -1.9214138984680176, 0.2680225670337677, -2.047983407974243, 1.005868911743164, 0.19277630746364594, -0.3098270893096924, 2.0919058322906494, 1.1699663400650024, -0.7442309856414795, -0.23393619060516357, 1.194966435432434, 0.38554301857948303, -0.1285402774810791, -1.6648998260498047, 0.7819913029670715, 1.5523760318756104, -1.4838582277297974, 0.25858786702156067, 0.2722376585006714, 2.6130130290985107, -0.15867258608341217, 1.4972349405288696, 1.3831311464309692, 0.6628785133361816, -0.5480559468269348, 0.7039471864700317, 1.0801948308944702, 1.3100123405456543, -0.9969757199287415, -1.3041446208953857, 1.4988011121749878, 0.3860509693622589, -1.36678946018219, -0.35657742619514465, -0.36513638496398926, -0.4942774772644043, -1.428418517112732, -0.796106219291687, -0.08413013815879822, 0.3525034785270691, 1.1941921710968018, -1.6576683521270752, -0.4337424337863922, -1.270621657371521, 1.0430867671966553, 0.04871524125337601, -0.24822275340557098, -0.2634138762950897, 0.9749042391777039, 0.4066787660121918, -0.41022422909736633, 0.23977039754390717, 2.1923375129699707, -1.1673918962478638, -0.00797722302377224, -0.7729182839393616, -0.32192444801330566, -0.578561544418335, 0.8065722584724426, -0.052181657403707504, 2.1425938606262207, -0.17818811535835266, -1.1826813220977783, -0.041667744517326355, 1.1177383661270142, -0.6478496789932251, 1.1941847801208496, -0.9677315354347229, -0.11438731849193573, 1.124487042427063, -0.6151832342147827, 0.3933275640010834, 0.8664956092834473, 0.5048004388809204, 1.9936885833740234, -0.6129202842712402, 1.2832386493682861, -0.17715021967887878, 0.48916515707969666, 0.41198256611824036, 0.3237835168838501, 1.117295265197754, -0.08623084425926208, -0.7218317985534668, 2.3356049060821533, 0.33601945638656616, 0.9284265041351318, -1.3149200677871704, 0.5316857695579529, 0.7784398198127747, -0.15123245120048523, -0.1667882204055786, -0.3679163157939911, -0.6770721077919006, 1.3377360105514526, -0.8488796353340149, 1.306244969367981, -0.6521270275115967, 1.7272192239761353, -1.0895313024520874, -0.6570799350738525, -0.2755638062953949, 0.3187102675437927, 0.4833146035671234, 0.5557458996772766, -2.4978139400482178, 0.47631222009658813, 2.489300489425659, -1.3662949800491333, 0.17564430832862854, -0.623250424861908, -0.6999706029891968, 0.4940539300441742, 0.6474714875221252, -1.458259105682373, -0.8381859064102173, -0.3710425794124603, -1.6690171957015991, 0.014500979334115982, 1.7682559490203857, -0.36216187477111816, 0.46851077675819397, -0.39909473061561584, -0.0848674327135086, 0.15711665153503418, 2.750246047973633, -0.6468931436538696, -1.746946930885315, 0.388652503490448, 0.8857260346412659, -1.0004948377609253, -1.8098112344741821, -0.9241867661476135, -0.7197577953338623, -0.9863176941871643, 1.0531669855117798, -1.8100579977035522, -0.24361418187618256, -0.6247897148132324, -0.24769896268844604, -0.7187947630882263, -1.9559406042099, 0.058708593249320984, -0.42018574476242065, 0.2411983460187912, -0.6186816096305847, 0.765960693359375, -1.0780318975448608, 2.5433642864227295, -1.145555019378662, -1.5251094102859497, 1.0468111038208008, 0.6078778505325317, -0.27432966232299805, 0.2660180330276489, -0.7052434086799622, 1.741920828819275, 0.16572165489196777, -0.4247802793979645, 1.0099595785140991, -0.10072405636310577, 0.6857130527496338, -0.21530455350875854, 0.7125222682952881, 0.4491751194000244, 0.2925553321838379, -0.7474978566169739, -0.1419709473848343, -0.15498727560043335, -0.20681273937225342, 1.5623929500579834, -0.7248813509941101, 0.6488111019134521, 0.24015355110168457, 1.0874630212783813, 0.286104291677475, 0.8777666091918945, -0.5463201999664307, 1.3739720582962036, 1.054800271987915, -0.3960651159286499, -0.6952677369117737, -2.2164905071258545, -0.4102153778076172, 0.24591045081615448, -0.19533267617225647, -2.3046040534973145, 2.0516738891601562, 0.26952242851257324, 1.4775773286819458, 0.7110602855682373, 0.37699684500694275, -0.23777350783348083, -0.6736634373664856, -0.15380512177944183, -0.34410956501960754, -0.33927369117736816, 0.9638551473617554, 0.15494056046009064, 0.822485625743866, -0.997615396976471, -0.6516914963722229, -1.0046638250350952, -1.139099359512329, 0.8672753572463989, -0.046773653477430344, -0.916985273361206, 0.19797854125499725, 1.1969257593154907, -0.990719199180603, -1.4810082912445068, -0.22017335891723633, -0.06410744041204453, -1.6877992153167725, -0.47492048144340515, 0.7843524217605591, 0.19649924337863922, 0.4467700719833374, -0.5198322534561157, 0.28188371658325195, 0.17480146884918213, -0.34260252118110657, 0.3774731457233429, 2.374725580215454, -2.168184995651245, -0.7602468132972717, 2.6054446697235107, -1.2110389471054077, -1.059645175933838, 0.11728503555059433, -1.0746711492538452, 0.6168954968452454, -1.983120083808899, -1.1670562028884888, 0.6836426854133606, -0.4910469949245453, 1.1168850660324097, -0.22545191645622253, -0.023151863366365433, -2.1799747943878174, 1.5502985715866089, 1.7139978408813477, 0.28193846344947815, 0.6154540181159973, -0.5871652364730835, 1.4189869165420532, 1.5979818105697632, -0.6020200252532959, -0.2547553777694702, 1.1118916273117065, -1.691437840461731, -1.1281118392944336, 1.2314176559448242, 0.9044811725616455, 0.6444177031517029, 1.0304926633834839, -0.7073022127151489, 1.109297275543213, 0.5379525423049927, 0.6918199062347412, -1.4364697933197021, -0.5809089541435242, -0.9631006717681885, -0.7850906252861023, 0.8846383094787598, 0.6126309037208557, 0.04125034436583519, -1.669087290763855, 0.4026234447956085, -0.12652206420898438, -0.6524174213409424, -0.6087422966957092, -0.9631116390228271, 0.3898312449455261, -0.7631741762161255, -0.2939082682132721, -0.5504720211029053, 1.256591558456421, 0.49522241950035095, 0.2625482976436615, -2.1530725955963135, -0.5957891941070557, -0.8056286573410034, 0.12025748938322067, 2.734497308731079, -0.12045285105705261, 0.5221111178398132, -0.13785387575626373, 0.9483235478401184, 0.40543046593666077, 1.21896493434906, 1.1840108633041382, -0.14435529708862305, 1.159456491470337, -2.579475164413452, 0.6744614243507385, -1.5058737993240356, 0.6543065905570984, 1.2591979503631592, -0.8548455238342285, -0.7542569637298584, -0.6470530033111572, 0.7661840915679932, -1.7947388887405396, -0.32181882858276367, 1.1089736223220825, -1.4966905117034912, 0.26267626881599426, -1.1504313945770264, 0.5260435342788696, -1.4305561780929565, -1.2190099954605103, 0.10373867303133011, 0.4356559216976166, 0.4561612606048584, 0.2516825199127197, 0.06111731007695198, -0.05763949081301689, 1.1308751106262207, 0.37980735301971436, 0.9276840686798096, -1.4127823114395142, 1.4554202556610107, -1.8397250175476074, -0.306706964969635, 1.1478244066238403, -0.6390500664710999, -0.8767624497413635, 0.14564037322998047, 1.5071772336959839, 1.3020455837249756, -1.5370947122573853, 0.40519464015960693, -0.4848625957965851, -1.338765263557434, -0.8611506223678589, -1.3161488771438599, -1.0379419326782227, -1.4677624702453613, 0.9860682487487793, -0.18763242661952972, 0.7609942555427551, 0.6059611439704895, -1.4948676824569702, 0.6895233988761902, -1.3509465456008911, -1.0561649799346924, 0.19055067002773285, -0.4862412214279175, -0.5373337268829346, 0.7266417741775513, 0.015275323763489723, -3.3660192489624023, 1.6658525466918945, 2.328258991241455, 0.6641473174095154, 0.9562974572181702, 0.37799787521362305, -0.9192537665367126, 0.07923293858766556, 0.26200422644615173, 0.38079994916915894, 0.45206424593925476, -0.38807693123817444, 0.7657690644264221, 1.9611879587173462, 1.4589953422546387, 0.5660702586174011, -0.6689872741699219, -0.8076024055480957, 0.035615961998701096, -1.1270172595977783, 0.11520783603191376, 0.10455857962369919, 0.8073688745498657, 1.2907508611679077, 0.5133639574050903, 0.38884520530700684, 0.44267934560775757, -0.6165624260902405, 0.5415668487548828, 1.567541480064392, 0.9019259810447693, -0.09666869044303894, 0.3073421120643616, 0.961487352848053, 0.8831873536109924, 1.1343377828598022, -1.6114563941955566, 0.8919211626052856, 0.6837582588195801, 0.11574114859104156, 0.253122478723526, -0.6976677775382996, -0.415179967880249, -0.19000402092933655, -0.8743863105773926, -0.5307075381278992, 0.3762577176094055, -0.017546091228723526, -1.3900467157363892, 2.3491435050964355, -2.7422468662261963, 0.6098591685295105, -0.649608314037323, 0.28640016913414, 0.5807389616966248, 0.40314429998397827, 0.16653943061828613, -0.11807642877101898, 1.4532263278961182, -0.9525416493415833, 1.3220391273498535, -0.5026089549064636, 0.25946134328842163, 0.6554285287857056, -0.008382970467209816, -0.9693905711174011, -0.30868101119995117, 0.10214236378669739, -1.2323209047317505, 0.12811331450939178, -1.1923456192016602, -0.7562195062637329, 0.35392332077026367, -1.0573796033859253, 0.9889820218086243, 0.21624131500720978, -1.278340458869934, -0.9248111248016357, 0.8924088478088379, 1.7778574228286743, 1.1761131286621094, -1.29912531375885, 1.0397562980651855, 1.8455660343170166, 1.5177619457244873, 0.45797300338745117, 1.0638383626937866, 0.5897607207298279, -0.3122597932815552, -0.614721953868866, -1.7305086851119995, 0.7111961245536804, -1.1889925003051758, 0.7068661451339722, 1.6285370588302612, 0.3880564272403717, -0.30012238025665283, -0.19282753765583038, 1.2023625373840332, -1.1577907800674438, 1.7606979608535767, -0.6707615852355957, -0.5122107863426208, 1.6570302248001099, -1.5560276508331299, -0.6073276400566101, 0.7571117877960205, 0.676876962184906, 0.41131842136383057, 0.023611335083842278, -0.8010706901550293, -0.023702245205640793, -1.20698082447052, -0.03840845450758934, -0.4900442659854889, -0.016301779076457024, 0.7302382588386536, -1.7679646015167236, 0.2685128152370453, -0.05490203574299812, -0.9616194367408752, 0.9513206481933594, -0.6213886737823486, -0.08354198932647705, -0.6447240114212036, -2.7731168270111084, 0.3686036169528961, -0.32158225774765015, 2.0450122356414795, -0.2747795581817627, -1.392232060432434, 1.3660892248153687, -0.5495470762252808, -0.6794924139976501, -0.4128171503543854, 1.8916252851486206, 1.830984115600586, -0.4616556763648987, 0.05579136684536934, 1.335412859916687, -0.40489867329597473, 0.4336412250995636, 0.04731478542089462, 2.0680291652679443, 1.6066677570343018, -0.12214670330286026, -0.7919251322746277, 0.24324552714824677, -0.05126709118485451, -1.3562723398208618, 0.005699239205569029, 1.0110392570495605, -0.373879075050354, 0.008183377794921398, -0.7863246202468872, 0.04948268458247185, 0.16887503862380981, 0.2894389033317566, -1.611111044883728, -0.6587757468223572, -1.3861368894577026, -0.3405214548110962, -1.2300840616226196, 0.8675264120101929, -0.780735969543457, -0.0006725532002747059, 2.258315324783325, -0.550434947013855, 0.10967258363962173, 0.34237831830978394, 0.5245143175125122, 0.009562545455992222, 0.6920536756515503, -0.32493311166763306, 0.7356528043746948, 0.2446100264787674, 1.3066390752792358, 0.950234055519104, 0.10460943728685379, 0.09227383136749268, -0.6789006590843201, 0.7356429696083069, 0.20925860106945038, -1.6640199422836304, 1.4323770999908447, -1.4044110774993896, -1.191109538078308, 0.3429482579231262, -1.4778715372085571, 0.26543787121772766, 0.062443945556879044, -0.11493179947137833, -1.6052064895629883, 0.03821765258908272, -2.1469035148620605, 1.6418367624282837, -1.0616849660873413, -0.6762055158615112, 1.5797030925750732, 1.0260612964630127, -0.6350606083869934, -1.5072011947631836, 1.2646499872207642, -0.21572455763816833, -0.6246398091316223, 1.3761872053146362, -0.3702092468738556, 1.175845742225647, -2.0704312324523926, -0.5546644330024719, 1.5097236633300781, 0.989252507686615, -0.01699831150472164, -0.4879954159259796, 1.1259669065475464, 1.6114673614501953, -1.0359935760498047, -1.6215298175811768, -0.0827474445104599, 0.31028980016708374, -0.16362448036670685, -1.5937552452087402, -0.6610833406448364, 0.4765526056289673, -1.2708698511123657, 1.0405027866363525, -0.12825217843055725, 0.3806227743625641, 0.011024721898138523, 0.21507461369037628, -2.079185724258423, -0.8283197283744812, -0.23902733623981476, -0.0002458461676724255, 0.08926176279783249, 1.0240895748138428, -1.0091017484664917, -1.524401307106018, -1.9127174615859985, -1.0173908472061157, -0.24509039521217346, 0.21540570259094238, -0.07634530961513519, -0.7934882640838623, 0.7244082093238831, -0.26726964116096497, 0.20690155029296875, -0.8830880522727966, 0.7379052639007568, -0.36337921023368835, 1.3179936408996582, 0.3532008230686188, 0.07871490716934204, -0.2355053871870041, -0.1569671779870987, -0.67653888463974, 1.1562159061431885, -0.37334078550338745, -1.3027399778366089, -0.631951093673706, 0.9922521710395813, 0.20326310396194458, 0.07214754819869995, -1.2179276943206787, 0.6859675049781799, -0.19596132636070251, -0.31891930103302, 0.22438156604766846, -0.5824577808380127, -0.8692101836204529, -2.0258045196533203, 1.3368855714797974, 0.6808653473854065, 2.0221283435821533, 2.3989109992980957, 0.5139979720115662, -0.5332359671592712, -0.2144361287355423, -1.1944025754928589, -0.6488991379737854, -0.05598565936088562, -0.7452701926231384, -0.04885204881429672, -0.32652562856674194, -0.0692167580127716, -1.778840184211731, 0.6877334713935852, -0.6659417748451233, -0.16863180696964264, 1.2406210899353027, -1.5894428491592407, 0.7149814963340759, 0.39176666736602783, -0.03418392315506935, -0.39394810795783997, -0.22981402277946472, -0.480991393327713, 0.626045823097229, -0.4373062252998352, -0.36754050850868225, -0.44221580028533936, 1.539339303970337, 1.8843706846237183, 0.8898724913597107, 0.5872921943664551, -0.9337216019630432, 0.520089864730835, 0.4344137907028198, -1.5092869997024536, 0.18399296700954437, -0.4151821434497833, -0.855750560760498, -0.09846324473619461, 1.7125366926193237, 1.0890134572982788, -0.803945004940033, -0.9618098735809326, 0.48896506428718567, -2.1333541870117188, 0.661486804485321, -2.037838935852051, 0.11291516572237015, 0.9608744382858276, -0.2596953511238098, 2.542245388031006, 0.5728853940963745, -0.32126978039741516, -0.8364658951759338, -0.899659276008606, 0.17062555253505707, 1.734087347984314, -0.866679847240448, -0.6859957575798035, 0.7364252805709839, -0.25666677951812744, -0.9248437881469727, -1.1556333303451538, -0.7021802663803101, -0.3420594036579132, 0.9046680927276611, 1.218825340270996, -0.32920774817466736, -1.4810932874679565, -0.005283698905259371, 0.6381887197494507, -0.3106062114238739, -0.896312415599823, -1.53032386302948, 0.6576551795005798, 0.15625527501106262, 0.7129688262939453, 0.9949472546577454, 1.5265520811080933, 1.9553539752960205, 0.016024645417928696, -0.9081898331642151, 0.8374579548835754, 0.41405048966407776, 0.773463785648346, -0.6417446732521057, 0.05329355224967003, 0.008012956939637661, -0.10334249585866928, 0.46046164631843567, -0.2067149579524994, -0.9739088416099548, -1.5288375616073608, 0.495045930147171, -2.359707832336426, 3.2363545894622803, 0.7067136168479919, -0.5664986371994019, -0.5063830614089966, 1.203994631767273, -0.844879150390625, -0.44862428307533264, -1.1653703451156616, 0.7339956760406494, 0.9360461235046387, 0.42173656821250916, -0.20870374143123627, -0.449767529964447, -1.279356837272644, -0.7827801704406738, -2.0884222984313965, -1.062211513519287, -0.2951807379722595, 1.1569408178329468, 1.709661602973938, 2.146202325820923, -0.10039655864238739, -0.4980183243751526, -0.4907749593257904, 1.5999103784561157, 1.9393409490585327, -0.18298546969890594, -0.10296246409416199, 0.9512609243392944, -0.9428897500038147, -2.171659231185913, -1.2059201002120972, -0.011062477715313435, -0.7841206789016724, -1.0266019105911255, 0.09260106831789017, -1.2727251052856445, 1.456926941871643, -0.976777970790863, -0.24847407639026642, -1.3730404376983643, 1.0799365043640137, 1.6537429094314575, -0.6362154483795166, -0.2862051725387573, 1.2168742418289185, 1.3973597288131714, 0.9107926487922668, -0.9409236907958984, -0.7888593673706055, -0.984201192855835, 0.5426540374755859, 0.01113828457891941, -1.060987114906311, 1.2288151979446411, -2.4340121746063232, 0.46341589093208313, -1.076485276222229, -0.5244843363761902, -0.6664490103721619, 1.6613496541976929, -1.6806291341781616, -0.8922966718673706, 1.217193603515625, -0.42195361852645874, 0.6670491099357605, 1.1689355373382568, -0.3345303535461426, 0.12009274959564209, 1.136976957321167, -0.215431809425354, 0.05270027369260788, -0.6429115533828735, 0.6383408308029175, 1.1192736625671387, 0.008204100653529167, 0.9137793779373169, -1.020473837852478, -0.7513718605041504, -1.1362870931625366, 0.7164668440818787, 1.42843759059906, 0.4686957597732544, 0.15129412710666656, -0.18734554946422577, 0.08301109075546265, -0.9413179755210876, -0.3923948407173157, -0.5148261189460754, -0.5352188944816589, -1.0999374389648438, 0.4812586307525635, 0.8269728422164917, 0.14311553537845612, 2.6706297397613525, 1.0589226484298706, -1.0157755613327026, -1.3459742069244385, -0.054285384714603424, 0.41325777769088745, 0.47855880856513977, -2.10296630859375, 0.7229671478271484, 0.37420785427093506, 0.22652290761470795, 1.1214206218719482, 0.504697859287262, -1.0793484449386597, 0.19349007308483124, -1.613405466079712, 0.36535391211509705, -0.12834808230400085, 0.328194797039032, 0.6277376413345337, -1.6235758066177368, 0.7374956011772156, -0.5162360072135925, 1.1448551416397095, -1.0967495441436768, 1.1323422193527222, -0.6992267370223999, 0.7099541425704956, -0.022410690784454346, -0.31845295429229736, -1.384436011314392, -0.8206668496131897, -0.8127269744873047, -0.7749282121658325, -0.8170716166496277, 0.5106935501098633, -1.0577729940414429, -0.3127473294734955, -1.824333667755127, -0.5999399423599243, 0.5326266884803772, 0.3079003691673279, -1.652338981628418, -0.8246626853942871, -1.7032508850097656, -1.050635814666748, 0.4396180212497711, 1.3421090841293335, -1.164071798324585, 0.9230782389640808, -0.6621811985969543, 1.2893903255462646, -1.1731783151626587, 1.3767551183700562, 0.4936848282814026, 0.24040041863918304, 0.30660027265548706, 0.10542947798967361, -1.1230084896087646, 0.9433709383010864, -0.5692316889762878, -0.487691193819046, -0.3959517478942871, -1.0505534410476685, -2.297417640686035, 1.1313954591751099, 0.20132778584957123, 0.2713053822517395, 1.0227296352386475, 0.7589854598045349, -0.3471319079399109, -0.9155307412147522, -0.5328895449638367, 1.6799894571304321, 1.3928059339523315, 0.03803684934973717, -0.033467795699834824, 0.3421178162097931, 0.28348207473754883, -0.1762877106666565, -0.04286564514040947, -1.2248320579528809, 0.14237667620182037, 0.34293800592422485, -0.7064568996429443, -1.2033350467681885, -0.22864098846912384, 2.4388184547424316, -0.9891409873962402, -0.2388814091682434, -1.847397804260254, 0.2261841595172882, 0.27204057574272156, -0.34935787320137024, 0.010991865769028664, 3.4399802684783936, 1.0316797494888306, -1.3993767499923706, 0.5337685942649841, -0.6437699198722839, 1.1870055198669434, -0.6721002459526062, -0.13276337087154388, 0.9592581391334534, -1.1262385845184326, -0.21295776963233948, -0.21430283784866333, 1.084276556968689, -0.27498725056648254, 1.7941137552261353, -1.0436224937438965, 0.21530508995056152, -0.945967435836792, 0.18681225180625916, -1.2696894407272339, 0.9474520087242126, 0.3987595736980438, -1.5645256042480469, 0.950954020023346, 1.3115788698196411, -0.705739438533783, 0.3627854287624359, -0.2674642503261566, -1.1316241025924683, 0.44691652059555054, -1.3392024040222168, -1.0618855953216553, -1.6951614618301392, 0.7773047685623169, 0.23119863867759705, -1.505828619003296, -2.056321382522583, 0.5692967176437378, -1.2146365642547607, 0.5391329526901245, 0.21094074845314026, 0.6532756090164185, -0.16641655564308167, 0.9230809211730957, -2.3979854583740234, -0.1652468740940094, -0.4293161928653717, 0.020481746643781662, 0.8421075344085693, 0.8436170220375061, 0.8215944766998291, 0.6078663468360901, 1.3580007553100586, 0.19374653697013855, 1.0436375141143799, 0.6533488035202026, 0.40472841262817383, -0.7519093751907349, -0.14277061820030212, -0.6181049942970276, 1.4971823692321777, -1.5388813018798828, 0.17171235382556915, -0.3766153156757355, 0.23532216250896454, -0.5046067833900452, -0.8893753886222839, -0.15741446614265442, -0.1029801070690155, -0.5809445977210999, -1.1853013038635254, -1.441489577293396, -0.8095394372940063, -0.4590688943862915, 0.3218116760253906, -0.8179781436920166, -0.8968713879585266, -1.7342413663864136, -0.6719807982444763, 1.7512024641036987, 1.0927025079727173, -1.6177111864089966, -0.23855066299438477, 0.11910141259431839, 1.3855228424072266, 0.2632395029067993, 0.3353530466556549, -0.3244927227497101, 1.2015570402145386, -1.363264560699463, -0.46090540289878845, 0.35716211795806885, -1.026268482208252, -0.6901109218597412, -1.545027732849121, 0.4188244342803955, -0.020115375518798828, 0.7979727983474731, -0.2960725426673889, -1.3338589668273926, 0.863822340965271, 0.7207117080688477, 0.2542136311531067, 1.4196614027023315, -1.6174298524856567, -1.3606008291244507, 0.10539592802524567, -1.960525393486023, -0.4443836808204651, 1.4494776725769043, -1.7739595174789429, 0.6889544725418091, 1.9094455242156982, -0.23672030866146088, -1.0722678899765015, 1.5895073413848877, -0.536018431186676, 2.183490753173828, 0.8706063628196716, -1.8834725618362427, 0.6234064102172852, -1.6727186441421509, 0.9536249041557312, -0.5020712614059448, 0.07242783904075623, -0.05201675742864609, 0.6477634310722351, -0.04132191836833954, 1.6517119407653809, -0.22123964130878448, 0.06619357317686081, -1.5570787191390991, 0.46907684206962585, -1.21172034740448, 0.0480661541223526, -0.204656720161438, 1.2948673963546753, -0.33079782128334045, -0.821450412273407, 1.0349422693252563, -1.7747852802276611, 0.002489612437784672, 0.06489549577236176, 1.1891223192214966, -1.7447575330734253, -0.3633395731449127, -0.006755325477570295, -0.9823882579803467, -0.4565870761871338, -2.0928359031677246, -0.19518223404884338, 1.0888373851776123, -0.966323733329773, -1.7021323442459106, -2.5846619606018066, -0.8777419924736023, -0.5138559341430664, 0.05640915408730507, -0.9308044910430908, 0.015029214322566986, -0.7167624235153198, 1.1883279085159302, 0.34484365582466125, -0.14427131414413452, 0.4281954765319824, 1.0313842296600342, 1.0995399951934814, -0.2784726023674011, -0.3008701205253601, -0.5071417689323425, 1.184777855873108, -0.06732378154993057, 0.9192821979522705, 0.49637287855148315, -2.538774013519287, -0.513709306716919, -1.6597590446472168, -1.1907787322998047, 1.3242385387420654, 0.4284838140010834, 0.2030823975801468, -0.17122462391853333, 0.4366151988506317, 0.35033491253852844, -0.8317424058914185, -1.089043378829956, 0.07095472514629364, -0.2612922787666321, -0.2402770221233368, 0.6560817360877991, -2.166518449783325, 0.3470277488231659, 1.300794243812561, -1.2280150651931763, 1.5333720445632935, 0.2840372323989868, 0.02558353915810585, -0.4879986047744751, 1.5358673334121704, -0.1544255018234253, -0.613797664642334, 0.568281352519989, 1.2503293752670288, -1.236205816268921, 0.47427040338516235, -0.9766901135444641, -1.1984232664108276, 0.3059501349925995, 0.4800780415534973, -0.5974692106246948, 1.2002507448196411, -0.3777756094932556, -0.03413034975528717, -0.6783639788627625, -2.064849615097046, 0.9031345844268799, -2.3059799671173096, -1.0042378902435303, -0.6471885442733765, -0.8668780326843262, 0.4076068699359894, -0.39279282093048096, 1.1185994148254395, -0.4463871717453003, -1.0860497951507568, 1.4349207878112793, -0.33448803424835205, -1.8946144580841064, 0.016416503116488457, 1.0258817672729492, -1.1942996978759766, -0.3405992090702057, 1.1705322265625, 0.45887792110443115, 1.420261025428772, 0.7069233655929565, -1.2121278047561646, 0.27569064497947693, -2.4610936641693115, -0.7524440884590149, -0.052073609083890915, 1.2760374546051025, -1.6627732515335083, -0.34969156980514526, -0.2208816409111023, -1.1566526889801025, 2.0925774574279785, -0.5639041066169739, -1.013340711593628, -0.5352775454521179, 0.5909217000007629, 1.018847107887268, -0.8726481795310974, 1.530163288116455, 1.5988895893096924, 1.1231251955032349, 0.43271368741989136, 0.009401952847838402, 0.513226330280304, 0.66451495885849, 0.22248680889606476, 0.8409407734870911, 0.5722282528877258, -0.6691312789916992, 0.04964068904519081, 1.4770125150680542, 0.11843866854906082, -0.9195000529289246, 1.1722155809402466, -0.09430725127458572, -0.9325881004333496, -0.15118053555488586, 1.4713695049285889, 2.249786376953125, 0.12019067257642746, 0.11645280569791794, -1.2696030139923096, 0.8388094902038574, -0.07291559129953384, 0.6355388760566711, -0.1552562564611435, 0.2544555962085724, 0.0789242759346962, 1.9488670825958252, -1.5945801734924316, -1.2042903900146484, -0.6203284859657288, 0.17834527790546417, -0.22604532539844513, -0.41015857458114624, 0.9309251308441162, -0.05460789054632187, -0.21955835819244385, 1.1537222862243652, 2.1953299045562744, -1.4260460138320923, 0.7101359963417053, -2.796689748764038, 1.393405795097351, 1.8198052644729614, -1.131363868713379, -0.08085572719573975, 0.7112930417060852, 0.2646885812282562, 0.2493402361869812, 0.9017235040664673, -0.6814718246459961, 0.9111030697822571, 0.88767009973526, 0.49720004200935364, 0.24057339131832123, -0.3599846065044403, 1.2129426002502441, 2.3040759563446045, 0.3390972912311554, 0.7244102358818054, 1.9356589317321777, -0.10496463626623154, -0.4038081765174866, 0.49761736392974854, -0.2846870720386505, 0.1379123330116272, 1.3227920532226562, 0.4360351860523224, -1.2379382848739624, 0.6668266654014587, 1.0500131845474243, 0.8525663614273071, -0.9995918273925781, -0.326080322265625, -0.11338987201452255, -2.088859796524048, -1.2732688188552856, 1.5785857439041138, 0.2937629520893097, 1.7744885683059692, 1.0478551387786865, -0.6667283177375793, 1.066428780555725, 0.045834168791770935, 0.5733726024627686, 0.7615579962730408, -0.7881637811660767, -0.9948196411132812, -0.9104479551315308, 1.1762800216674805, -0.35481584072113037, 0.0908958911895752, -1.4988062381744385, 0.75360506772995, -0.8101924657821655, 0.6472945213317871, -0.4191817343235016, 0.994048535823822, 1.0847359895706177, -0.2866981625556946, 0.9377965331077576, 0.1671569049358368, -1.281175136566162, -1.1224441528320312, 0.7540702819824219, -1.3864833116531372, -0.580064058303833, -1.5565800666809082, 0.1353408694267273, -1.5552177429199219, 0.31852081418037415, -0.1146094873547554, 0.3702564537525177, 0.5928157567977905, -0.8634839653968811, 0.8180496096611023, -0.731805682182312, 0.5576587319374084, -0.4058331847190857, 1.124463677406311, -0.17931967973709106, -0.5403923392295837, -0.059976495802402496, 0.443494588136673, 0.38049185276031494, -0.3059939444065094, 0.04250812157988548, -1.3171488046646118, 0.1930406242609024, -1.1117795705795288, 1.8568636178970337, 0.2686242163181305, -0.033912088721990585, -0.5148279666900635, 0.18169163167476654, 1.3543511629104614, 0.3741355836391449, 0.15932685136795044, 0.5321097373962402, 0.5410705208778381, 1.1241381168365479, -0.9623708128929138, -0.9144431948661804, 0.9266955852508545, 0.7573240995407104, -0.4269372820854187, 0.9216094017028809, 2.9204111099243164, -1.3414597511291504, -1.1590303182601929, -1.4059962034225464, 1.3189929723739624, 0.6532540321350098, 1.1570206880569458, -0.9319508075714111, -0.4766402542591095, 0.06665587425231934, 1.4517978429794312, 0.42294642329216003, -1.138427495956421, 0.9010571241378784, 0.8404373526573181, 0.154440775513649, -0.31126755475997925, -1.633825659751892, -1.312991976737976, -0.3064867854118347, -0.6712002158164978, 0.3130292296409607, 1.398019552230835, -0.1605917066335678, 0.10965917259454727, 0.37676534056663513, -0.10623137652873993, -1.7100275754928589, -0.7369678020477295, 0.16082151234149933, 0.0062384526245296, 1.3226375579833984, -0.3906497061252594, 1.778928279876709, -1.8089698553085327, -0.6414799094200134, 0.4513486325740814, -0.04488536715507507, 0.6636821031570435, -0.1507962942123413, -0.48134130239486694, -0.708073616027832, -0.42193326354026794, 1.471052646636963, 1.073845624923706, 0.20087578892707825, -0.5821544528007507, -0.17455339431762695, -1.6029415130615234, 1.1085618734359741, 0.030785439535975456, 0.8106206655502319, -0.9862696528434753, -0.6860988140106201, -0.8885257840156555, 0.490283727645874, 0.05405588820576668, -0.34063178300857544, -0.6003024578094482, -0.16988934576511383, 0.5017129182815552, -1.2304010391235352, 0.02086555026471615, 1.108886480331421, 0.07600197941064835, -0.9753072261810303, 1.0087952613830566, -2.2167177200317383, 0.19018061459064484, -0.5259542465209961, -0.38011908531188965, 0.55415278673172, -1.5536835193634033, -0.08138252794742584, 1.020307183265686, -0.2878114879131317, 1.505366563796997, -1.5877320766448975, -0.08972834050655365, -1.1518056392669678, 0.699259340763092, 0.1458098441362381, -0.4377431869506836, 0.27056393027305603, 0.9945642352104187, -0.3267586827278137, 0.2447846233844757, -0.7891567945480347, -0.12771129608154297, 1.5143953561782837, -0.284902960062027, 0.12701737880706787, 0.46284541487693787, 0.15641839802265167, 1.1462332010269165, 0.001966227311640978, -1.337554693222046, -1.2519383430480957, 0.23224832117557526, -0.8310104012489319, -1.4724394083023071, -0.2650283873081207, 1.0401768684387207, 0.34567439556121826, -2.3058054447174072, -0.3958854675292969, 0.39053648710250854, 0.08969991654157639, -2.1597890853881836, 0.46926140785217285, 0.5970431566238403, 0.8529700040817261, 0.930584192276001, -0.7358877062797546, -0.4999230206012726, -0.8559557795524597, -0.17442204058170319, -0.1888599395751953, 0.1307845413684845, 0.30117231607437134, -1.1680485010147095, -1.0227863788604736, 0.5149991512298584, 1.0780085325241089, 0.1908871978521347, 0.43344417214393616, -0.025130445137619972, -0.6944199204444885, 2.2615485191345215, 1.2120351791381836, 2.6029815673828125, 0.6470953226089478, -1.3176614046096802, 0.7777363061904907, -0.7859163284301758, 0.5525839328765869, 0.6112077832221985, -0.6710413694381714, 0.7625138759613037, 0.29259854555130005, 0.5015050172805786, 0.03896922990679741, 2.3291499614715576, -0.1390099823474884, -1.0944712162017822, -0.2605668604373932, 0.31403642892837524, 0.7506927847862244, -0.19885389506816864, -1.3111640214920044, 0.24171580374240875, -0.048057712614536285, 0.3525422215461731, 0.2993653118610382, 1.2221962213516235, 0.8022392988204956, 0.8032981157302856, 0.133571594953537, -1.7289382219314575, -0.0072916592471301556, 0.2354741096496582, -1.582431435585022, -1.043169379234314, -0.50648432970047, -0.8721634149551392, 0.3299238085746765, 0.28020766377449036, 0.27363041043281555, -1.2591955661773682, 1.0985912084579468, -0.6193689107894897, -0.4166891276836395, -0.6791438460350037, -0.15335498750209808, -0.5933610796928406, 0.025438042357563972, -0.5932837724685669, 1.1002768278121948, -2.36734676361084, 0.44871264696121216, 1.1164554357528687, -0.10735362023115158, -0.6461482048034668, -1.810980200767517, 0.41294530034065247, -0.6039702892303467, 1.2623668909072876, 0.4646873474121094, -0.9801568984985352, 0.697616696357727, 0.5839145183563232, -0.45476871728897095, 2.205500841140747, 1.8334287405014038, -2.2093124389648438, 0.02921815775334835, -1.0932695865631104, -0.012302037328481674, 0.8345724940299988, 1.2677891254425049, -1.4890493154525757, 1.5859774351119995, 0.6061033606529236, -3.7490406036376953, 0.6888012290000916, 0.6728232502937317, 1.6374350786209106, 0.6904560327529907, -1.815544843673706, 0.1558510810136795, 1.5179327726364136, -1.3619842529296875, 2.0486361980438232, -0.09868750721216202, 0.5933120250701904, 0.7429192662239075, -0.6801642775535583, -3.0283524990081787, -0.8458359241485596, -0.5416452288627625, -0.47029468417167664, 0.2813439965248108, -0.7280969619750977, -0.7657070755958557, 1.1010452508926392, -0.505820095539093, -0.7345246076583862, 0.5798590183258057, 0.07269231975078583, -0.42672184109687805, 0.3545769453048706, 1.3417786359786987, -1.9743469953536987, -0.5595939755439758, -0.03005685843527317, 0.45909175276756287, 0.15215323865413666, 1.0407580137252808, 1.6737910509109497, 0.8221412301063538, -0.5936598181724548, -0.09206240624189377, 1.3759515285491943, 0.15144644677639008, -1.2671422958374023, -0.47585880756378174, 1.447153925895691, 1.74116051197052, 0.7161998748779297, 0.28574076294898987, 0.10595621913671494, 0.7877230644226074, -0.5909456014633179, 0.004836553242057562, 0.8606547713279724, -1.929835319519043, 0.12608186900615692, -0.07076389342546463, 0.3359331786632538, 0.14325641095638275, 1.9586782455444336, -1.6403746604919434, 0.6990482211112976, 0.4012935757637024, -0.1595035195350647, -0.6368409991264343, -1.8567430973052979, -0.4640183746814728, 0.542281448841095, -0.6475242376327515, -0.7182794213294983, 1.2683134078979492, 0.5535072684288025, 0.28800031542778015, 0.48885267972946167, -0.6195751428604126, 0.6402856707572937, -0.3676314949989319, -1.907313346862793, -0.3816147446632385, 1.1331347227096558, 1.9562530517578125, -0.7539899945259094, -0.8243610262870789, 1.7262451648712158, -0.3073180317878723, 1.352034091949463, -0.03684188425540924, -0.20406125485897064, -0.012875836342573166, -0.8071743845939636, -1.8698890209197998, 0.4884239137172699, 0.5237534046173096, -0.502406895160675, 0.16870561242103577, 1.4422905445098877, -0.2942466139793396, 0.06107575446367264, 1.8155235052108765, -0.16931036114692688, 1.7104347944259644, 0.6318448185920715, -1.1047217845916748, -0.9108427166938782, 0.7697288393974304, -2.500281572341919, 1.528267741203308, -0.9184139370918274, -0.7492311596870422, 0.9036695957183838, 0.5977103114128113, 2.074956178665161, -0.579376220703125, 0.046277280896902084, 0.5725021958351135, 0.13288578391075134, -1.3721132278442383, -0.7021996378898621, 0.8208670020103455, -0.9493730068206787, 0.04545307531952858, -0.25722986459732056, -1.4086544513702393, 0.10423202067613602, 0.38345423340797424, -2.576920747756958, -0.5923125743865967, 1.1181156635284424, -0.8926628828048706, 1.474948763847351, 1.2727060317993164, -1.9143880605697632, -0.5259888172149658, -0.16595353186130524, -1.2183376550674438, 1.8236336708068848, 0.2444811910390854, -0.7244527339935303, -1.1772385835647583, -0.43615174293518066, -0.4460369050502777, -0.8778502345085144, -0.7656209468841553, 2.115889310836792, 0.12138540297746658, -0.7652015686035156, 0.9164065718650818, 1.0412912368774414, 0.026010382920503616, -0.3168112337589264, -0.4500276744365692, -0.8047758340835571, 1.0879764556884766, 2.848478317260742, -1.8502001762390137, -0.26037007570266724, 0.9001337289810181, -0.7495353817939758, -0.9005469083786011, -1.7853453159332275, -2.1280264854431152, 1.3059672117233276, 1.1773265600204468, -0.027085741981863976, 1.6865296363830566, 0.2355867326259613, 2.3119735717773438, 1.1848127841949463, -0.7597147822380066, -0.17614173889160156, 0.2536540627479553, 2.068742513656616, -1.331412672996521, -0.18855495750904083, -0.9066638350486755, -0.7669628262519836, 1.8028295040130615, 2.5876100063323975, 0.10221115499734879, -1.0634839534759521, -1.4806050062179565, 1.598171353340149, 1.2552459239959717, -0.43421098589897156, -0.0403442308306694, -0.3647333085536957, 1.612173080444336, 0.0111728236079216, 0.2640262544155121, -0.28455841541290283, -1.1346415281295776, -0.6331868767738342, -0.2610613703727722, -0.5579301714897156, -0.008703036233782768, -1.3479896783828735, 0.9955010414123535, -1.386734962463379, 0.010809101164340973, 0.46750956773757935, 1.3877149820327759, -2.000490427017212, -0.5948857665061951, 1.91456937789917, 0.5921320915222168, -0.097413070499897, -0.7278419733047485, 0.7108716368675232, 1.6490446329116821, -0.07254356890916824, -0.8014733791351318, -0.18582895398139954, -2.046112298965454, -0.9556812047958374, -0.28377246856689453, -0.31295594573020935, 0.6547421216964722, 0.6184942126274109, -0.7300668954849243, 1.6041462421417236, -0.12017872929573059, -0.7485383749008179, 0.054123010486364365, -0.837454617023468, -1.6279027462005615, -1.1326392889022827, -0.11989687383174896, 1.0748822689056396, -2.058478832244873, -0.6260658502578735, 0.6669113636016846, -0.7024955749511719, -0.04049283266067505, 0.13573943078517914, 1.0023761987686157, 0.11143721640110016, -0.1760084182024002, 2.2418813705444336, -0.6301291584968567, 0.2702856659889221, -3.3652865886688232, 0.0026107411831617355, -2.387458086013794, -0.47846370935440063, -2.947700262069702, -0.15035167336463928, 1.0192054510116577, -0.4044959843158722, -0.16882352530956268, 1.2531366348266602, -0.561224102973938, 0.3298380672931671, -1.4393746852874756, -0.9334384202957153, -2.357325792312622, 0.40810754895210266, -1.041817307472229, 0.10881348699331284, 0.4216797947883606, 0.435268759727478, -0.6838092803955078, 0.30300694704055786, -0.18322736024856567, -0.9655721783638, -1.590694546699524, -0.028370872139930725, -0.22586044669151306, 0.00943665113300085, -0.0074063437059521675, 0.12288948148488998, 0.8184366822242737, 1.4742602109909058, -0.41458648443222046, -0.6745707988739014, 0.5878909826278687, 0.08446675539016724, -0.35319826006889343, -0.03852174058556557, -0.7199504375457764, 0.7827072739601135, -0.5524682402610779, 1.2341996431350708, -0.9681223034858704, -0.19985130429267883, 0.8093665838241577, 0.8124493360519409, 0.09969866275787354, 0.03921542689204216, 1.3329675197601318, -0.9282236099243164, -2.5573458671569824, 1.3629436492919922, -0.5731582045555115, -0.8185078501701355, -0.014836634509265423, 0.8095337152481079, 0.028647564351558685, -0.10401997715234756, 0.15486203134059906, 1.5734350681304932, -0.007412651088088751, 0.8269698023796082, 0.8285654783248901, 0.7029123306274414, -0.32353395223617554, -0.8679145574569702, 1.6798771619796753, 1.7559894323349, 0.3464139401912689, -1.4221758842468262, 0.5222637057304382, 0.6271716952323914, 0.08894725888967514, -1.1883749961853027, 0.20364879071712494, -0.9378261566162109, -0.5856233239173889, 0.37563827633857727, -0.2472139298915863, 0.4682615101337433, -0.47030961513519287, 0.21322070062160492, 0.10997350513935089, 0.0862070843577385, 1.482102870941162, 0.9574704766273499, 0.2869165539741516, 0.9291006326675415, 2.301206350326538, -0.7692360281944275, 1.4729797840118408, -0.9485248923301697, -2.5273308753967285, -0.5877767205238342, 2.14471173286438, 1.0879539251327515, 1.025308609008789, -1.500324010848999, 3.1464920043945312, -0.33032867312431335, -0.2794433832168579, 1.7371057271957397, 0.2191193550825119, -1.5567030906677246, 0.17291763424873352, -0.4505856931209564, 0.5966894030570984, 0.1822904795408249, 0.44663766026496887, -0.7675304412841797, -2.368989944458008, -0.8399706482887268, -1.5619657039642334, -0.3307429254055023, -0.04488462954759598, -1.80880606174469, -0.5602694153785706, 1.6523995399475098, 0.7262890338897705, 0.7160658836364746, 0.4052943289279938, -0.8775001168251038, 2.1110713481903076, -1.0928537845611572, -0.1426444798707962, -0.33683153986930847, 0.2716589868068695, -1.3535643815994263, -1.4780998229980469, -2.216350793838501, -0.6707425713539124, -1.5930782556533813, -0.9200772047042847, 0.657149612903595, -0.8489448428153992, -0.010087192058563232, 0.22927263379096985, 0.5671122670173645, -0.7254976630210876, -0.11600062251091003, 0.0818478986620903, -1.6501520872116089, -1.0937612056732178, 0.7662957906723022, -0.5974608659744263, -0.2283601313829422, -0.1405850499868393, -1.4171398878097534, -0.05122203752398491, -0.7755612134933472, 0.9980236291885376, 1.5692973136901855, 1.3887523412704468, 1.1607857942581177, 0.781565248966217, 0.06970545649528503, 0.09331757575273514, 1.1436747312545776, -0.3316820561885834, 1.378421425819397, -0.06500054895877838, -1.085910677909851, -0.7342050075531006, -1.3485933542251587, 1.74961256980896, 0.5393688678741455, 0.5913384556770325, -0.2823488414287567, -1.527956485748291, -1.0251696109771729, 0.4379808008670807, -1.4372261762619019, -1.3172143697738647, 0.9675083756446838, -0.8570849299430847, 1.0858144760131836, -0.9099048376083374, -0.7404391169548035, -0.6306585073471069, 1.4338361024856567, 1.8775969743728638, 0.28927507996559143, 0.8441290259361267, -1.1951406002044678, -0.560271143913269, -0.17422239482402802, -1.155376672744751, -0.20626911520957947, 1.1461769342422485, 0.7745732665061951, -0.1760801374912262, -1.1676619052886963, 0.23065021634101868, 0.6584824323654175, 1.324562668800354, -0.009877345524728298, -0.35276198387145996, 0.333858847618103, 1.6290562152862549, -1.2063244581222534, 0.07120299339294434, 1.7576583623886108, 0.8372432589530945, -1.6816647052764893, -2.4429633617401123, 0.9923718571662903, -0.1392795890569687, 0.35784032940864563, 0.864284336566925, 1.2651978731155396, 0.4169448912143707, -0.736356794834137, 0.014938847161829472, 1.2742940187454224, -1.479832649230957, -1.5120675563812256, 0.23943892121315002, 0.4824756979942322, -0.6477785110473633, 1.4282217025756836, 0.3094874620437622, 1.00900137424469, -0.1410733014345169, 0.5262024402618408, -0.5146346092224121, -0.6862275004386902, -1.1825922727584839, -0.11405078321695328, 0.4842548072338104, 0.6807953119277954, -0.24633102118968964, -1.9999308586120605, 0.8394330143928528, 1.1693825721740723, 0.33260104060173035, 1.5693304538726807, 1.5369880199432373, -1.2892580032348633, 1.6853958368301392, -0.34591054916381836, -0.5281897187232971, -0.6560148000717163, -0.3032620847225189, 0.4496411681175232, 0.05016026645898819, 1.7226959466934204, 0.13873934745788574, 1.4046608209609985, -1.4269105195999146, -0.7834709286689758, 0.21880877017974854, -0.2727666199207306, 1.890929102897644, -1.4038357734680176, -0.6597940325737, 0.7621568441390991, 0.44876790046691895, -0.6442002654075623, -1.109034538269043, 0.12346245348453522, -0.10091224312782288, 1.1213610172271729, -1.0764154195785522, -1.0310660600662231, 1.2066861391067505, -0.6615270972251892, 1.533352255821228, -1.0354357957839966, 0.26033535599708557, 0.5908212065696716, -1.1993037462234497, 0.30921870470046997, 0.16926367580890656, 0.29962965846061707, -0.719188928604126, -0.5982891917228699, 0.20739148557186127, -1.094519853591919, -0.32133734226226807, -1.0162408351898193, 0.3889338970184326, 0.4402914047241211, 0.7248733639717102, 0.43394312262535095, 0.2405143529176712, 2.0858137607574463, 2.72709584236145, 1.053209900856018, 0.4582694470882416, 0.951450526714325, 0.019158393144607544, -1.153706431388855, -0.6382384300231934, 0.6011850237846375, -1.058443307876587, 0.8754989504814148, -0.9247841835021973, -0.8746116161346436, -0.8470256328582764, -0.10827770084142685, -1.2436519861221313, 1.8226133584976196, -0.7801300287246704, 0.8913984298706055, 0.3572287857532501, 0.031485751271247864, 0.5445228219032288, 1.0412896871566772, -0.47616147994995117, -0.4928244650363922, 1.054093837738037, -0.3381257951259613, -0.5881092548370361, -0.7155128717422485, 1.1397370100021362, 1.626592993736267, 1.0090410709381104, -0.40096673369407654, 2.1137709617614746, -1.2331072092056274, 0.9709633588790894, -0.7204899191856384, -0.7832475900650024, 0.9775804877281189, -1.2330280542373657, 0.08374620229005814, 0.5345967411994934, 0.3732927739620209, 0.631066620349884, 0.48999106884002686, -0.35587623715400696, -0.16379621624946594, -1.5446091890335083, -0.5856989622116089, 0.024182746186852455, -1.3868404626846313, 0.11712290346622467, -0.37938588857650757, 0.2557627856731415, 0.39315131306648254, -1.639596939086914, -0.18939058482646942, 1.2990870475769043, -0.44422706961631775, -0.42029938101768494, -0.02605525031685829, -0.08726196736097336, 1.2723166942596436, 1.758579969406128, -1.229793667793274, -1.0023173093795776, 1.9679933786392212, -0.24353191256523132, 0.6416472792625427, 0.12230533361434937, 0.24876543879508972, 0.7575832009315491, -1.3405390977859497, 0.04046347364783287, 1.327630639076233, -0.17602911591529846, 0.025003086775541306, 0.5300042033195496, -0.47122061252593994, 1.3536044359207153, 0.31436851620674133, 1.0328766107559204, -1.006130337715149, -0.20612601935863495, -1.142632246017456, -1.2716771364212036, 1.6150591373443604, -0.0684816837310791, -1.4400055408477783, -0.6923975944519043, -0.6617415547370911, -1.1136634349822998, 0.08763064444065094, 0.8639155030250549, 1.3053600788116455, -0.8667987585067749, 0.5344215035438538, 1.4746246337890625, 0.8614041209220886, 0.042319390922784805, -0.8988246917724609, 2.058741331100464, -0.1671382188796997, 0.4229089915752411, -0.7846723794937134, 0.5170933604240417, -0.4501166343688965, -0.8278093338012695, -0.051614757627248764, -0.007211350370198488, 2.1410012245178223, -0.07486838847398758, -1.251551628112793, 1.0156925916671753, 0.36999398469924927, -0.10908389091491699, -0.7778695225715637, 0.6304951906204224, -0.3586026728153229, 0.6077878475189209, 0.3705030381679535, 0.6492236852645874, -0.23660719394683838, 3.141127824783325, -0.09082786738872528, -1.5457547903060913, 0.3115004897117615, 0.4611653685569763, -0.36391887068748474, 0.5727981328964233, -0.3193208873271942, 0.36772775650024414, 1.1316618919372559, -0.16728536784648895, 1.0109219551086426, 0.8475689888000488, -2.3331267833709717, -0.9326637983322144, -0.5152154564857483, -0.05297510325908661, -0.18972371518611908, -0.19330133497714996, -1.2524712085723877, 1.1590523719787598, -0.536984920501709, -0.17953503131866455, 0.9447252154350281, -1.0890557765960693, -1.4330288171768188, 0.7053103446960449, -0.25481224060058594, -1.659636378288269, 0.34304943680763245, -0.41573700308799744, -0.3922463059425354, 1.391624927520752, 0.28503915667533875, 0.02492232620716095, 0.5936241149902344, -0.8948531150817871, 0.38787856698036194, -0.01852712780237198, -0.1587294191122055, -0.05192289128899574, -0.8909884095191956, -0.808718740940094, 1.314632773399353, 1.9673285484313965, -1.058344841003418, -1.599233627319336, 0.8708231449127197, -0.4685148000717163, 0.17656926810741425, 0.6088211536407471, 0.6057928204536438, 0.26236310601234436, 1.3032386302947998, 0.6696351170539856, 1.1931750774383545, -0.3553939461708069, 0.27188268303871155, 0.19292917847633362, 1.2637170553207397, 0.27978986501693726, 0.886006772518158, -0.03933180123567581, 0.46787965297698975, 0.6482694745063782, -0.1528944969177246, -1.2093150615692139, 0.812518835067749, -1.0625450611114502, -0.12214308232069016, -1.2209341526031494, 1.445733904838562, 0.6021465063095093, 0.3635264039039612, -1.3230795860290527, -0.7512364983558655, 0.1964588761329651, 0.33928728103637695, 0.9349398612976074, 0.31222954392433167, 0.06774821877479553, -1.0907374620437622, -0.25866055488586426, -0.9080476760864258, 1.044511079788208, -0.5131070017814636, 1.222408413887024, -0.2517062723636627, -0.02987331710755825, 1.541212558746338, -1.5747148990631104, 0.9313893914222717, 0.2193046361207962, -1.3352508544921875, 0.6001631617546082, -0.10297916829586029, -1.6550768613815308, 0.4531874358654022, 1.3953533172607422, 0.9087499976158142, -2.304154634475708, -0.5796934366226196, 0.3081700801849365, 0.12562718987464905, -0.18706925213336945, 0.47169211506843567, -0.6766178607940674, 1.7425824403762817, -0.14881569147109985, -0.7449934482574463, -0.7953739166259766, -1.7154840230941772, -0.10622920840978622, 0.6843132972717285, 1.9290046691894531, -0.515913188457489, -1.188791275024414, 1.029227375984192, 1.070327877998352, -0.5244888663291931, -0.3278094530105591, 1.8233851194381714, 0.7079766988754272, 1.259814739227295, 0.3585486114025116, 0.882588267326355, 0.30328187346458435, -1.0095123052597046, 1.4167360067367554, 1.8091158866882324, -0.2805015742778778, 1.049286961555481, 0.4196382761001587, 0.33791986107826233, -0.22057418525218964, 0.9744083285331726, 0.3190160393714905, -1.3444744348526, 0.7043449878692627, 0.21531063318252563, 1.170248031616211, 0.5260518193244934, 0.07851198315620422, -0.3082179129123688, 0.5473707914352417, -0.3437192142009735, 0.7718632221221924, -0.4868753254413605, -0.18745729327201843, 0.5902910828590393, 0.6724627017974854, -0.6541134715080261, 0.4615098536014557, 0.4575044810771942, -1.0400526523590088, 0.17734038829803467, 1.4315307140350342, -1.9962953329086304, -0.25276196002960205, 0.4148028492927551, 1.4788841009140015, -0.5913833379745483, 0.08921768516302109, -0.9738837480545044, -0.5439590215682983, 1.2570074796676636, -2.2818291187286377, -0.3050744831562042, 0.8791813850402832, -0.5866638422012329, -0.19729888439178467, -0.0025796727277338505, -0.23456472158432007, 0.9106429219245911, 1.8310261964797974, -0.8737010955810547, 0.827620804309845, -0.8100365996360779, -0.498666912317276, 2.3964929580688477, 1.0128799676895142, 1.2431137561798096, -2.180820941925049, -0.7792257070541382, 1.0165270566940308, -1.798916220664978, 1.3689326047897339, -0.9412240386009216, -0.6501768827438354, -0.8201509118080139, -0.7918254733085632, -0.877932608127594, 0.47240909934043884, -0.43540412187576294, -1.7018442153930664, 1.1300969123840332, 0.07020007818937302, -1.1697602272033691, 0.4740818738937378, 0.45807021856307983, -1.0432578325271606, -0.23126573860645294, 0.46279069781303406, 1.1446919441223145, -1.2687050104141235, 0.8636118173599243, -0.35739755630493164, 0.7664951682090759, -1.1347678899765015, -0.4720315635204315, 0.0037250560708343983, -1.7793446779251099, -0.8886452913284302, -1.666334867477417, -0.8401347994804382, 0.29984617233276367, -1.0038272142410278, -1.0832879543304443, 1.220294713973999, 0.6427050232887268, 1.1213583946228027, -0.334852397441864, -0.47384291887283325, -0.9877446889877319, 1.6843427419662476, -1.7659050226211548, 0.6863232851028442, 0.8446780443191528, -1.0150188207626343, -1.3655866384506226, -0.6529760956764221, -0.693239688873291, 0.882468044757843, -3.1800570487976074, 0.6477404236793518, -0.810769259929657, 0.3443303108215332, -0.8008763194084167, 1.5504661798477173, 0.9145922064781189, -0.603032112121582, 1.6908613443374634, -0.25688984990119934, 0.5522445440292358, -0.458251029253006, -1.290876030921936, 0.4970652163028717, -1.4433536529541016, -0.7611109614372253, -0.13361182808876038, -1.1167268753051758, 0.016046255826950073, -0.11946887522935867, 2.40818452835083, 0.8151846528053284, 0.4139866828918457, 0.03797227889299393, -0.34893354773521423, 1.204884648323059, -1.143868327140808, 0.4905110001564026, -0.9972974061965942, 0.2492755800485611, 0.8328572511672974, 1.0767260789871216, -1.151818871498108, 0.4166393280029297, 0.09555280208587646, 1.402963399887085, 0.3338741660118103, -2.378502368927002, 1.2362303733825684, 0.7003299593925476, 0.19292502105236053, -0.056041326373815536, 0.16907168924808502, -0.56770920753479, -1.0123486518859863, 0.3456563651561737, -0.3003554940223694, -0.6252076625823975, 0.6008721590042114, -0.8476753830909729, -0.25566932559013367, -0.2949274182319641, 1.2382862567901611, 0.20269733667373657, -0.07283852249383926, 0.701026976108551, -0.49490347504615784, -1.9698309898376465, -0.46999314427375793, -0.18184205889701843, -1.3595635890960693, 0.5579313039779663, 0.7192986607551575, -0.37854665517807007, -0.933048665523529, 0.6147521734237671, 0.7705161571502686, -0.947089672088623, 0.5776655077934265, 0.5926478505134583, 1.1320033073425293, 0.7757192850112915, -0.4408031404018402, -0.4231683313846588, 0.04221402481198311, 0.3741784989833832, 0.38881754875183105, -0.7525570392608643, 1.4706588983535767, -0.4882391393184662, 0.8607191443443298, 0.1301112323999405, -1.3042958974838257, -0.5862047076225281, 0.33932170271873474, 0.3385559022426605, -0.13398973643779755, -0.640155553817749, 1.7998591661453247, -1.293237328529358, -1.3043603897094727, 0.32426169514656067, 1.2104986906051636, 0.6073974967002869, -0.8421857357025146, -0.5363501310348511, 0.5548150539398193, -0.8824484348297119, -0.2634337246417999, -1.5742542743682861, -0.5891746282577515, -0.577958345413208, -1.037896990776062, -0.3446315824985504, -0.616769015789032, -0.3681032955646515, -1.4803804159164429, -0.5964298844337463, -1.4488542079925537, -0.11087022721767426, -0.2901117205619812, 0.431017130613327, -0.9732162356376648, 0.0014271429972723126, -0.00564511539414525, -1.4349414110183716, 0.5098928213119507, 0.3987478017807007, -0.4638347327709198, -0.44069844484329224, 0.5763387680053711, 0.3467826545238495, 0.8162006139755249, -1.0058709383010864, 0.9920852184295654, 0.9150201678276062, -1.6507729291915894, -0.21343903243541718, 0.7569667100906372, 0.01868770644068718, 0.5120618939399719, -1.1329188346862793, -0.08241133391857147, 0.3617171347141266, -0.9630151987075806, -0.5764431357383728, 0.4404814541339874, -0.8092737793922424, -1.157829761505127, 0.27123522758483887, -0.2675967812538147, 0.8331036567687988, -0.4204481244087219, 0.2472071647644043, -1.3565466403961182, 0.5562812089920044, -0.19865259528160095, -0.010018251836299896, -1.2215982675552368, 1.2351117134094238, -0.5267093181610107, -0.7078009247779846, 1.0423554182052612, -1.2256994247436523, -0.9368893504142761, -0.14663659036159515, 0.6964132189750671, 0.5707158446311951, -1.282088041305542, 0.08100827038288116, 0.9598376750946045, 1.011986494064331, -1.0966252088546753, -0.18637719750404358, -0.2549550235271454, -0.5524161458015442, -0.3386031985282898, -0.2779443562030792, -1.889654517173767, 0.9466730356216431, 0.6699289679527283, -0.524344265460968, -1.1710209846496582, 1.0649805068969727, 0.3192351162433624, 2.932912826538086, 0.5224400758743286, -2.0618951320648193, -0.0009244779357686639, -1.2588226795196533, -1.429254174232483, 0.2111915647983551, -0.24404536187648773, 0.16839946806430817, 0.44719773530960083, 0.856335461139679, -0.27838051319122314, -0.1914598047733307, 2.028204917907715, -1.8336440324783325, 1.507688283920288, 0.06772259622812271, -0.13198064267635345, 0.8505218029022217, 1.0350509881973267, -0.248270183801651, -0.9826463460922241, 0.6945169568061829, -0.4171833395957947, -1.0608994960784912, -0.8812442421913147, -1.570694088935852, -1.2377923727035522, 0.5202172994613647, -1.073571801185608, 1.5851249694824219, 0.04243814945220947, -0.5040790438652039, -0.08470635861158371, -1.0330778360366821, 1.694214940071106, 0.4869946539402008, 2.978924036026001, 0.7115088701248169, 0.05084021016955376, -2.272132158279419, 1.1913199424743652, 0.3705199360847473, -0.42473122477531433, -0.11879249662160873, -0.5779892802238464, 0.9936654567718506, 0.7599894404411316, -0.02749255858361721, 1.8995540142059326, -0.5835047364234924, -0.01983977109193802, -0.7393052577972412, -0.6746383309364319, 0.011899692937731743, -1.889062762260437, -1.8522343635559082, 2.189423084259033, -1.1967439651489258, 1.0925650596618652, -1.3555043935775757, 1.0032155513763428, 1.556401252746582, 3.2633142471313477, -2.486530303955078, -1.3849937915802002, 1.159729242324829, 0.7257177829742432, -1.693372368812561, 0.46977269649505615, -0.8644357323646545, -0.8857221007347107, -0.750678300857544, -0.874980092048645, 1.4639207124710083, 0.02739631198346615, -1.2777475118637085, -0.23971156775951385, -0.6478890180587769, 0.48101291060447693, 2.713981866836548, -0.1824345588684082, -1.899751901626587, -0.1322954297065735, -0.1680113524198532, 0.41789403557777405, -1.4001643657684326, -0.7957239747047424, -0.11329580843448639, -1.4424644708633423, -1.2004499435424805, -1.6053621768951416, 0.5822890996932983, 0.7384276390075684, 0.7353358268737793, -0.7409637570381165, -0.5963114500045776, 1.079367995262146, -0.2022385597229004, 0.24239540100097656, 0.1260100156068802, -0.35372394323349, -0.6941772103309631, 0.9532274603843689, 1.2420779466629028, 0.4137590527534485, 0.11531054228544235, -0.5778430700302124, -0.30837443470954895, -1.0434520244598389, 0.4900355041027069, 0.13951964676380157, 0.5168839693069458, -1.3279308080673218, 2.8679018020629883, 0.15247981250286102, -0.3568875789642334, -1.3763766288757324, -0.8816028833389282, 0.6886472105979919, 0.9311915040016174, 1.905781626701355, -1.0373287200927734, -0.610984742641449, -1.9536832571029663, -0.03771331161260605, 0.744036078453064, 2.0003092288970947, 1.5813497304916382, 0.6873915791511536, -0.15233156085014343, 0.5810502767562866, 0.8293691873550415, -0.9568465948104858, 1.6999030113220215, -0.7981995344161987, 0.9539733529090881, -1.5402547121047974, 0.11732490360736847, -0.12875865399837494, 0.40582743287086487, 1.080601453781128, -0.8009663820266724, 0.10673577338457108, 0.9833114147186279, -0.41684120893478394, 0.631583034992218, 0.1909143626689911, 2.391181707382202, 0.9483626484870911, 1.2692878246307373, -1.435640811920166, 0.5141074657440186, 0.8445183038711548, 1.359061598777771, 1.6400200128555298, -0.890222430229187, -0.10685694962739944, -1.0480996370315552, -0.16808128356933594, 0.7719356417655945, -0.3689377009868622, 0.049838680773973465, 0.539313793182373, -0.6375845074653625, -1.638596773147583, -0.4272097051143646, 0.9171552062034607, 0.857119083404541, -1.304264783859253, -0.24115271866321564, 1.5644505023956299, 0.8023303151130676, -0.6129562854766846, 1.3856042623519897, 0.29540401697158813, -0.19185870885849, -0.18503950536251068, -0.10298753529787064, 1.2846970558166504, -0.2761589586734772, -0.8017401099205017, 0.15132665634155273, 0.15736563503742218, 1.4424694776535034, 0.8138443231582642, -0.29045289754867554, -1.3969260454177856, 1.4543907642364502, 0.3929669260978699, 0.5553497076034546, 0.24555636942386627, -0.1278192698955536, -0.024278860539197922, 2.4757039546966553, 1.8093069791793823, 0.08187524229288101, 0.2629242241382599, -1.0765303373336792, 1.310573697090149, 1.3621747493743896, -0.0572977289557457, -2.1062209606170654, 2.3098888397216797, -0.16319186985492706, -0.6011064052581787, -0.8572262525558472, 0.4207751154899597, -1.102827548980713, 0.3822548985481262, 0.9153040051460266, 1.1186895370483398, -0.9251834154129028, 0.19065062701702118, 0.5353222489356995, 0.2545355260372162, -0.2032010555267334, -0.9796038866043091, -0.5457821488380432, 0.9838692545890808, 0.9398505687713623, -0.9641216993331909, -0.08544593304395676, -0.5657476782798767, 1.4193490743637085, -0.9623637795448303, 1.591849446296692, 0.5846052169799805, -1.229514479637146, -0.707206666469574, -0.19419220089912415, -0.11173807084560394, 1.920534372329712, 0.2891436219215393, -0.26041796803474426, -0.05655699595808983, -1.054498553276062, -0.1574007272720337, 0.9150384664535522, -0.15644364058971405, 1.1315900087356567, -0.11114402860403061, -0.3826734125614166, -2.242567539215088, -1.5342994928359985, -0.23245121538639069, -1.3690372705459595, -0.36024144291877747, 0.2842368483543396, -1.0827070474624634, 0.19166287779808044, -0.08590836822986603, 0.16071811318397522, -1.529322624206543, -1.1165393590927124, -0.57552570104599, -0.6443792581558228, -0.7468743920326233, -2.4556190967559814, 1.8374167680740356, 0.420602411031723, 1.933780312538147, 0.8071696758270264, -1.4668347835540771, -0.015858709812164307, -1.0628554821014404, 0.4456974267959595, -1.2364808320999146, -0.6318315267562866, 0.7444415092468262, -0.26767969131469727, 0.31129324436187744, 1.2699673175811768, -1.1798003911972046, 0.3462001085281372, 0.16920509934425354, 0.6797260642051697, -1.4648903608322144, -0.025113599374890327, -0.23538561165332794, 0.3768364191055298, 1.5506131649017334, -0.6507202982902527, -3.0014162063598633, -0.9893770813941956, -0.9474010467529297, 0.7828783392906189, 0.7488701939582825, -0.7908009886741638, 0.15852950513362885, -0.16736850142478943, -0.3672586977481842, 1.5489165782928467, -0.9818601012229919, 0.6349430680274963, 0.3961876332759857, 0.6317539215087891, -1.1299636363983154, -1.0146865844726562, 0.2574392855167389, 0.29820457100868225, -0.4148772954940796, -1.000431776046753, -1.003609538078308, -0.9055464267730713, -1.5484644174575806, 0.4681127965450287, 0.27507123351097107, 0.5732461214065552, -0.4075060784816742, 0.3656895160675049, -0.9629161357879639, -1.2734744548797607, 1.2254087924957275, -1.012712001800537, 0.4975428283214569, 0.8093356490135193, 0.5505187511444092, -0.22111521661281586, 0.72984778881073, 1.5254881381988525, 1.7853734493255615, 0.2831525504589081, 0.27369362115859985, 1.3566128015518188, -0.7355672717094421, -0.12412422895431519, -1.6273462772369385, 2.7442469596862793, 1.948844075202942, 1.700771450996399, -0.17314735054969788, 0.3378615379333496, -0.9244936108589172, -1.1905944347381592, -0.19960643351078033, 1.3969155550003052, 1.6423637866973877, -0.822717010974884, 0.29857146739959717, 0.7034040093421936, 0.2592802047729492, -0.013944385573267937, 0.263422429561615, -1.2312486171722412, -0.3403494358062744, -1.2422749996185303, 1.556087613105774, -0.9270965456962585, 0.529725968837738, -0.044798530638217926, 0.6442126631736755, -0.2841833531856537, 1.007509708404541, -1.2294087409973145, 2.1927242279052734, -0.6449803113937378, -0.9337954521179199, 1.9301596879959106, 0.34057438373565674, -1.0680571794509888, -1.4066787958145142, 0.9921483397483826, 0.06492801010608673, 0.5540438294410706, -0.1227274164557457, -2.6226348876953125, -0.6653223633766174, 3.1865241527557373, 0.9210699200630188, -0.06308238953351974, 1.2699024677276611, -0.7941576838493347, 1.0553481578826904, -0.07365227490663528, -1.5284144878387451, 1.4181288480758667, 1.3450835943222046, 0.814472496509552, 0.29541653394699097, -0.023893307894468307, -1.2396141290664673, -0.14732077717781067, -0.26122066378593445, -0.8158160448074341, -0.5682453513145447, -0.5391733646392822, 1.3254607915878296, -0.8255376815795898, 2.1024441719055176, -0.7251946926116943, -1.442917823791504, -1.4177417755126953, -1.049961805343628, 1.1324453353881836, 0.3268444240093231, -1.40959632396698, 0.23005852103233337, -0.7644059062004089, 1.154296636581421, -1.4270234107971191, -1.648206353187561, 0.6129335165023804, -0.35100412368774414, -0.5470098853111267, 1.3070199489593506, -0.7933192849159241, 0.2632029950618744, 0.9099840521812439, -0.7418670058250427, 0.9059901237487793, 0.09708821028470993, 0.7097779512405396, -0.6881433725357056, -1.5524308681488037, -1.0806175470352173, -0.2855944335460663, -0.6281599998474121, 0.9769405722618103, 0.35388144850730896, -1.5773569345474243, 1.4899674654006958, 0.06281019002199173, -1.5680954456329346, -1.2261666059494019, 0.6501659750938416, 0.3790178894996643, 0.3574136197566986, 0.5076532959938049, 0.16028328239917755, 1.517425537109375, -0.0686485543847084, -0.023148665204644203, 0.7060842514038086, 0.23504917323589325, 0.08268625289201736, 1.094160556793213, 0.07485257834196091, 2.8625612258911133, 0.17661653459072113, -1.0809283256530762, 0.37353330850601196, 1.8475638628005981, 0.47254762053489685, 0.5782954096794128, 0.7702692747116089, 1.1866363286972046, -0.9839362502098083, 0.24227258563041687, -0.25748637318611145, -0.09266330301761627, 0.039270609617233276, -0.43913090229034424, 0.07583299279212952, 2.3076658248901367, 0.30992212891578674, -0.417436420917511, -0.3080368936061859, 0.25313448905944824, 1.4643727540969849, 0.7240367531776428, 1.075669527053833, -0.6105254292488098, 0.5629537105560303, 0.9672962427139282, 0.4117412865161896, 1.4687000513076782, 0.8597769737243652, 0.7952724695205688, 0.8458490967750549, 0.22027097642421722, -1.085115909576416, -1.4595922231674194, -0.1494416445493698, -2.1935617923736572, 0.7392070293426514, 0.3072322905063629, -0.016757048666477203, 0.41605105996131897, -2.543989419937134, -0.33676642179489136, 0.7422778606414795, 1.6098568439483643, -0.03378758952021599, 1.186204195022583, -0.2384095937013626, 0.4518202841281891, 0.17526555061340332, -0.9131906628608704, -1.1507596969604492, -0.9211735725402832, -1.113899827003479, -0.9730199575424194, -0.04590662568807602, -0.14775346219539642, -0.2894214391708374, -0.2660068869590759, -0.7576815485954285, -0.10269795358181, 0.7543103098869324, -0.030789947137236595, -0.7736948728561401, -1.246199607849121, 1.0027306079864502, 1.7858071327209473, 0.5671119689941406, 0.8664771318435669, -0.021496566012501717, -1.0470243692398071, 0.5916492938995361, 1.3104033470153809, -0.8690069317817688, -0.47449207305908203, 0.03030744381248951, 0.20635128021240234, -1.4117769002914429, 0.11185789853334427, 0.3996575176715851, 0.13788817822933197, 1.9301912784576416, -0.22143688797950745, -0.504613995552063, 0.40985313057899475, -0.7572275996208191, -0.7678825855255127, 2.2765450477600098, -0.1809074431657791, -0.821127712726593, 2.098068952560425, -0.11210059374570847, -1.8735378980636597, -0.4633651375770569, 3.7545905113220215, 2.160597085952759, -1.7740923166275024, -0.8536866903305054, 1.1812572479248047, -1.3624228239059448, 0.5334473848342896, 0.7041444778442383, 0.7369796633720398, 0.014166994951665401, 0.26527276635169983, -0.8193018436431885, -0.05272471532225609, -0.5682211518287659, -0.8971980214118958, 1.6928002834320068, 0.7226894497871399, 1.631662130355835, 1.6240724325180054, 0.09453582018613815, -2.0015783309936523, -1.4580647945404053, 1.0092008113861084, -0.04460703209042549, 0.16180244088172913, 0.9288837909698486, -1.5436947345733643, -0.3263958692550659, 0.04836186021566391, -0.48350808024406433, -0.27063482999801636, 1.0248243808746338, 1.12433922290802, -0.474765807390213, 0.7016538977622986, 1.1117533445358276, -1.546800971031189, -0.8759575486183167, 0.6025012731552124, 1.1327269077301025, 0.48835793137550354, 0.3604067265987396, 1.107466220855713, 0.4063476324081421, 2.197256565093994, -1.3242396116256714, 1.8719642162322998, -0.08756859600543976, 1.2459514141082764, 0.13744154572486877, 0.9788733720779419, 0.0957435742020607, 0.2357671558856964, -0.874112069606781, -0.019480040296912193, 1.2530076503753662, 0.7869395613670349, -0.44475626945495605, 0.3020293712615967, -2.626277208328247, -0.0659426897764206, -1.5158847570419312, 1.6775317192077637, 1.8277441263198853, 1.8439974784851074, 0.08650172501802444, 0.058672841638326645, -0.13942615687847137, -0.2038014680147171, 1.1438473463058472, 0.5782312750816345, -1.2836006879806519, -1.4371706247329712, 1.3428761959075928, -0.501899778842926, -0.8623862862586975, -0.8156372904777527, -0.6811889410018921, -0.14893853664398193, 0.5835290551185608, -0.4784509539604187, 0.004154330585151911, 1.0029528141021729, 0.9853805303573608, 2.3028481006622314, -0.22907902300357819, 0.3524128496646881, -0.37886038422584534, 0.0686851367354393, -0.255321204662323, 1.9027236700057983, -0.3329075276851654, -0.154422789812088, -0.28601589798927307, -0.23582467436790466, 0.1581113636493683, 0.6192513108253479, -0.1202441155910492, -1.658586859703064, 2.3899285793304443, 0.18905510008335114, -0.5654679536819458, -0.5235117077827454, -0.6350369453430176, -1.7099064588546753, -1.1478520631790161, 0.9610949158668518, 1.0536620616912842, -0.04760909453034401, 0.24104636907577515, 0.8737420439720154, -0.21597547829151154, 0.7514007687568665, 0.18923114240169525, -0.877907931804657, -0.12755505740642548, 1.354197382926941, 0.019319219514727592, 1.4661213159561157, 0.39280903339385986, 0.005286120809614658, -1.4581667184829712, -0.2510814070701599, 0.7546224594116211, -0.43398407101631165, 0.770952045917511, -0.5603587627410889, 0.6022900342941284, 1.096903920173645, -0.029735147953033447, 0.35312384366989136, -0.6811121106147766, -0.09277995675802231, 1.7045575380325317, -0.3708450496196747, -0.9082017540931702, -0.28347304463386536, 1.032886266708374, -0.41911762952804565, -1.4714337587356567, -0.08066203445196152, -0.6032735109329224, 0.013695310801267624, -0.41622576117515564, -0.5863186120986938, -0.03450166806578636, -1.16361665725708, 0.24499407410621643, -1.4416615962982178, 2.441138982772827, 0.6952856779098511, -0.665790855884552, -0.2313290536403656, -1.038956880569458, 0.40328487753868103, 0.5965243577957153, 1.2398093938827515, 1.2558318376541138, -0.19938179850578308, 0.09086696058511734, -0.20301030576229095, 0.2201012521982193, 1.0433019399642944, -0.005017926450818777, -0.5195494294166565, 0.14664378762245178, -0.43045341968536377, -1.0355654954910278, -0.5716727375984192, -0.9444006681442261, -0.054481979459524155, 0.02933034859597683, -1.8755855560302734, 0.12529303133487701, 0.7069602608680725, 0.32000914216041565, -1.5595892667770386, -0.7192103266716003, -1.1856577396392822, 0.6024808883666992, 1.9492360353469849, 0.5517402291297913, 2.598172426223755, 0.09906533360481262, 0.4329985976219177, -0.5717892646789551, -0.04445084184408188, -2.4204249382019043, 0.6334881782531738, 1.2905604839324951, -2.730767250061035, 0.35811513662338257, -0.040823452174663544, 0.028773417696356773, -0.6623984575271606, 0.26030778884887695, -0.35217803716659546, -0.5028626322746277, 0.16236691176891327, -0.9347649812698364, 0.3683534860610962, 2.474059581756592, -1.418160319328308, -0.4155764877796173, -1.5645827054977417, -1.386925220489502, 0.034352611750364304, 0.12651661038398743, 0.17549721896648407, 0.018348002806305885, -0.5916258096694946, 0.9093884229660034, 0.4126776158809662, 0.3332895040512085, -0.7555898427963257, -0.4847029447555542, -0.5464232563972473, 0.6812087297439575, 1.0226434469223022, 1.287951111793518, -1.7345706224441528, -1.2039257287979126, 0.40165889263153076, 0.028043588623404503, 1.305286169052124, 0.7991899847984314, -0.4390918016433716, 0.5417144894599915, -0.674598217010498, 0.32737085223197937, -0.7548435926437378, -0.9082781076431274, 1.2784596681594849, 1.3079532384872437, -1.8244763612747192, 2.880695343017578, 0.4161255359649658, 1.1947546005249023, 0.17613963782787323, -0.16710014641284943, 1.5558176040649414, 0.5126445889472961, 0.42019158601760864, 1.602968692779541, -0.7242704033851624, 0.1650054156780243, 0.626230776309967, -0.10971743613481522, -0.5193496942520142, 1.509589672088623, 1.4445652961730957, -1.5091502666473389, 0.3216334283351898, -0.207960307598114, 1.010543704032898, 0.8128827810287476, -0.5454908609390259, -0.6391041874885559, -0.5728989839553833, -0.2728680372238159, 0.8010709285736084, -0.17867949604988098, -0.9892365336418152, 0.7175029516220093, 1.053977370262146, 1.2046884298324585, -0.06242114305496216, 0.8066504597663879, -0.4946655035018921, 1.5189236402511597, -1.4016361236572266, 0.9742858409881592, -0.48795634508132935, -0.4202723801136017, -0.4620126485824585, 0.6117432713508606, 0.3307538330554962, 1.719099760055542, -0.7908381819725037, 0.8784857988357544, -0.47951579093933105, 1.0802454948425293, 0.2824532687664032, 1.547455072402954, -0.12674593925476074, -1.0483514070510864, 2.168606996536255, 0.940952479839325, 0.7831851243972778, -0.15499186515808105, -0.21848852932453156, -0.49276164174079895, 0.49013692140579224, -0.5407795906066895, -1.914260745048523, 0.9994070529937744, 2.1128246784210205, 0.6923408508300781, -1.4110584259033203, -0.6620097756385803, 0.2575300335884094, 1.322436809539795, -0.2366291731595993, -1.317083716392517, -0.010801097378134727, -1.4721423387527466, -0.3210291564464569, 0.5935604572296143, 0.352486789226532, -0.6924688220024109, -0.42743632197380066, -0.5856929421424866, -0.39576616883277893, -0.4578709304332733, 0.4715898334980011, -0.18623052537441254, 0.5544925332069397, 0.8734636306762695, 1.363191843032837, 2.025125503540039, -0.13022653758525848, 1.3310446739196777, -0.963241696357727, 1.4485067129135132, 1.0459377765655518, -0.31058469414711, -0.001548977568745613, -1.427720308303833, -0.07755791395902634, -1.2124320268630981, 1.469999074935913, 0.6034814715385437, -0.17625464498996735, 0.4468453526496887, -0.1095811054110527, -0.05384023115038872, 0.5744187831878662, -0.513418972492218, -1.3597376346588135, 1.2610797882080078, 0.1622021645307541, 1.0659250020980835, -0.6257295608520508, 0.9943594336509705, -1.929425597190857, -1.1747099161148071, -0.5001630187034607, -1.2368146181106567, -0.3299981653690338, 0.4016243517398834, 0.39823925495147705, -0.9488258361816406, 0.5657315254211426, 0.9188989400863647, 0.06100428104400635, -0.8706819415092468, -1.0866105556488037, 0.5615800619125366, -0.43749287724494934, -1.0703085660934448, -0.7618817687034607, -0.953783392906189, 1.8416588306427002, -0.7265748977661133, 0.3985813856124878, 0.5797430276870728, 0.1289759874343872, -0.6001328825950623, 0.19157500565052032, -1.1096150875091553, -0.5099155902862549, 0.0015042823506519198, 0.6982952356338501, 0.29394665360450745, -0.4076201617717743, -0.06950968503952026, 0.07254332304000854, -0.5040217638015747, 0.1327599734067917, -1.045316219329834, -0.5656304359436035, -0.15512505173683167, 1.123058557510376, -1.097690463066101, 0.8297274112701416, -0.30077478289604187, -0.3215172290802002, 0.7626104950904846, 0.09895892441272736, 1.0377404689788818, -0.9279342889785767, -0.8638995885848999, -0.3008611500263214, -0.12544666230678558, 1.042510747909546, 1.3948190212249756, 2.3855342864990234, 0.09519315510988235, -0.28648465871810913, -1.7325999736785889, -0.15880808234214783, -0.02989904396235943, 1.606360912322998, 0.6196789145469666, 0.893474280834198, 0.5350542068481445, -0.8954226970672607, -1.2786463499069214, 0.2305578887462616, -0.5466042757034302, -1.3636795282363892, -0.0836743712425232, 0.5219612121582031, 0.6526864171028137, -1.275860071182251, 1.5139663219451904, -0.35137584805488586, 1.5727086067199707, -0.5589144825935364, 0.2851024866104126, -0.5638794898986816, 0.5935974717140198, -1.2036594152450562, 0.004781309049576521, 0.971307635307312, -0.09043430536985397, 0.8708606362342834, 0.1941695362329483, 1.2696675062179565, 1.5186153650283813, -0.8863188028335571, 0.0422031544148922, -1.1952134370803833, -0.13879798352718353, -0.3746224641799927, -0.2961420714855194, -1.0876222848892212, 2.1036970615386963, 0.16872933506965637, -1.2715858221054077, 0.17257677018642426, -0.0600522942841053, 0.7457751035690308, 2.3427605628967285, 1.895795464515686, -0.8921986222267151, 0.42326319217681885, 0.10647274553775787, 0.5399969220161438, -0.7632986307144165, -1.1476078033447266, 0.8240695595741272, 1.1224756240844727, -0.13144780695438385, 1.2686514854431152, -1.4415686130523682, -0.29990166425704956, 0.70405513048172, 0.24875588715076447, -0.37554094195365906, -0.16674432158470154, 0.8455114960670471, -2.0363996028900146, 1.1411443948745728, -0.9132367968559265, 0.17793340981006622, 0.6634895205497742, 0.29133671522140503, 0.34819456934928894, -0.8149849772453308, -1.340802550315857, 1.7122318744659424, 0.17274466156959534, 0.787248432636261, 0.8827417492866516, 2.1594862937927246, -1.2643367052078247, 2.4786338806152344, 0.4989679157733917, -1.4617087841033936, -1.1927217245101929, -0.7678903937339783, -0.9833356738090515, 0.13090503215789795, 0.12597812712192535, -0.9208294749259949, 0.128226637840271, 2.1825098991394043, -0.9891469478607178, 0.41121581196784973, -0.8094167113304138, 0.7086142897605896, 0.0007269051275216043, 1.561058521270752, -1.0608186721801758, -1.2098277807235718, -0.23781119287014008, 0.07851480692625046, 0.6444214582443237, 0.4054326117038727, -1.015671730041504, -0.9476436376571655, -0.9416036605834961, 0.8869562149047852, -0.8112519383430481, -0.052100662142038345, 1.0570225715637207, 0.4010462462902069, -0.33372822403907776, 0.6219961047172546, 0.7951924204826355, -0.37440183758735657, -0.4406386911869049, -1.7185693979263306, 0.7341101765632629, 1.0722265243530273, 0.44185805320739746, 0.7343413233757019, 0.2864750027656555, -0.7877167463302612, -0.01321711391210556, 2.097642183303833, 0.7285893559455872, -0.6734583377838135, -0.09775689244270325, 0.21009992063045502, 0.5349825620651245, 1.5255190134048462, -0.6564925312995911, 0.5944488644599915, -0.10979504883289337, -0.2563685476779938, -1.1611230373382568, -0.5334506630897522, -0.8268241286277771, 0.07706084847450256, 0.9245995283126831, -1.79026198387146, 0.726878821849823, 0.947252631187439, 0.4765002429485321, 0.9189902544021606, 1.0294126272201538, -0.42775759100914, 0.06108104810118675, -0.6519219875335693, 1.3057018518447876, 0.47094979882240295, -0.7177333831787109, -1.1499528884887695, 0.5728737115859985, 1.627364158630371, 0.5052794814109802, -0.22764882445335388, 0.5493429899215698, 1.7045947313308716, 0.6696248054504395, 0.8022226095199585, -0.2839326858520508, -0.1284743994474411, 0.3435373604297638, -0.6634503602981567, 0.4942722022533417, -0.7834247350692749, 2.309488296508789, 0.9379700422286987, 0.09456387162208557, 0.8286904096603394, 0.42229869961738586, 1.264223337173462, -0.49760064482688904, -0.21020589768886566, -0.7678739428520203, 0.7171269059181213, -0.4624868333339691, 0.10254267603158951, 0.7649878263473511, -0.8542622923851013, 2.581150531768799, 0.20574074983596802, -1.6334881782531738, 1.3368782997131348, -2.0698134899139404, -0.8324936032295227, -1.4497652053833008, 1.547279953956604, -0.7721930146217346, -0.7415318489074707, 0.47476401925086975, 0.14129725098609924, -0.498922199010849, -1.396567463874817, -1.362375259399414, 2.127784490585327, 0.7028341889381409, 1.574469804763794, -0.5705477595329285, 0.7261847257614136, 0.5441359877586365, 1.5105489492416382, 0.4052809178829193, 0.18724998831748962, 0.45098644495010376, -1.3905514478683472, -0.05148174986243248, -0.27509579062461853, 2.0225038528442383, -0.40389740467071533, 1.2886384725570679, 1.7476253509521484, -0.32737040519714355, -0.9801255464553833, -0.9850638508796692, 0.48044857382774353, -0.6371172070503235, -0.01014840416610241, 0.3733310103416443, 0.08426021784543991, -1.558786153793335, -1.8515253067016602, -0.200914666056633, -0.09325302392244339, -0.6511587500572205, 1.2411651611328125, 1.0139038562774658, 0.03858187794685364, -0.2254752218723297, 0.705336332321167, 0.081965371966362, -1.092596173286438, 1.1023430824279785, -0.43263038992881775, -1.1282604932785034, -0.7154102921485901, -1.4215904474258423, 0.37978997826576233, 0.478644996881485, -1.8022581338882446, 1.7526981830596924, 0.5375781059265137, 0.5240506529808044, -2.7090396881103516, 2.123511552810669, 2.4499850273132324, 0.8212078809738159, 0.02878270298242569, -0.8028417229652405, -0.8852238059043884, -1.5470821857452393, -0.8367324471473694, 0.7446463108062744, 0.6111398339271545, 0.6916578412055969, 2.199479818344116, 0.3045819401741028, -0.9473174810409546, 1.06771981716156, 0.6586836576461792, 0.41322755813598633, -0.25975438952445984, 1.2012661695480347, 0.3859541118144989, 1.513479232788086, 0.41686591506004333, -1.05305814743042, -0.3758375346660614, 1.3912492990493774, -0.42736804485321045, -0.44542595744132996, 1.340572476387024, -0.008340488187968731, -1.1319077014923096, -1.161372184753418, 0.03538142517209053, 0.5253288745880127, 0.41251465678215027, -0.2879208028316498, -1.7062922716140747, 1.1490094661712646, 0.2512683868408203, -0.18504273891448975, 0.41231584548950195, -1.453011393547058, -1.148270845413208, -0.06146597862243652, -0.6165775656700134, 0.6655429005622864, -0.23462098836898804, -0.10184692591428757, 0.31414246559143066, 0.22472336888313293, 0.057330742478370667, -0.2911575734615326, -0.28559064865112305, 0.4921071231365204, -0.6334257125854492, -1.132347583770752, -0.8758174777030945, 0.9589250087738037, -0.4263562560081482, 0.16420283913612366, 0.14000436663627625, 1.794144868850708, -0.7939204573631287, 1.1331992149353027, -0.8725574016571045, 0.6220085620880127, 1.6719824075698853, 1.096543788909912, 0.18043839931488037, 1.7548339366912842, -1.0919972658157349, -0.023213285952806473, -0.42239251732826233, 1.3486319780349731, 0.029535464942455292, 0.0012617438333109021, 0.7449546456336975, -1.0843534469604492, 2.0064682960510254, -0.886530876159668, -0.15870226919651031, -0.04305250570178032, 0.6507193446159363, 0.9048263430595398, -0.2806227505207062, -0.7466864585876465, 1.1647893190383911, 1.3172268867492676, 0.08914587646722794, -2.698672294616699, 1.2572643756866455, 1.7551847696304321, -1.5809667110443115, -0.6091871857643127, 0.9597101211547852, 0.7224075198173523, -0.04991799220442772, -1.8484777212142944, 0.33211788535118103, 0.6157134771347046, -0.9150558114051819, 0.037223391234874725, -0.11154726147651672, 0.2993164360523224, -0.7086700201034546, 1.9281985759735107, -1.37445068359375, -0.2663891613483429, 0.8613665103912354, 0.37095412611961365, -0.5357948541641235, -1.0719401836395264, -0.8806555271148682, 0.5635249614715576, 0.13599660992622375, -0.9358574151992798, -2.054690361022949, 0.47674524784088135, -0.050274722278118134, 0.8654171228408813, -2.393645763397217, 0.6167794466018677, -0.7046111226081848, 0.38958126306533813, -0.6504655480384827, 0.06498895585536957, -1.7493932247161865, 0.9211990237236023, 1.0037485361099243, -0.32412609457969666, -1.958371877670288, 1.33308744430542, 1.1328750848770142, 0.2587408125400543, 0.9884172081947327, -0.9687871336936951, 1.9005359411239624, 0.00985549483448267, 2.86179256439209, -0.37352126836776733, 0.5938159227371216, -0.2692272663116455, 0.2084045559167862, -0.8016931414604187, -0.09959963709115982, 0.5138639211654663, 0.40122073888778687, 0.9293825626373291, -1.1313037872314453, 1.1331714391708374, -2.111051559448242, -0.3604864776134491, 0.19846731424331665, -1.2077959775924683, -2.9180386066436768, -0.1568303257226944, -0.4671393930912018, 0.6604037880897522, 0.6489934921264648, -1.5652586221694946, 0.3425333797931671, -0.29151812195777893, -0.6398760676383972, 0.9977406859397888, -0.5202412605285645, 0.8652810454368591, -0.1749047040939331, -1.652940034866333, -0.09210321307182312, -1.1974760293960571, -0.8740143179893494, -1.9622143507003784, -0.5478892922401428, 1.0735944509506226, 0.5700066685676575, 0.18349769711494446, -0.5498981475830078, -1.535059928894043, -0.24759095907211304, 0.2743837237358093, -0.42601075768470764, -0.9602908492088318, 0.8861318826675415, 0.9400697946548462, 0.8325326442718506, 0.45431479811668396, 0.22062894701957703, 0.7449164390563965, -2.6603989601135254, 0.22416619956493378, 0.8294802308082581, 0.05832940712571144, -0.9769405722618103, -0.1687294989824295, 2.977581024169922, 1.2764455080032349, -0.9302740693092346, 0.8511533141136169, 1.791015386581421, -0.8693352937698364, 0.2244851142168045, -1.338088870048523, -0.1390024721622467, 0.37080317735671997, 0.9631152153015137, -1.2267258167266846, -0.6673558354377747, 0.5948562026023865, 1.4404501914978027, 0.8025141954421997, 1.3209643363952637, -0.19575177133083344, 2.1661720275878906, -1.0548686981201172, -0.11883620917797089, 0.36616528034210205, -1.949117660522461, -1.9768917560577393, 1.7638139724731445, 0.3219478130340576, 0.9300961494445801, -1.2549619674682617, -1.586333155632019, -1.8212629556655884, -0.39690542221069336, -1.098192811012268, 0.21320921182632446, 1.2173513174057007, 0.25473475456237793, 0.48812001943588257, -2.0160281658172607, -1.4605426788330078, 2.207674503326416, 0.13514915108680725, -0.5561659336090088, -1.7760878801345825, 2.2848777770996094, -1.4444833993911743, -0.6913607716560364, -0.47760164737701416, -1.081875205039978, 0.4240390658378601, 0.8840951323509216, 1.0509251356124878, 0.03792191669344902, 0.09059120714664459, -2.1326584815979004, -1.2035515308380127, -1.1553223133087158, 0.35743486881256104, -1.1416237354278564, 0.6834484338760376, -0.414051353931427, -1.5006132125854492, 0.12748752534389496, 1.2491804361343384, 0.79909348487854, -1.3214118480682373, 1.1702206134796143, -0.5781773328781128, 0.6296591758728027, -0.2938775420188904, 0.6393259763717651, -1.3581620454788208, -0.1609801948070526, -0.40059757232666016, 1.2707133293151855, -0.9297778606414795, -0.3191225528717041, -1.2941062450408936, -0.004668351728469133, 0.40292730927467346, -1.7929682731628418, -1.1170262098312378, 0.7086396217346191, 0.5895422101020813, 0.9729563593864441, 1.4866887331008911, 0.06667663156986237, -1.036597490310669, 1.2006776332855225, 0.9113695621490479, -1.026606798171997, -1.0161269903182983, 0.4326428174972534, 0.11256548017263412, 1.270849347114563, 0.3767235577106476, 0.179023876786232, -0.7001223564147949, 1.9900202751159668, -0.0452120304107666, -0.19891691207885742, 0.18248294293880463, -0.28082019090652466, -0.31760942935943604, 0.01991775445640087, 0.12653526663780212, -0.32907578349113464, -0.14953064918518066, -0.9450263381004333, -0.28994014859199524, 2.397185802459717, 0.7111168503761292, 0.08698765933513641, 0.8357647657394409, 0.4684104919433594, -1.030336856842041, 0.5200878977775574, -1.1662989854812622, 2.1611204147338867, -0.0905655026435852, 0.31704989075660706, -0.682661771774292, -0.8438066244125366, 0.3298890292644501, 0.5399900674819946, -1.1233761310577393, 0.07468333095312119, -2.070181369781494, 0.21538420021533966, -0.65260249376297, -0.048578765243291855, 1.3060352802276611, 1.276898741722107, 1.0129481554031372, 0.12491258978843689, -0.3741259276866913, 1.2206134796142578, 0.3154789209365845, 1.2584428787231445, 1.2133889198303223, 0.48620882630348206, -0.9453436732292175, -1.041632890701294, 0.8175331354141235, -1.192070484161377, 0.6488847732543945, -1.7131770849227905, -0.089823417365551, -1.2736390829086304, 0.25750732421875, -1.0384107828140259, 0.008930769748985767, -0.7440053820610046, -0.8403428792953491, -0.386893630027771, 0.10656683146953583, -0.8630138039588928, -0.6869087219238281, -0.007120821159332991, -0.9593808054924011, -1.0286685228347778, 0.38392820954322815, -0.45715856552124023, 0.46200111508369446, -0.42548444867134094, 0.9790414571762085, 1.5015062093734741, 0.5419349074363708, 0.0010868889512494206, -0.5904126763343811, 1.0364789962768555, -0.41312119364738464, 0.9307724833488464, -0.21967901289463043, 0.7096748352050781, -0.6499179601669312, -2.0347869396209717, 0.570990800857544, -0.472374826669693, 2.3030664920806885, -1.013085126876831, 0.7817960381507874, 1.6489850282669067, 0.46113118529319763, -0.2904636561870575, 1.8608592748641968, -0.26522064208984375, 1.3431025743484497, -3.09420108795166, 0.23057998716831207, -0.19447749853134155, -0.6668264865875244, -1.206852912902832, -1.0793139934539795, 1.5914313793182373, -0.04552832990884781, 0.08071563392877579, 0.7935624718666077, 0.6194254755973816, -1.4427953958511353, 1.286824107170105, -0.37935733795166016, 1.0995044708251953, -0.830790102481842, -0.8112942576408386, -0.06938832253217697, 1.3556413650512695, -0.29022103548049927, 0.7605552077293396, 0.3477635085582733, -1.9339443445205688, -0.482179194688797, -1.9973177909851074, -0.2115863859653473, 0.004010844975709915, 0.051128800958395004, -0.7974879741668701, 2.467653512954712, 0.9780679941177368, 0.29941490292549133, -0.8047670722007751, -1.7800508737564087, -0.28828951716423035, -0.00903377402573824, 0.09980174154043198, -0.8228432536125183, -0.8071406483650208, -0.5349491834640503, 0.24340878427028656, -0.3384351432323456, 0.2056809663772583, -0.18007780611515045, -1.993038296699524, 0.958365261554718, -1.6903746128082275, -1.1313680410385132, 0.059515755623579025, -1.655645489692688, -0.5334650278091431, -1.0648343563079834, 0.7761645913124084, 0.7115360498428345, 1.5836633443832397, 0.15964558720588684, 0.20240439474582672, -0.4814683496952057, -1.3692926168441772, 0.8965765237808228, 0.3216054439544678, -0.32773253321647644, 0.27597111463546753, -0.6252961754798889, 1.1548993587493896, -1.6821794509887695, -1.1816375255584717, -1.046510934829712, -0.04964692145586014, -0.8591837286949158, 1.7346075773239136, -0.5405699014663696, -0.05212768167257309, -1.1737600564956665, -0.47222498059272766, -0.7293120622634888, -0.10817720741033554, -1.8881055116653442, -1.4024369716644287, 1.7109113931655884, -0.9056031703948975, -0.07238274812698364, 1.407344937324524, -0.32481372356414795, 0.855211079120636, 0.48898524045944214, -0.3151620328426361, 0.5650559067726135, 1.5769292116165161, 0.4794299900531769, 0.4466516077518463, -0.5836386680603027, 0.42916926741600037, -0.020769864320755005, -1.067381501197815, -1.0822129249572754, -0.9937295913696289, 0.4257412850856781, -0.48852285742759705, -0.1529262214899063, -1.4030331373214722, 1.2461498975753784, -1.3626285791397095, 1.3974553346633911, 0.30319100618362427, -0.7391207814216614, 1.010229468345642, 0.6961972117424011, -0.28525489568710327, 0.4908912479877472, 1.8601562976837158, 0.028065018355846405, 0.3883231282234192, 0.16267095506191254, -0.38696110248565674, -0.09311620146036148, 0.9479560852050781, 0.1781514286994934, 0.1759747862815857, 0.9967004656791687, -0.34921014308929443, 1.2428789138793945, 0.7178913354873657, 0.396836519241333, 0.23967309296131134, -0.44156917929649353, -1.165794014930725, -0.3304893672466278, 0.2334575057029724, -0.7049907445907593, -0.9596838355064392, 0.4497007429599762, -0.5002169013023376, 0.7135840058326721, 0.4488650858402252, -0.5630092024803162, -0.25209230184555054, 0.7935229539871216, -0.8829476833343506, 0.6560383439064026, 0.11512833088636398, 0.23785026371479034, 1.0397570133209229, -0.46139073371887207, -2.1356301307678223, -0.09151952713727951, 0.4702100455760956, -0.8110668063163757, 2.96852970123291, 0.07379233837127686, -0.29451385140419006, 0.1400361955165863, 2.1323745250701904, -0.3037891089916229, 1.4929429292678833, 0.7187389135360718, -1.822044014930725, -2.7635281085968018, 3.1088974475860596, -0.6714189648628235, 1.7742440700531006, 0.8401896953582764, 2.3783602714538574, -0.4722394645214081, -1.5549628734588623, -1.321537733078003, -1.1384730339050293, -0.40786921977996826, 1.3025394678115845, -2.2526652812957764, -1.134633183479309, 1.1615124940872192, -0.11567389965057373, 0.46498024463653564, -0.6986736059188843, -2.5476677417755127, 0.20710226893424988, -0.19290708005428314, 0.10545984655618668, 0.13269317150115967, 0.2685379087924957, 0.01942451111972332, 0.3539404571056366, -1.0807663202285767, 1.8976088762283325, 1.107832670211792, -0.4740369915962219, -1.1043702363967896, 1.1123465299606323, 1.0686720609664917, -0.6592000722885132, -0.9556788206100464, 0.48190316557884216, -2.1535120010375977, -0.1491282731294632, -0.49590781331062317, 1.730772614479065, -0.9625909328460693, 0.34160149097442627, -0.8077349662780762, -2.8047616481781006, 1.8508226871490479, -1.348455786705017, -1.0941418409347534, 1.039678931236267, 0.37689143419265747, -0.7922956943511963, -0.6039923429489136, -0.16897624731063843, -1.4182299375534058, -0.3471343517303467, -0.27951306104660034, 0.8000428080558777, 0.9767970442771912, 1.664932370185852, 1.0946400165557861, 0.7940214276313782, 0.21046574413776398, -0.5238152146339417, -0.770175576210022, 0.1818896234035492, 0.06491602212190628, -0.33107849955558777, -1.1957008838653564, 0.6030930876731873, -0.6318342685699463, 0.9338067770004272, -0.253030925989151, 1.7419860363006592, 0.20007847249507904, 0.5107613205909729, -0.9752915501594543, 0.46706080436706543, 0.5789914131164551, -0.013652723282575607, -0.5696702003479004, -0.7244613766670227, 1.7043848037719727, -1.513835072517395, -0.31158319115638733, -0.7230713963508606, 1.1941932439804077, 0.1068049967288971, -0.6810458302497864, -1.671629786491394, 0.7737864851951599, 1.2480006217956543, -0.30551043152809143, 0.48210588097572327, 0.40569907426834106, 0.7244998812675476, 0.1796293407678604, -1.5450764894485474, 0.09294870495796204, 0.07107627391815186, -0.18804262578487396, 0.6643715500831604, 1.1021080017089844, 1.0943408012390137, 1.0145988464355469, 0.20188483595848083, 0.6197437047958374, -0.48985204100608826, -0.5971758961677551, 0.793407678604126, 0.9824494123458862, -0.12041948735713959, -0.9171987175941467, 0.8243722915649414, -0.47911933064460754, 0.17388483881950378, -1.5686978101730347, 0.2119312584400177, -0.5990415811538696, -0.5564956068992615, 1.0077213048934937, 0.06068456918001175, 0.025167331099510193, 0.5510475039482117, 0.45110490918159485, -0.8377856016159058, -0.5412324666976929, -0.252497136592865, -0.0010483141522854567, 0.07363682240247726, -2.03886079788208, 0.6347575187683105, 1.2964714765548706, -1.9854973554611206, -0.6681477427482605, -0.4690188467502594, -0.781452476978302, 0.0035302313044667244, -1.7321592569351196, 0.04024379327893257, -0.00016424716159235686, -1.4072493314743042, -2.115900754928589, 0.30852463841438293, -0.9470226764678955, 1.3366076946258545, 0.08500143885612488, -0.7593587636947632, 1.2899668216705322, 0.02153971791267395, -0.5880048871040344, -0.5340512990951538, -0.812881350517273, -0.2646431028842926, -0.7809235453605652, 0.13112977147102356, 0.631677508354187, -0.1647246927022934, 0.7774410843849182, -0.9439135789871216, -0.8530406951904297, -0.5879764556884766, -0.37484848499298096, 1.2267141342163086, -0.4150840640068054, -0.8463662266731262, 0.5445119142532349, -2.636843204498291, 0.7540192008018494, -1.891639232635498, -0.643644392490387, -0.572792112827301, -1.0421546697616577, -1.316263198852539, 1.2171107530593872, -2.7774336338043213, -0.5773075222969055, -0.8058428168296814, 0.22650004923343658, -2.2602500915527344, 0.3462602198123932, 1.8800811767578125, 0.6524237394332886, 0.6513301134109497, 0.08925744146108627, 0.24016845226287842, 0.24835263192653656, 0.30260559916496277, 0.34909769892692566, -0.5110004544258118, -1.8791571855545044, 1.4301621913909912, 1.2738665342330933, -0.19122380018234253, 1.6070188283920288, -2.104172468185425, -0.02902982197701931, 1.2189048528671265, -0.475911408662796, 1.009049654006958, -0.5831737518310547, 2.1954734325408936, 1.2904866933822632, 0.5581377744674683, -1.0374023914337158, -0.328338086605072, -0.7799020409584045, -0.7037825584411621, 0.0005521723651327193, 1.1118803024291992, -1.5933314561843872, 1.1048924922943115, -1.1856869459152222, -2.3288989067077637, 0.8117262721061707, 0.5990626215934753, -0.4146921932697296, -0.260614275932312, -0.6904717683792114, -0.14081920683383942, -0.8010792136192322, -0.4849558174610138, -0.642362117767334, 1.7079354524612427, 0.2852577567100525, -1.447368860244751, -0.17525383830070496, -1.1815091371536255, 0.22726836800575256, 0.295684278011322, 0.6938281059265137, -1.4073041677474976, 1.1122862100601196, 0.7581015825271606, 0.005540505982935429, 1.8129557371139526, -0.8256434202194214, 1.0499327182769775, 0.7917920351028442, -0.26509445905685425, -0.027607988566160202, -0.42344558238983154, -0.1753142923116684, 0.3506934642791748, 0.28156203031539917, 1.4291377067565918, -0.554897129535675, 0.6172391176223755, -0.08227526396512985, 0.9045560956001282, -0.7245833277702332, -0.014005755074322224, 0.5506020784378052, -1.205862283706665, 0.8830478191375732, 0.2106158584356308, -0.9523411989212036, -0.24264249205589294, -1.4264289140701294, 0.5919209718704224, -0.035958368331193924, -1.3632915019989014, 1.9183968305587769, 0.8139184713363647, 0.6912352442741394, -1.1060786247253418, -1.3279885053634644, 0.8930712342262268, 0.45117470622062683, 0.3090292811393738, -1.5054609775543213, 0.38911503553390503, 0.678068220615387, 1.1125521659851074, -1.8905545473098755, -0.8393733501434326, 0.5803723931312561, -1.2545429468154907, -1.0913734436035156, 1.1080429553985596, -0.39175036549568176, -0.07283896952867508, -0.8099192976951599, 0.33949998021125793, -1.1176103353500366, -1.4063470363616943, 0.28770768642425537, 0.049175046384334564, 0.12445104867219925, 0.6032111048698425, -0.2768835127353668, 0.09747668355703354, 2.8831615447998047, 0.5362674593925476, -0.3953608572483063, -0.3291723132133484, 0.05519359931349754, 0.38147398829460144, -0.38365691900253296, 0.8168781995773315, 0.19143112003803253, -0.09978233277797699, 0.7715660333633423, -1.207231879234314, -0.9477736949920654, 0.5808735489845276, -1.8308717012405396, -0.7312027812004089, -0.6638030409812927, 0.005306030623614788, -1.8534966707229614, 2.191554546356201, -0.9073959589004517, 0.045727912336587906, 1.3797714710235596, 1.3902515172958374, 0.2877681851387024, 0.8251571655273438, -0.8646895885467529, -0.9645817279815674, -0.9521104097366333, -0.584136426448822, -0.06856338679790497, 1.1165497303009033, 0.9427222609519958, 0.485003262758255, 1.366707444190979, 0.194069504737854, -0.0018582851625978947, 0.1368139535188675, 0.7810993790626526, -0.23472657799720764, 0.1663742959499359, 0.09246938675642014, -0.828126847743988, -0.4625411927700043, 0.228130504488945, 0.023995762690901756, -0.8044687509536743, 0.8513607382774353, 1.8220280408859253, -0.02640298195183277, -0.6419013142585754, -0.6837194561958313, 0.4836875796318054, 1.5931085348129272, -0.1575937569141388, -0.10329674184322357, -0.39087799191474915, -0.01866118423640728, 0.5085265040397644, -0.36690378189086914, 0.3342169225215912, -1.0426881313323975, -1.6472065448760986, 0.45222100615501404, -0.29437538981437683, 0.14926353096961975, 0.6788806915283203, 0.36435237526893616, -2.1169023513793945, 2.054488182067871, -0.14674368500709534, 0.21495398879051208, -0.7817699909210205, -0.03650953993201256, -0.37467315793037415, 0.8780931234359741, -1.2165995836257935, -0.3227832615375519, 0.47851625084877014, 0.2769787907600403, -1.022487998008728, -0.10143566131591797, -1.5273079872131348, -0.5196540951728821, 0.49105024337768555, 0.48036089539527893, -0.6432353258132935, 1.0643186569213867, 0.1573655605316162, 0.3989911377429962, -0.3989602327346802, -0.3958275020122528, -0.0925508365035057, -0.13002899289131165, -0.7706928253173828, 0.17705638706684113, -1.1993063688278198, -1.79475736618042, 0.7261229753494263, -1.480637550354004, -0.8267199993133545, -1.700175166130066, 1.6424461603164673, -1.1598260402679443, -0.39095863699913025, 0.1107177585363388, -0.09072522073984146, -0.7566426396369934, 0.10090538114309311, -0.24051609635353088, -1.4592689275741577, 0.17699608206748962, -0.01208486221730709, 0.7569180130958557, 0.15776799619197845, -1.1716476678848267, 0.09810660779476166, -0.6337353587150574, 1.4744395017623901, 2.1917905807495117, -1.328479290008545, -0.6298107504844666, -1.6234604120254517, 0.8882924318313599, 0.4652755856513977, -1.5073851346969604, 0.09929847717285156, 0.13940604031085968, -1.9254206418991089, -0.46409040689468384, -1.2537846565246582, -1.074873447418213, -0.3547980487346649, -1.1564325094223022, -1.1073707342147827, 0.4640437960624695, 0.12283675372600555, -0.6612945795059204, 0.8419765830039978, -1.845766544342041, 0.9037477970123291, -0.6594877243041992, 0.039953432977199554, -0.43466809391975403, -0.5299076437950134, 1.7909115552902222, 1.0271244049072266, -2.0898196697235107, 0.41927847266197205, -0.018170377239584923, -0.1045914813876152, -1.164932131767273, -0.7037194967269897, -1.587228775024414, 0.6902994513511658, -1.5271774530410767, 0.3830691874027252, 0.3414325416088104, -1.4421495199203491, 0.3521881699562073, 0.9412959218025208, -0.31095829606056213, -1.3687810897827148, -0.45291605591773987, 0.07037026435136795, -1.6491390466690063, -1.337557077407837, 0.648242712020874, -0.7836018800735474, -0.44464269280433655, -1.471659779548645, -0.9057486653327942, -0.9958221316337585, 0.1272449642419815, -0.40837687253952026, -1.0024700164794922, 0.4849955141544342, 0.3007809817790985, 2.7728934288024902, -0.09259457886219025, -0.48056861758232117, -1.4112541675567627, 1.3434946537017822, -1.4260202646255493, -0.33875882625579834, 0.5422768592834473, -0.46669861674308777, 0.733253002166748, 0.009383419528603554, 1.1042320728302002, 0.05153731256723404, 1.3069331645965576, 0.05970528721809387, 0.25717002153396606, 1.029707670211792, 1.2819838523864746, -0.6573824882507324, -1.5797295570373535, 0.4759277105331421, 1.3199628591537476, 0.4970085322856903, -0.03761213272809982, -1.409738540649414, 0.23624002933502197, -0.3585197627544403, 0.10868428647518158, 1.9872941970825195, -0.5601415634155273, 0.4162561595439911, -0.38593316078186035, 0.1400420367717743, -0.9092892408370972, -0.1304996907711029, 0.01321831438690424, 1.8224527835845947, -0.644195556640625, -1.0056735277175903, -0.24654924869537354, -0.917263925075531, -0.6691651940345764, -0.7546720504760742, -0.01894083432853222, -0.7954978942871094, 0.08745197206735611, 0.41709983348846436, -0.12330224364995956, -1.3289775848388672, 1.5585880279541016, -0.14756856858730316, 0.37792280316352844, -0.0656026154756546, 0.961126983165741, -0.6831126809120178, -0.2244388312101364, -1.4061411619186401, -0.006935416720807552, 0.8358990550041199, -0.34292155504226685, 0.029259420931339264, 0.8082229495048523, 0.7499920725822449, 0.04334905743598938, -0.8742557168006897, -0.6649061441421509, 0.37855759263038635, 0.09986475855112076, 0.6769981384277344, 0.541644811630249, 0.31280314922332764, -1.080645203590393, -1.6872127056121826, 0.4831497073173523, -1.5054305791854858, 0.16438645124435425, -0.21051427721977234, 1.1081602573394775, -1.6361697912216187, -0.5646700263023376, 0.0018348683370277286, -0.6533657908439636, -0.6310514807701111, 0.24409230053424835, 0.08163636177778244, -0.7913898229598999, 0.6240264177322388, 0.5542711615562439, 0.1548151820898056, -0.9712766408920288, 0.8554818630218506, -1.0802942514419556, -0.1429230272769928, 0.6926798224449158, -0.40655508637428284, 0.5760378837585449, 1.463870882987976, 1.2525599002838135, -0.5070398449897766, -1.3451858758926392, 1.19225013256073, 0.15919016301631927, 0.18688024580478668, -1.0014837980270386, 1.726847529411316, 0.38677334785461426, 0.0354803241789341, 0.907650887966156, -0.6733133792877197, -0.4968298375606537, -1.0524959564208984, 0.9749950766563416, 1.222572922706604, 0.35948464274406433, -0.18336640298366547, -0.27849280834198, -0.15066400170326233, -1.0818127393722534, -0.0705007016658783, -0.1837126910686493, 0.021552419289946556, 0.7063227295875549, -1.5606335401535034, 0.9036967754364014, -2.3585498332977295, -0.07385601103305817, 0.13400501012802124, 0.2764730155467987, 0.6165434122085571, 0.5816645622253418, -0.1721956878900528, -0.1647203266620636, 0.7245743870735168, -0.9145128726959229, 1.7208539247512817, -0.25184014439582825, -0.3160243332386017, 1.144935131072998, 0.5599855780601501, -0.4319770038127899, -1.021064043045044, 0.0007565958076156676, -1.2204073667526245, 0.6891810894012451, -1.2428014278411865, 1.532334804534912, 0.40342918038368225, 0.3572349548339844, -0.7530184388160706, -0.2569214701652527, 2.183046579360962, -0.7019652128219604, -0.7051798701286316, -0.839887797832489, 0.18316343426704407, 1.1796687841415405, -0.061110299080610275, 1.035402536392212, 2.691915988922119, 0.08871069550514221, -1.2244155406951904, -0.03995512053370476, 0.4030817151069641, -0.24466168880462646, 0.594307541847229, -0.2182890623807907, 0.6241633296012878, -1.34402596950531, -0.3454959988594055, 0.17152492702007294, 0.39375218749046326, 0.4111478626728058, 0.04674327000975609, 0.19074752926826477, -0.7779505848884583, 0.33213844895362854, 0.42779043316841125, 0.34812694787979126, -0.5515667796134949, -1.7312177419662476, 0.19756774604320526, 0.4045398235321045, 1.0847218036651611, -0.38402536511421204, -0.3499128222465515, -0.4235685467720032, 0.5410435795783997, -1.7539551258087158, -0.7823672890663147, -1.7989745140075684, 1.138404369354248, 0.40235936641693115, -1.2751163244247437, -1.4063719511032104, -0.6298544406890869, -1.902387261390686, -0.9246838688850403, 0.6569748520851135, 0.0600469633936882, 0.627687394618988, -0.23066525161266327, 0.10513556003570557, -1.4544121026992798, 1.5794473886489868, -0.6269994378089905, -1.1410058736801147, -1.729764699935913, -1.6328346729278564, 1.4146215915679932, 0.6434935331344604, 0.5232924818992615, 0.21307605504989624, -0.24444352090358734, 0.580935537815094, -0.3380068838596344, -0.1673399955034256, -0.532466471195221, 0.6211300492286682, -0.461355060338974, -1.1646647453308105, 1.512121319770813, 0.6430439949035645, 0.1150447204709053, -0.5267953872680664, 0.9950847625732422, 1.3892589807510376, -1.4811536073684692, 1.1646974086761475, 2.130697011947632, -0.20089088380336761, -0.6437857747077942, -0.25129154324531555, 0.31864920258522034, 1.2365000247955322, -1.8039367198944092, -0.608400285243988, -0.06383766233921051, -0.26052385568618774, -0.5794777870178223, 0.5601658821105957, -0.5350218415260315, -0.48181647062301636, -1.1867820024490356, 1.4401073455810547, -1.549237608909607, 0.8572079539299011, -0.1970854550600052, 0.059172630310058594, 0.34533706307411194, -0.7700024843215942, -0.4723722040653229, 0.7240921854972839, 0.39015993475914, 0.4227462410926819, 0.6099779605865479, 0.6426460146903992, -1.6158210039138794, -0.27694350481033325, 1.1052944660186768, 1.2947593927383423, -0.4171395003795624, 0.2859490215778351, -1.2521193027496338, 2.133043050765991, 0.9134650826454163, -0.4187653362751007, 1.5920048952102661, -1.4593263864517212, 0.8325005769729614, 0.2820514738559723, -0.16633829474449158, -1.4721676111221313, 1.4285236597061157, 0.15074773132801056, 0.6213761568069458, -0.7767300605773926, -1.3958531618118286, -0.4232049286365509, -0.06922105699777603, 0.5067950487136841, -1.5383362770080566, -0.1726432889699936, -1.556966781616211, -1.348219633102417, -1.0763487815856934, -1.4000153541564941, -0.042040444910526276, -0.2573997974395752, 0.6227280497550964, 0.06156530976295471, 0.9188748598098755, -0.7669249176979065, 0.4750640094280243, 0.1079612746834755, -1.178126573562622, -1.6124368906021118, -0.4435425102710724, 1.640251636505127, -0.5651054978370667, -1.1549899578094482, 1.2996065616607666, 0.1634681224822998, -0.4159010052680969, -0.9754438996315002, -0.6858494877815247, 0.7455165386199951, 1.9224238395690918, -0.885148823261261, -2.2093441486358643, 1.3034950494766235, 0.4943821132183075, -0.9952561855316162, -0.8178654909133911, -0.0858796164393425, -1.4051443338394165, -0.1079452857375145, 1.3217014074325562, 0.7340992093086243, -0.5437924265861511, 0.24037297070026398, -1.4045060873031616, -1.2862335443496704, 1.2553026676177979, 1.0851073265075684, 0.7146494388580322, -0.5854995846748352, 0.4670238196849823, -0.5394169688224792, -0.37213248014450073, 1.668764591217041, 0.3959773778915405, 1.0851839780807495, -0.7585936784744263, 0.14268024265766144, 0.5737593173980713, 0.6770831942558289, -0.2158583253622055, 0.41819947957992554, 2.8917768001556396, -0.8110544085502625, -0.043142449110746384, 0.8445443511009216, 1.4315037727355957, -0.7222245335578918, 0.6511109471321106, -0.7435749769210815, 1.1839556694030762, 0.4684910476207733, -0.8095899224281311, 0.6809945702552795, 0.14198459684848785, 0.9730215668678284, 0.9940242171287537, 0.019625501707196236, -0.39610257744789124, 0.35362890362739563, -0.3251388967037201, 0.2647498846054077, 0.19207395613193512, 1.2627571821212769, 0.9068818092346191, 1.4093856811523438, 0.16930654644966125, 0.16087622940540314, -0.7886802554130554, 1.6609828472137451, 2.163288116455078, -0.530454158782959, -0.24946023523807526, -2.1794631481170654, -1.3394536972045898, 1.235533595085144, 1.2005093097686768, 2.157397985458374, -0.6892791986465454, 1.9194691181182861, -1.059552550315857, -0.29675954580307007, 2.465264320373535, 1.5198618173599243, 0.46419307589530945, -0.41636988520622253, 1.7253350019454956, -1.4790812730789185, 0.5098555088043213, -0.5258503556251526, -0.15122392773628235, 0.30319875478744507, -1.1645140647888184, -1.7922027111053467, -0.7596058249473572, -0.1999943107366562, 0.11325215548276901, 0.8944586515426636, -2.0548901557922363, -0.20794115960597992, -0.1013580858707428, -0.8111453652381897, 0.49718600511550903, -0.8104513883590698, 0.26826831698417664, 1.8679754734039307, -1.6815084218978882, -0.34717002511024475, 0.6759158968925476, -0.01341166254132986, 0.06730464100837708, 2.0164694786071777, -1.6480381488800049, -1.1637582778930664, -1.0714590549468994, -0.29300612211227417, 1.3094958066940308, -1.2657922506332397, -0.3643020987510681, -0.03359861299395561, 0.7888378500938416, 0.39069655537605286, 1.5351823568344116, -1.2428604364395142, -1.5882703065872192, -0.03571806102991104, -1.073564887046814, -1.815059781074524, 0.9160414934158325, -0.32734420895576477, 0.11304952204227448, 0.6757737994194031, 0.23722201585769653, -0.24142947793006897, 0.9195752143859863, 1.115309238433838, 0.8240196704864502, -0.4775678217411041, -0.24594224989414215, -0.37144145369529724, 1.102490782737732, 1.6980494260787964, 1.700218677520752, -0.5501829385757446, 0.9958950281143188, 1.042339563369751, -2.3794546127319336, 0.21532344818115234, -0.3338446021080017, 0.7413772344589233, 0.5648232102394104, -0.683840274810791, -1.1319011449813843, 1.0639677047729492, 0.7928522229194641, -1.8548526763916016, -1.3518601655960083, 0.7621836066246033, 0.5114578008651733, -1.1132267713546753, -0.8797764182090759, 1.4902631044387817, 0.8339231610298157, -0.6892172694206238, -0.06872359663248062, -0.11243068426847458, 1.7412610054016113, 0.19486260414123535, 0.2793372869491577, -0.15811365842819214, 0.6353474259376526, 0.03743002936244011, -1.147719383239746, -1.7263360023498535, -2.2282769680023193, 1.5247366428375244, 0.05290522053837776, -1.8114608526229858, 2.764969825744629, -2.0057313442230225, -2.5473906993865967, 0.31401166319847107, 1.0587576627731323, -0.2155531495809555, -1.966373324394226, 1.0882315635681152, -0.3330991268157959, 0.4295791685581207, 1.0532559156417847, 1.3314759731292725, -0.09294534474611282, -1.146122694015503, 0.7311992645263672, -1.3216636180877686, -1.1917195320129395, 1.527401328086853, -2.354694366455078, -0.4006604552268982, 1.4712224006652832, -0.4284450113773346, -0.9941515922546387, -0.06022915244102478, 1.0017900466918945, 1.276907205581665, 0.4696454107761383, -1.7351016998291016, 0.38780540227890015, 0.15682978928089142, 1.4356855154037476, -2.937831163406372, -0.3299386501312256, 0.8619300127029419, 0.7864987850189209, -2.2766144275665283, 0.276741623878479, 0.2597939968109131, -1.7026475667953491, 0.6308276057243347, -0.8152629733085632, -0.111618772149086, 1.2469916343688965, -0.0194556787610054, 1.6148566007614136, -0.7180173397064209, 0.4233763515949249, -0.011768144555389881, -0.7212904691696167, -0.8066465854644775, 0.576575517654419, 1.4207721948623657, 0.5613623857498169, 0.3526446521282196, 1.6739797592163086, -0.7857711315155029, -1.4698143005371094, 0.5505262017250061, 0.038728538900613785, -2.698176622390747, -1.776882529258728, 1.0223416090011597, 2.1203932762145996, 0.7668820023536682, 1.204249620437622, -0.7466263771057129, 0.09431206434965134, -0.3896442949771881, -1.042178988456726, -0.45071473717689514, 0.5224319696426392, 1.1205532550811768, -0.1801530420780182, 0.8683047294616699, -0.09445953369140625, -1.6519931554794312, 0.43334391713142395, 0.3418201208114624, -1.2352294921875, 1.606021523475647, 2.830514430999756, 0.7551843523979187, -1.5831189155578613, -1.846557855606079, -0.6126821637153625, 0.7625900506973267, 0.3338198959827423, -1.4220203161239624, -0.4317198395729065, 0.7712830305099487, -1.3278396129608154, 0.06977882236242294, -0.8437961935997009, -0.2237006574869156, 1.2278175354003906, -0.3503439128398895, 1.16797935962677, -2.032290458679199, 0.8574094772338867, -0.6093283295631409, 0.549589991569519, -0.07673503458499908, 1.3739466667175293, -1.120009183883667, -2.128448247909546, -0.7365496158599854, 0.1930733621120453, -0.14913791418075562, -0.638110876083374, -1.1031662225723267, -0.9460402727127075, -0.10191325843334198, -1.1173428297042847, 1.645995855331421, -0.8436457514762878, 0.6300775408744812, 0.14969636499881744, 1.3287769556045532, 0.48808351159095764, 0.11859120428562164, 0.9408154487609863, -1.5467908382415771, -1.2999571561813354, 1.0289584398269653, 0.6877345442771912, -1.2817180156707764, -2.348128080368042, 0.4521540403366089, 1.2082269191741943, 1.0620301961898804, 0.674808919429779, -1.0021013021469116, 0.31960710883140564, -0.26228243112564087, 0.16107214987277985, -0.10824555903673172, 0.45211589336395264, -0.4424942433834076, -1.3263286352157593, 0.9523817300796509, 2.43459153175354, 0.4137382209300995, -0.3668740689754486, -0.27183064818382263, 1.5355045795440674, -0.6978620886802673, 0.17790278792381287, 1.1175096035003662, 1.1658399105072021, -0.4236833453178406, -1.258467197418213, 0.6484071612358093, -1.5410795211791992, 0.10892496258020401, 0.2837430238723755, 0.2083519548177719, -1.1359635591506958, 0.9007324576377869, 0.7677565217018127, -1.5918456315994263, 1.002771258354187, -1.0453622341156006, 0.12294268608093262, 0.7248280644416809, 0.05123598873615265, 0.02300250716507435, 0.06794432550668716, 0.35087889432907104, 0.15493658185005188, 1.0689654350280762, -1.486581802368164, 0.09467566013336182, 1.4613969326019287, -2.489609479904175, -0.06927413493394852, -0.4038732647895813, -1.295669674873352, 0.1030103787779808, -0.724543035030365, -2.035855531692505, 0.08315884321928024, 1.403908133506775, 0.32589486241340637, -0.029762543737888336, -0.9672757983207703, 0.06365068256855011, 0.07312550395727158, 0.40272822976112366, -1.1458756923675537, 0.31452661752700806, 0.49173447489738464, 0.2193344682455063, -0.99244225025177, 0.7912675738334656, 0.7695441842079163, 0.9438881874084473, 1.2353854179382324, 0.4238187372684479, -0.4292285740375519, 0.29821500182151794, -0.6337391138076782, 0.204397514462471, -0.33107054233551025, 1.3340462446212769, -1.5346704721450806, -1.2776471376419067, 0.1287221759557724, 0.10845351219177246, 0.5282504558563232, 0.17440104484558105, -0.617117166519165, 0.7810393571853638, 0.35267671942710876, -0.6715098023414612, -1.677207112312317, 0.6155229806900024, -0.6862978339195251, -0.5605531334877014, -0.25991320610046387, 0.06353499740362167, -1.6811140775680542, 0.16298602521419525, -1.0846481323242188, -1.1476316452026367, 0.17140968143939972, 1.542977213859558, 0.15923286974430084, 0.38504379987716675, 0.46362993121147156, -0.0758957713842392, -0.648506224155426, -0.5021474361419678, -0.08406799286603928, 0.986061692237854, 1.8659087419509888, 1.5645923614501953, -1.726081132888794, -0.8892182111740112, 0.3838292956352234, 1.025801181793213, 0.5227816104888916, -0.07018211483955383, 1.2382030487060547, 1.2550158500671387, 0.4468619227409363, -1.0867841243743896, -0.4336431622505188, -1.722650170326233, 1.0055434703826904, -1.81892991065979, -0.1625097095966339, -1.61785888671875, 0.2947910726070404, 0.08539474755525589, -0.35695594549179077, -0.39361101388931274, -0.792246401309967, 0.7948977947235107, -1.5883878469467163, 2.266390562057495, 2.1882681846618652, -0.16837561130523682, -0.009880278259515762, 0.2531445324420929, 1.5716978311538696, 0.05378001555800438, -1.5820382833480835, 0.21098098158836365, 0.24085049331188202, 0.23367248475551605, -0.9740201234817505, -1.7144356966018677, -0.812390148639679, 1.394023060798645, -1.2093194723129272, 0.15278778970241547, 0.4472920894622803, 1.5988929271697998, 0.8746757507324219, 1.3572951555252075, 1.109328269958496, -0.661304235458374, 0.5957000851631165, -1.245664358139038, 2.258204936981201, 1.0852891206741333, -1.4209998846054077, -0.9300857186317444, 0.653425395488739, -0.9180313348770142, 0.23817779123783112, 0.6896769404411316, -0.1471548229455948, -2.2510392665863037, -1.740260124206543, 0.691807746887207, -0.6976827383041382, -0.007262126076966524, 0.3225676119327545, -1.2754980325698853, 0.34659844636917114, -1.9381242990493774, -0.2959845960140228, 0.5817258954048157, 1.9521313905715942, 1.3140645027160645, 1.974884033203125, 1.299557089805603, 0.8265095949172974, 0.2727760076522827, 1.7288587093353271, -1.0197010040283203, -0.8883903622627258, -0.9943159818649292, -1.197710633277893, -1.0566301345825195, 0.21752868592739105, -1.4967166185379028, -0.2534613013267517, -0.2641121447086334, 0.1319870501756668, -0.5063642859458923, -0.7350525259971619, -0.2908066511154175, 0.7410039901733398, 0.23109814524650574, 0.05830013379454613, -0.3566206991672516, 1.1751350164413452, -1.4448579549789429, -2.2373206615448, -1.81161367893219, 1.1606078147888184, 0.0589078851044178, -1.2420765161514282, 0.6462399363517761, 0.4838264584541321, 0.4488161504268646, 0.018359245732426643, -0.26613649725914, 1.3747507333755493, 0.29387837648391724, 0.011442327871918678, -0.7330398559570312, 0.13389788568019867, 0.5513520836830139, -1.4916012287139893, 0.4338633120059967, -0.38700631260871887, -0.9239317178726196, 0.5368218421936035, 2.247563362121582, -1.1154927015304565, -0.6039007306098938, -1.8673217296600342, 0.9551874995231628, 0.661564290523529, -0.8017662167549133, 1.4892102479934692, 0.8410003185272217, -0.07781556993722916, 0.441607803106308, 0.2877582609653473, 1.3217461109161377, -1.9957720041275024, 1.3739194869995117, -1.4815560579299927, 1.519698977470398, 0.555057168006897, 0.14802183210849762, -0.8140326142311096, -0.22385726869106293, 0.4982047975063324, 0.7954748272895813, 1.5949511528015137, -0.5223003625869751, -0.5655446648597717, -0.35807284712791443, -0.3417026400566101, 0.20218636095523834, -0.6382884979248047, 0.4165228307247162, -1.3043243885040283, -0.547444224357605, 0.5052507519721985, -1.4276431798934937, -0.791780948638916, 0.44584691524505615, -0.04314348101615906, -0.3885185122489929, 1.6931626796722412, -0.17536792159080505, 1.0761502981185913, 0.15714715421199799, -0.8176491260528564, -0.41951075196266174, 0.38985002040863037, 0.5920450091362, -1.0725243091583252, -0.3523346483707428, 1.2688140869140625, 0.28882208466529846, -1.4626110792160034, -0.11964297294616699, 0.17225807905197144, -0.9195331931114197, -1.2197753190994263, 0.3635430634021759, -0.21246905624866486, 1.8911337852478027, -1.0999797582626343, 0.1906764954328537, 0.4076322019100189, -1.2097251415252686, 0.50749671459198, -0.7401384711265564, 0.8895146250724792, -1.5856051445007324, -0.3732719123363495, 0.12643831968307495, 0.714486837387085, -1.043555498123169, -1.603877067565918, 0.6515832543373108, -0.44418251514434814, 0.725369393825531, -1.3824952840805054, -0.9134519100189209, -0.994240403175354, -0.87492835521698, 0.4787808358669281, -0.8759537935256958, 0.6093623638153076, -0.8604093790054321, -1.2868938446044922, 0.3006233274936676, 0.37618133425712585, 2.0287671089172363, -2.600975751876831, -0.8406355977058411, 0.3101212680339813, -0.28808820247650146, 1.7563748359680176, -0.4817017912864685, 0.2488914579153061, -0.536908745765686, -2.164926528930664, -2.7600533962249756, -1.6843734979629517, -2.469153881072998, 0.5822861790657043, -0.11048127710819244, -1.9656671285629272, 0.23983389139175415, 1.2580015659332275, -0.7392769455909729, -1.2642344236373901, -0.8005959391593933, -0.0036191255785524845, 1.1959726810455322, 0.565572202205658, 0.44971656799316406, 0.09543583542108536, -0.6542319059371948, -1.756625771522522, 0.35055941343307495, 0.3217517137527466, -0.754188597202301, -0.19716797769069672, -0.9955018162727356, -1.0251888036727905, 0.6317073702812195, -0.7034367322921753, 0.011991696432232857, -0.4997899830341339, -0.3730678856372833, 1.1648684740066528, -2.0422894954681396, -0.5738495588302612, 2.188246726989746, 0.3143080472946167, 0.2191867083311081, 0.7311874032020569, 1.145680546760559, 0.4063992202281952, -0.6386862397193909, -0.24890801310539246, 1.692233920097351, -0.6711574196815491, -1.6659809350967407, 0.13228286802768707, 0.41366660594940186, 0.45507943630218506, 0.15737532079219818, -0.8172154426574707, 1.289679765701294, 1.9054129123687744, -0.9669414758682251, 1.736833095550537, 0.5594533681869507, -1.1902936697006226, 0.04632464051246643, 0.18741483986377716, 0.7083494663238525, 0.485451340675354, 0.8939657211303711, 0.33009618520736694, 1.2471810579299927, -0.17968526482582092, 0.7403998374938965, -1.5131500959396362, -0.07777278125286102, 2.002448320388794, -1.0774863958358765, 2.9384682178497314, 2.466109275817871, -0.04586417227983475, -1.6524152755737305, 0.3369765281677246, -1.5479536056518555, 1.0719578266143799, -1.0868351459503174, -0.21420563757419586, -1.8068833351135254, -0.30993151664733887, -0.06765174865722656, 0.4957166016101837, 1.0847066640853882, 1.6154234409332275, 0.15538626909255981, -0.8665551543235779, 0.9136213064193726, -1.3917632102966309, 1.0826153755187988, -0.930596113204956, -0.25413280725479126, -0.952540397644043, -1.4854835271835327, -0.6652039289474487, 0.22094596922397614, -1.3744066953659058, 1.199278473854065, -0.49366679787635803, -0.1952829509973526, -0.439778208732605, 0.011491550132632256, 0.3002108633518219, 0.4214594066143036, -0.24837364256381989, 2.284590482711792, 1.0506917238235474, 0.6144401431083679, -0.14106470346450806, -0.3301955759525299, 1.4958617687225342, -1.3265373706817627, -0.5747475624084473, -0.48009398579597473, 0.9503005743026733, 0.7457674741744995, 1.2579175233840942, -0.5754433274269104, 0.5606024265289307, 0.29328131675720215, 0.1717132180929184, 0.6180459856987, 1.126302719116211, 0.5262008905410767, 0.7493679523468018, -1.0848180055618286, 0.2372211366891861, -0.7182983756065369, -0.12202909588813782, 0.7903457880020142, -2.7705063819885254, -0.7041110992431641, 1.0670990943908691, 0.9895842671394348, -1.3876969814300537, -0.738437294960022, -1.6065428256988525, 1.1997151374816895, -0.28850501775741577, -0.12408001720905304, -1.4494863748550415, -0.2846980094909668, -0.5099372267723083, -2.7688167095184326, -1.5255074501037598, -2.981712818145752, -0.345665842294693, -0.4203084409236908, 1.4699851274490356, -2.4547488689422607, 1.3023018836975098, 0.3893396258354187, 0.717572033405304, -0.18033918738365173, -0.6192570328712463, 1.4030303955078125, -1.691902756690979, 1.2195693254470825, 1.0319222211837769, 0.5066291093826294, -0.23757584393024445, 0.764606237411499, 0.38164883852005005, -1.4347765445709229, 0.28124332427978516, 0.03216835856437683, 0.47405678033828735, 0.9497878551483154, 1.0308243036270142, 0.4952124059200287, -0.16008701920509338, -0.06109614297747612, -0.5112041234970093, -1.2474229335784912, 0.633922278881073, 0.3996618986129761, 0.42950373888015747, 0.564572811126709, 0.1912776231765747, -0.7335266470909119, -0.3972913920879364, 0.14653639495372772, 1.6647013425827026, 0.6974106431007385, 0.4111306369304657, 0.08065754175186157, 0.11127801239490509, -0.03214649856090546, -0.957160234451294, 2.353066921234131, -1.2476001977920532, 0.19646140933036804, -0.7034879922866821, -0.8804206252098083, 2.0259010791778564, 0.3334009051322937, -1.869641661643982, 1.0814388990402222, 2.511021614074707, 0.7050632834434509, -1.6869211196899414, 0.8198149800300598, -0.3135402798652649, 0.7766656875610352, 0.9759955406188965, 0.031153056770563126, -1.301391363143921, 0.002882617060095072, -0.864704430103302, 1.0666483640670776, 0.14152075350284576, -0.3517150282859802, -1.7494832277297974, -1.7143827676773071, 0.16470597684383392, 0.6715195178985596, 0.4938163459300995, 1.984713077545166, 0.7521482706069946, -1.3559921979904175, 2.002753734588623, 1.3838257789611816, -0.5151078104972839, -0.42506593465805054, 1.9816545248031616, 1.0822254419326782, 1.1243948936462402, -1.0778697729110718, -0.3094760477542877, -0.993646502494812, -0.6530218124389648, -0.5713019371032715, 0.6459251046180725, 0.7571723461151123, -0.5417315363883972, 1.9603278636932373, 0.05084004998207092, -0.19311270117759705, 0.648151695728302, -2.0143375396728516, 0.18532878160476685, 0.38367971777915955, -0.6179466247558594, 0.8181671500205994, 0.7857585549354553, 0.9846933484077454, 0.971111536026001, 1.632079005241394, 0.5985841751098633, -0.5407556891441345, 0.05072811245918274, 0.428242564201355, -0.3267712891101837, 1.4248261451721191, 1.6348756551742554, -0.9736685752868652, -0.3765887916088104, 0.10363530367612839, -0.0993359312415123, -1.3292641639709473, 1.08529531955719, 1.2003169059753418, 1.078646183013916, -1.8283263444900513, 0.10772042721509933, -1.8693633079528809, 0.07242367416620255, 1.5743521451950073, 0.06698082387447357, -0.9772310256958008, -1.1192692518234253, 0.7059117555618286, 1.3650728464126587, 0.28618717193603516, 0.5088871121406555, 0.8336738348007202, 0.23915590345859528, -0.26974424719810486, 0.8184224963188171, 1.505191683769226, -0.40397271513938904, 1.1331872940063477, -1.270864725112915, -1.5615066289901733, 0.12111131846904755, 1.1249805688858032, -0.21252119541168213, 0.34929338097572327, 0.7565247416496277, 0.20155197381973267, 0.6535778641700745, 0.964544415473938, 0.7209293842315674, -1.6137663125991821, -0.3911270201206207, -0.04526209831237793, 1.78336501121521, 0.40928569436073303, 0.8372264504432678, 0.8326970338821411, -0.24878205358982086, 0.861754298210144, 0.9152221083641052, 1.0039196014404297, -1.9283359050750732, 1.1045249700546265, 0.11166827380657196, -0.7552402019500732, 0.05986257269978523, 0.5467034578323364, 0.33502137660980225, 0.41275614500045776, -1.6374751329421997, -0.6176034212112427, 1.6006019115447998, 0.7669300436973572, 0.3531596064567566, -1.3154540061950684, 0.09189524501562119, 1.0507184267044067, 0.378611296415329, -0.4717326760292053, 0.2883109152317047, 0.5816047787666321, -0.8719629645347595, -0.9425174593925476, 1.3953089714050293, -0.5076507329940796, 0.42582932114601135, -1.7403837442398071, 1.2024545669555664, -0.6974551677703857, -0.38714468479156494, -0.007289333269000053, -0.6776732802391052, 0.8761478066444397, 0.9179368615150452, -0.3892427086830139, 0.38819608092308044, -0.24317553639411926, 1.5437973737716675, -0.0409938208758831, 1.3991304636001587, 0.26970353722572327, 1.1454713344573975, 0.9003570675849915, 0.9119595289230347, -0.09370800107717514, -0.23791080713272095, -1.8611705303192139, 0.3939945101737976, 0.5758221745491028, -1.3005554676055908, -1.9832334518432617, 0.01373556163161993, -0.7591463923454285, 0.5097705125808716, 3.1918563842773438, -0.03204956278204918, 0.8679468035697937, 0.8759178519248962, -0.888192355632782, -1.0698561668395996, -0.37862536311149597, 1.4798457622528076, 0.16385719180107117, 2.511021852493286, 0.7484445571899414, 0.2630501091480255, -0.8672106266021729, 1.0716248750686646, -2.2984535694122314, -0.03921768441796303, 0.08124884217977524, 2.1735799312591553, 1.5197604894638062, 0.13122470676898956, -0.4808948040008545, 1.1237748861312866, 0.12382945418357849, -0.2645317018032074, 2.6415815353393555, -0.9568057060241699, 0.8518780469894409, -0.052272334694862366, 1.7058801651000977, -1.5874673128128052, -1.5836211442947388, -0.6561729311943054, -0.7888287901878357, 0.071555495262146, -1.1951708793640137, 0.25761300325393677, -0.6694763898849487, 0.020138133317232132, 1.9906878471374512, -0.8811867237091064, 0.032434165477752686, -0.5063203573226929, 0.6838266849517822, 0.4255801737308502, -0.0633796900510788, -1.1423860788345337, 0.3318638503551483, 0.28248080611228943, 0.14936061203479767, 0.5419828295707703, 2.39859938621521, -1.5377508401870728, -1.0753540992736816, -2.3561952114105225, -0.4894341230392456, -0.40394553542137146, -2.0389368534088135, 0.5671156048774719, -0.6018934845924377, 1.7712271213531494, -0.39596670866012573, 0.2881534695625305, -0.2742651402950287, -1.4285588264465332, 1.4593515396118164, 0.7640348672866821, -1.3513542413711548, -0.53242027759552, 1.7638840675354004, -1.6109514236450195, -1.3000797033309937, -0.9603842496871948, -0.9679659008979797, -0.9549384117126465, 0.47863778471946716, -0.4895903468132019, -1.1977870464324951, -0.6100513339042664, 1.9848742485046387, 1.8488117456436157, -0.024742895737290382, -0.397635281085968, 0.15958072245121002, -1.809446096420288, -0.3610536754131317, -1.3125550746917725, 0.5120975375175476, -1.0206011533737183, -0.24239253997802734, -0.23221011459827423, -0.07764500379562378, 0.32677483558654785, 1.9331508874893188, -0.8018123507499695, -0.1304389238357544, 0.7450947761535645, 0.29860660433769226, 0.8844764232635498, 0.9891070127487183, -1.2468699216842651, 0.5373407602310181, -0.3454706370830536, 0.42865875363349915, 1.3253849744796753, 0.47495388984680176, 0.9957704544067383, -1.663960576057434, 0.4914518892765045, -0.1215878576040268, 0.1354624480009079, -0.29386216402053833, 0.5332263112068176, -0.22131098806858063, -1.6884405612945557, -0.23604445159435272, -0.5627837777137756, -0.520246148109436, 0.297910213470459, 1.3924959897994995, 0.14897790551185608, 3.2605807781219482, 1.429301381111145, -0.38803690671920776, -0.7367172241210938, 1.7333564758300781, 1.3557054996490479, 1.4565794467926025, 1.6431108713150024, 1.2034859657287598, -0.5533969402313232, -0.5922225117683411, -0.5556549429893494, -0.5952415466308594, -0.8442329168319702, -1.4185744524002075, -2.1255762577056885, -0.24991244077682495, 0.3844072222709656, 1.7590711116790771, 1.2489346265792847, -1.1197651624679565, -0.21775038540363312, 0.36170560121536255, -0.5576170086860657, -1.811985969543457, 0.29704222083091736, 0.06835416704416275, 0.18753619492053986, 0.2992722690105438, -0.9011162519454956, 0.48546966910362244, 0.4114774465560913, -0.25288155674934387, -0.9028388261795044, -1.0903247594833374, 0.48575863242149353, 0.2023448944091797, -1.3559317588806152, 0.08901496231555939, 0.41917628049850464, -2.0914831161499023, 0.20074203610420227, 0.04114663600921631, -2.1080162525177, 0.27016398310661316, -0.14700336754322052, -0.25159579515457153, -1.2014049291610718, -0.4761340022087097, 0.39365988969802856, 0.16246488690376282, -1.53462815284729, 0.19442413747310638, -0.36049413681030273, 1.2785930633544922, 1.882266879081726, -1.898266077041626, 1.1291882991790771, 0.7183877229690552, 0.9184252619743347, 1.3555876016616821, -0.9627693295478821, 2.3679888248443604, 2.805894374847412, -1.1261742115020752, 1.8871532678604126, 0.6928099989891052, 1.7143551111221313, 1.1959474086761475, 0.2656201422214508, 0.36159852147102356, -0.21622073650360107, -0.1804133504629135, -0.22679713368415833, -0.1609942764043808, 1.558529257774353, 1.1254196166992188, -1.6394327878952026, -0.7811004519462585, 2.28682804107666, 0.463192880153656, -0.0014380449429154396, 0.5390143394470215, -0.25313180685043335, 1.1938717365264893, 0.2632448375225067, 0.23393748700618744, 0.28357020020484924, 1.1480839252471924, 0.5683402419090271, -1.520421028137207, 0.15801052749156952, -1.9505385160446167, 0.9603298306465149, -0.17339468002319336, -1.9048455953598022, -0.5345818996429443, 0.6286154389381409, -0.6464714407920837, -0.2348410040140152, 0.8377131223678589, 2.19988751411438, 0.3175088167190552, 1.3408294916152954, -0.5028790831565857, 0.05547294765710831, -0.35822173953056335, 0.14928652346134186, 2.4421069622039795, -0.4144344925880432, 0.7964881658554077, 0.6770954728126526, -0.07665497064590454, -1.306041955947876, -0.999545156955719, -0.21853794157505035, 1.7537846565246582, 1.0589568614959717, -0.44105085730552673, -0.0831449031829834, -1.6981562376022339, -0.8393661975860596, 0.4890024662017822, -0.7549655437469482, -0.3033987283706665, 0.4937599301338196, 0.7913530468940735, 2.170301914215088, 0.15170833468437195, -0.6426019072532654, 0.4161474108695984, 1.6073423624038696, 0.4746098518371582, 0.5923330187797546, -0.5841712355613708, -0.07413838803768158, 0.712123453617096, -0.8927935361862183, 0.34227070212364197, 1.7610273361206055, 0.4661853611469269, -0.16913814842700958, -0.016878660768270493, 1.4530144929885864, -0.3085763454437256, -1.5659085512161255, -1.5489190816879272, 1.1387463808059692, 0.951460063457489, 0.7105292677879333, 0.9596329927444458, 0.397687703371048, 0.6479960083961487, -0.5224999785423279, -0.9608180522918701, 1.4919177293777466, -1.22390878200531, 1.8308392763137817, 1.111789345741272, 0.9957287311553955, 1.2631624937057495, -1.119734287261963, -0.5587321519851685, -0.2767731547355652, 1.4767022132873535, -0.3281194269657135, 0.33851611614227295, -0.05591706559062004, -0.10544466972351074, -1.3003122806549072, 0.40424713492393494, 0.34716033935546875, 0.0005554936942644417, 0.027857786044478416, 0.7195689678192139, -0.12106563150882721, 0.7784875631332397, -0.26750680804252625, 0.34078869223594666, -1.1278860569000244, -0.34431684017181396, 0.33375272154808044, -0.23218201100826263, -0.38088351488113403, -0.19302481412887573, -0.67910236120224, 0.333161860704422, 1.0801100730895996, 1.3827382326126099, -0.12333495169878006, -0.7114970088005066, -0.5028367042541504, -0.016308678314089775, 0.28417858481407166, -0.06915165483951569, 0.7062626481056213, -0.4138045012950897, 0.26622676849365234, -0.07592765986919403, -1.0281035900115967, -1.0795081853866577, 1.396469235420227, 1.328722357749939, 1.4061124324798584, 2.072958469390869, 1.1952983140945435, 1.5474448204040527, -0.6548864841461182, 0.10407040268182755, -0.5774788856506348, 0.5669749975204468, -2.0089166164398193, -0.5382843017578125, 0.14618465304374695, -1.8783812522888184, 1.3326727151870728, -1.0008397102355957, 0.3384149968624115, -1.254077672958374, -0.23190315067768097, 0.8438881039619446, 0.7858874201774597, -1.7356771230697632, -0.14569906890392303, -0.0969250500202179, -0.31512632966041565, 0.2610941231250763, 1.4403445720672607, 0.04331548139452934, 1.3358699083328247, -1.1144789457321167, 1.2040420770645142, -0.21883758902549744, -0.7105646729469299, 0.3006296157836914, -0.2047630399465561, 0.21019475162029266, 0.5008648037910461, -0.37860581278800964, -0.827191174030304, 0.6545153260231018, 1.8619004487991333, 0.6844220161437988, -0.601478099822998, -1.9354526996612549, 0.7072392702102661, 2.4177730083465576, -0.1741953343153, 0.017825864255428314, -1.924646019935608, -0.4473404884338379, -0.3469984829425812, -0.22206801176071167, 1.3491653203964233, -0.7252739071846008, -1.490857720375061, 0.5173566937446594, -0.46234366297721863, -0.8244308829307556, -0.351791650056839, -0.5212371945381165, -0.2439795583486557, 0.6454957127571106, 0.20403780043125153, -1.3567415475845337, 0.1523982733488083, 0.568221926689148, -0.9791455268859863, 2.0667684078216553, 1.6556988954544067, -0.05611753091216087, -1.7267931699752808, -1.6222221851348877, -0.044826630502939224, 0.4708004891872406, 0.8068974614143372, -0.6192606091499329, 0.4646504521369934, 0.17278769612312317, 0.8069787621498108, 0.3223337233066559, -1.2414441108703613, 1.0134668350219727, 0.3088182806968689, 0.9873359799385071, -0.7084733843803406, 0.7750965356826782, -0.7780454754829407, -0.10328322649002075, -0.6969770789146423, -0.36092156171798706, 0.2765319049358368, -1.226103663444519, -1.1618893146514893, 1.3858983516693115, -0.8963788747787476, 0.5734605193138123, -1.8046848773956299, 0.4474916458129883, 0.8282909393310547, -0.5270226001739502, -0.4593992531299591, -0.5960585474967957, 0.7562808990478516, 1.54812490940094, 0.6049963235855103, -0.0035043407697230577, -1.965027093887329, 1.2606388330459595, 1.3275344371795654, 1.650688648223877, 0.1143346056342125, -0.9443110227584839, 0.5807506442070007, 0.5995644927024841, -0.2808294892311096, -0.8244168162345886, 0.6921457648277283, 0.6840089559555054, -0.8769158124923706, 0.2884831726551056, -0.3373187184333801, -0.9157685041427612, -0.31728702783584595, -0.06904439628124237, 1.4468048810958862, -1.6144863367080688, 0.34549862146377563, -0.025299230590462685, 0.648374080657959, -1.056593418121338, -1.5809987783432007, 2.0338704586029053, -1.170409083366394, -0.9321979880332947, -1.7077937126159668, 0.9405695796012878, 0.1184430792927742, -1.8205853700637817, 0.08880840241909027, 2.637526273727417, -0.6606269478797913, 0.3151310682296753, -0.05196134373545647, 0.11958079040050507, -2.024534225463867, -0.9178933501243591, -1.8885245323181152, 0.09228202700614929, 1.8156194686889648, -1.1930264234542847, 0.507929265499115, 0.9763310551643372, 0.6355945467948914, -0.10421761870384216, -0.11542994529008865, -0.6109566688537598, 2.681284189224243, 0.2178967297077179, 0.29970431327819824, 0.14330139756202698, 1.074090838432312, 0.5359312891960144, 0.507987916469574, 2.487351417541504, -0.10783462226390839, 0.9274989366531372, 1.0424141883850098, -0.9954321980476379, -0.6379263997077942, 1.3108704090118408, -1.2243235111236572, 1.6122045516967773, -0.24193896353244781, -0.36194270849227905, -0.5510807633399963, 0.6233330965042114, -1.3176060914993286, -0.3624251186847687, 0.004635340068489313, 1.1928555965423584, 0.7469760775566101, 1.0801165103912354, -0.22155746817588806, 0.013117793016135693, -0.13768082857131958, 1.032599925994873, 0.5614415407180786, 0.8287119269371033, 0.601574718952179, 0.01698227971792221, 0.21819399297237396, 1.5766119956970215, 0.39284223318099976, 1.6074762344360352, -0.8411681056022644, -0.20298825204372406, -0.29710397124290466, -2.039069890975952, 0.6379614472389221, -1.0197511911392212, 0.23160827159881592, 0.5271843671798706, 0.3726716935634613, 0.22476615011692047, -0.09135009348392487, 0.7333868741989136, -0.4734139144420624, 1.1535216569900513, 0.15058328211307526, -0.8088114261627197, -1.2447925806045532, 1.5490068197250366, 0.2821041941642761, -1.1111292839050293, 0.020790893584489822, 0.518920361995697, 1.4143751859664917, -2.093676805496216, -2.9201807975769043, 0.8187131285667419, -0.5364447236061096, -1.880205512046814, -0.746749222278595, -2.0526070594787598, -0.6235450506210327, -0.527405858039856, -0.07565044611692429, 0.7587063312530518, -2.0147511959075928, -2.4645676612854004, -0.36327555775642395, -0.6343874335289001, 1.0505356788635254, -0.4085988402366638, 1.5051857233047485, -0.27551326155662537, -0.11363039165735245, 0.4451833665370941, 0.7445854544639587, 1.666689157485962, 0.020694434642791748, -0.08455429971218109, -0.36859259009361267, 1.351860523223877, -1.3851603269577026, 1.0551657676696777, -1.014696478843689, -0.5381258130073547, 0.9790154099464417, -0.16129888594150543, -1.1924182176589966, -0.1367144137620926, -0.4511609375476837, 0.004396640229970217, 0.18571221828460693, 0.9592229723930359, 0.3871864974498749, -1.0939035415649414, -1.6969704627990723, -0.5550876259803772, 1.3889755010604858, 0.3473759889602661, 1.8121811151504517, -0.9187291264533997, -0.3163723945617676, -1.526667594909668, 0.37734365463256836, 0.678606390953064, 1.1964713335037231, -1.3893802165985107, -1.459288477897644, 0.5435578227043152, -0.3119731843471527, -1.8747472763061523, -0.42974740266799927, -0.06516040861606598, -1.6971529722213745, -0.6920297145843506, 0.08180849999189377, -0.5225794911384583, 0.199692502617836, 0.14400488138198853, -0.8540441393852234, 0.2855721414089203, -1.1976066827774048, -1.134508728981018, 1.2406741380691528, -0.7470465898513794, 1.319879174232483, 0.2070544958114624, 1.477077603340149, 0.13988742232322693, 0.812926709651947, -0.42501407861709595, 1.647939682006836, 1.7446939945220947, -0.6056336760520935, 0.7352038025856018, -0.07419486343860626, 0.9678552746772766, -1.4382823705673218, 0.11575613915920258, 0.914422869682312, 0.5232232213020325, -0.6192014217376709, -2.325732469558716, -0.27987203001976013, -0.38225990533828735, -0.6179214715957642, -0.41666659712791443, 1.4822710752487183, 0.6247348189353943, -0.8270338177680969, 1.4600428342819214, -1.1892907619476318, -0.2851915955543518, 1.3389583826065063, 1.1434415578842163, -0.32481494545936584, 1.7702327966690063, 0.7706169486045837, -0.7946378588676453, 2.0185766220092773, -0.602687418460846, 0.9978245496749878, -1.0006699562072754, -2.2639753818511963, 0.5127293467521667, -1.1796588897705078, 2.2652649879455566, -0.1373605728149414, 1.3473178148269653, 0.8474147319793701, 1.7817434072494507, 0.1817164570093155, 2.3410401344299316, -0.22765956819057465, -0.3439977765083313, -0.5381550192832947, -0.5182314515113831, 1.2690354585647583, -1.5594855546951294, -2.7472481727600098, -1.752937912940979, -0.8820528984069824, -2.0867507457733154, -0.7046137452125549, -1.2275532484054565, 0.5269913673400879, 0.33500802516937256, -0.2139955759048462, 1.0709097385406494, 0.2052765041589737, -1.353052020072937, 2.337608814239502, 1.100825548171997, -0.18268240988254547, -0.16378694772720337, 0.4060560464859009, -2.3558926582336426, -1.5721946954727173, 0.21860529482364655, 1.294670820236206, 1.7935564517974854, 0.4595530927181244, 1.5253901481628418, 0.12211234867572784, 0.5557069182395935, -0.36668288707733154, -1.405814290046692, -0.8357160687446594, 0.6345207095146179, -0.5779303908348083, -1.4407018423080444, 0.24924376606941223, 2.6847033500671387, 1.634706735610962, -1.5375151634216309, 1.292893409729004, 1.348042368888855, -0.03514249250292778, 0.8104054927825928, 0.6978165507316589, 1.0887322425842285, -0.17127588391304016, -0.14276906847953796, -1.0657620429992676, -0.22972320020198822, 0.6872656345367432, 0.6928166747093201, 0.059498682618141174, 0.02373761683702469, -0.043640196323394775, 0.8530226349830627, -0.8121101260185242, -0.5512983202934265, -1.1079137325286865, -0.9658628702163696, 1.7665257453918457, -0.43132254481315613, 1.1532530784606934, -1.5915825366973877, 0.02357509918510914, 0.015646012499928474, 1.1336650848388672, -0.8343954682350159, -0.3538512885570526, 0.23571287095546722, -0.5896095037460327, 1.6503719091415405, -1.2182036638259888, 1.366613507270813, -1.805932641029358, 2.755619764328003, -0.12843570113182068, 0.9187455773353577, 0.13441777229309082, 0.5017005801200867, 0.12639328837394714, 0.7767465114593506, -1.3048646450042725, -0.06508409231901169, -0.3858901560306549, 0.4969216287136078, -2.1017048358917236, -0.5584258437156677, -0.6280712485313416, 1.7449487447738647, 1.1092127561569214, 0.8348028659820557, 0.21892404556274414, 0.17392221093177795, -1.2466033697128296, -0.9749394655227661, 0.6020357012748718, 0.8108869194984436, -0.8538556694984436, 0.831892728805542, -0.8950169086456299, 0.26600778102874756, -0.3237507939338684, 0.6455522775650024, 0.043322958052158356, -0.2996077835559845, 0.5343952775001526, 0.06404734402894974, 0.453594446182251, -0.27835556864738464, 2.454411745071411, -1.2003742456436157, 0.9886402487754822, -0.2810100317001343, 1.1159319877624512, -0.32442083954811096, 0.9258794784545898, 2.6694304943084717, -0.9051360487937927, -1.3189738988876343, 1.4397461414337158, -0.4895765781402588, -2.1216297149658203, -0.21184702217578888, -1.3884857892990112, -1.090085744857788, 0.6258749961853027, -0.035761479288339615, 0.8246563673019409, -0.29264700412750244, 0.17265209555625916, -0.4870496094226837, 0.4633949100971222, 1.2396982908248901, -0.3398243486881256, 2.372955799102783, -0.6897061467170715, -1.7849056720733643, -0.654575526714325, 1.8513530492782593, -0.19403067231178284, 0.4848315119743347, 2.097362995147705, 0.42836859822273254, 0.04190254211425781, -1.173960566520691, -1.3529947996139526, -0.016448305919766426, 1.1547645330429077, -1.4383518695831299, -0.27889400720596313, -1.500440239906311, 1.052250623703003, -0.027472687885165215, -0.714005708694458, -0.27775201201438904, 2.2283308506011963, 0.08555468171834946, 2.673696279525757, -0.1659928262233734, -0.7391124963760376, 0.2966899275779724, 1.8984678983688354, -0.3331044316291809, 0.23505660891532898, 2.1185567378997803, 1.3951925039291382, 0.6042458415031433, -0.2299034148454666, -1.6872326135635376, -0.5239519476890564, 0.3973168432712555, -0.40602046251296997, 0.17542439699172974, -0.9425041675567627, -0.05401301383972168, -0.12780453264713287, 1.2226272821426392, 0.3629244863986969, 0.36897799372673035, -0.2512767016887665, 0.13207247853279114, 0.5541733503341675, 0.19708584249019623, -0.046521902084350586, 0.5842717885971069, 0.4957689642906189, 0.8650994896888733, 0.06105181202292442, -1.2294855117797852, 0.2294754534959793, -0.2106582522392273, 1.1137912273406982, -1.0727664232254028, -0.5790515542030334, 0.1538182646036148, 1.116265058517456, 0.2704755961894989, 0.5311604738235474, 0.4431697428226471, 0.4406416416168213, 1.8105542659759521, 0.2080727070569992, -0.6141895651817322, 0.558457612991333, 0.3727478086948395, 0.006733720190823078, -0.7478307485580444, 1.627138614654541, 0.06353633105754852, -1.63107168674469, -0.4739901125431061, -1.3001041412353516, -1.1140085458755493, -0.49231410026550293, 1.2935841083526611, 1.6327214241027832, -0.17493687570095062, -0.9495290517807007, -0.05027291551232338, 0.27789318561553955, 0.10696835815906525, -0.125026136636734, -0.29195520281791687, 0.17017874121665955, -1.0337650775909424, -0.1869879961013794, -0.42413821816444397, 0.7027121782302856, 0.9934093952178955, -2.4579954147338867, 1.4690139293670654, 1.327958583831787, -0.7080667614936829, -1.6241555213928223, 1.2779942750930786, -1.2115126848220825, -0.020419185981154442, 0.7266874313354492, -0.8489572405815125, 0.6161370277404785, 0.3671504557132721, -0.8918173909187317, -0.7419595122337341, -1.2309253215789795, 0.7832388281822205, 0.07540316879749298, 0.5110421180725098, 0.4702819287776947, -1.3827632665634155, 0.12992632389068604, 0.6643840074539185, -0.3293497562408447, -0.8900421261787415, 1.0948388576507568, 0.5016492605209351, -0.6012323498725891, 0.5583955645561218, 1.179351568222046, -1.1588631868362427, -4.023889064788818, 1.8350059986114502, 0.6550946831703186, 0.21091464161872864, -0.0518912710249424, -1.8308829069137573, -0.4528232216835022, -1.0607683658599854, 0.33470866084098816, 0.5045276880264282, -0.8931831121444702, -1.231337308883667, -1.2264748811721802, 1.0283280611038208, -0.6796553134918213, 1.7217514514923096, 0.13155150413513184, 0.4654778838157654, 1.2314817905426025, 0.18191266059875488, -0.21106921136379242, 1.0503309965133667, -1.2475810050964355, -0.48578429222106934, -0.16410788893699646, -0.0806041955947876, -0.049258239567279816, 0.29427430033683777, -0.24506445229053497, 0.1492432951927185, 1.8550689220428467, -1.0836601257324219, 0.24131999909877777, -0.10100027173757553, -0.21340182423591614, -0.7914363741874695, -1.3214490413665771, 2.1622352600097656, -0.7098560333251953, 0.37237247824668884, -0.3077773451805115, -0.4288070499897003, 0.7096017599105835, -1.1800556182861328, 0.5154109597206116, -0.6091048717498779, 0.12328164279460907, 1.9343916177749634, -0.0008725334191694856, -1.3183355331420898, -1.2830191850662231, 0.8641238808631897, 0.32158249616622925, 1.0246614217758179, 0.2073606550693512, 0.08407779783010483, -0.7524262070655823, -1.1675668954849243, 0.29821211099624634, 0.6378746628761292, -0.003945319913327694, 0.5901219844818115, -0.9827567934989929, -1.127629041671753, 1.71513032913208, 0.9394354224205017, -0.12323293089866638, 0.22206470370292664, -0.6744037866592407, 1.2477422952651978, -0.29218029975891113, -0.30580076575279236, -0.4756348133087158, -0.6138166189193726, 0.028973689302802086, 0.01564689539372921, -0.19719842076301575, 0.37578627467155457, 0.38440632820129395, -0.2030658721923828, 0.05218358710408211, -0.5939542055130005, 0.48600512742996216, 0.24865153431892395, 1.7413270473480225, 0.34508419036865234, -1.021872639656067, 0.5810531377792358, 0.10200836509466171, -0.3017948567867279, 2.400177001953125, 0.9265992641448975, -0.7282293438911438, 1.6959800720214844, 0.27557694911956787, 0.42530304193496704, -1.5694122314453125, 0.7692527770996094, -0.003852797206491232, -0.4372217357158661, 1.128618836402893, -1.0123820304870605, 0.48110368847846985, 0.34016016125679016, -1.0069066286087036, 1.2656362056732178, -0.40786683559417725, -0.5064010620117188, 0.1328715682029724, -0.35833659768104553, -0.9859869480133057, -0.341802716255188, -0.28920021653175354, -1.2996082305908203, -0.12369677424430847, 0.923089861869812, 1.1258658170700073, 0.5171814560890198, 0.651850163936615, 0.7999390959739685, 0.09892889112234116, 1.4385898113250732, -0.4936028718948364, -0.5689167380332947, 0.20005613565444946, 0.7159045338630676, -0.785685122013092, -0.14419785141944885, 0.9565882086753845, 1.0417159795761108, 1.3774231672286987, 1.4438204765319824, 0.16782814264297485, -0.7852239012718201, 0.27899783849716187, 0.9586000442504883, -1.6538934707641602, -2.0375709533691406, 1.3188822269439697, -0.5543436408042908, 0.1612129956483841, 0.7095350027084351, 0.35850071907043457, 0.5037276148796082, -0.8662235736846924, 1.5377860069274902, 1.862043023109436, 0.016113029792904854, 0.7864935398101807, -1.9360407590866089, 1.1230173110961914, 0.5652502775192261, 0.15276020765304565, 0.9289658069610596, 0.8224817514419556, -0.34406930208206177, 0.19672206044197083, -0.48859167098999023, -0.18369445204734802, -0.9120064377784729, 0.028488177806138992, 0.144779235124588, -1.9249638319015503, -0.7107139825820923, 1.332243800163269, 0.3704599142074585, -0.6394636034965515, -1.4019482135772705, 0.5645148158073425, 0.40701404213905334, -0.5297476053237915, 0.09533893316984177, 1.3380683660507202, -1.6590282917022705, -0.32666224241256714, -0.758118212223053, -0.04174191504716873, -1.2488648891448975, -1.1994190216064453, -0.773511528968811, 0.6823253631591797, -0.3874795436859131, -0.052486155182123184, -0.5023442506790161, -0.5130866169929504, -0.7735939025878906, 1.2256925106048584, 1.1993080377578735, 2.1111953258514404, 0.2797226905822754, 0.060717545449733734, 0.32018935680389404, 0.6789566874504089, -0.12899336218833923, -1.4185073375701904, 0.7009814977645874, -0.752640962600708, -0.975640058517456, 0.5575070977210999, -0.39833423495292664, 1.461382269859314, 0.46403926610946655, 0.46276187896728516, 1.4340267181396484, 0.7144790887832642, -1.2131240367889404, -1.8002341985702515, 0.6994110345840454, -0.35295379161834717, -0.5183676481246948, -0.7957909107208252, -0.36101672053337097, -2.404808521270752, 0.02121567539870739, -0.29357030987739563, 0.6972755193710327, -1.0067499876022339, -1.4017465114593506, -0.40061259269714355, 0.9715200662612915, -0.5344617962837219, 0.5642703771591187, 1.5807033777236938, 0.6925009489059448, -0.943081259727478, -0.2759508490562439, -0.23222333192825317, -0.014378504827618599, -0.2922587990760803, -2.555047035217285, 0.5109870433807373, -1.400836706161499, 0.8026031255722046, -1.2541441917419434, -0.19932043552398682, 0.5590956211090088, 1.4807548522949219, 0.2612275779247284, 0.08521240204572678, 0.3176702558994293, -1.0175329446792603, -0.43776950240135193, 0.8085232377052307, 0.6348850727081299, -0.5907103419303894, -1.181140661239624, -0.7693197727203369, -0.917129397392273, 1.1994738578796387, 0.311284601688385, 2.2578189373016357, 0.8946730494499207, 0.4350021481513977, 1.0495080947875977, 0.8705738186836243, 1.2660590410232544, -0.21471837162971497, -1.0238101482391357, 0.4274204969406128, 0.051464978605508804, -0.29052650928497314, 0.7967418432235718, 1.7938740253448486, 0.6623108386993408, 0.5443024039268494, 0.12987682223320007, -0.49971142411231995, 1.321210503578186, 0.13648267090320587, -0.7309901714324951, 0.0023333861026912928, 1.5768083333969116, 0.3688845932483673, -0.20497570931911469, -0.5496179461479187, 1.0368194580078125, 0.42792537808418274, 0.09340299665927887, -0.7042489647865295, 0.10425468534231186, 0.5097887516021729, -0.2744397819042206, 0.7180579304695129, 0.6992683410644531, -1.9857816696166992, 1.4140342473983765, -1.0161200761795044, 0.13160055875778198, -1.061594009399414, -2.4547369480133057, 0.6341308355331421, -0.6592295169830322, 0.6281509399414062, -0.873860239982605, -0.1492534577846527, 1.74795663356781, 0.3535999059677124, -1.2502604722976685, 0.9077931046485901, -1.1833693981170654, -0.6885464787483215, -1.3651841878890991, 0.7906712293624878, -0.679135799407959, -0.4038558304309845, -0.8654788732528687, -1.9051384925842285, 0.8360745906829834, 0.9352989792823792, 0.1862390786409378, 0.4079729914665222, 1.5374099016189575, -0.7949997782707214, -0.7681766748428345, -0.2259558141231537, -0.12541598081588745, 1.180289387702942, -0.8673226833343506, -0.40705201029777527, -1.1390326023101807, 0.7509545087814331, 2.1299657821655273, 0.03046959824860096, -1.44932222366333, 1.7591081857681274, 1.149670958518982, 0.7978695631027222, -1.0122631788253784, 1.2337757349014282, 0.3233235776424408, -0.2584822177886963, -0.3247636556625366, 2.3169236183166504, -0.6009141802787781, 0.03226647898554802, -1.077439546585083, -1.0146335363388062, -0.8007129430770874, 0.5067716240882874, -0.2540320158004761, -0.5107738971710205, -0.33891987800598145, 0.2969205677509308, -0.567510724067688, 0.5125366449356079, 1.4358748197555542, 0.28383544087409973, 0.3883918225765228, -0.9266955256462097, -0.8616152405738831, 0.765824019908905, 0.10517792403697968, 0.1272164136171341, 0.47338661551475525, -0.8596817255020142, 0.11246059834957123, -0.9084906578063965, 1.1578551530838013, -0.05884512513875961, 0.11806890368461609, 0.8199263215065002, -0.280163437128067, 1.8964080810546875, -0.42779117822647095, 0.33949917554855347, -0.2843874990940094, -0.1267504096031189, -0.2980055510997772, 1.5205955505371094, -0.8139581680297852, -0.49466609954833984, -0.9328704476356506, 0.29901161789894104, 1.7236279249191284, 0.06423439085483551, -0.35134217143058777, 1.7174283266067505, 1.9227888584136963, -0.7749377489089966, -0.7287750840187073, -1.7355362176895142, 0.656193196773529, 2.2750158309936523, -0.5489107966423035, 0.39874252676963806, 0.9186110496520996, -0.292520135641098, 1.5091356039047241, 0.30790552496910095, 1.6174858808517456, -1.728888988494873, -0.37292933464050293, 1.441009283065796, -0.08097710460424423, -0.49463415145874023, -1.6706771850585938, -1.1641185283660889, -0.679137647151947, 0.3502865135669708, -0.36736345291137695, 1.2835177183151245, -1.2679240703582764, -0.21474607288837433, 0.41022664308547974, -0.7286906242370605, 0.21146121621131897, -0.26452985405921936, -0.16976326704025269, -0.4889998435974121, -0.5438706278800964, -1.017903208732605, 1.1401796340942383, 0.1788109689950943, -0.9112181663513184, -0.3252708613872528, -0.38741806149482727, -0.11295055598020554, 0.7788584232330322, 2.417418956756592, 0.3607836961746216, 0.7801625728607178, 1.4554532766342163, 1.1305896043777466, 0.8225911259651184, -0.042120322585105896, -0.589830756187439, 2.1134188175201416, -0.019955145195126534, -0.5040011405944824, 1.6146695613861084, -1.104091763496399, -0.7637442946434021, 0.4320387542247772, 0.6008292436599731, -1.9740396738052368, 0.6895055770874023, 0.19485321640968323, -0.9982305765151978, -0.030567586421966553, 2.5498266220092773, 0.6909704208374023, -0.23447322845458984, -1.3913052082061768, 1.3844528198242188, 0.7291591167449951, 0.4652974307537079, 1.262664794921875, 0.8435492515563965, 0.8198339939117432, -1.1799885034561157, 0.8404229879379272, -0.32122939825057983, -2.3236782550811768, -1.965484619140625, -0.9096238017082214, 0.9735767245292664, 2.163379192352295, -1.1468044519424438, -1.776357650756836, 0.5756712555885315, -1.6277300119400024, 0.6561123132705688, -0.40336406230926514, 0.38239771127700806, 1.2617871761322021, -0.5580965876579285, 0.9896392226219177, -1.0593854188919067, -0.68567955493927, -1.6376854181289673, -1.0052804946899414, 1.8267039060592651, 0.0452692024409771, -0.8976036310195923, 1.0611426830291748, 0.8189363479614258, 0.9555003046989441, 0.07153719663619995, -0.23373441398143768, 0.10738686472177505, 0.6091178059577942, -2.136347532272339, -1.161346435546875, 0.28325995802879333, 0.10535168647766113, -0.28215110301971436, 1.0611664056777954, 0.28867480158805847, -1.8224238157272339, 0.08356348425149918, 1.3227226734161377, 0.12014767527580261, 0.7110580801963806, 0.41336411237716675, 0.39399996399879456, -0.704393208026886, 1.9764906167984009, 0.2544139325618744, -0.9684781432151794, -0.012541188858449459, -1.0313889980316162, 0.9473108649253845, -0.8308702111244202, -0.3273015022277832, -0.813513457775116, -0.1962268352508545, -1.4531794786453247, -0.6582670211791992, -1.03826904296875, -0.48433414101600647, -1.2064859867095947, -1.7721781730651855, -0.6749807000160217, 0.8717024326324463, 0.26748576760292053, 2.2319345474243164, 0.3219696283340454, 0.803540825843811, 0.5937315821647644, -1.1376678943634033, -1.273202896118164, 0.7701507210731506, -0.2376057356595993, 1.3600797653198242, 0.6140503883361816, -0.33936047554016113, -0.5485855340957642, 0.44943174719810486, 1.1987755298614502, -0.020879041403532028, 1.7918307781219482, -0.05032194405794144, 0.6139934659004211, -0.23868051171302795, 2.523383855819702, -1.9692003726959229, 0.1435449719429016, 0.2543344795703888, -1.1029877662658691, 1.2948834896087646, 0.46238139271736145, -0.4028625786304474, 1.1357922554016113, 1.7000876665115356, -0.005988330114632845, -1.1237859725952148, 0.2871800661087036, -0.48134729266166687, -0.820238471031189, 0.3639642596244812, -0.9464924931526184, -0.4401889443397522, -0.6861838102340698, -2.549450635910034, 0.5890542268753052, 0.46598368883132935, 1.4877076148986816, 0.8603625893592834, 0.5234678983688354, 1.411971926689148, -0.012995832599699497, -0.8506551384925842, -1.6067304611206055, 0.19141660630702972, -0.15698695182800293, -0.6407449841499329, 0.45397406816482544, 1.116400957107544, -1.557573914527893, -0.3569810390472412, -0.8119767308235168, -0.6583108305931091, 0.22017648816108704, 1.4277039766311646, -0.35621902346611023, 0.39081329107284546, 0.5853956937789917, -1.2213822603225708, 0.4037838578224182, -1.4302856922149658, 1.4405841827392578, 0.37119653820991516, 0.7672760486602783, 2.598022937774658, -0.12495653331279755, -1.185691237449646, -0.4659308195114136, 1.63821280002594, 0.6790518164634705, -0.05349921062588692, 2.857250452041626, 0.3786410689353943, 2.2057456970214844, 0.506767988204956, -1.090197205543518, 0.5623497366905212, 0.06257578730583191, -0.5678114295005798, -0.8482515215873718, -1.4899927377700806, -0.08928105235099792, -0.5565006136894226, -0.0492573156952858, 0.2656250596046448, 0.46544837951660156, 0.13529561460018158, 1.0828120708465576, -0.5152488946914673, 1.8166799545288086, 0.38586392998695374, 1.1942403316497803, -1.6697988510131836, 0.450723260641098, 2.236339569091797, 0.5284872651100159, -0.8699309229850769, -0.09814759343862534, -1.0799225568771362, -0.6702656149864197, -0.5960955619812012, -1.5042567253112793, 0.21825581789016724, -0.44191983342170715, 0.5672245621681213, 0.5713778734207153, 1.1344759464263916, 0.15094581246376038, 0.250661164522171, 0.04426807910203934, 2.3729453086853027, -1.0713257789611816, -1.6737059354782104, -0.5812190771102905, 0.3259201943874359, 0.06235616281628609, -0.3211696445941925, 0.6071818470954895, -1.1054290533065796, -1.470836877822876, -1.463175892829895, 1.5945584774017334, 0.14383834600448608, 0.10218512266874313, -1.3440488576889038, 0.8555316925048828, 1.0731728076934814, -0.16107764840126038, -0.6071434020996094, 1.4534235000610352, 1.0362776517868042, 1.784949779510498, 1.0070573091506958, 0.4021152853965759, 1.76524817943573, 0.2676483988761902, -0.5393209457397461, 1.6516741514205933, -0.16099779307842255, 0.8415421843528748, 1.5267927646636963, -0.6496158838272095, -0.4437173306941986, 0.5099347829818726, 0.5773559808731079, -0.9755963087081909, -0.3736696243286133, 0.09003733098506927, 0.35694459080696106, -0.26087242364883423, -0.3256400227546692, -1.4300899505615234, -1.4610087871551514, -0.995805025100708, -0.25363337993621826, -0.5662125945091248, 0.4670659303665161, -0.08204777538776398, 0.6808890700340271, -1.6838645935058594, -0.5669540166854858, -1.5096358060836792, 0.16963574290275574, 0.4220358431339264, 1.8801774978637695, 1.0459480285644531, 2.1228504180908203, -0.3594316244125366, -2.4395086765289307, 0.23665562272071838, -1.2758288383483887, 0.789622962474823, 1.3259435892105103, 0.20116764307022095, -0.522826611995697, 0.6133145689964294, -1.0780978202819824, -1.1930640935897827, -0.2000173032283783, 1.4341658353805542, 0.9574041962623596, 0.2512364089488983, 0.8930121064186096, -1.2718491554260254, 0.6822529435157776, -1.5237590074539185, -0.5666240453720093, 0.31001561880111694, 0.41453617811203003, 0.7000404596328735, 0.2930688261985779, -2.0182533264160156, 0.8933807015419006, -0.3056323826313019, -0.38434576988220215, 0.6710090637207031, -0.2196219265460968, -1.3272514343261719, 0.8395134210586548, -0.009404756128787994, -2.2670013904571533, 0.8185943365097046, -0.6723504066467285, 0.4836233854293823, -1.5903394222259521, 1.7029894590377808, -0.9096994400024414, 0.08620722591876984, -1.1338791847229004, 3.0389111042022705, 1.1603294610977173, 0.665594220161438, 0.5402476191520691, -4.009869575500488, 0.7748368382453918, -0.9893496632575989, -0.3867564797401428, 0.161997452378273, 0.2628032863140106, 0.8509207367897034, -0.5988951921463013, -0.04408811032772064, -0.572694718837738, -0.39455440640449524, 1.165468454360962, 0.11579783260822296, 0.48275789618492126, -0.7565105557441711, 0.25840240716934204, 0.5366093516349792, 0.44966796040534973, 0.9166591763496399, 1.4412721395492554, 0.5466428399085999, -1.668561577796936, -0.05801486223936081, -0.6091936826705933, 0.38937222957611084, 0.7016289830207825, 1.3240957260131836, -0.6368206143379211, -0.7535066604614258, -1.0903046131134033, -0.7842862606048584, -0.8045253753662109, 0.9748955368995667, -1.410884141921997, -0.17973269522190094, -0.28792738914489746, -1.6575335264205933, -0.08199361711740494, -0.6646639108657837, -0.774258017539978, -0.5054036974906921, 0.6980558037757874, -0.45594754815101624, 0.2903386354446411, 0.4125041961669922, -0.2835600972175598, 0.38221392035484314, -0.10973434895277023, 1.0656815767288208, -0.2308972179889679, -0.08695570379495621, 0.323961079120636, 0.006597969215363264, -0.09539350122213364, 1.5765268802642822, -0.1838994175195694, -1.1510480642318726, 0.8250682353973389, 1.052981972694397, -0.7767958045005798, -1.5440378189086914, -0.3313240110874176, 0.2883780598640442, 0.8313236832618713, -1.8371727466583252, 0.7593984007835388, 0.906252384185791, 1.6760927438735962, -0.36582037806510925, 0.028947288170456886, -1.0547879934310913, 2.4980080127716064, -0.3884139955043793, 0.25746950507164, 0.4405912756919861, -2.2834179401397705, -1.2506333589553833, 1.5536516904830933, -1.3697550296783447, -0.5615829825401306, -1.754393219947815, -1.810571551322937, 1.4253182411193848, 1.167500615119934, -0.8523988127708435, 0.3036983013153076, -0.3824409246444702, 1.200542688369751, -0.223196879029274, -0.5564466118812561, -1.3255301713943481, 0.01822555810213089, -0.8003758788108826, -0.18609589338302612, 0.5009996294975281, 1.0624680519104004, 0.400812566280365, -1.294636607170105, -2.2787208557128906, -1.18973970413208, 0.9201677441596985, 1.5378050804138184, -0.41924527287483215, -0.880224347114563, 1.2584599256515503, 0.15078113973140717, -0.33805903792381287, 0.40044882893562317, 1.340358853340149, -0.24918220937252045, -0.8095788955688477, 0.6758635640144348, -1.3020856380462646, -0.15856905281543732, 0.6022634506225586, 1.479243516921997, -1.0754783153533936, -0.8397020697593689, 0.7046645879745483, -0.45450448989868164, -0.16039493680000305, -0.095054991543293, 0.9844350218772888, -1.9109941720962524, 0.5681086182594299, -0.6890646815299988, 0.10427597165107727, 0.2883285582065582, -1.3233520984649658, 0.17554804682731628, 0.5137053728103638, 1.5746444463729858, -0.4368470311164856, -0.24276532232761383, 0.05096418783068657, 1.6506460905075073, 1.630861520767212, -0.7386249303817749, -1.9955915212631226, -0.44527214765548706, 1.4117043018341064, -0.22905568778514862, -1.018355369567871, 0.5014851689338684, -0.7204133868217468, 0.5272627472877502, 1.4462329149246216, -0.4237446188926697, -0.1917094588279724, -1.0685911178588867, 0.060108501464128494, 1.7435303926467896, 1.1603891849517822, 0.7726275324821472, -0.29563459753990173, -0.7389194369316101, 0.02260305918753147, 0.21092116832733154, -0.956707775592804, -0.3788515627384186, 0.6698975563049316, 1.4148610830307007, -2.315169095993042, -0.5590934157371521, -0.06249712407588959, -0.5053567886352539, -0.9507376551628113, -0.604608952999115, 1.7173771858215332, 0.5456147193908691, -1.306547999382019, -0.19924409687519073, 1.987121820449829, 0.8079807758331299, -1.058716058731079, 0.0036232839338481426, -0.43971195816993713, 0.07832443714141846, -0.9640861749649048, 0.3849652409553528, 1.3412537574768066, -0.05717958137392998, 0.12563243508338928, 0.7549288272857666, -0.5388621687889099, -1.5879031419754028, -1.1830912828445435, 0.8412424325942993, 0.44811657071113586, 1.071799397468567, 0.6251472234725952, -0.0387142151594162, -0.3153130114078522, -0.34322455525398254, -0.23970189690589905, -1.1852130889892578, -1.5310955047607422, -1.3370285034179688, -0.2739446759223938, -0.8980713486671448, -0.5635215640068054, -0.5744501948356628, -0.7365420460700989, -1.3556358814239502, 0.8951015472412109, -1.6176578998565674, -0.3160950541496277, -0.6073072552680969, 0.3890746235847473, 0.9995114803314209, -0.9410274624824524, 0.8087924122810364, -1.0109103918075562, -0.8853089213371277, -0.4117686152458191, -1.204655408859253, -0.28257662057876587, 0.19086405634880066, 1.039374828338623, 0.9050760269165039, 1.1560308933258057, -1.1603890657424927, 0.46062400937080383, 0.3181462585926056, -0.4820282459259033, 2.662978410720825, -1.8642215728759766, 0.11057452857494354, -1.3162615299224854, -0.03879069164395332, 0.5129857063293457, 0.2447476089000702, 0.8669275045394897, 0.6228206753730774, 0.7997635006904602, -0.3042772114276886, -0.39318785071372986, -0.41228315234184265, -0.7945440411567688, -2.302886962890625, 0.5917810797691345, -0.3392362892627716, 1.7022323608398438, 0.06558477878570557, 0.20410284399986267, -1.4421257972717285, 0.15870238840579987, 0.06717565655708313, -0.7169013023376465, -0.39024195075035095, 0.12144628167152405, -1.6326804161071777, -2.173137664794922, 1.666619896888733, -0.21977204084396362, 0.06510767340660095, -0.9255166053771973, 0.9183664917945862, 1.6787360906600952, -1.0527347326278687, 1.0267071723937988, 1.626762866973877, -1.2699135541915894, -1.0191829204559326, 0.3764317035675049, 0.7844062447547913, -0.6883473992347717, 1.0226682424545288, 0.7633385062217712, -1.8494758605957031, -0.5998110175132751, -0.8096626996994019, 1.7812528610229492, -0.9320394992828369, 0.19294768571853638, -0.7071273922920227, 1.4492563009262085, -0.18748433887958527, -1.600466251373291, -0.1036151647567749, 1.1964755058288574, -0.564695417881012, 0.2945440411567688, -0.4563518166542053, -1.2841475009918213, 0.3288952112197876, 1.0710033178329468, 1.8880040645599365, 1.575494647026062, -0.9875901937484741, -0.48614397644996643, -0.43617287278175354, -0.1032378152012825, -0.049286242574453354, 0.7333593964576721, 0.9039545059204102, -0.7502629160881042, 1.075830340385437, 0.003872114699333906, -0.05528179928660393, 0.8517953157424927, -0.12566998600959778, 0.22714510560035706, 0.8963081240653992, 0.5301247835159302, -0.5135346055030823, -0.20050451159477234, -0.5200034379959106, 0.8367485404014587, 0.9041244387626648, -0.38343557715415955, 2.032392978668213, 0.6354747414588928, 1.2317049503326416, 0.9921294450759888, -0.36327236890792847, 1.3023813962936401, -0.508516252040863, -0.8277432322502136, 0.296862930059433, 0.2582179307937622, -0.9173373579978943, -0.089591845870018, 3.220625162124634, 0.14123617112636566, 0.8894318342208862, -0.8738709092140198, 0.11722008883953094, -0.14073322713375092, 0.602075457572937, -0.32451456785202026, -0.07649946212768555, -0.022134121507406235, 1.7436630725860596, 0.9705146551132202, -0.20746533572673798, 0.6989296078681946, -0.4262044131755829, -1.242477536201477, 0.42945703864097595, 0.18536600470542908, -0.18071700632572174, -0.8416637182235718, -1.1728541851043701, 0.1939162313938141, 0.46102362871170044, 0.3610829710960388, 0.39834171533584595, -0.3719608187675476, -0.344663143157959, -0.716894805431366, -0.6788580417633057, -1.338962197303772, -0.502403736114502, 0.4895433783531189, -1.656043529510498, 0.5361753106117249, 0.13550472259521484, -0.4355325996875763, -1.9370359182357788, -0.6774973273277283, 0.9586169719696045, -1.1814243793487549, -0.5001108646392822, -1.3301135301589966, 0.6942260265350342, -0.031177416443824768, -0.2853785753250122, 0.40390992164611816, 0.03313208371400833, 1.361219048500061, -0.31205639243125916, 0.20567163825035095, -0.09312168508768082, -2.306030035018921, 0.03652653843164444, -0.37738025188446045, -0.7891831398010254, 0.23848196864128113, -0.08184440433979034, -1.3187048435211182, -0.26657766103744507, -0.49549704790115356, -0.0025524974334985018, 0.7547307014465332, -0.24657954275608063, 0.016358118504285812, 0.7469179630279541, -1.3660368919372559, -0.8576256632804871, -1.982079267501831, 2.079982042312622, -1.2094379663467407, -0.2924533486366272, 0.43223294615745544, -0.3735100030899048, -0.8076034188270569, -1.1471400260925293, -0.41143471002578735, -0.00553514901548624, -1.2943562269210815, -0.6760528683662415, 2.0691981315612793, 0.2447412759065628, 2.041032314300537, 0.8867286443710327, -1.0852025747299194, 1.3607375621795654, -0.11382018029689789, 0.23732879757881165, 0.027693066745996475, -1.4662317037582397, -1.5939682722091675, 1.0332859754562378, 0.15544386208057404, -0.22679798305034637, 1.0980712175369263, -0.65547776222229, 0.6435108184814453, 2.317047357559204, -1.9848805665969849, 2.016146183013916, -0.3077251613140106, -0.47822996973991394, 1.2623533010482788, -0.37924525141716003, 0.40325963497161865, -0.5342373251914978, 0.20040598511695862, 2.4253785610198975, -0.18493451178073883, -1.4119046926498413, -0.46800416707992554, 0.1273290514945984, -0.7988114356994629, -0.07848253101110458, -1.6837903261184692, -1.286149024963379, -1.3330953121185303, 0.05122704803943634, 0.2786497473716736, 0.2931724190711975, -0.22711458802223206, 0.5333088040351868, -0.9660214781761169, -0.9857478737831116, -0.7315292954444885, 0.3481466472148895, -0.1498592346906662, 1.7701586484909058, -0.4046269655227661, 0.2808101773262024, -0.43796178698539734, -0.5025380849838257, -1.0463932752609253, 0.10963068157434464, 0.11644898355007172, -0.07677394896745682, 0.2523408532142639, -0.10422442108392715, -0.4415571689605713, 0.901329517364502, 1.302785873413086, -1.4367151260375977, -0.21727201342582703, -0.16252680122852325, 0.4872787892818451, 0.5578277111053467, 0.5915714502334595, 1.5488187074661255, -0.1180860698223114, 0.6009745001792908, 0.12091991305351257, -2.4298436641693115, -0.8018500208854675, 1.5306570529937744, -0.0833669900894165, 1.3190206289291382, -0.4429263770580292, 0.874699592590332, -0.15870891511440277, 0.8372701406478882, -0.5976212620735168, 0.5323631167411804, -1.1740739345550537, 0.21339361369609833, 0.404167115688324, -1.137555480003357, 1.7118103504180908, -0.554524302482605, 0.09565021097660065, 1.992106318473816, -0.764118492603302, 0.23841580748558044, -0.7282887101173401, 0.19394633173942566, -0.07182197272777557, -0.3476918339729309, -1.9167569875717163, -0.03673470765352249, -0.5485442280769348, 0.3229328691959381, -0.54071044921875, -0.6599725484848022, -0.3037644326686859, -0.3467783033847809, -1.167272686958313, 1.9013456106185913, 0.6527412533760071, -1.8585273027420044, -1.5771644115447998, 1.03816819190979, 0.588490903377533, -0.006065999157726765, 0.4751916527748108, 1.5200543403625488, -0.3576442003250122, 2.091392755508423, 1.458184003829956, 1.3121552467346191, 0.3449932634830475, 0.022438276559114456, 0.10596717149019241, 0.26848849654197693, 0.5693750381469727, -0.36579039692878723, -0.5668138265609741, 1.8888288736343384, 1.1766501665115356, 0.5238719582557678, -0.5643908977508545, 0.8030316233634949, -0.7135840654373169, 0.748611569404602, -0.10824970155954361, -1.0444350242614746, 0.37783777713775635, -0.745919942855835, -1.1951326131820679, 1.468966007232666, 0.7626032829284668, 0.7249035835266113, -0.36458200216293335, 1.516045331954956, 0.5747963786125183, -0.629179060459137, 0.9130244255065918, -1.1677227020263672, -0.14339075982570648, 1.2139770984649658, -0.6950072050094604, -1.9501962661743164, 0.542930006980896, -0.7917932271957397, -1.349891185760498, -0.2840479016304016, 1.0352295637130737, 0.6735486388206482, 1.2557995319366455, -0.5235954523086548, -0.8543421626091003, -0.6671690344810486, 1.1196507215499878, -1.2905746698379517, 0.9422374367713928, 1.0978070497512817, -0.9713641405105591, -0.5139659643173218, 0.6906929612159729, 1.2386126518249512, -0.03710222616791725, -0.3000011146068573, -0.052269939333200455, 1.953505277633667, 0.4590606689453125, 0.9812326431274414, 0.7503272891044617, 0.016946088522672653, -0.9597927331924438, -0.10020221024751663, -0.5706336498260498, -0.5797598361968994, 0.7848669290542603, -0.10091566294431686, -2.251840829849243, -0.621372640132904, 0.46075019240379333, 1.7284152507781982, -0.15609878301620483, -0.873856782913208, -1.4743421077728271, -0.7659716606140137, -0.20279447734355927, 1.1535253524780273, -0.36666861176490784, 1.2553989887237549, 1.4333748817443848, -1.5310277938842773, 1.0287299156188965, 1.3671743869781494, 0.2157285511493683, 0.468569278717041, -0.8929482102394104, 0.2575056850910187, -0.06658212840557098, 0.5069475173950195, 1.0169097185134888, 1.7020225524902344, 0.48130643367767334, 0.0044777896255254745, 1.8328782320022583, 1.8200567960739136, 0.6633699536323547, 1.261907696723938, 0.29469746351242065, -0.7019959688186646, 1.1237716674804688, 0.12959565222263336, -1.1617926359176636, -1.5800230503082275, -0.41964924335479736, 1.8738189935684204, -0.7749502658843994, -0.327880859375, -2.8690097332000732, -0.16402190923690796, 0.11378935724496841, -0.7964664697647095, -0.22323957085609436, -0.06471472978591919, 0.4508940279483795, 0.6107978224754333, -1.811559796333313, 1.285079836845398, -0.18175257742404938, 2.7832350730895996, 1.7132346630096436, -0.31668540835380554, -0.505645751953125, -1.4550747871398926, 1.543052077293396, -1.8114778995513916, 0.04378893971443176, 0.7756555080413818, 0.13698671758174896, -0.4364147186279297, -0.1664378046989441, -0.7591180205345154, -0.4139571785926819, 0.6071743369102478, 0.6739818453788757, 0.48674896359443665, 0.48240429162979126, 0.47209325432777405, 0.30740708112716675, 0.30380213260650635, 0.44008398056030273, 0.6751828789710999, -0.3980139195919037, -1.017146110534668, -1.2872099876403809, 1.3582149744033813, -0.5559400916099548, -1.3532651662826538, -1.2807759046554565, 0.10172668844461441, 0.4462323486804962, -0.4863033890724182, -0.6627179980278015, -0.26527804136276245, 0.09781096130609512, 1.1214088201522827, 0.8693507313728333, -1.1590282917022705, -1.3267570734024048, -1.3207595348358154, 0.7432542443275452, -1.221901535987854, 0.9352620840072632, 0.10350649803876877, -0.40715664625167847, 0.49179714918136597, -0.40440401434898376, -0.662121593952179, -2.181912422180176, 0.0472152978181839, -0.6012957096099854, 0.5028010606765747, -0.7289372086524963, -0.49926483631134033, -1.0161786079406738, 0.7443501353263855, 0.7230129837989807, 1.2880936861038208, 1.6476202011108398, 2.2729945182800293, -0.006777357310056686, -0.5689036250114441, -2.041499137878418, -1.0118361711502075, -0.6340281963348389, 1.4412343502044678, -2.5068936347961426, 0.375969260931015, 0.7214213013648987, -0.08269581198692322, -0.3671095371246338, -0.31172096729278564, 1.0849924087524414, -0.39355573058128357, 0.5447525382041931, 0.5029905438423157, -1.5230306386947632, 1.3951094150543213, 0.5965358018875122, -0.02539958246052265, 0.40477317571640015, 1.2272354364395142, -1.1143825054168701, 0.07978679984807968, 1.1058971881866455, 0.5153540968894958, -0.24940161406993866, -0.6101164817810059, 0.4258357584476471, 0.2344687283039093, -0.6730928421020508, -0.9005348682403564, -0.5899239182472229, -1.5630204677581787, -0.43101730942726135, -0.1966298222541809, 0.5787699818611145, -0.2556804120540619, 0.8393411636352539, -0.5717569589614868, -2.8085415363311768, 0.48209047317504883, -0.7922437191009521, -0.2809033989906311, -1.4455265998840332, 1.4298069477081299, 1.954393744468689, 0.9512761235237122, -1.2506365776062012, -2.2817153930664062, -0.808798611164093, -0.6503036022186279, 0.31462711095809937, 0.8905078768730164, 1.6470675468444824, -1.2641373872756958, 0.6185299754142761, 0.6127364635467529, -0.003708516014739871, -1.70572829246521, -0.6811551451683044, 1.1491526365280151, -0.9639497995376587, 0.6138075590133667, -0.7804048657417297, -0.5515170693397522, 0.5952267646789551, 2.082597017288208, 1.614495873451233, -0.13497214019298553, 0.651212215423584, 0.7504191994667053, 0.4524950087070465, 0.03291953355073929, 0.7788712382316589, 1.0630205869674683, 0.5939504504203796, 0.33959442377090454, -0.1677936166524887, -0.6985142230987549, -0.8203595280647278, 0.3564254939556122, 1.5751147270202637, -1.187698483467102, -1.364661455154419, 0.796835720539093, 0.4550289511680603, -1.9260238409042358, -0.6704905033111572, 0.5028225779533386, -1.1879826784133911, 1.3077137470245361, 0.11855137348175049, -0.24952280521392822, 1.0651865005493164, -0.11892619729042053, 0.0195216815918684, -0.15793731808662415, -0.2844080924987793, -0.8988754153251648, 1.604205846786499, -1.361932396888733, 0.4798542857170105, -0.8209543824195862, 0.24756698310375214, -0.8958733677864075, 0.24139179289340973, 0.9636626243591309, 0.060161322355270386, -0.044173434376716614, 0.07252555340528488, 0.9092724919319153, -0.9919822216033936, 0.061316922307014465, 1.1916968822479248, 0.7349315881729126, 0.5645561814308167, -0.3507443070411682, -0.9651309251785278, 1.5714330673217773, -0.07539904862642288, 0.0018416730454191566, 0.7765771150588989, 0.1893942952156067, -0.7300037145614624, 0.9438152313232422, -0.5033249258995056, -0.006436409428715706, -0.19210349023342133, -0.8998745083808899, 0.5802339315414429, -1.351345419883728, 0.9874894618988037, 0.2844283878803253, -0.6952989101409912, 0.47960996627807617, -1.4776638746261597, 3.0881550312042236, -0.09718068689107895, -0.3245948851108551, -0.766045093536377, 0.8254354000091553, -0.27443382143974304, 1.20597505569458, 0.7353302240371704, 0.1459672749042511, -1.2241711616516113, -0.7552110552787781, -0.8402437567710876, 1.5246535539627075, 0.14366571605205536, 0.690287709236145, 0.6545976996421814, -1.2885453701019287, -1.5189073085784912, -0.6077395081520081, -2.701706886291504, -0.07415740191936493, -0.23164938390254974, -0.3874525725841522, 1.147491693496704, 0.7093045711517334, -0.21435318887233734, -0.39519333839416504, 0.19055485725402832, -0.1735822558403015, 0.522389829158783, 1.508697509765625, -0.12400813400745392, 1.2801620960235596, 0.7442874312400818, -0.031353216618299484, -0.20425771176815033, -0.8669273257255554, 0.531011700630188, -0.5894597768783569, -0.04701739177107811, 1.523608684539795, 1.3168299198150635, -0.19977059960365295, -0.599025547504425, 1.2069859504699707, -0.2901035249233246, 0.5785166025161743, -1.8968044519424438, 1.9741309881210327, -1.3445223569869995, 0.7773278951644897, 1.4107508659362793, -0.18872417509555817, 1.0036765336990356, 0.949551522731781, 0.36149320006370544, -0.4757373631000519, 0.02216433547437191, -1.5332553386688232, 0.6725612282752991, 1.0991212129592896, 1.7869211435317993, 0.6910924911499023, 0.08349140733480453, 0.743205189704895, -0.5366328954696655, -1.6528481245040894, -0.2322310358285904, 0.07270140200853348, 0.29145461320877075, 1.2608191967010498, 0.8187901973724365, 1.0071907043457031, 1.1828370094299316, -1.9676803350448608, 1.4319236278533936, 1.835747480392456, 0.055624570697546005, -0.18363741040229797, 0.5396456122398376, -0.009837500751018524, -0.5472429394721985, 0.39753827452659607, -0.9621347188949585, 1.387154221534729, -0.743014931678772, -0.8185508251190186, -0.4043969213962555, 1.699175238609314, -1.2105787992477417, 0.34095317125320435, -0.7850145101547241, -1.3987778425216675, 1.5268300771713257, 0.21507145464420319, 1.2769237756729126, 1.1635388135910034, -0.5320321917533875, -0.48314228653907776, -0.0567755401134491, -0.15821124613285065, 0.4968114197254181, 1.5424776077270508, -0.7670826315879822, -0.514076828956604, 1.2646218538284302, 1.4891350269317627, -0.3359571695327759, 0.3553617298603058, -0.2614453434944153, 1.641715168952942, 0.6055487394332886, 0.23574253916740417, 0.9461084604263306, 0.8661386966705322, 0.014353002421557903, 0.7728404998779297, 0.49074217677116394, 0.3015778660774231, 0.07186924666166306, 0.9797871708869934, 1.5035611391067505, -1.2079060077667236, 0.28200653195381165, -1.273789644241333, -0.49447232484817505, -0.8022061586380005, -1.1422147750854492, 0.3845566213130951, -0.07880831509828568, -0.2797462046146393, -0.2117997109889984, -0.7794296145439148, -0.25885093212127686, -0.6857959032058716, -0.1840469241142273, -0.5255358815193176, 0.09622218459844589, 0.21946051716804504, -0.14532431960105896, 1.2611384391784668, 0.282662957906723, -0.2267674058675766, 0.5794154405593872, -2.506547689437866, -1.1932603120803833, -0.28661951422691345, -0.37268608808517456, -0.7509964108467102, -0.47422224283218384, 0.4342886209487915, -0.580693244934082, 0.5164502859115601, 0.7713235020637512, -0.056726183742284775, 0.8488770127296448, -0.6729886531829834, 1.0417062044143677, 0.9070141315460205, 0.2591980993747711, 0.3750735819339752, -1.817098617553711, -1.0609043836593628, -0.8446151614189148, -0.023117007687687874, 0.41116437315940857, -0.23650896549224854, 0.6399446725845337, 0.579979419708252, -2.745056390762329, 0.2663349509239197, -0.1778038591146469, -0.900150716304779, -0.3102705776691437, 0.5989768505096436, 0.6780032515525818, -0.1633123904466629, -0.372787743806839, 0.4830649793148041, -0.7721054553985596, 1.585675835609436, -0.47952598333358765, -0.03826751932501793, -1.62811279296875, -0.019252339377999306, 0.26920974254608154, -1.615921139717102, -0.4785243272781372, 0.6743799448013306, 1.2658787965774536, 0.6977770328521729, 0.2395053505897522, 0.0255263838917017, 0.8157777786254883, 0.20511503517627716, -0.5004540085792542, -1.8715367317199707, -0.938057005405426, 0.1222919225692749, 0.5792638659477234, -0.44415587186813354, -0.09548776596784592, 0.09961912781000137, -0.2438475340604782, -1.0604742765426636, -0.37043577432632446, 0.5151888728141785, 0.8797526955604553, -0.5194955468177795, -2.295898675918579, -1.4077913761138916, -2.059330701828003, -1.6580246686935425, 0.3398677408695221, -0.535933792591095, 1.3181771039962769, -1.0468428134918213, 0.03360067680478096, 0.14650218188762665, 0.3161962628364563, -1.7830109596252441, -0.38035619258880615, 0.9328957200050354, -1.824619174003601, 2.4176032543182373, 0.05620897561311722, 0.32961705327033997, 1.295322299003601, 0.076776884496212, 0.9897280335426331, -2.3055200576782227, -0.2163306623697281, -1.2842813730239868, 0.8382542133331299, -0.18160472810268402, -0.6860418915748596, 0.7826744318008423, -2.131570339202881, 1.0977790355682373, -0.8405455350875854, -0.6743128895759583, -2.415963888168335, 1.0965465307235718, 1.165540099143982, -0.2364213615655899, -0.48562493920326233, 0.13329526782035828, 1.8002870082855225, -0.6350920796394348, -0.5395434498786926, -2.5219082832336426, -2.1231887340545654, -0.3124339282512665, -0.5729196071624756, 0.8533511757850647, -0.5683585405349731, 0.4552546739578247, 0.49115702509880066, 1.9040759801864624, -0.44925013184547424, 0.3635425269603729, 1.065608024597168, -1.5035876035690308, 0.42536547780036926, -1.2618591785430908, -0.5671479105949402, -1.3753377199172974, -0.35356128215789795, 0.22020097076892853, 1.568140983581543, -0.2120458036661148, 0.04965806007385254, -0.7376015186309814, -0.11092355847358704, 2.0685744285583496, -0.14693346619606018, -0.0642213523387909, -0.8882770538330078, 0.5668491721153259, 0.816791832447052, 1.15696120262146, -0.47981634736061096, -0.6745662689208984, 1.2690201997756958, -0.025837915018200874, -0.027157265692949295, 0.7223012447357178, 0.22398890554904938, -1.1308881044387817, 0.922382116317749, 0.21909092366695404, -2.0574467182159424, -0.7202380299568176, -0.4272400140762329, -1.6312682628631592, -0.6428537368774414, -0.03276880085468292, 0.3281231224536896, 0.059970077127218246, -1.4508767127990723, 2.6595821380615234, 0.6596378087997437, 0.5055230259895325, 0.5309200882911682, -0.521687924861908, -1.4076288938522339, -0.19864284992218018, 0.20106665790081024, 0.1485731601715088, 0.5604261159896851, 1.090460181236267, -1.6225873231887817, -1.185227394104004, -1.2123740911483765, -0.14287538826465607, -0.41884875297546387, 0.06832647323608398, -0.4793623685836792, 0.42367202043533325, -0.3771705627441406, 0.5358504056930542, -1.0751312971115112, -0.15462976694107056, -0.9408121109008789, 0.5517317652702332, 0.23616020381450653, -0.293455570936203, -0.2943045198917389, 0.019552353769540787, 0.2648095488548279, -1.424153208732605, -2.477937936782837, -0.05648721009492874, 0.7093430161476135, 0.26831674575805664, -0.8508980870246887, 1.1323968172073364, -1.204554557800293, -1.5385011434555054, -0.9420133829116821, 0.542188286781311, 0.9586615562438965, -0.4668351411819458, -1.0935144424438477, -0.5600430369377136, 0.5475242137908936, 0.3658609986305237, 0.43610307574272156, 0.9139801263809204, 1.4618197679519653, 0.08562278002500534, -0.0794532299041748, -0.9460894465446472, -0.47479650378227234, -1.928501009941101, 0.03534305468201637, -1.7782341241836548, -1.2334063053131104, 0.7086160778999329, 0.16141599416732788, -0.3502286374568939, 0.18193255364894867, 0.7829709053039551, -0.6715757250785828, -1.5602309703826904, 3.1302731037139893, 1.1999797821044922, 0.23170535266399384, -1.0873098373413086, 0.9609846472740173, -0.6542973518371582, -1.119627594947815, -0.08508910983800888, -0.38308146595954895, -0.2305998057126999, 0.3165000379085541, 0.7937287092208862, -0.44971442222595215, -0.9482747912406921, 0.37491917610168457, 1.401442050933838, 0.6051344275474548, 0.5308927297592163, -2.424321413040161, 0.8395231366157532, 0.770418107509613, -1.8848494291305542, 0.20809419453144073, 2.1188535690307617, 0.8030587434768677, -1.226096510887146, 1.1239272356033325, -0.6133702993392944, -1.0142648220062256, 0.01637391746044159, -1.2323949337005615, -0.975182056427002, 2.2155046463012695, -2.0845606327056885, 1.2404117584228516, 1.4000617265701294, -0.37422120571136475, 0.7019509673118591, -0.05947740748524666, -0.1372123807668686, 0.010648838244378567, 1.5789762735366821, 1.415154218673706, -0.034286078065633774, -0.7015349864959717, -0.7448868155479431, -0.6894184350967407, 1.5360785722732544, 0.39733636379241943, 1.6213339567184448, 0.6437059640884399, 1.279622197151184, -1.6277321577072144, 1.1801104545593262, -1.4723716974258423, 0.8876132965087891, 1.2666562795639038, -1.1273006200790405, 0.701999306678772, -0.10820434987545013, 0.08032199740409851, 1.393279790878296, -0.30645951628685, -1.2733657360076904, 1.3296735286712646, 0.8834463953971863, 0.24200454354286194, -1.9533261060714722, -0.7905541658401489, 0.4576599895954132, -1.7353435754776, 1.6527961492538452, 0.5227649807929993, -0.8548425436019897, -1.1645249128341675, -0.17947348952293396, 1.0480997562408447, -2.2876763343811035, -0.433449387550354, -0.0950261577963829, 0.7321255207061768, -0.7719886898994446, 0.9442986249923706, -0.018111957237124443, -0.7550591230392456, 1.6773898601531982, -0.1680290549993515, -0.6755351424217224, -0.01133478432893753, 0.6211205124855042, 0.13898234069347382, 0.4643740653991699, 0.7891255021095276, -0.5476727485656738, -1.8372292518615723, -0.052075568586587906, -1.679836392402649, 0.12236287444829941, -0.19870200753211975, -1.520155668258667, 0.07702682167291641, 0.8067070841789246, 1.0258796215057373, 1.871594786643982, -1.084438681602478, 0.8292240500450134, 0.2339237928390503, -0.28753623366355896, 0.8050916790962219, -0.5244303941726685, 1.1573668718338013, -1.711353063583374, 0.05860935524106026, -1.199881672859192, -0.05604347959160805, -0.48670798540115356, 0.1977744996547699, -1.7229893207550049, 0.7120625972747803, -1.8053700923919678, 0.708832323551178, 0.661520779132843, 0.03671848401427269, 2.0957815647125244, -0.5563021302223206, 1.5333099365234375, 0.16184620559215546, -0.6303080320358276, 1.3821340799331665, -0.5697161555290222, -0.21212804317474365, -0.22861360013484955, 0.20780837535858154, -0.3026712238788605, -0.5049097537994385, 0.15344573557376862, -0.2730654776096344, -0.681043803691864, -0.7394797801971436, 0.8580759763717651, 0.3595867455005646, -0.5463042259216309, 0.7535814642906189, 0.07399840652942657, 1.1669539213180542, -0.6791395545005798, 0.4252035319805145, -0.4332229495048523, -0.7045774459838867, -2.104712724685669, -0.4828242063522339, 0.4678173363208771, -0.7499746084213257, -0.11358998715877533, 1.134636402130127, -1.4090512990951538, 0.6484652757644653, -1.9863054752349854, 0.7081549167633057, 0.5417935252189636, 1.7500865459442139, -0.25813916325569153, 0.3130358159542084, -1.8116217851638794, 0.9230989217758179, -1.09224534034729, -2.4940474033355713, 0.14580382406711578, 0.7452004551887512, -1.2160990238189697, -0.2841338515281677, -0.14315176010131836, 0.8995679616928101, 0.5438368916511536, -0.6116459965705872, 0.6913975477218628, 0.6102914214134216, 1.380023717880249, 0.8317825198173523, 0.844399094581604, -0.5000507831573486, -0.3054315447807312, 1.4194566011428833, -0.7809796333312988, -0.03609707951545715, 1.148540735244751, -0.7827228903770447, 0.49925416707992554, -0.00792770180851221, 2.486891031265259, 2.083360433578491, 1.066318154335022, -0.7445684671401978, 1.128781795501709, -1.3605412244796753, 0.9026252627372742, -0.4592742919921875, -1.5610450506210327, 0.4416603446006775, 0.27717286348342896, 1.0924649238586426, 0.4453706443309784, 0.7495551705360413, -0.37009501457214355, -1.2766592502593994, -0.3654617369174957, -0.32780227065086365, -0.11832985281944275, 1.545218586921692, -0.43471601605415344, 0.1806495636701584, -0.21142084896564484, 0.6488543152809143, -1.2958629131317139, 0.12964583933353424, 1.0713061094284058, 0.5835492610931396, 0.8730122447013855, -0.11535753309726715, -0.5229129195213318, -1.2422469854354858, 1.198825716972351, -2.14821457862854, 0.33142945170402527, -0.7513898015022278, 1.1046892404556274, 0.11005441844463348, 0.11035091429948807, -1.1970651149749756, 1.748967170715332, 0.9211442470550537, 2.4682059288024902, 1.4823416471481323, -0.14949621260166168, 1.2571223974227905, 0.3519109785556793, 0.5845890045166016, -0.17625723779201508, -0.3021785616874695, 1.0585111379623413, 1.3177180290222168, -1.418432593345642, -0.01606173999607563, -1.1966780424118042, -0.022009264677762985, 0.2771218419075012, -1.034251093864441, 1.7580187320709229, 1.5240283012390137, 0.34167835116386414, -0.10496275871992111, 0.768083393573761, -0.036554962396621704, -0.5587934851646423, -0.49697786569595337, 1.4584616422653198, -0.07241977751255035, -0.45716455578804016, -0.5440634489059448, 0.8486869931221008, 0.3137184679508209, 0.9482271671295166, 0.6542477011680603, 0.9472557306289673, -0.002082705730572343, -2.1762893199920654, 0.8430951237678528, -0.5278239846229553, -0.5037903189659119, -0.7067850828170776, 0.832605242729187, -0.5156419277191162, -0.25618335604667664, -2.155061721801758, -0.40895527601242065, 0.9542035460472107, -0.19625233113765717, 0.39355164766311646, 0.9318352937698364, 1.2681671380996704, -0.020871581509709358, 1.2236398458480835, 0.20851777493953705, -0.21582375466823578, 0.9442683458328247, -1.7584612369537354, 0.697783887386322, 0.7590191960334778, -0.9836622476577759, 1.1162962913513184, 0.09802883118391037, 0.1324395090341568, -1.6504253149032593, 0.21562252938747406, 0.19245220720767975, -0.641697347164154, -0.3091772794723511, 0.3870202898979187, -0.6215901374816895, 0.23426738381385803, 0.6473957300186157, 0.19798831641674042, -0.14414553344249725, -0.1719684600830078, -2.0268261432647705, -0.8903615474700928, 0.7742050290107727, -0.011490018106997013, -0.8855294585227966, 0.1899072378873825, 1.0684101581573486, 0.06763599067926407, -0.8227362632751465, -0.43021780252456665, -1.0448187589645386, 1.1578365564346313, 0.8564876914024353, -0.5444551706314087, 0.5349248647689819, -0.9545013308525085, 0.07158441096544266, -2.0629913806915283, -0.44432005286216736, 1.756797432899475, 1.2428268194198608, -0.1959114372730255, 0.4423297345638275, -0.8889158964157104, 0.3318384885787964, 0.03841913491487503, 0.2599940896034241, -0.08467300981283188, -1.4068386554718018, -0.909942090511322, -0.25046834349632263, 0.9637173414230347, -2.499584913253784, -1.0374373197555542, -0.919711709022522, -0.2388239949941635, 0.05966068431735039, -2.157559633255005, 0.882551372051239, 1.3213558197021484, 1.547763466835022, 1.4913614988327026, 1.8407312631607056, -0.48469069600105286, 0.23360490798950195, -0.2500046193599701, 0.021403564140200615, 0.8932451605796814, 0.11657141149044037, 1.594622254371643, 2.0259149074554443, -2.510436773300171, 1.0999785661697388, -0.09135712683200836, 2.0110254287719727, 1.083349347114563, -0.5102695822715759, -1.172066330909729, 0.658111572265625, 0.5630711317062378, 1.5032364130020142, 0.5958184599876404, -0.8393847346305847, 0.1517830640077591, 1.4063489437103271, -0.32552722096443176, -0.8105606436729431, 1.6118282079696655, -0.4306069612503052, 0.5698582530021667, 0.319499671459198, 2.391166925430298, 1.939727783203125, 2.1000423431396484, -1.089426875114441, 0.029925765469670296, -3.237097978591919, -1.2415090799331665, -1.035822868347168, 0.35897842049598694, -0.6894528865814209, -1.8447669744491577, 1.3842267990112305, 0.05331944301724434, -0.3838063180446625, -1.3463939428329468, 0.12829647958278656, -0.06354330480098724, 0.9389973878860474, -1.3487437963485718, -0.20119349658489227, -0.5453379154205322, -1.3120907545089722, -0.06286722421646118, -0.28020310401916504, -1.661022424697876, 0.6705498695373535, -0.11133460700511932, -0.5132566690444946, -1.0536648035049438, 1.2347701787948608, -0.1316789984703064, 0.5732832551002502, 0.8519999980926514, -1.0469751358032227, 0.5348936319351196, -0.3932737708091736, -0.308383971452713, -1.8786742687225342, 0.13592813909053802, 0.23692595958709717, -0.7138814330101013, -1.2337130308151245, -1.2072854042053223, 0.1382429301738739, 0.4027647078037262, -0.48216959834098816, -1.1589651107788086, 0.41897305846214294, 0.3565202057361603, 0.7240332961082458, 0.23371078073978424, 0.44661104679107666, 0.4232490658760071, -2.3353428840637207, -1.4565515518188477, 1.4148139953613281, 0.3213365375995636, 1.5128859281539917, 0.7221654057502747, 0.1881287842988968, -0.2084309458732605, 0.5086467862129211, 1.2553120851516724, 0.03273915499448776, 0.4585888385772705, -0.4154229164123535, -0.2843814790248871, 0.10819333046674728, 0.5538269281387329, 0.8146688938140869, 0.5393084287643433, -0.7363662123680115, 0.0014896474312990904, -0.2859335243701935, 0.7677155137062073, 1.2323129177093506, 1.731413722038269, 0.3587796986103058, 0.4814854562282562, 0.11059239506721497, 1.0226233005523682, -0.6950884461402893, 0.5710785984992981, -0.28966471552848816, -0.9087542295455933, 1.354811429977417, -0.6004570722579956, -0.3725983500480652, -0.3823675215244293, 0.5808518528938293, 0.0542878732085228, 0.5156216025352478, -0.12940675020217896, 1.1257774829864502, -0.7015636563301086, 0.9896633625030518, -0.03017430752515793, 1.2266809940338135, -0.057797010987997055, -1.5641816854476929, -1.4538809061050415, -0.15181460976600647, 0.00856236182153225, -0.5070018768310547, -0.5758419036865234, 0.051470428705215454, -0.7030662894248962, -1.461174488067627, 0.4768659472465515, -0.6257140040397644, -0.2787543833255768, -2.3807389736175537, -0.3405193090438843, 0.1138676255941391, 0.7679755091667175, 0.3494984209537506, 0.09853767603635788, 1.107571005821228, -1.252393364906311, 1.3363733291625977, 0.6877326965332031, -0.8276675939559937, 0.7465289831161499, 0.7015155553817749, -0.25680118799209595, -1.7247508764266968, -1.4337880611419678, -0.5965878367424011, -0.04603644460439682, -0.26955074071884155, 1.3205194473266602, -0.9591459631919861, -0.07210661470890045, -1.4378461837768555, -0.5291209816932678, 0.21893540024757385, -0.5559883117675781, 1.250417709350586, 0.19884736835956573, 1.1518583297729492, 0.8827476501464844, 0.37402522563934326, 0.014726463705301285, 0.2600557506084442, -0.024775667116045952, -1.107936978340149, -0.3437410295009613, 0.02297825925052166, 0.5949034094810486, 0.3685712218284607, 1.1808360815048218, -1.2164233922958374, 2.0662827491760254, -0.6945061087608337, 0.028360804542899132, -0.33831945061683655, -0.8323983550071716, -0.9560925364494324, -0.044615983963012695, 0.9452469348907471, -0.410899817943573, -1.5000317096710205, 1.620188593864441, 0.7094306945800781, -0.12925086915493011, 1.6813782453536987, 0.6914003491401672, 0.3524070680141449, 3.1934125423431396, -0.6605128645896912, 0.6267614364624023, 0.23094087839126587, -0.7421813607215881, -1.3822264671325684, -0.8329029679298401, 0.23070353269577026, 1.6225011348724365, -0.713153600692749, 0.8700149059295654, -0.2100970447063446, -0.18142203986644745, -0.6496198177337646, -1.5872172117233276, -1.0488898754119873, -0.9381593465805054, -2.842759847640991, 0.8368681073188782, 0.028677061200141907, -0.18406642973423004, 1.8338898420333862, -2.1780264377593994, 0.11727269738912582, 0.8581454753875732, 0.19377568364143372, 0.09343480318784714, -0.16093988716602325, -0.7481544017791748, -2.0153238773345947, -2.4516570568084717, 1.203567624092102, 0.772503137588501, -1.3143556118011475, 1.712141990661621, 0.19064664840698242, -0.09328147768974304, -0.05452299118041992, -1.1874756813049316, 0.4720016121864319, -1.3606352806091309, -0.6370797753334045, -0.4455939531326294, -0.8042545318603516, -0.9968996047973633, -0.1335381418466568, 1.346902847290039, -0.522664487361908, 2.339191436767578, -1.453940749168396, 1.7359800338745117, 0.09634184837341309, -1.9976578950881958, 0.2555497884750366, -0.15037421882152557, 0.31652122735977173, -0.14571528136730194, -0.5591779947280884, -0.07957256585359573, 0.5704148411750793, -1.1486917734146118, -0.03263389691710472, 0.283066064119339, -1.1961253881454468, 1.7360948324203491, 0.5587741732597351, -0.22713139653205872, -2.375276803970337, -0.0750146433711052, 2.6668906211853027, 0.5919290781021118, -0.5628178715705872, -0.5953052639961243, -1.1277886629104614, 0.5829381942749023, 0.5790450572967529, -0.49859076738357544, -0.6439550518989563, -0.09932250529527664, -0.8882065415382385, 1.0022448301315308, 1.4307416677474976, -0.30198681354522705, 1.9124114513397217, 1.451066017150879, 0.3010737895965576, -0.40974950790405273, 1.393506646156311, -1.6219522953033447, 0.031126560643315315, -0.818152666091919, -0.5392107367515564, 1.1037770509719849, -1.3988429307937622, 0.3470095098018646, 1.547764539718628, -0.7479811906814575, -0.9787521958351135, 2.14506196975708, -1.9010043144226074, -0.03685692697763443, 0.22796276211738586, -1.0232881307601929, 0.2882900834083557, -1.078616976737976, -0.5597847700119019, 0.2355152666568756, -0.10466021299362183, 1.018897294998169, 1.1582082509994507, 1.2240198850631714, -2.305635929107666, -0.38370293378829956, -0.4294619858264923, 0.38523873686790466, -1.0892313718795776, -1.3606318235397339, 1.0985685586929321, -1.5069268941879272, -0.4109344184398651, -1.6072925329208374, 0.8558081984519958, -1.0988731384277344, 0.06516758352518082, -0.03437710925936699, 1.5303434133529663, 0.3795037269592285, -1.0078685283660889, -2.1212127208709717, 0.12244363129138947, 0.898246169090271, 2.5808053016662598, -0.5898218750953674, -1.0552802085876465, -0.9017239809036255, 1.8048564195632935, 0.3096015453338623, -0.5847292542457581, 1.2084745168685913, 0.666142463684082, -1.80678391456604, -2.346839427947998, 2.1404495239257812, 0.22265495359897614, -1.0631961822509766, -0.5561016201972961, 1.020684838294983, -0.8692542314529419, 1.166593074798584, 0.15492203831672668, -0.07452018558979034, -0.21600651741027832, -0.907005250453949, -2.2129738330841064, 1.2022264003753662, -0.4867984652519226, -0.49970942735671997, -0.8095400929450989, -0.9354225993156433, -0.6526533961296082, -2.286027431488037, -0.34734123945236206, -0.07636955380439758, -1.228002905845642, 0.2995317578315735, -0.053864043205976486, -0.8011204600334167, -0.7998402714729309, 0.4595440924167633, -0.5411890149116516, -1.1852163076400757, 0.7220003008842468, -0.19557823240756989, -0.06289451569318771, -0.9077610373497009, -0.039553362876176834, 1.0096153020858765, 0.11880114674568176, 0.3389618992805481, 0.09369229525327682, 0.11920144408941269, -2.2166197299957275, 0.571575939655304, 0.9186254143714905, -0.01948355883359909, 0.7101342678070068, -0.2000073790550232, 1.0655289888381958, -1.45989990234375, 0.4449117183685303, 1.0168888568878174, 1.100903034210205, 0.8990696668624878, 0.6015706062316895, -0.569839596748352, -0.8413155674934387, 0.6515614986419678, -0.46474340558052063, 0.43897736072540283, -1.1306158304214478, -0.018131013959646225, -1.247489333152771, -1.2351044416427612, 0.19654186069965363, 1.4210022687911987, 0.15847769379615784, 1.080812931060791, 0.8133547902107239, 0.4611338973045349, -0.28154513239860535, -0.0057786270044744015, 1.0169602632522583, 1.3641228675842285, 0.261398583650589, 0.6063838005065918, 1.656334638595581, 0.37323710322380066, -0.6746522188186646, -0.6111969351768494, -1.1284406185150146, -1.5450615882873535, -0.3589199185371399, 0.08049242943525314, 0.2774556577205658, 0.5133093595504761, -0.0071625555865466595, 1.681113362312317, -0.8536428213119507, -0.06854093819856644, 1.817050814628601, 1.148462176322937, -0.26195311546325684, 2.700927495956421, 1.128654956817627, -0.7636023163795471, -0.4000815749168396, 0.9187595844268799, -1.0967029333114624, 1.2058603763580322, 0.8721967339515686, -0.0006915118428878486, 2.3346402645111084, -1.1848078966140747, -0.8852417469024658, 1.0132436752319336, -0.8653731942176819, 1.3483718633651733, -0.7727973461151123, 1.400608777999878, 0.9723018407821655, 0.3057946562767029, -0.631202757358551, -0.7050416469573975, -0.02699524164199829, -2.0698177814483643, -0.4086543917655945, -0.37402039766311646, -0.18614168465137482, -0.9203819632530212, 1.0653752088546753, 0.8015848398208618, -0.4272143542766571, -0.3788924515247345, -1.8399760723114014, -0.36493492126464844, -0.15247312188148499, 1.1425548791885376, 1.3177462816238403, 0.7021995186805725, 0.04053781181573868, 0.9344161748886108, 1.1478631496429443, -0.6211599111557007, -0.7119845747947693, 0.5744881629943848, 0.2982548475265503, 0.0699075236916542, -0.07626360654830933, -1.1729828119277954, -0.00889402162283659, -0.5172264575958252, -0.31898272037506104, -0.5373129844665527, 1.6840155124664307, 0.9934906959533691, 1.4300771951675415, 0.027159690856933594, 1.69199800491333, -0.13598187267780304, -0.13120831549167633, 1.1722607612609863, 0.6599289178848267, 0.15668673813343048, 0.5663874745368958, 1.5526169538497925, 1.3087997436523438, -0.1378554254770279, 0.3017064332962036, -0.7815080881118774, -0.6397339105606079, 1.2991243600845337, 2.302767515182495, 1.2152118682861328, -0.4426921308040619, 0.9759657979011536, -0.15993165969848633, 0.5114891529083252, 0.6934341788291931, -0.8605181574821472, 0.078328438103199, 1.1562490463256836, -1.0534783601760864, 0.09976594150066376, 1.6745444536209106, 1.1166481971740723, -1.0369526147842407, -0.6111561059951782, -0.49639585614204407, 1.0230598449707031, -1.7828755378723145, -0.044819727540016174, 0.8330972194671631, -0.5081535577774048, 0.4921925961971283, -0.2717403471469879, -1.664930820465088, -1.478983759880066, -0.3359341323375702, 0.2919256091117859, 0.31349557638168335, -0.12405520677566528, 0.06165898218750954, -0.14662940800189972, -1.4545472860336304, 0.6847374439239502, 0.2328488677740097, 0.8717255592346191, 0.9553869366645813, 0.42307719588279724, 1.2786762714385986, 0.5301470160484314, 0.9403446912765503, 1.5291268825531006, -1.4496841430664062, -0.3073219060897827, 0.032451044768095016, 0.11779672652482986, -1.487315058708191, -0.6999922394752502, -1.1227694749832153, 0.995864987373352, 1.1141051054000854, -1.1550238132476807, -0.711844801902771, -0.336789608001709, 0.15347424149513245, 1.8503177165985107, 0.3265121579170227, -1.166179895401001, 0.5598036050796509, 0.4208827018737793, 1.7190502882003784, -0.35265442728996277, -2.0046067237854004, -0.35208696126937866, -1.6630604267120361, 0.4189176559448242, -1.7192819118499756, -0.35066813230514526, -0.6714525818824768, 1.0108232498168945, 0.2513614296913147, 0.6211129426956177, 0.9662559628486633, -0.35268843173980713, -0.6183351278305054, -0.02415807731449604, 0.8801090717315674, -0.4323929250240326, 0.8845801949501038, -1.1226271390914917, -0.438388466835022, -0.44004005193710327, -1.3304346799850464, 2.8226332664489746, 1.9629794359207153, -2.3611128330230713, 0.626083254814148, -0.4053053557872772, 0.029835712164640427, 1.721416711807251, -1.1703336238861084, -0.8166236281394958, 2.3906311988830566, 0.38040152192115784, -2.1043806076049805, -0.19528444111347198, 0.48794955015182495, 0.2167923003435135, -0.3057870864868164, 1.7521730661392212, 1.7266849279403687, -0.6705009937286377, 0.036823879927396774, -0.006877924781292677, -0.17251266539096832, 0.4004230797290802, -0.03710700199007988, 0.760823130607605, -0.7045655250549316, 0.16393235325813293, 0.7780940532684326, 1.2976521253585815, 0.200094074010849, -0.7134799361228943, 0.05976198986172676, -0.06215853616595268, -0.44169294834136963, 1.0592342615127563, -0.9932650923728943, 0.7908799052238464, 1.2576022148132324, -0.9270119667053223, -2.506868839263916, -1.2900110483169556, 0.3879116475582123, 0.2481730580329895, 0.8019136190414429, 1.4687942266464233, 0.088346928358078, -1.1230177879333496, 1.7948241233825684, -0.15216708183288574, -1.162682056427002, -0.7705379128456116, -3.203611135482788, 0.9346267580986023, -1.3929674625396729, -0.16429947316646576, 0.482909232378006, -1.1580575704574585, -0.0644911453127861, -0.6113926768302917, 0.4281253218650818, 0.4809589087963104, -1.6169774532318115, -1.6387690305709839, 0.6629692912101746, -0.9192995429039001, 0.8788498044013977, 0.5492196679115295, 2.322754383087158, 1.8896843194961548, 1.7840583324432373, 0.013028088957071304, -1.2449047565460205, -0.029295388609170914, 0.8015261888504028], "dtype": "float32"}], "e": {"__nd__": 281.4315185546875, "dtype": "float32"}}, {"a": [{"__nd__": [[0.3990272879600525, 0.3062388002872467, 0.04675423353910446, 0.25270676612854004, 0.8815069794654846, 0.15810969471931458, 0.7652770280838013, 0.29167571663856506, 0.2550857961177826, 0.21781101822853088, 0.6917876601219177, 0.31558549404144287, 0.6421387195587158, 0.5617464184761047, 0.12977658212184906, 0.1189427450299263, 0.6999958157539368, 0.39764147996902466, 0.341569721698761, 0.7488930225372314, 0.20334921777248383, 0.5295682549476624, 0.5626378059387207, 0.8853965401649475, 0.7735175490379333, 0.09581899642944336, 0.9205141067504883, 0.7747105956077576, 0.7992249727249146, 0.4180350601673126, 0.08316389471292496, 0.5609261393547058, 0.4928288459777832], [0.1633743792772293, 0.7770113348960876, 0.10635756701231003, 0.4176754057407379, 0.09209489077329636, 0.03515830263495445, 0.98354172706604, 0.7143140435218811, 0.2102736532688141, 0.9471349120140076, 0.8399497270584106, 0.04761604219675064, 0.9382888078689575, 0.10304886847734451, 0.7420282959938049, 0.9214446544647217, 0.880043089389801, 0.5649855732917786, 0.5794798135757446, 0.673119843006134, 0.9618493914604187, 0.2957971394062042, 0.3199900984764099, 0.6708024740219116, 0.14290834963321686, 0.9639161825180054, 0.4753570854663849, 0.9565082788467407, 0.1890970766544342, 0.14488941431045532, 0.5186076164245605, 0.4428238868713379, 0.6106193661689758], [0.3216562271118164, 0.34985223412513733, 0.875204861164093, 0.5843868255615234, 0.01663297973573208, 0.7566039562225342, 0.3547980785369873, 0.9912065863609314, 0.9706754088401794, 0.015017526224255562, 0.58090740442276, 0.8890866637229919, 0.9228233098983765, 0.24785764515399933, 0.9248365163803101, 0.33551594614982605, 0.036599669605493546, 0.2794024348258972, 0.09240607917308807, 0.3932205140590668, 0.7849104404449463, 0.3833346962928772, 0.9702295660972595, 0.7157942652702332, 0.334084689617157, 0.5664647221565247, 0.5484068393707275, 0.7055861353874207, 0.6979789137840271, 0.03166814520955086, 0.9696512818336487, 0.23763252794742584, 0.5692977905273438], [0.9939958453178406, 0.654061496257782, 0.44841787219047546, 0.5158318281173706, 0.9996006488800049, 0.3407192528247833, 0.5352590680122375, 0.6373739838600159, 0.26377490162849426, 0.29601967334747314, 0.5106828212738037, 0.6125268340110779, 0.3662007749080658, 0.046888191252946854, 0.05346290394663811, 0.9058173298835754, 0.4975641667842865, 0.27002957463264465, 0.17305997014045715, 0.9970057010650635, 0.5799113512039185, 0.2742607593536377, 0.6751596331596375, 0.5561946034431458, 0.9845550060272217, 0.5617810487747192, 0.11411811411380768, 0.2069765329360962, 0.036968767642974854, 0.48544833064079285, 0.955186665058136, 0.6940577030181885, 0.41662701964378357], [0.10033910721540451, 0.67352294921875, 0.007197883911430836, 0.17562468349933624, 0.04405232518911362, 0.879305362701416, 0.13741952180862427, 0.3737458288669586, 0.2858574390411377, 0.0858912467956543, 0.5118589401245117, 0.7087975740432739, 0.16711924970149994, 0.9897680878639221, 0.35677969455718994, 0.882945716381073, 0.6178689002990723, 0.5239418745040894, 0.44715505838394165, 0.808377742767334, 0.5079166889190674, 0.08958616852760315, 0.9925274848937988, 0.18802499771118164, 0.9727848172187805, 0.8334504961967468, 0.5614087581634521, 0.4093920886516571, 0.4007583558559418, 0.7061552405357361, 0.9975129961967468, 0.6400640606880188, 0.5743390917778015], [0.8928200006484985, 0.9130344986915588, 0.41667237877845764, 0.12802384793758392, 0.0018126684008166194, 0.9281905293464661, 0.22953394055366516, 0.788230836391449, 0.9423475861549377, 0.27295607328414917, 0.01596924476325512, 0.5106698870658875, 0.1562882661819458, 0.6669502854347229, 0.8207841515541077, 0.9900029897689819, 0.026679864153265953, 0.5327674150466919, 0.44559183716773987, 0.627137303352356, 0.6878668665885925, 0.4255930483341217, 0.591064453125, 0.3869636058807373, 0.6629732847213745, 0.7121336460113525, 0.1339518129825592, 0.5275132060050964, 0.5789439082145691, 0.8516265749931335, 0.46972528100013733, 0.12901127338409424, 0.016522319987416267], [0.19825565814971924, 0.6255425810813904, 0.513286828994751, 0.7697482705116272, 0.6061617136001587, 0.24166160821914673, 0.4167367219924927, 0.20340831577777863, 0.5118387341499329, 0.5756394863128662, 0.7334921360015869, 0.050924163311719894, 0.2745220959186554, 0.22894765436649323, 0.18133962154388428, 0.9584639072418213, 0.7124799489974976, 0.04429268836975098, 0.6014539003372192, 0.6345425844192505, 0.7836930751800537, 0.7762884497642517, 0.7565381526947021, 0.026872629299759865, 0.42933574318885803, 0.26557037234306335, 0.6987773180007935, 0.7120213508605957, 0.007315249647945166, 0.7286462187767029, 0.10538280755281448, 0.7756533622741699, 0.503460168838501], [0.42201095819473267, 0.37928855419158936, 0.6505798697471619, 0.3221953511238098, 0.8315542936325073, 0.5003508925437927, 0.659821629524231, 0.8158727884292603, 0.4338783025741577, 0.3795986771583557, 0.36728522181510925, 0.5409858226776123, 0.40879279375076294, 0.2732372283935547, 0.8622263073921204, 0.6343907713890076, 0.2730235755443573, 0.3289619982242584, 0.6077232360839844, 0.2594709098339081, 0.5760757923126221, 0.10790073871612549, 0.9158404469490051, 0.007778265047818422, 0.6707878112792969, 0.22594130039215088, 0.33007869124412537, 0.4124752879142761, 0.6898934841156006, 0.05681953206658363, 0.3217563331127167, 0.685387372970581, 0.033739108592271805], [0.5004889965057373, 0.6083292365074158, 0.5487381815910339, 0.4392488896846771, 0.7436261177062988, 0.7181880474090576, 0.3126705586910248, 0.7398322224617004, 0.9395053386688232, 0.6482237577438354, 0.16438494622707367, 0.14748744666576385, 0.27342432737350464, 0.5503707528114319, 0.7401224970817566, 0.4763791561126709, 0.0008326715906150639, 0.5611278414726257, 0.9582939743995667, 0.7592170834541321, 0.8406038284301758, 0.26153939962387085, 0.3804982602596283, 0.39828673005104065, 0.12028712779283524, 0.8599295020103455, 0.8787131309509277, 0.3103860318660736, 0.9153423309326172, 0.48986300826072693, 0.9839814901351929, 0.5782027244567871, 0.4709407389163971], [0.05829155072569847, 0.6613546013832092, 0.9369994401931763, 0.07617276906967163, 0.34964630007743835, 0.70937180519104, 0.18906182050704956, 0.2872121036052704, 0.43115073442459106, 0.2679007351398468, 0.5477223992347717, 0.5810788869857788, 0.32502081990242004, 0.37520939111709595, 0.8359323740005493, 0.1698089838027954, 0.4529430866241455, 0.9514951705932617, 0.567253589630127, 0.7822505831718445, 0.4600885212421417, 0.0952124372124672, 0.3621920347213745, 0.35100364685058594, 0.8914441466331482, 0.10818514227867126, 0.3205590546131134, 0.04763704538345337, 0.24425385892391205, 0.6079097390174866, 0.7505136132240295, 0.2202935665845871, 0.837562620639801], [0.05653274804353714, 0.3839651048183441, 0.19822730123996735, 0.4099116027355194, 0.45623108744621277, 0.43955323100090027, 0.3923400044441223, 0.6627179980278015, 0.813501238822937, 0.718079686164856, 0.4092647433280945, 0.5649360418319702, 0.6471803188323975, 0.8184647560119629, 0.5076162815093994, 0.6886765956878662, 0.5986995100975037, 0.4063500165939331, 0.5676659345626831, 0.078987717628479, 0.7227556109428406, 0.8931309580802917, 0.7735700011253357, 0.10128655284643173, 0.6194834113121033, 0.8964682221412659, 0.9911834597587585, 0.8289189338684082, 0.3005058765411377, 0.9069215655326843, 0.32973694801330566, 0.7890350222587585, 0.08943361043930054], [0.7655207514762878, 0.251391738653183, 0.9659703969955444, 0.5536219477653503, 0.9273802638053894, 0.008901470340788364, 0.53065025806427, 0.9170518517494202, 0.16596843302249908, 0.9605372548103333, 0.20750465989112854, 0.00489243445917964, 0.8030276894569397, 0.5497444868087769, 0.2955378592014313, 0.29137033224105835, 0.8007858395576477, 0.7971129417419434, 0.40366020798683167, 0.8096796274185181, 0.039469145238399506, 0.25876158475875854, 0.3058973550796509, 0.9239054322242737, 0.8450422286987305, 0.548126220703125, 0.38875851035118103, 0.5963591933250427, 0.7769040465354919, 0.8603604435920715, 0.30594733357429504, 0.12723110616207123, 0.4174362123012543], [0.5309644937515259, 0.008859196677803993, 0.7955297827720642, 0.30467990040779114, 0.5069310069084167, 0.5441827178001404, 0.08582843840122223, 0.06496811658143997, 0.7913726568222046, 0.2038612812757492, 0.6000251173973083, 0.4852293133735657, 0.26727303862571716, 0.07286199927330017, 0.40180400013923645, 0.5276394486427307, 0.010079625993967056, 0.49008801579475403, 0.8497461676597595, 0.950403094291687, 0.5880306363105774, 0.8432721495628357, 0.7161116003990173, 0.38660210371017456, 0.2899303734302521, 0.12602601945400238, 0.9255329966545105, 0.09224354475736618, 0.8740530014038086, 0.3562399744987488, 0.31244221329689026, 0.7251743078231812, 0.4694903492927551], [0.22963887453079224, 0.14867614209651947, 0.9230905771255493, 0.17166322469711304, 0.6246404051780701, 0.24535134434700012, 0.1596163958311081, 0.9996187686920166, 0.16587649285793304, 0.5463323593139648, 0.8472187519073486, 0.012542903423309326, 0.495470255613327, 0.7367487549781799, 0.7850443720817566, 0.26783910393714905, 0.9473314881324768, 0.22042354941368103, 0.08335889130830765, 0.20005224645137787, 0.9357821941375732, 0.8957774639129639, 0.32505249977111816, 0.8851833939552307, 0.3138483762741089, 0.5730559229850769, 0.5956702828407288, 0.880956768989563, 0.9190585613250732, 0.9514314532279968, 0.6178256273269653, 0.26565298438072205, 0.3117975890636444], [0.36332640051841736, 0.7618978023529053, 0.1494656354188919, 0.17102061212062836, 0.5691424012184143, 0.5481930375099182, 0.3305155038833618, 0.41504859924316406, 0.26237696409225464, 0.9323118925094604, 0.9852221608161926, 0.6302563548088074, 0.24420079588890076, 0.5713974237442017, 0.45041242241859436, 0.4712289869785309, 0.044483732432127, 0.6012251377105713, 0.7631343603134155, 0.010814514942467213, 0.7918308973312378, 0.4524592161178589, 0.7568826079368591, 0.010608887299895287, 0.002047855406999588, 0.6336851119995117, 0.5817386507987976, 0.06720864027738571, 0.7928131222724915, 0.2051306813955307, 0.6504044532775879, 0.11099399626255035, 0.5627517700195312], [0.0939374640583992, 0.016177982091903687, 0.9899014830589294, 0.8453205823898315, 0.01981615461409092, 0.018060453236103058, 0.5573729276657104, 0.3512902855873108, 0.5654992461204529, 0.3699561655521393, 0.7611066102981567, 0.9613543748855591, 0.7671500444412231, 0.06484295427799225, 0.6307486295700073, 0.5424960255622864, 0.051905304193496704, 0.07480774819850922, 0.1682554930448532, 0.1181151494383812, 0.5582788586616516, 0.8566848039627075, 0.6871163249015808, 0.5647446513175964, 0.6127524375915527, 0.1636948138475418, 0.9985334873199463, 0.9628602862358093, 0.6848658919334412, 0.47025570273399353, 0.3004048466682434, 0.189161479473114, 0.6411146521568298], [0.3202362656593323, 0.5235995054244995, 0.007603982929140329, 0.44210484623908997, 0.47623419761657715, 0.17157217860221863, 0.7660043239593506, 0.13468559086322784, 0.41756394505500793, 0.9922428727149963, 0.6793280243873596, 0.18822209537029266, 0.03764866292476654, 0.9828029274940491, 0.11010854691267014, 0.398998886346817, 0.49133890867233276, 0.09511003643274307, 0.6794543266296387, 0.9549739360809326, 0.6574270725250244, 0.8593769073486328, 0.3526182472705841, 0.70645672082901, 0.003768960712477565, 0.08043170720338821, 0.46824580430984497, 0.6317545175552368, 0.6525483727455139, 0.5902758240699768, 0.22566868364810944, 0.22657085955142975, 0.20465728640556335], [0.7063688039779663, 0.79514080286026, 0.35080698132514954, 0.07625497132539749, 0.423146516084671, 0.6170144081115723, 0.2922305166721344, 0.23832900822162628, 0.5003860592842102, 0.8634569644927979, 0.6651869416236877, 0.33447274565696716, 0.9864543676376343, 0.6209191679954529, 0.5455031394958496, 0.7232100367546082, 0.9292205572128296, 0.3977711796760559, 0.6591258645057678, 0.12174702435731888, 0.2136576771736145, 0.9326363801956177, 0.4531734883785248, 0.8787677884101868, 0.4800940752029419, 0.8478281497955322, 0.1301291584968567, 0.5048739314079285, 0.7276242971420288, 0.9740216732025146, 0.21763615310192108, 0.7516530156135559, 0.39333227276802063], [0.9759794473648071, 0.5527073740959167, 0.6189295053482056, 0.6750281453132629, 0.5125730633735657, 0.9524143934249878, 0.05434933677315712, 0.7411561012268066, 0.14935486018657684, 0.6576022505760193, 0.9346913695335388, 0.572304368019104, 0.21762238442897797, 0.03349748253822327, 0.7026915550231934, 0.4137159287929535, 0.17441801726818085, 0.5299738049507141, 0.19983233511447906, 0.4577040672302246, 0.08735685795545578, 0.8605007529258728, 0.6329854726791382, 0.05937660485506058, 0.5493021607398987, 0.18051691353321075, 0.5401993989944458, 0.3226674497127533, 0.8030288219451904, 0.3828786015510559, 0.42291513085365295, 0.5791674256324768, 0.0717819482088089], [0.08913568407297134, 0.8354327082633972, 0.4671407639980316, 0.09142966568470001, 0.49449747800827026, 0.4026394784450531, 0.6400197148323059, 0.4910537600517273, 0.21367190778255463, 0.9593689441680908, 0.4842849373817444, 0.533692479133606, 0.08914205431938171, 0.2512255012989044, 0.6717414259910583, 0.11857149004936218, 0.6879104971885681, 0.47192737460136414, 0.3587906062602997, 0.1748117357492447, 0.9537899494171143, 0.6430096626281738, 0.8857066035270691, 0.45362138748168945, 0.2870468199253082, 0.44335657358169556, 0.42190325260162354, 0.5933530330657959, 0.12160912156105042, 0.7682849764823914, 0.76593017578125, 0.307849258184433, 0.2411133348941803], [0.5283865332603455, 0.6067095994949341, 0.9068922996520996, 0.011109081096947193, 0.14239215850830078, 0.5939083695411682, 0.42128926515579224, 0.28713133931159973, 0.7621514201164246, 0.18433108925819397, 0.7253497242927551, 0.9336585998535156, 0.3678719699382782, 0.8272494673728943, 0.34064745903015137, 0.684252917766571, 0.6342863440513611, 0.022223422303795815, 0.8521581888198853, 0.7867536544799805, 0.5349091291427612, 0.9800954461097717, 0.4040859341621399, 0.9657274484634399, 0.8406163454055786, 0.5575148463249207, 0.5876502394676208, 0.02344351075589657, 0.16581222414970398, 0.5690093636512756, 0.2787710428237915, 0.04053356498479843, 0.08201095461845398], [0.3060152530670166, 0.33074724674224854, 0.5604969263076782, 0.327473521232605, 0.6570689678192139, 0.025812340900301933, 0.4736934304237366, 0.2718712091445923, 0.033306561410427094, 0.7017632722854614, 0.6714295148849487, 0.7685577869415283, 0.7893276214599609, 0.6589659452438354, 0.7705581784248352, 0.615669846534729, 0.21071217954158783, 0.7229281067848206, 0.39795374870300293, 0.1893908530473709, 0.23976904153823853, 0.23389291763305664, 0.39542946219444275, 0.7455685138702393, 0.2090873420238495, 0.40377721190452576, 0.8037723302841187, 0.9918278455734253, 0.386531800031662, 0.03641761466860771, 0.8104153275489807, 0.7348771095275879, 0.8946640491485596], [0.6843320727348328, 0.8774188160896301, 0.9438649415969849, 0.20337216556072235, 0.8488933444023132, 0.6095830798149109, 0.6877322793006897, 0.31289246678352356, 0.29558858275413513, 0.3380484879016876, 0.8872112035751343, 0.1779056042432785, 0.1589137315750122, 0.13479013741016388, 0.36190515756607056, 0.8469228148460388, 0.6952295303344727, 0.5404502153396606, 0.9931246042251587, 0.1697617471218109, 0.6826705932617188, 0.7002599239349365, 0.6686615943908691, 0.38997453451156616, 0.9762033224105835, 0.2183426320552826, 0.24664533138275146, 0.057731226086616516, 0.8883318901062012, 0.004233242012560368, 0.4465242624282837, 0.9545079469680786, 0.42398104071617126], [0.22169752418994904, 0.8043429255485535, 0.14505517482757568, 0.7280899286270142, 0.3518085479736328, 0.7059710621833801, 0.7060559988021851, 0.10570301860570908, 0.7449948787689209, 0.28369003534317017, 0.2248048633337021, 0.4190531373023987, 0.2668091058731079, 0.7461705803871155, 0.866722822189331, 0.9644052386283875, 0.14188170433044434, 0.7694505453109741, 0.5502961874008179, 0.5635251402854919, 0.9433279037475586, 0.25393107533454895, 0.4423283636569977, 0.6442030668258667, 0.872798502445221, 0.4824203848838806, 0.8412512540817261, 0.23030202090740204, 0.8728878498077393, 0.6293946504592896, 0.017195751890540123, 0.6028148531913757, 0.08719237893819809], [0.8500202298164368, 0.6252431273460388, 0.5345199704170227, 0.24207493662834167, 0.6443270444869995, 0.9846641421318054, 0.41656580567359924, 0.5573058724403381, 0.7276405692100525, 0.5764711499214172, 0.7491207718849182, 0.09912100434303284, 0.8746764063835144, 0.4746634066104889, 0.42749130725860596, 0.855324387550354, 0.05529012903571129, 0.0871347039937973, 0.23131802678108215, 0.07483342289924622, 0.3541586399078369, 0.17770764231681824, 0.44869518280029297, 0.2807503640651703, 0.031646180897951126, 0.43923571705818176, 0.12618185579776764, 0.07768992334604263, 0.4808835983276367, 0.8526893258094788, 0.3654816150665283, 0.8904410004615784, 0.7511177659034729], [0.7348195314407349, 0.6804242730140686, 0.3526045083999634, 0.5036563277244568, 0.19250783324241638, 0.9631514549255371, 0.9905397295951843, 0.45096153020858765, 0.7942067384719849, 0.5130549669265747, 0.027826815843582153, 0.3852795958518982, 0.9715324640274048, 0.8480288982391357, 0.8369308710098267, 0.4183993935585022, 0.5413814187049866, 0.8722326755523682, 0.9184820652008057, 0.787706196308136, 0.9341581463813782, 0.42494043707847595, 0.9166451096534729, 0.7932616472244263, 0.32198062539100647, 0.13758668303489685, 0.08841151744127274, 0.2391647845506668, 0.2967931628227234, 0.7539857625961304, 0.38526496291160583, 0.029223309829831123, 0.003103388473391533], [0.07946065813302994, 0.7239371538162231, 0.33478057384490967, 0.17881357669830322, 0.6883598566055298, 0.1807858794927597, 0.6224673986434937, 0.502160370349884, 0.6861979961395264, 0.23497213423252106, 0.3877786099910736, 0.2571510672569275, 0.4885972738265991, 0.7971978187561035, 0.3487890660762787, 0.9038094878196716, 0.3668769598007202, 0.31441381573677063, 0.03219660744071007, 0.3338056802749634, 0.2780141830444336, 0.6336434483528137, 0.03230200707912445, 0.33087611198425293, 0.34895527362823486, 0.5671747326850891, 0.054474081844091415, 0.19022272527217865, 0.7056068181991577, 0.6650605201721191, 0.7706084251403809, 0.9434875845909119, 0.9131064414978027], [0.1699998378753662, 0.2827513515949249, 0.6718708276748657, 0.14145714044570923, 0.48991575837135315, 0.7191883325576782, 0.6444776654243469, 0.9881539940834045, 0.6905325055122375, 0.17943736910820007, 0.26430028676986694, 0.7149980068206787, 0.9485820531845093, 0.715474009513855, 0.058026108890771866, 0.19685643911361694, 0.2668207883834839, 0.6211011409759521, 0.4836607277393341, 0.7965341210365295, 0.8986937403678894, 0.10715240985155106, 0.009067177772521973, 0.4431782364845276, 0.15360523760318756, 0.35142043232917786, 0.024108370766043663, 0.8702331781387329, 0.08233216404914856, 0.22752849757671356, 0.643526554107666, 0.9785332679748535, 0.9203817844390869], [0.5862040519714355, 0.9331385493278503, 0.1992940604686737, 0.9419175386428833, 0.4893830418586731, 0.07049804925918579, 0.12367913872003555, 0.4408140182495117, 0.37401360273361206, 0.8201822638511658, 0.792587161064148, 0.8924868106842041, 0.8835654258728027, 0.5300081968307495, 0.634220540523529, 0.17929907143115997, 0.9745869636535645, 0.3756987154483795, 0.09399837255477905, 0.9585312008857727, 0.13186347484588623, 0.18180882930755615, 0.23773261904716492, 0.33317652344703674, 0.36716052889823914, 0.26885807514190674, 0.6074862480163574, 0.5967057347297668, 0.6589207649230957, 0.029154054820537567, 0.7918145060539246, 0.9716468453407288, 0.4873449206352234], [0.2487349659204483, 0.5972122550010681, 0.3199276924133301, 0.41751059889793396, 0.479507178068161, 0.8672856688499451, 0.7305540442466736, 0.817719578742981, 0.46448883414268494, 0.052361443638801575, 0.475652277469635, 0.21003039181232452, 0.7341836094856262, 0.1912911832332611, 0.3015017509460449, 0.6593908071517944, 0.8299797177314758, 0.43047553300857544, 0.4657725691795349, 0.47735315561294556, 0.7226634621620178, 0.49210596084594727, 0.20858700573444366, 0.7856774926185608, 0.677203893661499, 0.20108069479465485, 0.9368849396705627, 0.3348597288131714, 0.4874136745929718, 0.012006028555333614, 0.1804739087820053, 0.9690226912498474, 0.7789011597633362], [0.3752175569534302, 0.9611548185348511, 0.04489264264702797, 0.18161799013614655, 0.2227507084608078, 0.8309327363967896, 0.8776053786277771, 0.7643247246742249, 0.5807889103889465, 0.27365177869796753, 0.5794982314109802, 0.12109196186065674, 0.9189985394477844, 0.27504879236221313, 0.8654509782791138, 0.8628355860710144, 0.2334403544664383, 0.438816100358963, 0.7874502539634705, 0.1921868473291397, 0.7668770551681519, 0.73572838306427, 0.2690078318119049, 0.6856669187545776, 0.4061284065246582, 0.8253417015075684, 0.5513679385185242, 0.2597372531890869, 0.028469694778323174, 0.4822116494178772, 0.057907771319150925, 0.024322913959622383, 0.7411069869995117], [0.6989961266517639, 0.294024258852005, 0.2642829418182373, 0.3585479259490967, 0.002734899753704667, 0.6701306700706482, 0.8763246536254883, 0.13712285459041595, 0.9476525783538818, 0.90412837266922, 0.2526785731315613, 0.11770081520080566, 0.13670410215854645, 0.8524746298789978, 0.39224979281425476, 0.023452386260032654, 0.8067572712898254, 0.454569011926651, 0.35013601183891296, 0.10947049409151077, 0.6863368153572083, 0.6470364332199097, 0.21449516713619232, 0.7323115468025208, 0.8829377889633179, 0.14882811903953552, 0.041473448276519775, 0.9151949882507324, 0.8866210579872131, 0.05972260981798172, 0.520210325717926, 0.038522716611623764, 0.6668333411216736], [0.15375205874443054, 0.5033584237098694, 0.7129331231117249, 0.5854567289352417, 0.9558247923851013, 0.9739304184913635, 0.406982958316803, 0.4463198184967041, 0.7018582820892334, 0.09026876837015152, 0.6035270094871521, 0.3343474566936493, 0.9371346235275269, 0.08064254373311996, 0.7261836528778076, 0.702488362789154, 0.564769446849823, 0.5936120748519897, 0.43734461069107056, 0.5730896592140198, 0.44961389899253845, 0.9331740140914917, 0.12021765857934952, 0.5104582905769348, 0.9000033736228943, 0.7595533728599548, 0.5737899541854858, 0.35647016763687134, 0.5267340540885925, 0.3019890785217285, 0.39413005113601685, 0.6030115485191345, 0.038767583668231964], [0.22022975981235504, 0.9075276255607605, 0.5726447105407715, 0.6345463991165161, 0.4058137834072113, 0.8141549229621887, 0.5668248534202576, 0.4035932719707489, 0.6497690081596375, 0.32735827565193176, 0.9398460984230042, 0.7301946878433228, 0.3731216490268707, 0.7779538035392761, 0.10782494395971298, 0.4018012583255768, 0.7491204738616943, 0.9679228067398071, 0.005114578641951084, 0.2969786524772644, 0.21794240176677704, 0.6708723902702332, 0.9501006007194519, 0.027601709589362144, 0.3670390248298645, 0.5499986410140991, 0.9726559519767761, 0.08320531994104385, 0.7493882775306702, 0.03228696063160896, 0.25311776995658875, 0.9832008481025696, 0.9669715166091919], [0.9691950082778931, 0.8578870296478271, 0.6590673923492432, 0.18575312197208405, 0.36609557271003723, 0.4776231348514557, 0.7697092890739441, 0.3352844715118408, 0.03565871715545654, 0.28152167797088623, 0.03543469309806824, 0.07392989844083786, 0.19679497182369232, 0.48329445719718933, 0.22464019060134888, 0.391095906496048, 0.27082547545433044, 0.9418267607688904, 0.6989975571632385, 0.3754647970199585, 0.4793049991130829, 0.17459022998809814, 0.47275322675704956, 0.3981420695781708, 0.3781791925430298, 0.8484081625938416, 0.972175657749176, 0.28794652223587036, 0.12015477567911148, 0.8814273476600647, 0.911670982837677, 0.6306008100509644, 0.865375816822052], [0.45740941166877747, 0.7412102222442627, 0.6218396425247192, 0.3330906629562378, 0.5500121712684631, 0.656278133392334, 0.477287620306015, 0.5174628496170044, 0.1247788518667221, 0.055266011506319046, 0.37219563126564026, 0.37988558411598206, 0.8814166188240051, 0.12754863500595093, 0.07834340631961823, 0.7325267791748047, 0.5421744585037231, 0.6130734086036682, 0.39764297008514404, 0.6030219793319702, 0.03769815340638161, 0.7054615616798401, 0.5153653025627136, 0.09586441516876221, 0.4921131134033203, 0.5056207180023193, 0.5507448315620422, 0.6255810856819153, 0.3388742506504059, 0.6297202706336975, 0.14138679206371307, 0.606393575668335, 0.7314079403877258], [0.6466141939163208, 0.9123488664627075, 0.4653298258781433, 0.3998173475265503, 0.6712758541107178, 0.6261184215545654, 0.3182940185070038, 0.5810110569000244, 0.23019398748874664, 0.48656967282295227, 0.030659226700663567, 0.288106232881546, 0.08853011578321457, 0.6304283738136292, 0.142461895942688, 0.12605591118335724, 0.6079950928688049, 0.754447877407074, 0.37755218148231506, 0.9147534370422363, 0.6218507289886475, 0.2961093485355377, 0.3635442554950714, 0.5522348880767822, 0.05924820899963379, 0.4162755310535431, 0.3394850194454193, 0.7064377069473267, 0.7390971779823303, 0.6811598539352417, 0.8289053440093994, 0.5349892377853394, 0.6757518649101257], [0.3406328558921814, 0.08271729201078415, 0.22691263258457184, 0.4720151722431183, 0.9440807700157166, 0.1653735339641571, 0.1623619794845581, 0.11688129603862762, 0.5453977584838867, 0.5568360090255737, 0.6148732304573059, 0.3928855061531067, 0.8804075717926025, 0.9450667500495911, 0.05080317705869675, 0.7871553897857666, 0.8418071866035461, 0.6458526849746704, 0.6103933453559875, 0.1426561027765274, 0.7525954842567444, 0.06321755051612854, 0.2186710387468338, 0.19032755494117737, 0.9519526958465576, 0.9689574837684631, 0.9266439080238342, 0.12449420988559723, 0.8964025974273682, 0.37238171696662903, 0.03243822976946831, 0.0687328428030014, 0.3807234466075897], [0.1419585943222046, 0.6382367610931396, 0.21456755697727203, 0.7095741033554077, 0.13571800291538239, 0.41707441210746765, 0.6982078552246094, 0.6838521957397461, 0.6365762948989868, 0.7872752547264099, 0.5077461004257202, 0.877763569355011, 0.14158108830451965, 0.8308716416358948, 0.49518918991088867, 0.029018240049481392, 0.35675105452537537, 0.514849066734314, 0.05655987560749054, 0.20996317267417908, 0.35354217886924744, 0.2808835208415985, 0.08442903310060501, 0.5167044401168823, 0.6740903854370117, 0.5978381633758545, 0.5254716277122498, 0.888545572757721, 0.7742204070091248, 0.7207757830619812, 0.3898031711578369, 0.5838878154754639, 0.6806490421295166], [0.10980462282896042, 0.04503967985510826, 0.8591476082801819, 0.6025887727737427, 0.190341517329216, 0.8476330637931824, 0.8126360774040222, 0.7024285197257996, 0.33223050832748413, 0.18845437467098236, 0.5813489556312561, 0.7722976803779602, 0.5213969945907593, 0.90290367603302, 0.7436901330947876, 0.15800438821315765, 0.6047898530960083, 0.946150004863739, 0.2988553047180176, 0.17624861001968384, 0.3409815728664398, 0.8304018378257751, 0.26219242811203003, 0.10430734604597092, 0.9068756103515625, 0.43214482069015503, 0.37307536602020264, 0.8724772930145264, 0.3416816294193268, 0.9976994395256042, 0.016144627705216408, 0.560955822467804, 0.7299643158912659], [0.8648256063461304, 0.233936607837677, 0.7326096296310425, 0.475544273853302, 0.3346143066883087, 0.5070204734802246, 0.7644873857498169, 0.783496081829071, 0.014592033810913563, 0.6454200148582458, 0.7677753567695618, 0.35502365231513977, 0.7603424191474915, 0.12525789439678192, 0.5816183686256409, 0.2869180142879486, 0.7924160361289978, 0.7501757740974426, 0.8379417657852173, 0.7216767072677612, 0.831305205821991, 0.14011284708976746, 0.9373515248298645, 7.519771315855905e-05, 0.7012453079223633, 0.570401132106781, 0.18036404252052307, 0.4059397876262665, 0.89121413230896, 0.3998613953590393, 0.6944704055786133, 0.0913257971405983, 0.17341725528240204], [0.9089205861091614, 0.6923034191131592, 0.7999520301818848, 0.9274201989173889, 0.5474811792373657, 0.06881414353847504, 0.9401720762252808, 0.0007544768159277737, 0.24319353699684143, 0.6788289546966553, 0.7232391238212585, 0.3818523585796356, 0.3503195643424988, 0.7708736658096313, 0.9915291666984558, 0.5727607607841492, 0.32737699151039124, 0.2897094488143921, 0.8509109616279602, 0.63283771276474, 0.10521059483289719, 0.4022124409675598, 0.21117854118347168, 0.47107335925102234, 0.8837384581565857, 0.9985295534133911, 0.04897508770227432, 0.47095537185668945, 0.10254859924316406, 0.3533666133880615, 0.02716662362217903, 0.8434903621673584, 0.6674598455429077], [0.45509636402130127, 0.9780026078224182, 0.874128520488739, 0.07419376075267792, 0.08686002343893051, 0.1334780752658844, 0.9787814617156982, 0.0714297816157341, 0.8309021592140198, 0.003617569338530302, 0.16732513904571533, 0.12873575091362, 0.6339213848114014, 0.044804271310567856, 0.9802572131156921, 0.9960508942604065, 0.4115924835205078, 0.10787610709667206, 0.3006928265094757, 0.016702130436897278, 0.770911693572998, 0.7595383524894714, 0.2788558304309845, 0.45072898268699646, 0.9969214797019958, 0.955086350440979, 0.7692425847053528, 0.20797903835773468, 0.7574837803840637, 0.8838458061218262, 0.24855898320674896, 0.8770363926887512, 0.12104445695877075], [0.2389872521162033, 0.34579169750213623, 0.7576637864112854, 0.2817428410053253, 0.614061713218689, 0.3225902318954468, 0.09052208065986633, 0.3891177773475647, 0.6856200098991394, 0.3640840947628021, 0.4252694249153137, 0.4686581492424011, 0.5860639214515686, 0.015099010430276394, 0.743859589099884, 0.7265777587890625, 0.46787288784980774, 0.25210219621658325, 0.875419557094574, 0.9485334157943726, 0.589237630367279, 0.5019130706787109, 0.3064223527908325, 0.004019320011138916, 0.7515904307365417, 0.3821999132633209, 0.32115933299064636, 0.5764889717102051, 0.30524083971977234, 0.3680438697338104, 0.8681864738464355, 0.23531962931156158, 0.29821157455444336], [0.40494441986083984, 0.2351163625717163, 0.34709617495536804, 0.5383601784706116, 0.5075124502182007, 0.9631556272506714, 0.7219356894493103, 0.027072744444012642, 0.9273224472999573, 0.6014608144760132, 0.96724534034729, 0.5065885782241821, 0.6005174517631531, 0.3916192948818207, 0.926313579082489, 0.8695856332778931, 0.45657879114151, 0.3267576992511749, 0.07832375168800354, 0.7596514225006104, 0.03435763716697693, 0.018009742721915245, 0.09595268219709396, 0.8900193572044373, 0.8951902389526367, 0.19854377210140228, 0.799180269241333, 0.3865989148616791, 0.4598682224750519, 0.8548062443733215, 0.12603357434272766, 0.4675431549549103, 0.2766619026660919], [0.9425433874130249, 0.3447315990924835, 0.9642256498336792, 0.4369613528251648, 0.40357398986816406, 0.4737725853919983, 0.8124240040779114, 0.49045854806900024, 0.8675580620765686, 0.26741287112236023, 0.07935238629579544, 0.9418080449104309, 0.8289141058921814, 0.8039888143539429, 0.6453459858894348, 0.992680549621582, 0.4657389223575592, 0.46275606751441956, 0.8160549998283386, 0.07874466478824615, 0.5543463826179504, 0.5967952013015747, 0.40175944566726685, 0.8823546171188354, 0.6998167037963867, 0.2528292238712311, 0.17639289796352386, 0.7721918821334839, 0.8104514479637146, 0.4826793670654297, 0.6844757199287415, 0.26878538727760315, 0.26097041368484497], [0.6569876670837402, 0.14393974840641022, 0.7719110250473022, 0.8788518905639648, 0.2159583419561386, 0.31528377532958984, 0.34649357199668884, 0.26815032958984375, 0.12455356121063232, 0.17808733880519867, 0.443425714969635, 0.16046392917633057, 0.296408087015152, 0.11575710773468018, 0.8225117921829224, 0.2523251473903656, 0.11893826723098755, 0.7379432916641235, 0.5054773092269897, 0.5253536701202393, 0.03791436553001404, 0.19924460351467133, 0.5295023918151855, 0.5774703025817871, 0.17308510839939117, 0.8546820878982544, 0.20555724203586578, 0.34989285469055176, 0.9191733598709106, 0.389877051115036, 0.334819495677948, 0.8398614525794983, 0.5202327966690063], [0.8753296136856079, 0.5467597842216492, 0.44710439443588257, 0.5126938223838806, 0.34070906043052673, 0.01103260274976492, 0.49916568398475647, 0.6503124833106995, 0.5257232189178467, 0.018351536244153976, 0.16695347428321838, 0.4306928515434265, 0.34904128313064575, 0.9394109845161438, 0.13910284638404846, 0.1224716305732727, 0.0756087452173233, 0.7233214974403381, 0.03271046653389931, 0.3052648603916168, 0.4215348958969116, 0.4763699173927307, 0.4714398980140686, 0.6270439028739929, 0.12691350281238556, 0.8657010197639465, 0.5652691721916199, 0.9499019384384155, 0.09426620602607727, 0.8125446438789368, 0.5230333209037781, 0.9995020627975464, 0.45856377482414246], [0.4423130452632904, 0.7556753158569336, 0.07839503139257431, 0.1592148393392563, 0.5720738768577576, 0.6740617156028748, 0.2162981629371643, 0.6082373261451721, 0.5126040577888489, 0.6099053621292114, 0.19573137164115906, 0.05587177351117134, 0.9008628129959106, 0.19796155393123627, 0.7671068906784058, 0.2229246199131012, 0.461407333612442, 0.36785751581192017, 0.18076744675636292, 0.25967368483543396, 0.21227328479290009, 0.8516637086868286, 0.46087226271629333, 0.7315426468849182, 0.03672492131590843, 0.22005343437194824, 0.4031996726989746, 0.07831522822380066, 0.6976633071899414, 0.5867329239845276, 0.5815332531929016, 0.215679332613945, 0.39689674973487854], [0.9192360043525696, 0.5709115266799927, 0.46527495980262756, 0.8434658646583557, 0.8147768974304199, 0.41195905208587646, 0.10607598721981049, 0.4451107084751129, 0.8139369487762451, 0.07088817656040192, 0.7267917394638062, 0.80047607421875, 0.5507925748825073, 0.8114058971405029, 0.07415007799863815, 0.28199079632759094, 0.7676832675933838, 0.6284667253494263, 0.29880291223526, 0.8498741984367371, 0.9677808284759521, 0.17810922861099243, 0.28635722398757935, 0.6535900235176086, 0.8066579103469849, 0.7796340584754944, 0.581892192363739, 0.906690239906311, 0.7928276062011719, 0.25145968794822693, 0.06704427301883698, 0.3749426007270813, 0.5938771367073059], [0.5583426356315613, 0.9380441904067993, 0.21980006992816925, 0.6245949268341064, 0.8705360293388367, 0.651892364025116, 0.2444520890712738, 0.6924558877944946, 0.5371083617210388, 0.20574124157428741, 0.24878382682800293, 0.9294573068618774, 0.5139424204826355, 0.019034307450056076, 0.9524025321006775, 0.9701068997383118, 0.3726872503757477, 0.7495287656784058, 0.73709636926651, 0.8885483145713806, 0.5303965210914612, 0.3051849603652954, 0.35857459902763367, 0.06298910081386566, 0.8664069771766663, 0.6161187887191772, 0.28026533126831055, 0.14282341301441193, 0.5367914438247681, 0.4165250062942505, 0.7218329310417175, 0.31277403235435486, 0.9071686863899231], [0.05660288408398628, 0.649901807308197, 0.44549447298049927, 0.3515450954437256, 0.06724783033132553, 0.329093873500824, 0.8599181771278381, 0.1720649003982544, 0.03005848079919815, 0.19560693204402924, 0.0022794518154114485, 0.20198240876197815, 0.11050866544246674, 0.6647064089775085, 0.928713321685791, 0.630717396736145, 0.908455491065979, 0.2527357041835785, 0.1802339255809784, 0.6267369985580444, 0.6030654907226562, 0.1787184625864029, 0.4215860366821289, 0.1846504509449005, 0.8863728642463684, 0.5072717666625977, 0.3828776776790619, 0.15819835662841797, 0.912134051322937, 0.4013442397117615, 0.3752061128616333, 0.36380085349082947, 0.024435006082057953], [0.5391608476638794, 0.26079726219177246, 0.989437460899353, 0.9446746110916138, 0.8582215905189514, 0.14704640209674835, 0.945655345916748, 0.37999042868614197, 0.10816309601068497, 0.8229103088378906, 0.8002197742462158, 0.9283966422080994, 0.3149639666080475, 0.6307000517845154, 0.5388695597648621, 0.31338807940483093, 0.6820777058601379, 0.701988697052002, 0.24708783626556396, 0.6190257668495178, 0.4799109995365143, 0.22631444036960602, 0.5490569472312927, 0.22602567076683044, 0.555195689201355, 0.1396426558494568, 0.40669047832489014, 0.6802964806556702, 0.0308640506118536, 0.5757488012313843, 0.36995139718055725, 0.9713420271873474, 0.9996581673622131], [0.24954864382743835, 0.39334923028945923, 0.2519516348838806, 0.7595200538635254, 0.422492116689682, 0.6507759094238281, 0.8672692775726318, 0.6443617939949036, 0.7818678617477417, 0.7134817838668823, 0.26406198740005493, 0.6159677505493164, 0.8977261781692505, 0.9846797585487366, 0.16002890467643738, 0.17458365857601166, 0.5848019123077393, 0.7633023262023926, 0.8975711464881897, 0.8895040154457092, 0.4792390465736389, 0.6928419470787048, 0.42458951473236084, 0.6291894912719727, 0.46403852105140686, 0.0982937142252922, 0.753383219242096, 0.12666139006614685, 0.709235429763794, 0.7454211711883545, 0.9341285824775696, 0.4852134585380554, 0.5255170464515686], [0.3178996443748474, 0.6772956848144531, 0.5460627675056458, 0.3738482594490051, 0.01617608778178692, 0.9607183933258057, 0.35564014315605164, 0.27572178840637207, 0.9899322986602783, 0.6625705361366272, 0.610419511795044, 0.18811370432376862, 0.9932423233985901, 0.4633833169937134, 0.8332255482673645, 0.8930695056915283, 0.0942022055387497, 0.1953723430633545, 0.5109226107597351, 0.09289171546697617, 0.3175848424434662, 0.2968897223472595, 0.11178848892450333, 0.34281888604164124, 0.8407461047172546, 0.6035380363464355, 0.7379907965660095, 0.45008525252342224, 0.7984130382537842, 0.3286304473876953, 0.11431334167718887, 0.5863755345344543, 0.913818359375], [0.5986048579216003, 0.13826780021190643, 0.8906441926956177, 0.7937458157539368, 0.7934850454330444, 0.4128156006336212, 0.09255018085241318, 0.13732855021953583, 0.6215639114379883, 0.8195128440856934, 0.8095940947532654, 0.599166214466095, 0.10505764186382294, 0.4284650683403015, 0.7224177122116089, 0.14615383744239807, 0.5899935960769653, 0.5507128834724426, 0.570633590221405, 0.6811034679412842, 0.1570097953081131, 0.0471971333026886, 0.30489882826805115, 0.6935704946517944, 0.9834439754486084, 0.11239007115364075, 0.9457948207855225, 0.7100690007209778, 0.32010361552238464, 0.5431034564971924, 0.33358484506607056, 0.9152398705482483, 0.3730930984020233], [0.9923697113990784, 0.15659154951572418, 0.5341977477073669, 0.5361968278884888, 0.013737894594669342, 0.7051215767860413, 0.8484951853752136, 0.8380136489868164, 0.1935282200574875, 0.6109411120414734, 0.6165076494216919, 0.8398985266685486, 0.5836140513420105, 0.434224009513855, 0.29231390357017517, 0.7101906538009644, 0.32315340638160706, 0.7404627203941345, 0.8703826069831848, 0.6810388565063477, 0.8751083612442017, 0.7687230110168457, 0.6392303109169006, 0.3211645185947418, 0.5232254862785339, 0.6883522272109985, 0.029353944584727287, 0.8437955975532532, 0.1443636566400528, 0.9002501368522644, 0.3699881434440613, 0.8268909454345703, 0.3280234932899475], [0.6911010146141052, 0.4283696711063385, 0.20759513974189758, 0.7410603761672974, 0.6413823962211609, 0.6516481637954712, 0.8361057043075562, 0.9357650876045227, 0.4510681629180908, 0.5891628861427307, 0.43925297260284424, 0.3699766993522644, 0.22864575684070587, 0.9125956296920776, 0.8030175566673279, 0.4121217727661133, 0.21929308772087097, 0.8909087181091309, 0.6176779270172119, 0.4618426263332367, 0.854017436504364, 0.4548996388912201, 0.16889721155166626, 0.2109776735305786, 0.005975206848233938, 0.521201491355896, 0.595270037651062, 0.617607831954956, 0.3620949983596802, 0.11197371035814285, 0.11324740201234818, 0.39421403408050537, 0.9134977459907532], [0.1891392469406128, 0.2858952581882477, 0.15800665318965912, 0.26776444911956787, 0.8718243837356567, 0.4164210557937622, 0.7093784809112549, 0.26417699456214905, 0.5676358342170715, 0.884928286075592, 0.925845742225647, 0.25164297223091125, 0.14751434326171875, 0.9732888340950012, 0.3632934093475342, 0.3981250524520874, 0.6759009957313538, 0.3142218291759491, 0.9899204969406128, 0.8581609129905701, 0.9367008805274963, 0.5357182621955872, 0.9735701680183411, 0.19602710008621216, 0.06312637776136398, 0.6767145395278931, 0.5318338871002197, 0.9973369836807251, 0.994705080986023, 0.8917007446289062, 0.00428598839789629, 0.1386321783065796, 0.9174647927284241], [0.02824314311146736, 0.07074417173862457, 0.38280555605888367, 0.00928577221930027, 0.022443514317274094, 0.04105834662914276, 0.8023388981819153, 0.05384012684226036, 0.04683980718255043, 0.4191977381706238, 0.17885196208953857, 0.2977575361728668, 0.5834174156188965, 0.961304247379303, 0.17114613950252533, 0.5555242300033569, 0.7751487493515015, 0.756467878818512, 0.4986325800418854, 0.3227144777774811, 0.013010709546506405, 0.861415445804596, 0.6073898673057556, 0.5052707195281982, 0.7683352828025818, 0.45648664236068726, 0.20093484222888947, 0.10625796020030975, 0.49977514147758484, 0.04151086509227753, 0.9570067524909973, 0.20549380779266357, 0.04443991556763649], [0.0069809709675610065, 0.14276723563671112, 0.7768384218215942, 0.11539722234010696, 0.7459051012992859, 0.9483171105384827, 0.9492954611778259, 0.7000489830970764, 0.7622539401054382, 0.004408702719956636, 0.14819218218326569, 0.02213307097554207, 0.5578976273536682, 0.06963036209344864, 0.5108416080474854, 0.6262182593345642, 0.6464316248893738, 0.9462939500808716, 0.013767357915639877, 0.8151795268058777, 0.7960485816001892, 0.39348429441452026, 0.8161969184875488, 0.09827443957328796, 0.410444051027298, 0.7997267246246338, 0.27170026302337646, 0.19905275106430054, 0.31795451045036316, 0.9759929776191711, 0.7235763669013977, 0.7682957053184509, 0.43067991733551025], [0.37784281373023987, 0.7324651479721069, 0.19396309554576874, 0.29277414083480835, 0.27633678913116455, 0.7623490691184998, 0.9384623169898987, 0.3351266086101532, 0.7169370055198669, 0.4541163146495819, 0.6435487270355225, 0.7005767822265625, 0.25551143288612366, 0.4470355212688446, 0.12909632921218872, 0.09965922683477402, 0.5922145247459412, 0.02069459855556488, 0.6738738417625427, 0.019940536469221115, 0.6466838717460632, 0.09316712617874146, 0.7117818593978882, 0.5732336044311523, 0.8817713856697083, 0.6038415431976318, 0.2978035807609558, 0.037983812391757965, 0.423369437456131, 0.936213493347168, 0.5800231099128723, 0.8659514784812927, 0.8576922416687012], [0.5839089155197144, 0.935329258441925, 0.38040801882743835, 0.7222987413406372, 0.26760178804397583, 0.8001824617385864, 0.6523013710975647, 0.8353994488716125, 0.7044125199317932, 0.39698565006256104, 0.49288710951805115, 0.9077608585357666, 0.3726905286312103, 0.0987316370010376, 0.442197322845459, 0.4849398732185364, 0.2379145473241806, 0.8362937569618225, 0.9304485321044922, 0.3319298028945923, 0.8147296905517578, 0.4539676606655121, 0.26015958189964294, 0.5972462892532349, 0.15473826229572296, 0.14813368022441864, 0.21864667534828186, 0.6096957325935364, 0.21161527931690216, 0.3290974795818329, 0.03042430616915226, 0.6684389114379883, 0.9185229539871216], [0.8514468669891357, 0.40548962354660034, 0.9465474486351013, 0.509296178817749, 0.04083101823925972, 0.22792162001132965, 0.809622049331665, 0.9325941205024719, 0.42457282543182373, 0.7035183906555176, 0.9281702041625977, 0.936430811882019, 0.230310320854187, 0.9739501476287842, 0.2834559679031372, 0.5010294318199158, 0.06324777752161026, 0.27436211705207825, 0.7309691905975342, 0.8232431411743164, 0.7960032820701599, 0.36124512553215027, 0.5268939137458801, 0.9415923357009888, 0.8795486688613892, 0.08451740443706512, 0.6766148209571838, 0.9736335873603821, 0.3609032928943634, 0.860668957233429, 0.6153534650802612, 0.20144492387771606, 0.19223587214946747]], "dtype": "float32"}], "e": {"__nd__": 1053.434814453125, "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(vector_sum, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")